# Caso práctico: clasificación de sentimientos en reseñas de Yelp

**Estudiante:** Julio César Mendoza  
**Docente:** Héctor Manuel Rojas  
**Asignatura:** Natural Language Processing  
**Programa:** Especialización en Inteligencia Artificial para Analítica de Datos  
**Semilla principal:** 42

---

## Objetivo

Desarrollar y comparar modelos de clasificación de sentimientos sobre reseñas de Yelp utilizando exclusivamente el texto como predictor.

La variable objetivo se construye como **etiqueta proxy** a partir de las estrellas:

- 1–2 estrellas → **NEGATIVO (0)**
- 3 estrellas → **NEUTRAL (1)**
- 4–5 estrellas → **POSITIVO (2)**

El flujo experimental incluye:

1. Auditoría y preparación de datos.
2. Split agrupado por `business_id`.
3. Preprocesamiento diferenciado.
4. Baselines.
5. Word2Vec + BiLSTM.
6. Fine-tuning de BERT.
7. Evaluación final homogénea.
8. Análisis de errores.
9. Conclusiones.

> Los resultados incluidos en este notebook corresponden a ejecuciones realizadas durante el caso práctico. La comparación principal se gobierna por F1 macro.


# Caso práctico: clasificación de sentimientos en reseñas de Yelp

Este notebook desarrolla el caso práctico con dos modelos comparables:

1. **Modelo secuencial:** Word2Vec + BiLSTM.
2. **Modelo Transformer:** fine-tuning de BERT para clasificación de tres clases.

Las etiquetas se construyen de forma explícita a partir de `stars`:

- 1–2 estrellas → **NEGATIVO**
- 3 estrellas → **NEUTRAL**
- 4–5 estrellas → **POSITIVO**

Los dos modelos usan exactamente los mismos splits y se comparan con accuracy, precisión, recall, F1 macro, F1 ponderado y balanced accuracy. Las cifras académicas solo son válidas después de ejecutar todas las celdas.

## 1. Configurar Google Colab

Seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. Suba `yelp_dataset.xlsx` al directorio `/content` y ejecute las celdas en orden.

In [4]:
!pip install -q pandas==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 66.6 MB/s eta 0:00:00


In [1]:
import pandas as pd
import sklearn
import torch

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pandas: 3.0.5
scikit-learn: 1.9.0
torch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [2]:
from pathlib import Path
import pandas as pd

DATASET_PATH = Path("/content/yelp_dataset.xlsx")

print("Archivo existe:", DATASET_PATH.exists())
print("Ruta:", DATASET_PATH)

if DATASET_PATH.exists():
    xls = pd.ExcelFile(DATASET_PATH)
    print("Hojas:", xls.sheet_names)

Archivo existe: False
Ruta: /content/yelp_dataset.xlsx


In [3]:
from google.colab import files

uploaded = files.upload()

Saving yelp_dataset.xlsx to yelp_dataset.xlsx


In [4]:
from pathlib import Path

DATASET_PATH = Path("/content/yelp_dataset.xlsx")

print("Archivo existe:", DATASET_PATH.exists())
print("Ruta:", DATASET_PATH)

Archivo existe: True
Ruta: /content/yelp_dataset.xlsx


In [5]:
import pandas as pd

xls = pd.ExcelFile(DATASET_PATH)

print("=== HOJAS DEL EXCEL ===")
print(xls.sheet_names)

sheet_name = xls.sheet_names[0]

df_original = pd.read_excel(
    DATASET_PATH,
    sheet_name=sheet_name
)

print("\n=== DIMENSIONES ===")
print("Hoja analizada:", sheet_name)
print("Filas:", df_original.shape[0])
print("Columnas:", df_original.shape[1])

=== HOJAS DEL EXCEL ===
['Hoja 1_yelp']

=== DIMENSIONES ===
Hoja analizada: Hoja 1_yelp
Filas: 10000
Columnas: 10


In [6]:
print("=== COLUMNAS Y TIPOS ===")

schema = pd.DataFrame({
    "columna": df_original.columns,
    "tipo": [str(dtype) for dtype in df_original.dtypes]
})

print(schema.to_string(index=False))

print("\n=== VALORES NULOS POR COLUMNA ===")

nulls = pd.DataFrame({
    "nulos": df_original.isna().sum(),
    "porcentaje": (df_original.isna().mean() * 100).round(4)
})

print(nulls.to_string())

=== COLUMNAS Y TIPOS ===
    columna  tipo
business_id   str
       date   str
  review_id   str
      stars int64
       text   str
       type   str
    user_id   str
       cool int64
     useful int64
      funny int64

=== VALORES NULOS POR COLUMNA ===
             nulos  porcentaje
business_id      0         0.0
date             0         0.0
review_id        0         0.0
stars            0         0.0
text             0         0.0
type             0         0.0
user_id          0         0.0
cool             0         0.0
useful           0         0.0
funny            0         0.0


In [7]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 2 y 3: revisar columnas, tipos de datos y valores nulos
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
# 1. Muestra los nombres de las 10 columnas del dataset.
# 2. Identifica el tipo de dato que pandas asignó a cada columna.
#    Por ejemplo:
#       - object: normalmente texto
#       - int64 / float64: valores numéricos
#       - datetime: fechas, si pandas ya las reconoció como tales
# 3. Cuenta cuántos valores vacíos (nulos) hay en cada columna.
# 4. Calcula qué porcentaje representan esos nulos.
#
# IMPORTANTE:
# Esta celda SOLO inspecciona los datos.
# No modifica ni elimina ninguna fila del dataset original.


# ------------------------------------------------------------
# 1. NOMBRES Y TIPOS DE LAS COLUMNAS
# ------------------------------------------------------------

print("=== COLUMNAS Y TIPOS ===")

# df_original.columns contiene los nombres de las columnas.
# df_original.dtypes contiene el tipo de dato detectado por pandas.
schema = pd.DataFrame({
    "columna": df_original.columns,
    "tipo": [str(dtype) for dtype in df_original.dtypes]
})

# Mostramos la tabla completa sin el índice 0, 1, 2...
print(schema.to_string(index=False))


# ------------------------------------------------------------
# 2. VALORES NULOS
# ------------------------------------------------------------

print("\n=== VALORES NULOS POR COLUMNA ===")

# isna() identifica las celdas vacías.
# sum() cuenta cuántas hay en cada columna.
#
# mean() calcula la proporción de valores nulos.
# Multiplicamos por 100 para expresarla como porcentaje.
nulls = pd.DataFrame({
    "nulos": df_original.isna().sum(),
    "porcentaje": (df_original.isna().mean() * 100).round(4)
})

print(nulls.to_string())

=== COLUMNAS Y TIPOS ===
    columna  tipo
business_id   str
       date   str
  review_id   str
      stars int64
       text   str
       type   str
    user_id   str
       cool int64
     useful int64
      funny int64

=== VALORES NULOS POR COLUMNA ===
             nulos  porcentaje
business_id      0         0.0
date             0         0.0
review_id        0         0.0
stars            0         0.0
text             0         0.0
type             0         0.0
user_id          0         0.0
cool             0         0.0
useful           0         0.0
funny            0         0.0


In [8]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 4 y 5: comprobar duplicados
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Vamos a buscar dos tipos distintos de duplicados:
#
# A) review_id duplicados
#    review_id es el identificador de una reseña.
#    En principio esperamos que cada reseña tenga un ID único.
#
# B) textos duplicados exactos
#    Dos review_id diferentes podrían contener exactamente
#    el mismo texto.
#
# IMPORTANTE:
# En esta celda NO eliminamos nada.
# Solo detectamos, contamos y mostramos los casos encontrados.


# ------------------------------------------------------------
# A. DUPLICADOS DE review_id
# ------------------------------------------------------------

# duplicated(keep=False) marca TODAS las filas que pertenecen
# a un grupo cuyo review_id aparece más de una vez.
review_id_dup_mask = df_original["review_id"].duplicated(
    keep=False
)

# Extraemos esas filas para poder inspeccionarlas.
review_id_duplicate_rows = df_original.loc[
    review_id_dup_mask
].copy()

# Este segundo cálculo cuenta únicamente las apariciones
# redundantes después de conservar la primera.
#
# Ejemplo:
# Si un ID aparece dos veces:
# - filas pertenecientes al grupo duplicado = 2
# - duplicados adicionales = 1
n_review_id_extra = int(
    df_original["review_id"]
    .duplicated(keep="first")
    .sum()
)

print("=== DUPLICADOS DE review_id ===")

print(
    "Filas pertenecientes a grupos de review_id duplicados:",
    len(review_id_duplicate_rows)
)

print(
    "Duplicados adicionales de review_id:",
    n_review_id_extra
)

# Si encontramos alguno, mostramos sus datos.
if len(review_id_duplicate_rows) > 0:
    print("\nDetalle de review_id duplicados:")

    print(
        review_id_duplicate_rows[
            [
                "review_id",
                "business_id",
                "user_id",
                "stars",
                "date"
            ]
        ]
        .sort_values("review_id")
        .to_string(index=False)
    )


# ------------------------------------------------------------
# B. TEXTOS DUPLICADOS EXACTOS
# ------------------------------------------------------------

# Aquí hacemos la misma comprobación sobre la columna text.
#
# keep=False:
# marca TODAS las filas cuyo texto aparece más de una vez.
text_dup_group_mask = df_original["text"].duplicated(
    keep=False
)

# keep="first":
# conserva conceptualmente la primera aparición y marca
# únicamente las apariciones redundantes.
text_dup_extra_mask = df_original["text"].duplicated(
    keep="first"
)

print("\n=== TEXTOS DUPLICADOS EXACTOS ===")

print(
    "Filas pertenecientes a grupos de textos duplicados:",
    int(text_dup_group_mask.sum())
)

print(
    "Filas redundantes si conservamos la primera aparición:",
    int(text_dup_extra_mask.sum())
)


# ------------------------------------------------------------
# C. MOSTRAR LOS CASOS ENCONTRADOS
# ------------------------------------------------------------

# Solo mostramos el detalle si realmente existen duplicados.
if text_dup_group_mask.any():

    duplicate_text_rows = df_original.loc[
        text_dup_group_mask,
        [
            "review_id",
            "business_id",
            "user_id",
            "stars",
            "date",
            "text"
        ]
    ].copy()

    print("\n=== DETALLE DE LOS TEXTOS DUPLICADOS ===")

    # Ordenamos por texto para que las copias idénticas
    # aparezcan juntas y podamos compararlas fácilmente.
    print(
        duplicate_text_rows
        .sort_values("text")
        .to_string(index=False)
    )

=== DUPLICADOS DE review_id ===
Filas pertenecientes a grupos de review_id duplicados: 0
Duplicados adicionales de review_id: 0

=== TEXTOS DUPLICADOS EXACTOS ===
Filas pertenecientes a grupos de textos duplicados: 4
Filas redundantes si conservamos la primera aparición: 2

=== DETALLE DE LOS TEXTOS DUPLICADOS ===
             review_id            business_id                user_id  stars       date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [9]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 6 y 7:
# - contar negocios y usuarios únicos
# - determinar fecha mínima y fecha máxima
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Cuenta cuántos business_id distintos hay.
#    Esto nos dice cuántos negocios diferentes aparecen.
#
# 2. Cuenta cuántos user_id distintos hay.
#    Esto nos dice cuántos usuarios diferentes escribieron reseñas.
#
# 3. Convierte temporalmente la columna 'date' a tipo fecha.
#
# 4. Comprueba si alguna fecha no puede interpretarse.
#
# 5. Obtiene la fecha más antigua y la más reciente.
#
# IMPORTANTE:
# - No modificamos el archivo Excel original.
# - No creamos splits.
# - No hacemos todavía ningún preprocesamiento lingüístico.


# ------------------------------------------------------------
# 1. NEGOCIOS ÚNICOS
# ------------------------------------------------------------

# nunique() cuenta valores distintos.
# dropna=True ignora valores nulos, aunque ya comprobamos
# que en este dataset no existen nulos.
n_businesses = df_original["business_id"].nunique(dropna=True)


# ------------------------------------------------------------
# 2. USUARIOS ÚNICOS
# ------------------------------------------------------------

n_users = df_original["user_id"].nunique(dropna=True)


print("=== ENTIDADES ÚNICAS ===")
print("Negocios únicos:", n_businesses)
print("Usuarios únicos:", n_users)


# ------------------------------------------------------------
# 3. CONVERTIR LA COLUMNA date A FECHA
# ------------------------------------------------------------

# La columna 'date' fue leída inicialmente como texto (str).
#
# pd.to_datetime() intenta convertir cada valor a una fecha real.
#
# errors="coerce" significa:
# si encuentra un valor que NO puede interpretar como fecha,
# lo convierte en NaT ("Not a Time") en lugar de provocar un error.
#
# Guardamos el resultado en una variable nueva llamada dates.
# Por tanto, NO modificamos df_original["date"].
dates = pd.to_datetime(
    df_original["date"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. COMPROBAR FECHAS NO INTERPRETABLES
# ------------------------------------------------------------

invalid_dates = int(dates.isna().sum())


# ------------------------------------------------------------
# 5. FECHA MÍNIMA Y MÁXIMA
# ------------------------------------------------------------

min_date = dates.min()
max_date = dates.max()


print("\n=== RANGO DE FECHAS ===")
print("Fechas no interpretables:", invalid_dates)
print("Fecha mínima:", min_date)
print("Fecha máxima:", max_date)

=== ENTIDADES ÚNICAS ===
Negocios únicos: 4174
Usuarios únicos: 6403

=== RANGO DE FECHAS ===
Fechas no interpretables: 0
Fecha mínima: 2005-04-18 00:00:00
Fecha máxima: 2013-01-05 00:00:00


In [1]:
# ============================================================
# ETAPA Y NOMBRE
# Paso X: objetivo concreto
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
# Explicación sencilla y numerada.

# IMPORTANTE:
# Qué modifica.
# Qué NO modifica.
# Qué riesgo controla.

# ------------------------------------------------------------
# 1. PRIMER BLOQUE
# ------------------------------------------------------------

# Explicación de la función utilizada
# y de por qué se usa.

# Código...

# ------------------------------------------------------------
# 2. SEGUNDO BLOQUE
# ------------------------------------------------------------

# Explicación...

# Código...

In [2]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 9: crear la etiqueta proxy de sentimiento
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea una COPIA de trabajo del dataset original.
#
# 2. A partir de la columna 'stars', crea dos nuevas columnas:
#
#       sentiment
#       sentiment_id
#
# 3. Aplica la regla oficial del caso:
#
#       1–2 estrellas -> NEGATIVO (0)
#       3 estrellas   -> NEUTRAL  (1)
#       4–5 estrellas -> POSITIVO (2)
#
# 4. Comprueba que todas las filas hayan recibido
#    correctamente una etiqueta.
#
# IMPORTANTE:
#
# - Esta etiqueta NO existe originalmente como una anotación
#   independiente realizada por una persona.
#
# - La estamos derivando de 'stars'.
#
# - Por eso se denomina ETIQUETA PROXY.
#
# - NO modificamos df_original.
#
# - NO modificamos ni sobrescribimos yelp_dataset.xlsx.
#
# - Todavía NO eliminamos textos duplicados.
#
# - Todavía NO creamos train, validation ni test.


# ------------------------------------------------------------
# 1. CREAR UNA COPIA DE TRABAJO
# ------------------------------------------------------------

# copy(deep=True) crea una copia independiente del DataFrame.
#
# A partir de este momento:
#
# df_original -> conserva los datos tal como fueron leídos
#                desde el Excel.
#
# df_work     -> copia sobre la que podemos añadir columnas
#                o hacer transformaciones documentadas.

df_work = df_original.copy(deep=True)


# ------------------------------------------------------------
# 2. DEFINIR LA REGLA stars -> sentimiento
# ------------------------------------------------------------

# Creamos una función que recibe el número de estrellas
# y devuelve:
#
# nombre del sentimiento + código numérico.
#
# Ejemplo:
# 2 estrellas -> ("NEGATIVO", 0)

def map_sentiment(stars):

    # 1 o 2 estrellas = sentimiento negativo
    if stars in (1, 2):
        return "NEGATIVO", 0

    # 3 estrellas = sentimiento neutral
    elif stars == 3:
        return "NEUTRAL", 1

    # 4 o 5 estrellas = sentimiento positivo
    elif stars in (4, 5):
        return "POSITIVO", 2

    # Este caso solo debería ocurrir si existe
    # algún valor de stars no válido.
    else:
        return None, None


# ------------------------------------------------------------
# 3. APLICAR LA REGLA A TODAS LAS RESEÑAS
# ------------------------------------------------------------

# apply() ejecuta map_sentiment sobre cada valor de stars.
mapped_sentiment = df_work["stars"].apply(map_sentiment)


# ------------------------------------------------------------
# 4. CREAR LA COLUMNA DE TEXTO
# ------------------------------------------------------------

# str[0] toma el primer elemento devuelto por la función:
#
# NEGATIVO
# NEUTRAL
# POSITIVO

df_work["sentiment"] = mapped_sentiment.str[0]


# ------------------------------------------------------------
# 5. CREAR LA COLUMNA NUMÉRICA
# ------------------------------------------------------------

# str[1] toma el código numérico:
#
# NEGATIVO -> 0
# NEUTRAL  -> 1
# POSITIVO -> 2
#
# Int64 permite trabajar con enteros y, si existiera
# algún valor faltante, representarlo correctamente.

df_work["sentiment_id"] = (
    mapped_sentiment
    .str[1]
    .astype("Int64")
)


# ------------------------------------------------------------
# 6. MOSTRAR LA REGLA UTILIZADA
# ------------------------------------------------------------

print("=== REGLA DE ETIQUETADO PROXY ===")

print("1–2 estrellas -> NEGATIVO (0)")
print("3 estrellas   -> NEUTRAL  (1)")
print("4–5 estrellas -> POSITIVO (2)")


# ------------------------------------------------------------
# 7. COMPROBAR QUE TODAS LAS FILAS FUERON ETIQUETADAS
# ------------------------------------------------------------

missing_sentiment = int(
    df_work["sentiment"].isna().sum()
)

missing_sentiment_id = int(
    df_work["sentiment_id"].isna().sum()
)

print("\n=== VALIDACIÓN DE LA ETIQUETA ===")

print(
    "Filas sin etiqueta textual:",
    missing_sentiment
)

print(
    "Filas sin etiqueta numérica:",
    missing_sentiment_id
)


# ------------------------------------------------------------
# 8. COMPROBAR QUE EL ORIGINAL SIGUE IGUAL
# ------------------------------------------------------------

print("\n=== CONTROL DEL DATASET ORIGINAL ===")

print(
    "Filas en df_original:",
    len(df_original)
)

print(
    "Filas en df_work:",
    len(df_work)
)

print(
    "¿sentiment existe en df_original?:",
    "sentiment" in df_original.columns
)

print(
    "¿sentiment existe en df_work?:",
    "sentiment" in df_work.columns
)


# ------------------------------------------------------------
# 9. MOSTRAR SOLO UN EJEMPLO ESTRUCTURAL
# ------------------------------------------------------------

# Mostramos las primeras filas para comprobar visualmente
# que stars, sentiment y sentiment_id son coherentes.
#
# Esto NO es todavía un análisis de resultados.

print("\n=== EJEMPLO DE LA NUEVA ETIQUETA ===")

print(
    df_work[
        [
            "review_id",
            "stars",
            "sentiment",
            "sentiment_id"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. MENSAJE METODOLÓGICO
# ------------------------------------------------------------

print(
    "\nNOTA METODOLÓGICA:"
)

print(
    "El sentimiento es una etiqueta proxy derivada de 'stars'; "
    "no corresponde a una anotación textual independiente."
)

In [3]:
# ============================================================
# G2 — RECUPERACIÓN DEL ENTORNO
# Volver a cargar el dataset después de reiniciar Colab
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# El error anterior indicó:
#
#     NameError: df_original is not defined
#
# Esto significa que Colab perdió la variable df_original
# que habíamos creado anteriormente.
#
# Esta celda:
#
# 1. Comprueba que yelp_dataset.xlsx sigue disponible.
# 2. Abre la hoja que ya identificamos: "Hoja 1_yelp".
# 3. Vuelve a crear df_original.
# 4. Comprueba sus dimensiones.
# 5. NO modifica el archivo Excel.
#
# IMPORTANTE:
#
# Esto NO significa comenzar G2 desde cero.
# Los resultados que ya ejecutamos y registramos siguen siendo
# evidencia de la auditoría.
#
# Simplemente reconstruimos en memoria la variable necesaria
# para continuar trabajando.


# ------------------------------------------------------------
# 1. IMPORTAR LAS LIBRERÍAS NECESARIAS
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINIR LA RUTA DEL ARCHIVO ORIGINAL
# ------------------------------------------------------------

# Esta variable solamente apunta al archivo.
# No realiza ninguna modificación sobre él.

DATASET_PATH = Path("/content/yelp_dataset.xlsx")


# ------------------------------------------------------------
# 3. COMPROBAR QUE EL ARCHIVO EXISTE
# ------------------------------------------------------------

print("=== COMPROBACIÓN DEL ARCHIVO ===")
print("Archivo existe:", DATASET_PATH.exists())
print("Ruta:", DATASET_PATH)


# ------------------------------------------------------------
# 4. DETENERNOS SI EL ARCHIVO YA NO ESTÁ EN COLAB
# ------------------------------------------------------------

# Si el archivo desapareció al reiniciar la sesión,
# este assert producirá un mensaje claro.
#
# En ese caso solo tendremos que volver a subir el Excel.

assert DATASET_PATH.exists(), (
    "No se encuentra yelp_dataset.xlsx. "
    "Debes volver a subirlo a Colab."
)


# ------------------------------------------------------------
# 5. ABRIR EL EXCEL Y COMPROBAR SUS HOJAS
# ------------------------------------------------------------

xls = pd.ExcelFile(DATASET_PATH)

print("\n=== HOJAS DISPONIBLES ===")
print(xls.sheet_names)


# ------------------------------------------------------------
# 6. SELECCIONAR LA HOJA YA AUDITADA
# ------------------------------------------------------------

sheet_name = "Hoja 1_yelp"


# ------------------------------------------------------------
# 7. RECONSTRUIR df_original
# ------------------------------------------------------------

# read_excel() lee el archivo y crea un DataFrame en memoria.
#
# No estamos escribiendo sobre el Excel.
# Por tanto, yelp_dataset.xlsx permanece intacto.

df_original = pd.read_excel(
    DATASET_PATH,
    sheet_name=sheet_name
)


# ------------------------------------------------------------
# 8. COMPROBAR QUE LA RECUPERACIÓN FUE CORRECTA
# ------------------------------------------------------------

print("\n=== DATASET RECUPERADO ===")
print("Hoja:", sheet_name)
print("Filas:", df_original.shape[0])
print("Columnas:", df_original.shape[1])


# ------------------------------------------------------------
# 9. COMPROBAR QUE df_original SIGUE SIN NUEVAS ETIQUETAS
# ------------------------------------------------------------

# sentiment todavía NO debería existir en df_original,
# porque esa columna debe crearse solamente en df_work.

print(
    "¿sentiment existe en df_original?:",
    "sentiment" in df_original.columns
)

print(
    "¿sentiment_id existe en df_original?:",
    "sentiment_id" in df_original.columns
)


print("\nRecuperación terminada correctamente.")

=== COMPROBACIÓN DEL ARCHIVO ===
Archivo existe: False
Ruta: /content/yelp_dataset.xlsx


In [4]:
# ============================================================
# G2 — RECUPERACIÓN DEL ENTORNO
# Paso: volver a subir yelp_dataset.xlsx
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Abre el selector de archivos de Colab.
# 2. Permite volver a subir yelp_dataset.xlsx.
# 3. Guarda temporalmente el archivo en /content/.
#
# IMPORTANTE:
# - No modifica el archivo original de tu computador.
# - Solo crea una copia temporal dentro de la sesión de Colab.
# - Todavía no hacemos ninguna auditoría nueva.

from google.colab import files

uploaded = files.upload()

Saving yelp_dataset.xlsx to yelp_dataset.xlsx


In [5]:
# ============================================================
# G2 — RECUPERACIÓN DEL ENTORNO
# Paso: comprobar que el dataset volvió a cargarse
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Verifica que el archivo exista nuevamente en /content/.
# Todavía no lo lee ni modifica.

from pathlib import Path

DATASET_PATH = Path("/content/yelp_dataset.xlsx")

print("Archivo existe:", DATASET_PATH.exists())
print("Ruta:", DATASET_PATH)

Archivo existe: True
Ruta: /content/yelp_dataset.xlsx


In [6]:
# ============================================================
# G2 — RECUPERACIÓN DEL ENTORNO
# Paso: reconstruir df_original
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Abre la hoja "Hoja 1_yelp" del archivo Excel.
# 2. Carga sus datos en memoria dentro de una variable llamada
#    df_original.
# 3. Comprueba que el número de filas y columnas coincide con
#    la auditoría que ya habíamos realizado.
# 4. Verifica que las columnas sentiment y sentiment_id todavía
#    NO existen en df_original.
#
# IMPORTANTE:
#
# - Esta celda NO modifica yelp_dataset.xlsx.
# - Esta celda NO elimina duplicados.
# - Esta celda NO crea la etiqueta de sentimiento todavía.
# - Esta celda NO crea splits.
# - Solo reconstruye el DataFrame original en memoria.


# ------------------------------------------------------------
# 1. IMPORTAR PANDAS
# ------------------------------------------------------------

import pandas as pd


# ------------------------------------------------------------
# 2. DEFINIR LA HOJA QUE YA AUDITAMOS
# ------------------------------------------------------------

sheet_name = "Hoja 1_yelp"


# ------------------------------------------------------------
# 3. LEER EL EXCEL Y CREAR df_original
# ------------------------------------------------------------

# read_excel() lee el contenido del archivo.
# No escribe nada sobre él.

df_original = pd.read_excel(
    DATASET_PATH,
    sheet_name=sheet_name
)


# ------------------------------------------------------------
# 4. COMPROBAR DIMENSIONES
# ------------------------------------------------------------

print("=== DATASET RECUPERADO ===")
print("Hoja:", sheet_name)
print("Filas:", df_original.shape[0])
print("Columnas:", df_original.shape[1])


# ------------------------------------------------------------
# 5. COMPROBAR QUE SIGUE SIENDO EL ORIGINAL
# ------------------------------------------------------------

print(
    "\n¿sentiment existe en df_original?:",
    "sentiment" in df_original.columns
)

print(
    "¿sentiment_id existe en df_original?:",
    "sentiment_id" in df_original.columns
)

=== DATASET RECUPERADO ===
Hoja: Hoja 1_yelp
Filas: 10000
Columnas: 10

¿sentiment existe en df_original?: False
¿sentiment_id existe en df_original?: False


In [7]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 9: crear la etiqueta proxy de sentimiento
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea una COPIA de trabajo llamada df_work.
#
# 2. Usa la columna 'stars' para crear dos columnas nuevas:
#
#       sentiment
#       sentiment_id
#
# 3. Aplica la regla oficial del caso:
#
#       1–2 estrellas -> NEGATIVO (0)
#       3 estrellas   -> NEUTRAL  (1)
#       4–5 estrellas -> POSITIVO (2)
#
# 4. Comprueba que todas las filas hayan recibido
#    correctamente una etiqueta.
#
# 5. Verifica que df_original siga intacto.
#
# IMPORTANTE:
#
# - La etiqueta de sentimiento es una ETIQUETA PROXY.
# - Se deriva de 'stars'.
# - No es una anotación textual independiente.
# - No modificamos yelp_dataset.xlsx.
# - No eliminamos todavía textos duplicados.
# - No creamos splits.
# - No hacemos preprocesamiento lingüístico.


# ------------------------------------------------------------
# 1. CREAR UNA COPIA DE TRABAJO
# ------------------------------------------------------------

# copy(deep=True) crea una copia independiente de df_original.
#
# df_original:
#     conserva los datos tal como fueron leídos del Excel.
#
# df_work:
#     será la copia sobre la que añadiremos nuevas columnas.

df_work = df_original.copy(deep=True)


# ------------------------------------------------------------
# 2. DEFINIR LA REGLA stars -> sentimiento
# ------------------------------------------------------------

def map_sentiment(stars):

    # 1 o 2 estrellas
    if stars in (1, 2):
        return "NEGATIVO", 0

    # 3 estrellas
    elif stars == 3:
        return "NEUTRAL", 1

    # 4 o 5 estrellas
    elif stars in (4, 5):
        return "POSITIVO", 2

    # Este caso solo debería aparecer si stars
    # contiene un valor inesperado.
    else:
        return None, None


# ------------------------------------------------------------
# 3. APLICAR LA REGLA A TODAS LAS FILAS
# ------------------------------------------------------------

# apply() ejecuta la función map_sentiment
# sobre cada valor de la columna stars.

mapped_sentiment = df_work["stars"].apply(
    map_sentiment
)


# ------------------------------------------------------------
# 4. CREAR LA COLUMNA sentiment
# ------------------------------------------------------------

# str[0] toma el primer elemento de cada resultado:
#
# NEGATIVO
# NEUTRAL
# POSITIVO

df_work["sentiment"] = (
    mapped_sentiment.str[0]
)


# ------------------------------------------------------------
# 5. CREAR LA COLUMNA sentiment_id
# ------------------------------------------------------------

# str[1] toma el segundo elemento:
#
# NEGATIVO -> 0
# NEUTRAL  -> 1
# POSITIVO -> 2
#
# Int64 permite manejar enteros y posibles valores faltantes.

df_work["sentiment_id"] = (
    mapped_sentiment
    .str[1]
    .astype("Int64")
)


# ------------------------------------------------------------
# 6. MOSTRAR LA REGLA UTILIZADA
# ------------------------------------------------------------

print("=== REGLA DE ETIQUETADO PROXY ===")
print("1–2 estrellas -> NEGATIVO (0)")
print("3 estrellas   -> NEUTRAL  (1)")
print("4–5 estrellas -> POSITIVO (2)")


# ------------------------------------------------------------
# 7. COMPROBAR QUE TODAS LAS FILAS TIENEN ETIQUETA
# ------------------------------------------------------------

missing_sentiment = int(
    df_work["sentiment"].isna().sum()
)

missing_sentiment_id = int(
    df_work["sentiment_id"].isna().sum()
)

print("\n=== VALIDACIÓN DE LA ETIQUETA ===")
print(
    "Filas sin etiqueta textual:",
    missing_sentiment
)
print(
    "Filas sin etiqueta numérica:",
    missing_sentiment_id
)


# ------------------------------------------------------------
# 8. COMPROBAR QUE df_original SIGUE INTACTO
# ------------------------------------------------------------

print("\n=== CONTROL DEL DATASET ORIGINAL ===")

print(
    "Filas en df_original:",
    len(df_original)
)

print(
    "Filas en df_work:",
    len(df_work)
)

print(
    "¿sentiment existe en df_original?:",
    "sentiment" in df_original.columns
)

print(
    "¿sentiment existe en df_work?:",
    "sentiment" in df_work.columns
)


# ------------------------------------------------------------
# 9. MOSTRAR UNA MUESTRA PARA COMPROBAR COHERENCIA
# ------------------------------------------------------------

print("\n=== EJEMPLO DE LA NUEVA ETIQUETA ===")

print(
    df_work[
        [
            "review_id",
            "stars",
            "sentiment",
            "sentiment_id"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. DEJAR REGISTRADA LA NOTA METODOLÓGICA
# ------------------------------------------------------------

print("\n=== NOTA METODOLÓGICA ===")

print(
    "El sentimiento es una etiqueta proxy derivada de 'stars'; "
    "no corresponde a una anotación textual independiente."
)

=== REGLA DE ETIQUETADO PROXY ===
1–2 estrellas -> NEGATIVO (0)
3 estrellas   -> NEUTRAL  (1)
4–5 estrellas -> POSITIVO (2)

=== VALIDACIÓN DE LA ETIQUETA ===
Filas sin etiqueta textual: 0
Filas sin etiqueta numérica: 0

=== CONTROL DEL DATASET ORIGINAL ===
Filas en df_original: 10000
Filas en df_work: 10000
¿sentiment existe en df_original?: False
¿sentiment existe en df_work?: True

=== EJEMPLO DE LA NUEVA ETIQUETA ===
             review_id  stars sentiment  sentiment_id
fWKvX83p0-ka4JS3dc6E5A      5  POSITIVO             2
IjZ33sJrzXqU-0X6U8NwyA      5  POSITIVO             2
IESLBzqUCLdSzSqm0eCSxQ      4  POSITIVO             2
G-WvGaISbqqaMHlNnByodA      5  POSITIVO             2
1uJFq2r5QfJG_6ExMRCaGw      5  POSITIVO             2
m2CKSsepBCoRYWxiRUsxAg      4  POSITIVO             2
riFQ3vxNpP4rWLk_CSri2A      5  POSITIVO             2
JL7GXJ9u4YMx7Rzs05NfiQ      4  POSITIVO             2
XtnfnYmnJYi71yIuGsXIUA      4  POSITIVO             2
jJAIXA46pU1swYyRCdfXtQ      5  POSI

In [8]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 10: distribución de las clases de sentimiento
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Cuenta cuántas reseñas pertenecen a cada sentimiento:
#
#       NEGATIVO (0)
#       NEUTRAL  (1)
#       POSITIVO (2)
#
# 2. Calcula qué porcentaje del dataset representa cada clase.
#
# 3. Comprueba que la suma de las tres clases coincide con
#    el número total de filas de df_work.
#
# 4. Comprueba también que existe correspondencia exacta entre:
#
#       NEGATIVO <-> 0
#       NEUTRAL  <-> 1
#       POSITIVO <-> 2
#
# ¿POR QUÉ LO HACEMOS?
#
# Necesitamos conocer la distribución de la variable objetivo
# antes de entrenar cualquier modelo.
#
# Si una clase aparece mucho más que las demás, existe
# desbalance de clases. Eso afectará posteriormente a la
# interpretación de accuracy y justificará utilizar F1 macro
# como métrica principal.
#
# IMPORTANTE:
#
# - Esta celda NO modifica df_work.
# - NO modifica df_original.
# - NO modifica yelp_dataset.xlsx.
# - NO elimina duplicados todavía.
# - NO calcula pesos de clase.
# - NO crea train, validation ni test.
# - NO entrena ningún modelo.


# ------------------------------------------------------------
# 1. DEFINIR EL ORDEN DE LAS CLASES
# ------------------------------------------------------------

# Establecemos manualmente el orden para que las tablas
# siempre aparezcan de la misma manera:
#
# NEGATIVO -> NEUTRAL -> POSITIVO

sentiment_order = [
    "NEGATIVO",
    "NEUTRAL",
    "POSITIVO"
]


# ------------------------------------------------------------
# 2. CONTAR RESEÑAS POR SENTIMIENTO
# ------------------------------------------------------------

# value_counts() cuenta cuántas veces aparece cada clase.
#
# reindex(sentiment_order) obliga a mostrar las clases
# en el orden definido anteriormente.
#
# fillna(0) serviría para mostrar 0 si alguna clase
# inesperadamente no estuviera presente.

sentiment_distribution = (
    df_work["sentiment"]
    .value_counts()
    .reindex(sentiment_order)
    .fillna(0)
    .astype(int)
    .rename_axis("sentimiento")
    .reset_index(name="cantidad")
)


# ------------------------------------------------------------
# 3. CALCULAR PORCENTAJES
# ------------------------------------------------------------

# Para cada sentimiento:
#
# porcentaje = cantidad / total de reseñas * 100

sentiment_distribution["porcentaje"] = (
    sentiment_distribution["cantidad"]
    / len(df_work)
    * 100
).round(2)


# ------------------------------------------------------------
# 4. MOSTRAR LA DISTRIBUCIÓN
# ------------------------------------------------------------

print("=== DISTRIBUCIÓN POR SENTIMIENTO ===")

print(
    sentiment_distribution.to_string(index=False)
)


# ------------------------------------------------------------
# 5. COMPROBAR QUE TODAS LAS FILAS ESTÁN CONTABILIZADAS
# ------------------------------------------------------------

total_sentiment = int(
    sentiment_distribution["cantidad"].sum()
)

print("\n=== COMPROBACIÓN DEL TOTAL ===")

print(
    "Total contabilizado:",
    total_sentiment
)

print(
    "Total en df_work:",
    len(df_work)
)

print(
    "¿Coinciden los totales?:",
    total_sentiment == len(df_work)
)


# ------------------------------------------------------------
# 6. COMPROBAR LA CORRESPONDENCIA TEXTO <-> CÓDIGO
# ------------------------------------------------------------

# Creamos una tabla cruzada.
#
# Las filas serán:
# NEGATIVO, NEUTRAL y POSITIVO.
#
# Las columnas serán:
# 0, 1 y 2.
#
# Si la codificación es correcta, cada sentimiento debería
# aparecer únicamente bajo su código correspondiente.

label_check = pd.crosstab(
    df_work["sentiment"],
    df_work["sentiment_id"]
)

print("\n=== COMPROBACIÓN sentiment <-> sentiment_id ===")

print(label_check)


# ------------------------------------------------------------
# 7. MOSTRAR LA REGLA PARA FACILITAR LA INTERPRETACIÓN
# ------------------------------------------------------------

print("\n=== CODIFICACIÓN UTILIZADA ===")

print("NEGATIVO -> 0")
print("NEUTRAL  -> 1")
print("POSITIVO -> 2")


# ------------------------------------------------------------
# 8. RECORDATORIO METODOLÓGICO
# ------------------------------------------------------------

print("\n=== RECORDATORIO ===")

print(
    "Estas clases son etiquetas proxy derivadas de stars."
)

print(
    "Todavía no se han calculado pesos de clase ni creado splits."
)

=== DISTRIBUCIÓN POR SENTIMIENTO ===
sentimiento  cantidad  porcentaje
   NEGATIVO      1676       16.76
    NEUTRAL      1461       14.61
   POSITIVO      6863       68.63

=== COMPROBACIÓN DEL TOTAL ===
Total contabilizado: 10000
Total en df_work: 10000
¿Coinciden los totales?: True

=== COMPROBACIÓN sentiment <-> sentiment_id ===
sentiment_id     0     1     2
sentiment                     
NEGATIVO      1676     0     0
NEUTRAL          0  1461     0
POSITIVO         0     0  6863

=== CODIFICACIÓN UTILIZADA ===
NEGATIVO -> 0
NEUTRAL  -> 1
POSITIVO -> 2

=== RECORDATORIO ===
Estas clases son etiquetas proxy derivadas de stars.
Todavía no se han calculado pesos de clase ni creado splits.


In [9]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 8: distribución original de reseñas por estrellas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Esta comprobación quedó pendiente después del reinicio
# de la sesión de Colab.
#
# 1. Cuenta por separado las reseñas de:
#       1 estrella
#       2 estrellas
#       3 estrellas
#       4 estrellas
#       5 estrellas
#
# 2. Calcula el porcentaje correspondiente a cada valor.
#
# 3. Comprueba que no existen valores de stars fuera
#    del intervalo permitido 1–5.
#
# 4. Comprueba que la suma coincide con las 10.000 filas.
#
# ¿POR QUÉ ES IMPORTANTE?
#
# La variable 'stars' es la fuente de nuestra etiqueta proxy.
# Por eso debemos documentar su distribución ORIGINAL antes
# de agrupar:
#
#       1–2 -> NEGATIVO
#       3   -> NEUTRAL
#       4–5 -> POSITIVO
#
# IMPORTANTE:
#
# - Esta celda SOLO inspecciona df_original.
# - NO modifica df_original.
# - NO modifica el Excel.
# - NO crea nuevas etiquetas.
# - NO elimina duplicados.
# - NO crea splits.


# ------------------------------------------------------------
# 1. CONTAR RESEÑAS DE CADA NÚMERO DE ESTRELLAS
# ------------------------------------------------------------

# value_counts() cuenta las apariciones de cada valor.
#
# sort_index() ordena el resultado:
# 1, 2, 3, 4 y 5.

stars_distribution = (
    df_original["stars"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("stars")
    .reset_index(name="cantidad")
)


# ------------------------------------------------------------
# 2. CALCULAR EL PORCENTAJE
# ------------------------------------------------------------

# porcentaje =
# número de reseñas de esa categoría / total * 100

stars_distribution["porcentaje"] = (
    stars_distribution["cantidad"]
    / len(df_original)
    * 100
).round(2)


# ------------------------------------------------------------
# 3. MOSTRAR LOS RESULTADOS
# ------------------------------------------------------------

print("=== DISTRIBUCIÓN POR ESTRELLAS ===")

print(
    stars_distribution.to_string(index=False)
)


# ------------------------------------------------------------
# 4. COMPROBAR QUE stars SOLO CONTIENE VALORES DE 1 A 5
# ------------------------------------------------------------

valid_stars_mask = (
    df_original["stars"].isin([1, 2, 3, 4, 5])
)

invalid_stars = int(
    (~valid_stars_mask).sum()
)

print("\n=== VALIDACIÓN DEL RANGO DE stars ===")

print(
    "Valores fuera del rango 1–5:",
    invalid_stars
)


# ------------------------------------------------------------
# 5. COMPROBAR EL TOTAL
# ------------------------------------------------------------

total_stars = int(
    stars_distribution["cantidad"].sum()
)

print("\n=== COMPROBACIÓN DEL TOTAL ===")

print(
    "Total contabilizado:",
    total_stars
)

print(
    "Total en df_original:",
    len(df_original)
)

print(
    "¿Coinciden los totales?:",
    total_stars == len(df_original)
)


# ------------------------------------------------------------
# 6. RECORDATORIO METODOLÓGICO
# ------------------------------------------------------------

print("\n=== RECORDATORIO ===")

print(
    "stars se utiliza únicamente para construir la etiqueta proxy."
)

print(
    "stars NO será utilizada como predictor de los modelos."
)

=== DISTRIBUCIÓN POR ESTRELLAS ===
 stars  cantidad  porcentaje
     1       749        7.49
     2       927        9.27
     3      1461       14.61
     4      3526       35.26
     5      3337       33.37

=== VALIDACIÓN DEL RANGO DE stars ===
Valores fuera del rango 1–5: 0

=== COMPROBACIÓN DEL TOTAL ===
Total contabilizado: 10000
Total en df_original: 10000
¿Coinciden los totales?: True

=== RECORDATORIO ===
stars se utiliza únicamente para construir la etiqueta proxy.
stars NO será utilizada como predictor de los modelos.


In [10]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 11: analizar la longitud de las reseñas en palabras
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Toma el texto ORIGINAL de cada reseña.
#
# 2. Cuenta aproximadamente cuántas palabras contiene cada
#    reseña utilizando espacios como separadores.
#
# 3. Calcula las estadísticas solicitadas:
#
#       - promedio
#       - mediana
#       - percentil 90 (P90)
#       - percentil 95 (P95)
#       - mínimo
#       - máximo
#
# ¿POR QUÉ LO HACEMOS?
#
# Queremos conocer qué tan largas son las reseñas antes de
# diseñar posteriormente las entradas de los modelos.
#
# Los percentiles son especialmente útiles:
#
# P90 = el 90 % de las reseñas tiene una longitud igual
#       o inferior a ese valor.
#
# P95 = el 95 % de las reseñas tiene una longitud igual
#       o inferior a ese valor.
#
# Esto será información útil posteriormente para justificar
# las longitudes máximas de secuencia.
#
# IMPORTANTE:
#
# - Esto NO es todavía tokenización NLP.
# - NO eliminamos stopwords.
# - NO eliminamos puntuación.
# - NO aplicamos lematización.
# - NO modificamos el texto.
# - NO modificamos df_original.
# - NO creamos splits.
#
# Simplemente realizamos una medición descriptiva utilizando
# separación por espacios.


# ------------------------------------------------------------
# 1. CONTAR PALABRAS DE CADA RESEÑA
# ------------------------------------------------------------

# fillna("") sería una protección frente a textos vacíos.
# Ya sabemos por nuestra auditoría que text no contiene nulos,
# pero dejamos esta protección para que el código sea robusto.
#
# astype(str) garantiza que trabajamos con cadenas.
#
# str.split() separa cada texto utilizando espacios.
#
# str.len() cuenta cuántos elementos resultaron de esa
# separación.

word_lengths = (
    df_original["text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)


# ------------------------------------------------------------
# 2. CALCULAR EL PROMEDIO
# ------------------------------------------------------------

# mean() calcula la media aritmética de palabras por reseña.

mean_words = word_lengths.mean()


# ------------------------------------------------------------
# 3. CALCULAR LA MEDIANA
# ------------------------------------------------------------

# median() identifica el valor central de la distribución.
#
# La mediana suele ser especialmente útil cuando existen
# algunas reseñas extremadamente largas.

median_words = word_lengths.median()


# ------------------------------------------------------------
# 4. CALCULAR EL PERCENTIL 90
# ------------------------------------------------------------

# quantile(0.90) determina la longitud por debajo de la cual
# se encuentra aproximadamente el 90 % de las reseñas.

p90_words = word_lengths.quantile(0.90)


# ------------------------------------------------------------
# 5. CALCULAR EL PERCENTIL 95
# ------------------------------------------------------------

# quantile(0.95) determina la longitud por debajo de la cual
# se encuentra aproximadamente el 95 % de las reseñas.

p95_words = word_lengths.quantile(0.95)


# ------------------------------------------------------------
# 6. CALCULAR MÍNIMO Y MÁXIMO
# ------------------------------------------------------------

min_words = word_lengths.min()
max_words = word_lengths.max()


# ------------------------------------------------------------
# 7. AGRUPAR LOS RESULTADOS EN UNA TABLA
# ------------------------------------------------------------

length_statistics = pd.DataFrame({
    "estadística": [
        "Promedio",
        "Mediana",
        "Percentil 90",
        "Percentil 95",
        "Mínimo",
        "Máximo"
    ],
    "palabras": [
        mean_words,
        median_words,
        p90_words,
        p95_words,
        min_words,
        max_words
    ]
})


# ------------------------------------------------------------
# 8. MOSTRAR LOS RESULTADOS
# ------------------------------------------------------------

print("=== LONGITUD DE LAS RESEÑAS EN PALABRAS ===")

print(
    length_statistics.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 9. COMPROBAR CUÁNTAS RESEÑAS FUERON MEDIDAS
# ------------------------------------------------------------

print("\n=== COMPROBACIÓN ===")

print(
    "Reseñas medidas:",
    len(word_lengths)
)

print(
    "Total de filas en df_original:",
    len(df_original)
)

print(
    "¿Coinciden los totales?:",
    len(word_lengths) == len(df_original)
)


# ------------------------------------------------------------
# 10. COMPROBAR POSIBLES TEXTOS DE LONGITUD CERO
# ------------------------------------------------------------

# Aunque ya comprobamos que text no tiene valores nulos,
# también verificamos si existe algún texto cuya longitud
# calculada sea cero.

zero_length_reviews = int(
    (word_lengths == 0).sum()
)

print(
    "Reseñas con longitud 0:",
    zero_length_reviews
)


# ------------------------------------------------------------
# 11. RECORDATORIO METODOLÓGICO
# ------------------------------------------------------------

print("\n=== RECORDATORIO ===")

print(
    "Estas longitudes se calcularon sobre el texto original."
)

print(
    "No se ha aplicado todavía preprocesamiento lingüístico."
)

=== LONGITUD DE LAS RESEÑAS EN PALABRAS ===
 estadística  palabras
    Promedio  131.0396
     Mediana  101.0000
Percentil 90  272.0000
Percentil 95  348.0000
      Mínimo    1.0000
      Máximo  945.0000

=== COMPROBACIÓN ===
Reseñas medidas: 10000
Total de filas en df_original: 10000
¿Coinciden los totales?: True
Reseñas con longitud 0: 0

=== RECORDATORIO ===
Estas longitudes se calcularon sobre el texto original.
No se ha aplicado todavía preprocesamiento lingüístico.


In [11]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 12: analizar el desbalance de clases
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Cuenta cuántas reseñas hay en cada clase:
#
#       NEGATIVO
#       NEUTRAL
#       POSITIVO
#
# 2. Identifica cuál es la clase con MÁS reseñas
#    (clase mayoritaria).
#
# 3. Identifica cuál es la clase con MENOS reseñas
#    (clase minoritaria).
#
# 4. Calcula el ratio entre la clase mayoritaria
#    y la clase minoritaria.
#
# 5. Calcula qué accuracy obtendríamos si un clasificador
#    muy simple predijera SIEMPRE la clase mayoritaria.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Porque nuestro dataset no tiene necesariamente la misma
# cantidad de ejemplos en NEGATIVO, NEUTRAL y POSITIVO.
#
# Cuando una clase domina sobre las demás, decimos que existe
# DESBALANCE DE CLASES.
#
# Esto es importante porque la accuracy puede ser engañosa.
#
# Ejemplo conceptual:
#
# Si una clase fuese muy frecuente, un modelo podría acertar
# muchas reseñas simplemente prediciendo siempre esa clase,
# aunque fuera malo distinguiendo las otras dos.
#
# Por esa razón, en este caso práctico:
#
#       F1 macro será la métrica principal.
#
# F1 macro calcula el desempeño de cada clase por separado
# y después les da la misma importancia.
#
#
# IMPORTANTE:
#
# - Esta celda NO entrena ningún modelo.
# - NO modifica df_work.
# - NO modifica df_original.
# - NO modifica yelp_dataset.xlsx.
# - NO crea train, validation ni test.
# - NO calcula todavía pesos de clase.
# - NO hace sobremuestreo ni submuestreo.
#
# Los pesos de clase se calcularán MÁS ADELANTE,
# exclusivamente utilizando TRAIN.
#
# Hacerlo ahora con todo el dataset provocaría fuga
# de información hacia validation/test.


# ------------------------------------------------------------
# 1. CONTAR CUÁNTAS RESEÑAS HAY EN CADA CLASE
# ------------------------------------------------------------

# value_counts() cuenta cuántas veces aparece
# cada sentimiento dentro de df_work.

class_counts = (
    df_work["sentiment"]
    .value_counts()
)

print("=== CANTIDAD POR CLASE ===")
print(class_counts)


# ------------------------------------------------------------
# 2. IDENTIFICAR LA CLASE MAYORITARIA
# ------------------------------------------------------------

# idxmax() devuelve el NOMBRE de la clase
# que tiene el mayor número de reseñas.

majority_class = class_counts.idxmax()


# max() devuelve la CANTIDAD de reseñas
# correspondiente a esa clase.

majority_count = int(
    class_counts.max()
)


# ------------------------------------------------------------
# 3. IDENTIFICAR LA CLASE MINORITARIA
# ------------------------------------------------------------

# idxmin() devuelve el nombre de la clase
# que tiene menos reseñas.

minority_class = class_counts.idxmin()


# min() devuelve su cantidad.

minority_count = int(
    class_counts.min()
)


# ------------------------------------------------------------
# 4. CALCULAR EL PORCENTAJE DE LA CLASE MAYORITARIA
# ------------------------------------------------------------

# Dividimos la cantidad de la clase mayoritaria
# entre el total de reseñas y multiplicamos por 100.

majority_percentage = (
    majority_count
    / len(df_work)
    * 100
)


# ------------------------------------------------------------
# 5. CALCULAR EL RATIO MAYORÍA / MINORÍA
# ------------------------------------------------------------

# Este ratio permite expresar numéricamente el desbalance.
#
# Por ejemplo:
#
# ratio = 1
# significaría que ambas clases tienen el mismo tamaño.
#
# ratio = 2
# significaría que la clase mayoritaria tiene el doble
# de ejemplos que la minoritaria.
#
# Nosotros NO anticipamos el resultado:
# lo calculará Colab con los datos reales.

imbalance_ratio = (
    majority_count
    / minority_count
)


# ------------------------------------------------------------
# 6. CALCULAR EL BASELINE DESCRIPTIVO MAYORITARIO
# ------------------------------------------------------------

# Imaginemos un clasificador extremadamente simple que
# respondiera SIEMPRE con la clase mayoritaria.
#
# Su accuracy sería:
#
# cantidad de la clase mayoritaria / total de reseñas
#
# IMPORTANTE:
#
# Esto NO es todavía el baseline formal de G5.
#
# Aquí lo calculamos únicamente como referencia descriptiva
# para entender por qué accuracy puede ser engañosa.

majority_baseline_accuracy = (
    majority_count
    / len(df_work)
)


# ------------------------------------------------------------
# 7. MOSTRAR LOS RESULTADOS DEL DESBALANCE
# ------------------------------------------------------------

print("\n=== DESBALANCE DE CLASES ===")

print(
    "Clase mayoritaria:",
    majority_class
)

print(
    "Cantidad clase mayoritaria:",
    majority_count
)

print(
    "Porcentaje clase mayoritaria:",
    round(majority_percentage, 2),
    "%"
)

print()

print(
    "Clase minoritaria:",
    minority_class
)

print(
    "Cantidad clase minoritaria:",
    minority_count
)

print()

print(
    "Ratio mayoría/minoría:",
    round(imbalance_ratio, 4)
)


# ------------------------------------------------------------
# 8. MOSTRAR EL BASELINE DESCRIPTIVO
# ------------------------------------------------------------

print(
    "\n=== BASELINE DESCRIPTIVO DE CLASE MAYORITARIA ==="
)

print(
    "Si predijéramos siempre:",
    majority_class
)

print(
    "Accuracy:",
    round(majority_baseline_accuracy, 4)
)

print(
    "Accuracy en porcentaje:",
    round(majority_baseline_accuracy * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 9. DEJAR REGISTRADA LA IMPLICACIÓN METODOLÓGICA
# ------------------------------------------------------------

print("\n=== IMPLICACIÓN METODOLÓGICA ===")

print(
    "La distribución de clases está siendo auditada antes "
    "de cualquier entrenamiento."
)

print(
    "Accuracy deberá interpretarse frente al baseline "
    "de clase mayoritaria."
)

print(
    "F1 macro será la métrica principal para comparar "
    "los modelos."
)

print(
    "Los pesos de clase se calcularán únicamente con TRAIN "
    "después de crear los splits."
)


# ------------------------------------------------------------
# 10. CONTROL DE COMPUERTA
# ------------------------------------------------------------

print("\n=== CONTROL DE COMPUERTA ===")

print("G2: EN EJECUCIÓN")
print("G3: NO INICIADO")
print("Modelos entrenados: NINGUNO")

=== CANTIDAD POR CLASE ===
sentiment
POSITIVO    6863
NEGATIVO    1676
NEUTRAL     1461
Name: count, dtype: int64

=== DESBALANCE DE CLASES ===
Clase mayoritaria: POSITIVO
Cantidad clase mayoritaria: 6863
Porcentaje clase mayoritaria: 68.63 %

Clase minoritaria: NEUTRAL
Cantidad clase minoritaria: 1461

Ratio mayoría/minoría: 4.6975

=== BASELINE DESCRIPTIVO DE CLASE MAYORITARIA ===
Si predijéramos siempre: POSITIVO
Accuracy: 0.6863
Accuracy en porcentaje: 68.63 %

=== IMPLICACIÓN METODOLÓGICA ===
La distribución de clases está siendo auditada antes de cualquier entrenamiento.
Accuracy deberá interpretarse frente al baseline de clase mayoritaria.
F1 macro será la métrica principal para comparar los modelos.
Los pesos de clase se calcularán únicamente con TRAIN después de crear los splits.

=== CONTROL DE COMPUERTA ===
G2: EN EJECUCIÓN
G3: NO INICIADO
Modelos entrenados: NINGUNO


In [12]:
# ============================================================
# G2 — AUDITORÍA DEL DATASET
# Paso 13: eliminar textos duplicados SOLO de la copia de trabajo
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba cuántas filas tiene df_original.
#
# 2. Comprueba cuántas filas tiene df_work antes
#    de eliminar duplicados.
#
# 3. Identifica textos exactamente repetidos.
#
# 4. Conserva la PRIMERA aparición de cada texto.
#
# 5. Elimina únicamente las apariciones posteriores
#    de esos textos dentro de df_work.
#
# 6. Registra exactamente cuántas filas fueron eliminadas.
#
# 7. Comprueba que después de la operación no quede
#    ningún texto duplicado exacto.
#
# 8. Comprueba que df_original siga intacto.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Ya comprobamos anteriormente que existen textos exactamente
# repetidos.
#
# No queremos que copias idénticas del mismo texto puedan
# terminar posteriormente en conjuntos diferentes.
#
# Por eso retiramos las apariciones redundantes ANTES
# de crear train, validation y test.
#
#
# IMPORTANTE:
#
# - SOLO modificamos df_work.
# - NO modificamos df_original.
# - NO sobrescribimos yelp_dataset.xlsx.
# - NO hacemos preprocesamiento lingüístico.
# - NO creamos todavía ningún split.
# - NO entrenamos ningún modelo.


# ------------------------------------------------------------
# 1. REGISTRAR EL TAMAÑO ANTES DE LA DEDUPLICACIÓN
# ------------------------------------------------------------

original_rows_before = len(df_original)
work_rows_before = len(df_work)

print("=== ANTES DE ELIMINAR DUPLICADOS ===")

print(
    "Filas en df_original:",
    original_rows_before
)

print(
    "Filas en df_work:",
    work_rows_before
)


# ------------------------------------------------------------
# 2. IDENTIFICAR LAS FILAS QUE SERÍAN ELIMINADAS
# ------------------------------------------------------------

# duplicated(subset=["text"], keep="first") funciona así:
#
# - La primera aparición de un texto se conserva.
# - Las apariciones posteriores del mismo texto
#   se consideran redundantes.
#
# Todavía no eliminamos nada en este bloque.
# Primero guardamos un registro de qué filas serán retiradas.

duplicate_mask = df_work.duplicated(
    subset=["text"],
    keep="first"
)

removed_rows_log = df_work.loc[
    duplicate_mask,
    [
        "review_id",
        "business_id",
        "user_id",
        "stars",
        "sentiment",
        "sentiment_id"
    ]
].copy()


print("\n=== REGISTRO DE EXCLUSIONES ===")

print(
    "Filas identificadas como redundantes:",
    len(removed_rows_log)
)

if len(removed_rows_log) > 0:

    print(
        removed_rows_log.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 3. ELIMINAR LOS DUPLICADOS SOLO DE df_work
# ------------------------------------------------------------

# drop_duplicates() elimina las apariciones posteriores
# de un texto exactamente idéntico.
#
# subset=["text"]:
# únicamente usamos el texto para definir duplicación.
#
# keep="first":
# conservamos la primera aparición.
#
# .copy():
# crea nuevamente un DataFrame independiente.

df_work = (
    df_work
    .drop_duplicates(
        subset=["text"],
        keep="first"
    )
    .copy()
)


# ------------------------------------------------------------
# 4. REINICIAR EL ÍNDICE DE LA COPIA DE TRABAJO
# ------------------------------------------------------------

# Después de eliminar filas, los índices originales pueden
# quedar con saltos.
#
# reset_index(drop=True) crea un índice limpio:
# 0, 1, 2, 3...
#
# Esto NO altera el contenido de las reseñas.

df_work.reset_index(
    drop=True,
    inplace=True
)


# ------------------------------------------------------------
# 5. CALCULAR CUÁNTAS FILAS SE ELIMINARON
# ------------------------------------------------------------

work_rows_after = len(df_work)

removed_count = (
    work_rows_before
    - work_rows_after
)


print("\n=== DESPUÉS DE ELIMINAR DUPLICADOS ===")

print(
    "Filas en df_work antes:",
    work_rows_before
)

print(
    "Filas en df_work después:",
    work_rows_after
)

print(
    "Filas eliminadas:",
    removed_count
)


# ------------------------------------------------------------
# 6. COMPROBAR QUE NO QUEDEN TEXTOS DUPLICADOS EXACTOS
# ------------------------------------------------------------

remaining_duplicates = int(
    df_work["text"]
    .duplicated(keep=False)
    .sum()
)

print(
    "Filas en grupos de texto duplicado restantes:",
    remaining_duplicates
)


# ------------------------------------------------------------
# 7. COMPROBAR QUE df_original NO CAMBIÓ
# ------------------------------------------------------------

print("\n=== CONTROL DE INTEGRIDAD DEL ORIGINAL ===")

print(
    "Filas actuales en df_original:",
    len(df_original)
)

print(
    "Filas originales registradas:",
    original_rows_before
)

print(
    "¿df_original conserva el mismo número de filas?:",
    len(df_original) == original_rows_before
)


# ------------------------------------------------------------
# 8. COMPROBAR QUE LAS ETIQUETAS SIGUEN PRESENTES
# ------------------------------------------------------------

print("\n=== CONTROL DE LA COPIA DE TRABAJO ===")

print(
    "¿sentiment existe en df_work?:",
    "sentiment" in df_work.columns
)

print(
    "¿sentiment_id existe en df_work?:",
    "sentiment_id" in df_work.columns
)


# ------------------------------------------------------------
# 9. CONTROL FINAL DE ESTA OPERACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE DEDUPLICACIÓN ===")

print(
    "Número registrado de exclusiones:",
    len(removed_rows_log)
)

print(
    "Número calculado de filas eliminadas:",
    removed_count
)

print(
    "¿Coinciden ambos controles?:",
    len(removed_rows_log) == removed_count
)


# ------------------------------------------------------------
# 10. RECORDATORIO DE COMPUERTA
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print(
    "Dataset original: NO MODIFICADO"
)

print(
    "Copia de trabajo: deduplicación exacta de text aplicada"
)

print(
    "G3: NO INICIADO"
)

print(
    "Splits creados: NO"
)

print(
    "Modelos entrenados: NINGUNO"
)

=== ANTES DE ELIMINAR DUPLICADOS ===
Filas en df_original: 10000
Filas en df_work: 10000

=== REGISTRO DE EXCLUSIONES ===
Filas identificadas como redundantes: 2
             review_id            business_id                user_id  stars sentiment  sentiment_id
M_GC_TG9TpSzMAUQ_TAimw kkBMTNET2xgHCW-cnNwKxA 9VmTOyq01oIUk5zuxOj1GA      5  POSITIVO             2
mutQE6UfjLIpJ8Wozpq5UA rIonUa02zMz_ki8eF-Adug KLekdmo4FdNnP0huUhzZNw      2  NEGATIVO             0

=== DESPUÉS DE ELIMINAR DUPLICADOS ===
Filas en df_work antes: 10000
Filas en df_work después: 9998
Filas eliminadas: 2
Filas en grupos de texto duplicado restantes: 0

=== CONTROL DE INTEGRIDAD DEL ORIGINAL ===
Filas actuales en df_original: 10000
Filas originales registradas: 10000
¿df_original conserva el mismo número de filas?: True

=== CONTROL DE LA COPIA DE TRABAJO ===
¿sentiment existe en df_work?: True
¿sentiment_id existe en df_work?: True

=== CONTROL DE DEDUPLICACIÓN ===
Número registrado de exclusiones: 2
Número calcul

In [13]:
# ============================================================
# G3 — PREPARACIÓN DE LA PARTICIÓN
# Paso 0: guardar los artefactos finales de G2
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Antes de crear train, validation y test vamos a guardar
# físicamente el resultado auditado de G2.
#
# Guardaremos:
#
# 1. df_work
#    Es nuestra COPIA DE TRABAJO después de retirar
#    únicamente los textos duplicados exactos.
#
# 2. removed_rows_log
#    Es el registro de las filas que fueron excluidas
#    durante la deduplicación.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Las variables de Google Colab viven temporalmente
# en memoria. Si la sesión se reinicia, podemos perderlas.
#
# Guardar estos artefactos permite:
#
# - reproducir el trabajo;
# - conservar evidencia de las exclusiones;
# - recuperar exactamente la entrada que utilizaremos en G3.
#
#
# IMPORTANTE:
#
# - NO modificamos yelp_dataset.xlsx.
# - NO sobrescribimos el dataset original.
# - NO creamos todavía los splits.
# - NO hacemos preprocesamiento lingüístico.
# - NO entrenamos ningún modelo.


# ------------------------------------------------------------
# 1. IMPORTAR LAS HERRAMIENTAS NECESARIAS
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 2. CREAR UNA CARPETA PARA LOS ARTEFACTOS
# ------------------------------------------------------------

# /content/artifacts_yelp será una carpeta separada
# del archivo Excel original.
#
# mkdir() crea la carpeta.
#
# exist_ok=True significa que, si ya existe,
# Python no producirá un error.

ARTIFACT_DIR = Path(
    "/content/artifacts_yelp"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. DEFINIR LOS ARCHIVOS DE SALIDA
# ------------------------------------------------------------

# Utilizamos CSV porque es un formato sencillo,
# portable y fácil de volver a leer.

WORK_PATH = (
    ARTIFACT_DIR
    / "g2_dataset_trabajo_deduplicado.csv"
)

EXCLUSIONS_PATH = (
    ARTIFACT_DIR
    / "g2_registro_exclusiones_duplicados.csv"
)


# ------------------------------------------------------------
# 4. GUARDAR LA COPIA DE TRABAJO
# ------------------------------------------------------------

# index=False evita guardar el índice interno de pandas
# como una columna adicional.
#
# encoding="utf-8" conserva correctamente los caracteres
# del texto.

df_work.to_csv(
    WORK_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. GUARDAR EL REGISTRO DE EXCLUSIONES
# ------------------------------------------------------------

removed_rows_log.to_csv(
    EXCLUSIONS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. COMPROBAR QUE LOS DOS ARCHIVOS EXISTEN
# ------------------------------------------------------------

print("=== ARTEFACTOS GUARDADOS ===")

print(
    "Dataset de trabajo existe:",
    WORK_PATH.exists()
)

print(
    "Registro de exclusiones existe:",
    EXCLUSIONS_PATH.exists()
)

print(
    "\nRuta dataset de trabajo:",
    WORK_PATH
)

print(
    "Ruta registro de exclusiones:",
    EXCLUSIONS_PATH
)


# ------------------------------------------------------------
# 7. VOLVER A LEER EL DATASET GUARDADO
# ------------------------------------------------------------

# No basta con comprobar que el archivo existe.
#
# Lo volvemos a leer para verificar que realmente
# puede recuperarse correctamente.

df_check = pd.read_csv(
    WORK_PATH
)


# ------------------------------------------------------------
# 8. COMPROBAR EL NÚMERO DE FILAS
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Filas actuales en df_work:",
    len(df_work)
)

print(
    "Filas recuperadas desde CSV:",
    len(df_check)
)

print(
    "¿Coinciden?:",
    len(df_work) == len(df_check)
)


# ------------------------------------------------------------
# 9. COMPROBAR DUPLICADOS EN EL ARCHIVO RECUPERADO
# ------------------------------------------------------------

remaining_duplicates_check = int(
    df_check["text"]
    .duplicated(keep=False)
    .sum()
)

print(
    "Textos duplicados exactos recuperados:",
    remaining_duplicates_check
)


# ------------------------------------------------------------
# 10. COMPROBAR LAS ETIQUETAS
# ------------------------------------------------------------

print(
    "¿sentiment existe?:",
    "sentiment" in df_check.columns
)

print(
    "¿sentiment_id existe?:",
    "sentiment_id" in df_check.columns
)


# ------------------------------------------------------------
# 11. CONTROL DE G3
# ------------------------------------------------------------

print("\n=== CONTROL DE G3 ===")

print(
    "Splits creados:",
    "NO"
)

print(
    "business_id repartidos entre splits:",
    "NO"
)

print(
    "Test utilizado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)

=== ARTEFACTOS GUARDADOS ===
Dataset de trabajo existe: True
Registro de exclusiones existe: True

Ruta dataset de trabajo: /content/artifacts_yelp/g2_dataset_trabajo_deduplicado.csv
Ruta registro de exclusiones: /content/artifacts_yelp/g2_registro_exclusiones_duplicados.csv

=== CONTROL DE RECUPERACIÓN ===
Filas actuales en df_work: 9998
Filas recuperadas desde CSV: 9998
¿Coinciden?: True
Textos duplicados exactos recuperados: 0
¿sentiment existe?: True
¿sentiment_id existe?: True

=== CONTROL DE G3 ===
Splits creados: NO
business_id repartidos entre splits: NO
Test utilizado: NO
Modelos entrenados: NINGUNO


In [14]:
# ============================================================
# G3 — PARTICIÓN TRAIN / VALIDATION / TEST
# Paso 1: crear una asignación candidata reproducible
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Vamos a dividir df_work en tres conjuntos:
#
#       TRAIN
#       VALIDATION
#       TEST
#
# utilizando StratifiedGroupKFold.
#
#
# ¿QUÉ SIGNIFICA "STRATIFIED"?
#
# Intentamos mantener una distribución de sentimientos
# razonablemente parecida entre los conjuntos.
#
#
# ¿QUÉ SIGNIFICA "GROUP"?
#
# Utilizamos business_id como grupo.
#
# Todas las reseñas pertenecientes al MISMO negocio deben
# permanecer juntas.
#
# Por tanto, un business_id NO debe aparecer simultáneamente
# en train, validation o test.
#
#
# ¿POR QUÉ ES IMPORTANTE?
#
# Si reseñas del mismo negocio aparecieran en entrenamiento
# y evaluación, podríamos introducir información relacionada
# con ese negocio en distintos conjuntos.
#
# Nuestro protocolo exige evitar esa fuga.
#
#
# ESTRATEGIA DE PARTICIÓN
#
# Utilizaremos 5 folds agrupados y estratificados.
#
# - 1 fold se reservará como TEST.
#
# - Sobre los 4 folds restantes realizaremos una segunda
#   partición agrupada/estratificada para obtener VALIDATION.
#
# Esto busca aproximadamente:
#
#       TRAIN       ~ 60 %
#       VALIDATION  ~ 20 %
#       TEST        ~ 20 %
#
# Debido al agrupamiento por business_id, los porcentajes
# exactos pueden variar ligeramente.
#
#
# SEMILLA
#
# random_state = 42
#
# Esto permite reproducir la asignación.
#
#
# IMPORTANTE:
#
# - SOLO utilizamos sentiment para estratificar.
#
# - SOLO utilizamos business_id para formar los grupos.
#
# - 'stars' NO se utiliza como predictor.
#
# - NO estamos entrenando ningún modelo.
#
# - NO hacemos preprocesamiento lingüístico.
#
# - El TEST todavía NO se evalúa.
#
# En esta celda únicamente construimos una ASIGNACIÓN
# CANDIDATA. En la siguiente celda la auditaremos antes
# de considerarla válida.


# ------------------------------------------------------------
# 1. IMPORTAR StratifiedGroupKFold
# ------------------------------------------------------------

from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. FIJAR LA SEMILLA DEL CASO
# ------------------------------------------------------------

SEED = 42

np.random.seed(SEED)

print("=== CONFIGURACIÓN ===")
print("Semilla:", SEED)
print("Método: StratifiedGroupKFold")
print("Variable de estratificación: sentiment")
print("Variable de agrupación: business_id")


# ------------------------------------------------------------
# 3. CREAR UNA COPIA PARA LA ASIGNACIÓN
# ------------------------------------------------------------

# Partimos de df_work, que ya contiene:
#
# - textos deduplicados;
# - sentiment;
# - sentiment_id.
#
# Creamos otra copia para añadir posteriormente
# una columna llamada split.

df_split = df_work.copy(deep=True)

print("\n=== ENTRADA A G3 ===")
print("Filas disponibles:", len(df_split))
print(
    "Negocios únicos:",
    df_split["business_id"].nunique()
)


# ------------------------------------------------------------
# 4. PREPARAR LAS VARIABLES PARA EL PRIMER SPLIT
# ------------------------------------------------------------

# X:
# StratifiedGroupKFold exige un argumento X.
#
# Aquí utilizamos únicamente text como entrada estructural.
# El contenido de text NO se transforma ni se modela.
#
# y:
# sentiment_id es la clase que queremos estratificar.
#
# groups:
# business_id obliga a mantener juntas las reseñas
# pertenecientes al mismo negocio.

X = df_split[["text"]]

y = df_split["sentiment_id"].astype(int)

groups = df_split["business_id"]


# ------------------------------------------------------------
# 5. CREAR EL PRIMER StratifiedGroupKFold
# ------------------------------------------------------------

# 5 folds permiten reservar aproximadamente 1/5
# de los datos para test.
#
# shuffle=True permite barajar los grupos antes de asignarlos.
#
# random_state=42 garantiza reproducibilidad.

sgkf_test = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)


# ------------------------------------------------------------
# 6. OBTENER UN FOLD PARA TEST
# ------------------------------------------------------------

# split() genera varias posibilidades.
#
# Seleccionamos reproduciblemente el PRIMER fold.
#
# trainval_idx:
# índices que seguirán disponibles para train + validation.
#
# test_idx:
# índices reservados para el candidato a test.

trainval_idx, test_idx = next(
    sgkf_test.split(
        X=X,
        y=y,
        groups=groups
    )
)


# ------------------------------------------------------------
# 7. CREAR EL CONJUNTO TEMPORAL TRAIN + VALIDATION
# ------------------------------------------------------------

trainval_df = (
    df_split
    .iloc[trainval_idx]
    .copy()
)

test_df = (
    df_split
    .iloc[test_idx]
    .copy()
)


# ------------------------------------------------------------
# 8. PREPARAR LA SEGUNDA PARTICIÓN
# ------------------------------------------------------------

# Ahora debemos separar train y validation
# dentro de trainval_df.
#
# Volvemos a utilizar:
#
# sentiment_id -> estratificación
# business_id  -> agrupación

X_trainval = trainval_df[["text"]]

y_trainval = (
    trainval_df["sentiment_id"]
    .astype(int)
)

groups_trainval = (
    trainval_df["business_id"]
)


# ------------------------------------------------------------
# 9. CREAR LA PARTICIÓN TRAIN / VALIDATION
# ------------------------------------------------------------

# Utilizamos 4 folds sobre el bloque train+validation.
#
# Reservar 1 de esos 4 folds para validation representa
# aproximadamente el 25 % de train+validation.
#
# Como train+validation representa aproximadamente el 80 %
# del dataset:
#
#       25 % de 80 % ≈ 20 %
#
# quedando aproximadamente:
#
#       TRAIN      ≈ 60 %
#       VALIDATION ≈ 20 %
#       TEST       ≈ 20 %

sgkf_val = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=SEED
)


train_rel_idx, val_rel_idx = next(
    sgkf_val.split(
        X=X_trainval,
        y=y_trainval,
        groups=groups_trainval
    )
)


# ------------------------------------------------------------
# 10. CREAR LOS TRES DATAFRAMES CANDIDATOS
# ------------------------------------------------------------

train_df = (
    trainval_df
    .iloc[train_rel_idx]
    .copy()
)

val_df = (
    trainval_df
    .iloc[val_rel_idx]
    .copy()
)


# ------------------------------------------------------------
# 11. ASIGNAR EL NOMBRE DEL SPLIT
# ------------------------------------------------------------

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"


# ------------------------------------------------------------
# 12. MOSTRAR ÚNICAMENTE LOS TAMAÑOS CANDIDATOS
# ------------------------------------------------------------

print("\n=== SPLITS CANDIDATOS ===")

print(
    "TRAIN:",
    len(train_df)
)

print(
    "VALIDATION:",
    len(val_df)
)

print(
    "TEST:",
    len(test_df)
)

print(
    "TOTAL:",
    len(train_df) + len(val_df) + len(test_df)
)


# ------------------------------------------------------------
# 13. CALCULAR PORCENTAJES
# ------------------------------------------------------------

total_rows = len(df_split)

print("\n=== PORCENTAJES CANDIDATOS ===")

print(
    "TRAIN:",
    round(len(train_df) / total_rows * 100, 2),
    "%"
)

print(
    "VALIDATION:",
    round(len(val_df) / total_rows * 100, 2),
    "%"
)

print(
    "TEST:",
    round(len(test_df) / total_rows * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 14. CONTROL DEL ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print("Asignación candidata creada: SÍ")

print(
    "Asignación auditada completamente:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)

=== CONFIGURACIÓN ===
Semilla: 42
Método: StratifiedGroupKFold
Variable de estratificación: sentiment
Variable de agrupación: business_id

=== ENTRADA A G3 ===
Filas disponibles: 9998
Negocios únicos: 4173

=== SPLITS CANDIDATOS ===
TRAIN: 5943
VALIDATION: 1956
TEST: 2099
TOTAL: 9998

=== PORCENTAJES CANDIDATOS ===
TRAIN: 59.44 %
VALIDATION: 19.56 %
TEST: 20.99 %

=== ESTADO ===
Asignación candidata creada: SÍ
Asignación auditada completamente: NO
Test evaluado: NO
Modelos entrenados: NINGUNO


In [15]:
# ============================================================
# G3 — PARTICIÓN TRAIN / VALIDATION / TEST
# Paso 2: auditar la asignación candidata
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba que train + validation + test suman
#    exactamente las 9.998 filas disponibles.
#
# 2. Comprueba que NINGÚN business_id aparezca en más
#    de un split.
#
# 3. Comprueba que NINGÚN texto exacto aparezca en más
#    de un split.
#
# 4. Muestra la distribución de sentimientos dentro
#    de train, validation y test.
#
# 5. Comprueba que las tres clases estén presentes
#    en los tres conjuntos.
#
# ¿POR QUÉ ES IMPORTANTE?
#
# Esta es la prueba de ausencia de fuga.
#
# El requisito principal de G3 es:
#
#     ningún business_id puede cruzar splits.
#
# Además, como eliminamos textos duplicados exactos antes
# de crear la partición, tampoco debería existir el mismo
# texto exacto en dos conjuntos diferentes.
#
# IMPORTANTE:
#
# - Esta celda NO cambia la partición.
# - NO entrena modelos.
# - NO usa el test para seleccionar modelos.
# - Solo audita la asignación candidata.


# ------------------------------------------------------------
# 1. COMPROBAR EL TOTAL DE FILAS
# ------------------------------------------------------------

total_split_rows = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)

print("=== CONTROL DEL TOTAL ===")

print(
    "Filas en train:",
    len(train_df)
)

print(
    "Filas en validation:",
    len(val_df)
)

print(
    "Filas en test:",
    len(test_df)
)

print(
    "Total combinado:",
    total_split_rows
)

print(
    "Total esperado:",
    len(df_work)
)

print(
    "¿Coinciden los totales?:",
    total_split_rows == len(df_work)
)


# ------------------------------------------------------------
# 2. OBTENER LOS business_id DE CADA SPLIT
# ------------------------------------------------------------

train_business = set(
    train_df["business_id"]
)

val_business = set(
    val_df["business_id"]
)

test_business = set(
    test_df["business_id"]
)


# ------------------------------------------------------------
# 3. COMPROBAR CRUCES DE business_id
# ------------------------------------------------------------

train_val_overlap = (
    train_business
    .intersection(val_business)
)

train_test_overlap = (
    train_business
    .intersection(test_business)
)

val_test_overlap = (
    val_business
    .intersection(test_business)
)


print("\n=== FUGA POR business_id ===")

print(
    "business_id compartidos train-validation:",
    len(train_val_overlap)
)

print(
    "business_id compartidos train-test:",
    len(train_test_overlap)
)

print(
    "business_id compartidos validation-test:",
    len(val_test_overlap)
)


# ------------------------------------------------------------
# 4. CONTROL GLOBAL DE business_id
# ------------------------------------------------------------

business_leakage = (
    len(train_val_overlap)
    + len(train_test_overlap)
    + len(val_test_overlap)
)

print(
    "Total de cruces detectados:",
    business_leakage
)

print(
    "¿Existe fuga por business_id?:",
    business_leakage > 0
)


# ------------------------------------------------------------
# 5. OBTENER LOS TEXTOS DE CADA SPLIT
# ------------------------------------------------------------

train_texts = set(
    train_df["text"]
)

val_texts = set(
    val_df["text"]
)

test_texts = set(
    test_df["text"]
)


# ------------------------------------------------------------
# 6. COMPROBAR TEXTOS EXACTOS ENTRE SPLITS
# ------------------------------------------------------------

text_train_val_overlap = (
    train_texts
    .intersection(val_texts)
)

text_train_test_overlap = (
    train_texts
    .intersection(test_texts)
)

text_val_test_overlap = (
    val_texts
    .intersection(test_texts)
)


print("\n=== TEXTOS EXACTOS ENTRE SPLITS ===")

print(
    "Textos compartidos train-validation:",
    len(text_train_val_overlap)
)

print(
    "Textos compartidos train-test:",
    len(text_train_test_overlap)
)

print(
    "Textos compartidos validation-test:",
    len(text_val_test_overlap)
)


# ------------------------------------------------------------
# 7. MOSTRAR DISTRIBUCIÓN POR SENTIMIENTO EN CADA SPLIT
# ------------------------------------------------------------

def split_distribution(df, split_name):
    """
    Devuelve cantidad y porcentaje de cada clase
    para un split concreto.
    """

    counts = (
        df["sentiment"]
        .value_counts()
        .reindex(
            ["NEGATIVO", "NEUTRAL", "POSITIVO"]
        )
        .fillna(0)
        .astype(int)
    )

    percentages = (
        counts
        / len(df)
        * 100
    ).round(2)

    result = pd.DataFrame({
        "split": split_name,
        "sentimiento": counts.index,
        "cantidad": counts.values,
        "porcentaje": percentages.values
    })

    return result


train_distribution = split_distribution(
    train_df,
    "train"
)

val_distribution = split_distribution(
    val_df,
    "validation"
)

test_distribution = split_distribution(
    test_df,
    "test"
)


split_class_distribution = pd.concat(
    [
        train_distribution,
        val_distribution,
        test_distribution
    ],
    ignore_index=True
)


print("\n=== DISTRIBUCIÓN DE SENTIMIENTO POR SPLIT ===")

print(
    split_class_distribution
    .to_string(index=False)
)


# ------------------------------------------------------------
# 8. COMPROBAR QUE LAS TRES CLASES EXISTAN EN CADA SPLIT
# ------------------------------------------------------------

def has_all_classes(df):
    """
    Comprueba que estén presentes los códigos:
    0 = NEGATIVO
    1 = NEUTRAL
    2 = POSITIVO
    """

    observed = set(
        df["sentiment_id"]
        .astype(int)
        .unique()
    )

    expected = {0, 1, 2}

    return observed == expected


print("\n=== PRESENCIA DE LAS TRES CLASES ===")

print(
    "Train contiene 0,1,2:",
    has_all_classes(train_df)
)

print(
    "Validation contiene 0,1,2:",
    has_all_classes(val_df)
)

print(
    "Test contiene 0,1,2:",
    has_all_classes(test_df)
)


# ------------------------------------------------------------
# 9. CONTROL FINAL DE LA ASIGNACIÓN CANDIDATA
# ------------------------------------------------------------

split_ok = (
    total_split_rows == len(df_work)
    and business_leakage == 0
    and len(text_train_val_overlap) == 0
    and len(text_train_test_overlap) == 0
    and len(text_val_test_overlap) == 0
    and has_all_classes(train_df)
    and has_all_classes(val_df)
    and has_all_classes(test_df)
)


print("\n=== RESULTADO DE LA AUDITORÍA DEL SPLIT ===")

print(
    "¿Asignación candidata supera los controles básicos?:",
    split_ok
)


# ------------------------------------------------------------
# 10. RECORDATORIO DEL TEST
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST ===")

print(
    "Test creado:",
    "SÍ"
)

print(
    "Test utilizado para seleccionar modelos:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)

=== CONTROL DEL TOTAL ===
Filas en train: 5943
Filas en validation: 1956
Filas en test: 2099
Total combinado: 9998
Total esperado: 9998
¿Coinciden los totales?: True

=== FUGA POR business_id ===
business_id compartidos train-validation: 0
business_id compartidos train-test: 0
business_id compartidos validation-test: 0
Total de cruces detectados: 0
¿Existe fuga por business_id?: False

=== TEXTOS EXACTOS ENTRE SPLITS ===
Textos compartidos train-validation: 0
Textos compartidos train-test: 0
Textos compartidos validation-test: 0

=== DISTRIBUCIÓN DE SENTIMIENTO POR SPLIT ===
     split sentimiento  cantidad  porcentaje
     train    NEGATIVO       993       16.71
     train     NEUTRAL       847       14.25
     train    POSITIVO      4103       69.04
validation    NEGATIVO       330       16.87
validation     NEUTRAL       306       15.64
validation    POSITIVO      1320       67.48
      test    NEGATIVO       352       16.77
      test     NEUTRAL       308       14.67
      test   

In [16]:
# ============================================================
# G3 — PARTICIÓN TRAIN / VALIDATION / TEST
# Paso 3: guardar y sellar la asignación definitiva
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Une train, validation y test en una única tabla
#    de asignación.
#
# 2. Conserva review_id como identificador de cada reseña.
#
# 3. Guarda la columna split:
#
#       train
#       validation
#       test
#
# 4. Guarda también business_id y las etiquetas
#    para poder auditar la partición posteriormente.
#
# 5. Exporta la asignación completa a CSV.
#
# 6. Guarda además tres archivos separados:
#
#       train.csv
#       validation.csv
#       test.csv
#
# 7. Vuelve a leer el archivo de asignación y comprueba
#    que las 9.998 reseñas siguen presentes una sola vez.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Esta asignación debe ser ÚNICA para todo el caso.
#
# Word2Vec + BiLSTM y BERT deberán utilizar exactamente
# los mismos review_id en train, validation y test.
#
# Así evitamos que cada modelo use una partición diferente.
#
#
# ¿QUÉ SIGNIFICA "SELLAR EL TEST"?
#
# A partir de este momento:
#
# - test queda físicamente definido;
# - NO se utilizará para elegir hiperparámetros;
# - NO se utilizará para decidir épocas;
# - NO se utilizará para escoger arquitectura;
# - NO se utilizará para calcular pesos de clase;
#
# El test se evaluará UNA SOLA VEZ al final, en G8,
# cuando las decisiones de modelo ya estén terminadas.
#
#
# IMPORTANTE:
#
# - NO cambiamos ningún split.
# - NO volvemos a ejecutar StratifiedGroupKFold.
# - NO hacemos preprocesamiento.
# - NO entrenamos ningún modelo.
# - NO calculamos métricas sobre test.


# ------------------------------------------------------------
# 1. CREAR UNA TABLA ÚNICA DE ASIGNACIÓN
# ------------------------------------------------------------

# Concatenamos los tres DataFrames que ya fueron auditados.
#
# ignore_index=True crea un índice nuevo y continuo.

split_assignment = pd.concat(
    [
        train_df,
        val_df,
        test_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 2. ORDENAR LA ASIGNACIÓN PARA FACILITAR SU AUDITORÍA
# ------------------------------------------------------------

# Ordenamos primero por split y luego por review_id.
#
# Esto NO cambia qué fila pertenece a cada conjunto.
# Solo facilita leer y comparar el archivo posteriormente.

split_assignment = (
    split_assignment
    .sort_values(
        ["split", "review_id"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. DEFINIR LAS RUTAS DE LOS ARTEFACTOS
# ------------------------------------------------------------

SPLIT_ASSIGNMENT_PATH = (
    ARTIFACT_DIR
    / "g3_split_assignment.csv"
)

TRAIN_PATH = (
    ARTIFACT_DIR
    / "g3_train.csv"
)

VALIDATION_PATH = (
    ARTIFACT_DIR
    / "g3_validation.csv"
)

TEST_PATH = (
    ARTIFACT_DIR
    / "g3_test_SELLADO.csv"
)


# ------------------------------------------------------------
# 4. GUARDAR LA ASIGNACIÓN COMPLETA
# ------------------------------------------------------------

split_assignment.to_csv(
    SPLIT_ASSIGNMENT_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. GUARDAR LOS TRES SPLITS POR SEPARADO
# ------------------------------------------------------------

train_df.to_csv(
    TRAIN_PATH,
    index=False,
    encoding="utf-8"
)

val_df.to_csv(
    VALIDATION_PATH,
    index=False,
    encoding="utf-8"
)

test_df.to_csv(
    TEST_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. COMPROBAR QUE LOS ARCHIVOS EXISTEN
# ------------------------------------------------------------

print("=== ARTEFACTOS DE G3 ===")

print(
    "Asignación completa existe:",
    SPLIT_ASSIGNMENT_PATH.exists()
)

print(
    "Train existe:",
    TRAIN_PATH.exists()
)

print(
    "Validation existe:",
    VALIDATION_PATH.exists()
)

print(
    "Test sellado existe:",
    TEST_PATH.exists()
)


# ------------------------------------------------------------
# 7. VOLVER A LEER LA ASIGNACIÓN GUARDADA
# ------------------------------------------------------------

# Esta comprobación demuestra que el archivo que acabamos
# de escribir puede recuperarse correctamente.

split_check = pd.read_csv(
    SPLIT_ASSIGNMENT_PATH
)


# ------------------------------------------------------------
# 8. COMPROBAR EL TOTAL DE FILAS RECUPERADAS
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Filas guardadas:",
    len(split_assignment)
)

print(
    "Filas recuperadas:",
    len(split_check)
)

print(
    "Total esperado:",
    len(df_work)
)

print(
    "¿Coinciden los totales?:",
    len(split_check) == len(df_work)
)


# ------------------------------------------------------------
# 9. COMPROBAR QUE review_id SIGUE SIENDO ÚNICO
# ------------------------------------------------------------

# Cada review_id debe aparecer una sola vez en la
# asignación completa.

review_duplicates = int(
    split_check["review_id"]
    .duplicated()
    .sum()
)

print(
    "review_id duplicados en asignación:",
    review_duplicates
)


# ------------------------------------------------------------
# 10. COMPROBAR LOS TAMAÑOS GUARDADOS DE CADA SPLIT
# ------------------------------------------------------------

saved_split_counts = (
    split_check["split"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\n=== TAMAÑOS GUARDADOS ===")

print(saved_split_counts)


# ------------------------------------------------------------
# 11. COMPROBAR DE NUEVO LOS business_id
# ------------------------------------------------------------

# Aunque ya lo auditamos, hacemos una última comprobación
# sobre el artefacto QUE QUEDA GUARDADO EN DISCO.
#
# Esto es importante porque este CSV será la referencia
# oficial para las etapas posteriores.

saved_train_business = set(
    split_check.loc[
        split_check["split"] == "train",
        "business_id"
    ]
)

saved_val_business = set(
    split_check.loc[
        split_check["split"] == "validation",
        "business_id"
    ]
)

saved_test_business = set(
    split_check.loc[
        split_check["split"] == "test",
        "business_id"
    ]
)


saved_train_val_overlap = (
    saved_train_business
    & saved_val_business
)

saved_train_test_overlap = (
    saved_train_business
    & saved_test_business
)

saved_val_test_overlap = (
    saved_val_business
    & saved_test_business
)


print("\n=== CONTROL FINAL DE business_id ===")

print(
    "Cruces train-validation:",
    len(saved_train_val_overlap)
)

print(
    "Cruces train-test:",
    len(saved_train_test_overlap)
)

print(
    "Cruces validation-test:",
    len(saved_val_test_overlap)
)


# ------------------------------------------------------------
# 12. COMPROBACIÓN FORMAL FINAL DE G3
# ------------------------------------------------------------

g3_controls_ok = (
    len(split_check) == len(df_work)
    and review_duplicates == 0
    and len(saved_train_val_overlap) == 0
    and len(saved_train_test_overlap) == 0
    and len(saved_val_test_overlap) == 0
)


print("\n=== RESULTADO FINAL DE CONTROLES G3 ===")

print(
    "¿Controles de G3 superados?:",
    g3_controls_ok
)


# ------------------------------------------------------------
# 13. REGISTRAR EL SELLADO DEL TEST
# ------------------------------------------------------------

print("\n=== TEST SELLADO ===")

print(
    "Archivo:",
    TEST_PATH.name
)

print(
    "Puede utilizarse para selección de modelos:",
    "NO"
)

print(
    "Puede utilizarse para hiperparámetros:",
    "NO"
)

print(
    "Puede utilizarse para pesos de clase:",
    "NO"
)

print(
    "Evaluación prevista:",
    "G8 — una sola vez"
)


# ------------------------------------------------------------
# 14. ESTADO DE COMPUERTAS
# ------------------------------------------------------------

print("\n=== ESTADO DE COMPUERTAS ===")

print("G2: APROBADO")
print("G3: pendiente de aprobación final")
print("G4: NO INICIADO")
print("Modelos entrenados: NINGUNO")

=== ARTEFACTOS DE G3 ===
Asignación completa existe: True
Train existe: True
Validation existe: True
Test sellado existe: True

=== CONTROL DE RECUPERACIÓN ===
Filas guardadas: 9998
Filas recuperadas: 9998
Total esperado: 9998
¿Coinciden los totales?: True
review_id duplicados en asignación: 0

=== TAMAÑOS GUARDADOS ===
split
train         5943
validation    1956
test          2099
Name: count, dtype: int64

=== CONTROL FINAL DE business_id ===
Cruces train-validation: 0
Cruces train-test: 0
Cruces validation-test: 0

=== RESULTADO FINAL DE CONTROLES G3 ===
¿Controles de G3 superados?: True

=== TEST SELLADO ===
Archivo: g3_test_SELLADO.csv
Puede utilizarse para selección de modelos: NO
Puede utilizarse para hiperparámetros: NO
Puede utilizarse para pesos de clase: NO
Evaluación prevista: G8 — una sola vez

=== ESTADO DE COMPUERTAS ===
G2: APROBADO
G3: pendiente de aprobación final
G4: NO INICIADO
Modelos entrenados: NINGUNO


In [17]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 1: preparar las herramientas de la rama secuencial
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Esta celda prepara las herramientas que utilizaremos
# posteriormente para limpiar el texto destinado a:
#
#       Word2Vec + BiLSTM
#
# Concretamente:
#
# 1. Importa librerías estándar de Python para:
#       - decodificar HTML;
#       - trabajar con expresiones regulares;
#       - manipular texto.
#
# 2. Prepara NLTK.
#
# 3. Descarga los recursos lingüísticos necesarios para:
#       - stopwords en inglés;
#       - lematización con WordNet.
#
# 4. Construye una lista de stopwords.
#
# 5. ELIMINA de esa lista los negadores que debemos conservar:
#
#       no
#       not
#       nor
#       never
#       n't
#
#
# ¿POR QUÉ CONSERVAMOS LAS NEGACIONES?
#
# Porque pueden cambiar completamente el sentimiento.
#
# Ejemplo:
#
#       "good"     -> puede expresar algo positivo
#       "not good" -> expresa algo negativo
#
# Si elimináramos "not", perderíamos una señal importante
# para la clasificación de sentimiento.
#
#
# IMPORTANTE:
#
# - Esta celda NO modifica df_work.
# - NO modifica train_df.
# - NO modifica validation.
# - NO modifica test.
# - NO crea todavía texto procesado.
# - NO entrena Word2Vec.
# - NO entrena la BiLSTM.
# - NO carga ni entrena BERT.
#
# En este paso únicamente PREPARAMOS las herramientas.


# ------------------------------------------------------------
# 1. IMPORTAR LIBRERÍAS ESTÁNDAR
# ------------------------------------------------------------

# html:
# permite convertir entidades HTML.
#
# Por ejemplo:
#     &amp;  -> &
#     &quot; -> "
#
# re:
# permite utilizar expresiones regulares.
#
# Las utilizaremos posteriormente para detectar URLs,
# etiquetas HTML, caracteres no deseados, etc.

import html
import re


# ------------------------------------------------------------
# 2. IMPORTAR NLTK
# ------------------------------------------------------------

# NLTK = Natural Language Toolkit.
#
# Es una biblioteca muy utilizada para procesamiento
# de lenguaje natural.

import nltk


# ------------------------------------------------------------
# 3. DESCARGAR RECURSOS LINGÜÍSTICOS
# ------------------------------------------------------------

# stopwords:
# contiene palabras frecuentes del inglés como:
#
#     the, a, an, is...
#
# wordnet:
# recurso utilizado por el lematizador.
#
# omw-1.4:
# recurso complementario de WordNet.
#
# Estas descargas NO modifican el dataset.

nltk.download(
    "stopwords",
    quiet=True
)

nltk.download(
    "wordnet",
    quiet=True
)

nltk.download(
    "omw-1.4",
    quiet=True
)


# ------------------------------------------------------------
# 4. IMPORTAR STOPWORDS Y LEMATIZADOR
# ------------------------------------------------------------

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# ------------------------------------------------------------
# 5. OBTENER LAS STOPWORDS DEL INGLÉS
# ------------------------------------------------------------

# set() crea un conjunto.
#
# Los conjuntos permiten comprobar rápidamente si una
# palabra pertenece o no a nuestra lista.

english_stopwords = set(
    stopwords.words("english")
)


# ------------------------------------------------------------
# 6. DEFINIR LOS NEGADORES QUE QUEREMOS CONSERVAR
# ------------------------------------------------------------

# Estas palabras NO deben desaparecer durante
# el preprocesamiento secuencial.

NEGATORS_TO_KEEP = {
    "no",
    "not",
    "nor",
    "never",
    "n't"
}


# ------------------------------------------------------------
# 7. RETIRAR LOS NEGADORES DE LAS STOPWORDS
# ------------------------------------------------------------

# difference() significa:
#
# "toma las stopwords, pero quita de ese conjunto
# las palabras que aparecen en NEGATORS_TO_KEEP".
#
# De esta manera, cuando posteriormente eliminemos
# stopwords, los negadores seguirán en el texto.

sequential_stopwords = (
    english_stopwords
    .difference(NEGATORS_TO_KEEP)
)


# ------------------------------------------------------------
# 8. CREAR EL LEMATIZADOR
# ------------------------------------------------------------

# El lematizador intentará reducir posteriormente
# palabras a una forma léxica más básica.
#
# Ejemplo conceptual:
#
#     cars -> car
#
# IMPORTANTE:
# aquí solamente creamos la herramienta.
# Todavía NO estamos lematizando el dataset.

lemmatizer = WordNetLemmatizer()


# ------------------------------------------------------------
# 9. VALIDAR QUE LOS NEGADORES IMPORTANTES SE CONSERVAN
# ------------------------------------------------------------

print("=== CONTROL DE NEGACIONES ===")

for negator in sorted(NEGATORS_TO_KEEP):

    print(
        negator,
        "-> ¿se eliminaría como stopword?:",
        negator in sequential_stopwords
    )


# ------------------------------------------------------------
# 10. MOSTRAR INFORMACIÓN DE CONTROL
# ------------------------------------------------------------

print("\n=== CONFIGURACIÓN RAMA SECUENCIAL ===")

print(
    "Stopwords originales:",
    len(english_stopwords)
)

print(
    "Stopwords después de proteger negadores:",
    len(sequential_stopwords)
)

print(
    "Lematizador:",
    type(lemmatizer).__name__
)


# ------------------------------------------------------------
# 11. RECORDAR QUÉ MODELO USARÁ ESTA RAMA
# ------------------------------------------------------------

print("\n=== DESTINO DE ESTA RAMA ===")

print(
    "Modelo previsto: Word2Vec + BiLSTM"
)

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)


# ------------------------------------------------------------
# 12. CONTROL DE BERT
# ------------------------------------------------------------

# BERT NO utilizará estas stopwords ni este lematizador.
#
# Su rama conservará mucho más del texto natural.

print("\n=== CONTROL DE BERT ===")

print(
    "BERT utilizará esta lista de stopwords:",
    "NO"
)

print(
    "BERT utilizará este lematizador:",
    "NO"
)

print(
    "BERT eliminará puntuación:",
    "NO"
)


# ------------------------------------------------------------
# 13. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST ===")

print(
    "Test utilizado en esta celda:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)

=== CONTROL DE NEGACIONES ===
n't -> ¿se eliminaría como stopword?: False
never -> ¿se eliminaría como stopword?: False
no -> ¿se eliminaría como stopword?: False
nor -> ¿se eliminaría como stopword?: False
not -> ¿se eliminaría como stopword?: False

=== CONFIGURACIÓN RAMA SECUENCIAL ===
Stopwords originales: 198
Stopwords después de proteger negadores: 195
Lematizador: WordNetLemmatizer

=== DESTINO DE ESTA RAMA ===
Modelo previsto: Word2Vec + BiLSTM
Word2Vec entrenado: NO
BiLSTM entrenada: NO

=== CONTROL DE BERT ===
BERT utilizará esta lista de stopwords: NO
BERT utilizará este lematizador: NO
BERT eliminará puntuación: NO

=== CONTROL DEL TEST ===
Test utilizado en esta celda: NO
Test evaluado: NO
Modelos entrenados: NINGUNO


In [18]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 2: definir y probar la limpieza de la rama secuencial
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Vamos a crear la función que utilizará posteriormente
# la rama:
#
#       Word2Vec + BiLSTM
#
# La función realizará, en este orden:
#
# 1. Decodificar entidades HTML.
# 2. Convertir el texto a minúsculas.
# 3. Eliminar URLs.
# 4. Eliminar etiquetas HTML.
# 5. Eliminar caracteres de control.
# 6. Normalizar algunas contracciones negativas.
# 7. Eliminar puntuación y caracteres especiales.
# 8. Tokenizar mediante separación por espacios.
# 9. Eliminar stopwords, EXCEPTO negadores.
# 10. Lematizar los tokens.
#
# ¿POR QUÉ PROBAMOS PRIMERO CON FRASES PEQUEÑAS?
#
# Porque antes de procesar miles de reseñas debemos comprobar
# que la función hace exactamente lo que esperamos.
#
# En particular, queremos verificar que las negaciones
# sobreviven.
#
# IMPORTANTE:
#
# - Esta celda NO procesa todavía train_df completo.
# - NO procesa validation.
# - NO procesa test.
# - NO entrena Word2Vec.
# - NO entrena BiLSTM.
# - NO toca BERT.
# - Solo define la función y la prueba con ejemplos controlados.


# ------------------------------------------------------------
# 1. DEFINIR LA FUNCIÓN DE LIMPIEZA SECUENCIAL
# ------------------------------------------------------------

def clean_text_sequential(text):
    """
    Preprocesamiento para la rama Word2Vec + BiLSTM.

    Entrada:
        texto original de una reseña

    Salida:
        texto limpio y normalizado como cadena
    """

    # --------------------------------------------------------
    # 1.1. ASEGURAR QUE TRABAJAMOS CON TEXTO
    # --------------------------------------------------------

    # Convertimos cualquier entrada a str por robustez.

    text = str(text)


    # --------------------------------------------------------
    # 1.2. DECODIFICAR ENTIDADES HTML
    # --------------------------------------------------------

    # Ejemplo:
    #   &amp;  -> &
    #   &quot; -> "

    text = html.unescape(text)


    # --------------------------------------------------------
    # 1.3. CONVERTIR A MINÚSCULAS
    # --------------------------------------------------------

    text = text.lower()


    # --------------------------------------------------------
    # 1.4. NORMALIZAR CONTRACCIONES NEGATIVAS
    # --------------------------------------------------------

    # Queremos conservar la información de negación.
    #
    # Ejemplos:
    #   don't   -> do not
    #   didn't  -> did not
    #   isn't   -> is not
    #
    # Esto ayuda a que "not" sobreviva claramente
    # al resto del preprocesamiento.

    text = re.sub(
        r"n['’]t\b",
        " not",
        text
    )


    # --------------------------------------------------------
    # 1.5. ELIMINAR URLs
    # --------------------------------------------------------

    # Detectamos enlaces que empiezan por:
    #
    # http://
    # https://
    # www.

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )


    # --------------------------------------------------------
    # 1.6. ELIMINAR ETIQUETAS HTML
    # --------------------------------------------------------

    # Ejemplo:
    #
    # <br>
    # <p>
    # </div>

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )


    # --------------------------------------------------------
    # 1.7. ELIMINAR CARACTERES DE CONTROL
    # --------------------------------------------------------

    # Incluye saltos y caracteres no imprimibles de control.
    # Los reemplazamos por espacios.

    text = re.sub(
        r"[\x00-\x1F\x7F]",
        " ",
        text
    )


    # --------------------------------------------------------
    # 1.8. ELIMINAR PUNTUACIÓN Y CARACTERES ESPECIALES
    # --------------------------------------------------------

    # Conservamos:
    # - letras
    # - números
    # - espacios
    #
    # Sustituimos lo demás por espacios.

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )


    # --------------------------------------------------------
    # 1.9. NORMALIZAR ESPACIOS
    # --------------------------------------------------------

    # Reemplazamos múltiples espacios por uno solo.

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    # --------------------------------------------------------
    # 1.10. TOKENIZAR
    # --------------------------------------------------------

    # Aquí usamos una tokenización sencilla por espacios.
    #
    # Cada palabra queda como un token.

    tokens = text.split()


    # --------------------------------------------------------
    # 1.11. ELIMINAR STOPWORDS, CONSERVANDO NEGADORES
    # --------------------------------------------------------

    # sequential_stopwords ya fue creada en el Paso 1.
    #
    # Como quitamos de esa lista:
    #
    # no, not, nor, never, n't
    #
    # esas palabras se conservan.

    tokens = [
        token
        for token in tokens
        if token not in sequential_stopwords
    ]


    # --------------------------------------------------------
    # 1.12. LEMATIZAR
    # --------------------------------------------------------

    # Aplicamos WordNetLemmatizer a cada token.

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]


    # --------------------------------------------------------
    # 1.13. RECONSTRUIR EL TEXTO
    # --------------------------------------------------------

    # Unimos los tokens procesados con espacios.

    cleaned_text = " ".join(tokens)

    return cleaned_text


# ------------------------------------------------------------
# 2. CREAR FRASES DE PRUEBA
# ------------------------------------------------------------

# No usamos todavía reseñas reales del dataset.
#
# Primero comprobamos el comportamiento con ejemplos
# cuya interpretación conocemos.

test_sentences = [
    "I do not like this restaurant.",
    "I don't like this restaurant.",
    "This is never good.",
    "The food was AMAZING!!!",
    "Visit https://example.com now",
    "<p>The service was not bad.</p>",
    "Cars were running quickly."
]


# ------------------------------------------------------------
# 3. APLICAR LA FUNCIÓN A LOS EJEMPLOS
# ------------------------------------------------------------

print("=== PRUEBAS DE LA FUNCIÓN SECUENCIAL ===")

for sentence in test_sentences:

    cleaned = clean_text_sequential(
        sentence
    )

    print("\nORIGINAL:")
    print(sentence)

    print("PROCESADO:")
    print(cleaned)


# ------------------------------------------------------------
# 4. PRUEBAS AUTOMÁTICAS DE NEGACIÓN
# ------------------------------------------------------------

# Estas comprobaciones son especialmente importantes.
#
# Queremos confirmar que "not" y "never" continúan
# presentes después del procesamiento.

negation_test_1 = clean_text_sequential(
    "I do not like this restaurant."
)

negation_test_2 = clean_text_sequential(
    "I don't like this restaurant."
)

negation_test_3 = clean_text_sequential(
    "This is never good."
)


print("\n=== CONTROL AUTOMÁTICO DE NEGACIONES ===")

print(
    "'not' se conserva en 'do not':",
    "not" in negation_test_1.split()
)

print(
    "'not' se conserva en \"don't\":",
    "not" in negation_test_2.split()
)

print(
    "'never' se conserva:",
    "never" in negation_test_3.split()
)


# ------------------------------------------------------------
# 5. CONTROL DE URLs Y HTML
# ------------------------------------------------------------

url_test = clean_text_sequential(
    "Good food https://example.com"
)

html_test = clean_text_sequential(
    "<p>Very good service</p>"
)

print("\n=== CONTROL DE RUIDO ===")

print(
    "URL eliminada:",
    "http" not in url_test
)

print(
    "Etiqueta HTML eliminada:",
    "<p>" not in html_test
)


# ------------------------------------------------------------
# 6. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print(
    "Función secuencial definida:",
    "SÍ"
)

print(
    "Dataset completo procesado:",
    "NO"
)

print(
    "Validation procesada:",
    "NO"
)

print(
    "Test procesado:",
    "NO"
)

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "BERT entrenado:",
    "NO"
)

=== PRUEBAS DE LA FUNCIÓN SECUENCIAL ===

ORIGINAL:
I do not like this restaurant.
PROCESADO:
not like restaurant

ORIGINAL:
I don't like this restaurant.
PROCESADO:
not like restaurant

ORIGINAL:
This is never good.
PROCESADO:
never good

ORIGINAL:
The food was AMAZING!!!
PROCESADO:
food amazing

ORIGINAL:
Visit https://example.com now
PROCESADO:
visit

ORIGINAL:
<p>The service was not bad.</p>
PROCESADO:
service not bad

ORIGINAL:
Cars were running quickly.
PROCESADO:
car running quickly

=== CONTROL AUTOMÁTICO DE NEGACIONES ===
'not' se conserva en 'do not': True
'not' se conserva en "don't": True
'never' se conserva: True

=== CONTROL DE RUIDO ===
URL eliminada: True
Etiqueta HTML eliminada: True

=== ESTADO ===
Función secuencial definida: SÍ
Dataset completo procesado: NO
Validation procesada: NO
Test procesado: NO
Word2Vec entrenado: NO
BiLSTM entrenada: NO
BERT entrenado: NO


In [19]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 3: definir y probar la limpieza ligera para BERT
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Vamos a crear la función de limpieza que utilizará
# posteriormente la rama:
#
#       BERT
#
# A diferencia de la rama secuencial, BERT necesita conservar
# mucho más del texto natural.
#
# Esta función hará SOLO:
#
# 1. Convertir la entrada a texto.
# 2. Decodificar entidades HTML.
# 3. Eliminar caracteres de control.
# 4. Normalizar espacios.
#
#
# ¿QUÉ NO HAREMOS PARA BERT?
#
# - NO convertimos manualmente a minúsculas aquí.
#   bert-base-uncased ya utiliza un tokenizer uncased.
#
# - NO eliminamos stopwords.
#
# - NO eliminamos puntuación.
#
# - NO eliminamos negaciones.
#
# - NO aplicamos stemming.
#
# - NO aplicamos lematización.
#
# - NO utilizamos el texto procesado de Word2Vec + BiLSTM.
#
#
# ¿POR QUÉ?
#
# BERT fue preentrenado sobre texto natural y utiliza
# información contextual.
#
# Si elimináramos stopwords, puntuación o negaciones
# podríamos destruir señales importantes del contexto.
#
#
# IMPORTANTE:
#
# - Esta celda NO procesa todavía todo el dataset.
# - NO tokeniza con BERT todavía.
# - NO descarga bert-base-uncased.
# - NO entrena BERT.
# - NO utiliza test para tomar decisiones.
#
# Primero probamos la función con ejemplos controlados.


# ------------------------------------------------------------
# 1. DEFINIR LA FUNCIÓN DE LIMPIEZA PARA BERT
# ------------------------------------------------------------

def clean_text_bert(text):
    """
    Limpieza ligera para la rama BERT.

    Entrada:
        texto original

    Salida:
        texto natural con limpieza mínima
    """

    # --------------------------------------------------------
    # 1.1. ASEGURAR QUE LA ENTRADA SEA TEXTO
    # --------------------------------------------------------

    text = str(text)


    # --------------------------------------------------------
    # 1.2. DECODIFICAR ENTIDADES HTML
    # --------------------------------------------------------

    # Ejemplo:
    # &amp; -> &
    # &quot; -> "

    text = html.unescape(text)


    # --------------------------------------------------------
    # 1.3. ELIMINAR CARACTERES DE CONTROL
    # --------------------------------------------------------

    # Sustituimos caracteres de control por espacios.
    #
    # NO eliminamos puntuación normal.

    text = re.sub(
        r"[\x00-\x1F\x7F]",
        " ",
        text
    )


    # --------------------------------------------------------
    # 1.4. NORMALIZAR ESPACIOS
    # --------------------------------------------------------

    # Reemplazamos secuencias de múltiples espacios
    # por un único espacio.

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    # --------------------------------------------------------
    # 1.5. DEVOLVER EL TEXTO
    # --------------------------------------------------------

    return text


# ------------------------------------------------------------
# 2. CREAR EJEMPLOS DE PRUEBA
# ------------------------------------------------------------

bert_test_sentences = [
    "I do not like this restaurant.",
    "I don't like this restaurant!",
    "This is never good.",
    "The food was AMAZING!!!",
    "<p>The service was not bad.</p>",
    "Good &amp; affordable.",
    "This   has     many     spaces.",
    "First line.\nSecond line."
]


# ------------------------------------------------------------
# 3. APLICAR LA FUNCIÓN A LOS EJEMPLOS
# ------------------------------------------------------------

print("=== PRUEBAS DE LA FUNCIÓN BERT ===")

for sentence in bert_test_sentences:

    cleaned = clean_text_bert(
        sentence
    )

    print("\nORIGINAL:")
    print(sentence)

    print("PROCESADO:")
    print(cleaned)


# ------------------------------------------------------------
# 4. CONTROL DE NEGACIONES
# ------------------------------------------------------------

# Queremos comprobar que BERT conserva las negaciones
# exactamente dentro del texto natural.

bert_negation_1 = clean_text_bert(
    "I do not like this restaurant."
)

bert_negation_2 = clean_text_bert(
    "I don't like this restaurant!"
)

bert_negation_3 = clean_text_bert(
    "This is never good."
)


print("\n=== CONTROL DE NEGACIONES EN BERT ===")

print(
    "'not' se conserva:",
    "not" in bert_negation_1
)

print(
    "\"don't\" se conserva:",
    "don't" in bert_negation_2
)

print(
    "'never' se conserva:",
    "never" in bert_negation_3
)


# ------------------------------------------------------------
# 5. CONTROL DE PUNTUACIÓN
# ------------------------------------------------------------

bert_punctuation_test = clean_text_bert(
    "Amazing!!! Really?"
)

print("\n=== CONTROL DE PUNTUACIÓN ===")

print(
    "Texto procesado:",
    bert_punctuation_test
)

print(
    "¿Se conserva !?:",
    "!" in bert_punctuation_test
)

print(
    "¿Se conserva ?:",
    "?" in bert_punctuation_test
)


# ------------------------------------------------------------
# 6. CONTROL DE HTML DECODIFICADO
# ------------------------------------------------------------

bert_html_entity_test = clean_text_bert(
    "Good &amp; affordable."
)

print("\n=== CONTROL DE ENTIDADES HTML ===")

print(
    "Texto procesado:",
    bert_html_entity_test
)

print(
    "¿&amp; fue decodificado?:",
    "&amp;" not in bert_html_entity_test
)


# ------------------------------------------------------------
# 7. CONTROL DE ESPACIOS
# ------------------------------------------------------------

bert_space_test = clean_text_bert(
    "This   has     many     spaces."
)

print("\n=== CONTROL DE ESPACIOS ===")

print(
    "Texto procesado:",
    bert_space_test
)

print(
    "¿Quedan espacios múltiples?:",
    "  " in bert_space_test
)


# ------------------------------------------------------------
# 8. COMPARAR LAS DOS RAMAS SOBRE UNA MISMA FRASE
# ------------------------------------------------------------

comparison_sentence = (
    "I don't like this AMAZING restaurant!!!"
)

sequential_version = clean_text_sequential(
    comparison_sentence
)

bert_version = clean_text_bert(
    comparison_sentence
)


print("\n=== COMPARACIÓN DE LAS DOS RAMAS ===")

print(
    "ORIGINAL:"
)

print(
    comparison_sentence
)

print(
    "\nRAMA Word2Vec + BiLSTM:"
)

print(
    sequential_version
)

print(
    "\nRAMA BERT:"
)

print(
    bert_version
)


# ------------------------------------------------------------
# 9. RECORDATORIO METODOLÓGICO
# ------------------------------------------------------------

print("\n=== DIFERENCIA ENTRE RAMAS ===")

print(
    "Word2Vec + BiLSTM:"
)

print(
    "limpieza fuerte + stopwords + lematización"
)

print()

print(
    "BERT:"
)

print(
    "limpieza mínima, conservando texto natural"
)


# ------------------------------------------------------------
# 10. CONTROL DEL ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print(
    "Función secuencial definida:",
    "SÍ"
)

print(
    "Función BERT definida:",
    "SÍ"
)

print(
    "Dataset completo procesado:",
    "NO"
)

print(
    "Tokenizer BERT utilizado:",
    "NO"
)

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "BERT entrenado:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)

=== PRUEBAS DE LA FUNCIÓN BERT ===

ORIGINAL:
I do not like this restaurant.
PROCESADO:
I do not like this restaurant.

ORIGINAL:
I don't like this restaurant!
PROCESADO:
I don't like this restaurant!

ORIGINAL:
This is never good.
PROCESADO:
This is never good.

ORIGINAL:
The food was AMAZING!!!
PROCESADO:
The food was AMAZING!!!

ORIGINAL:
<p>The service was not bad.</p>
PROCESADO:
<p>The service was not bad.</p>

ORIGINAL:
Good &amp; affordable.
PROCESADO:
Good & affordable.

ORIGINAL:
This   has     many     spaces.
PROCESADO:
This has many spaces.

ORIGINAL:
First line.
Second line.
PROCESADO:
First line. Second line.

=== CONTROL DE NEGACIONES EN BERT ===
'not' se conserva: True
"don't" se conserva: True
'never' se conserva: True

=== CONTROL DE PUNTUACIÓN ===
Texto procesado: Amazing!!! Really?
¿Se conserva !?: True
¿Se conserva ?: True

=== CONTROL DE ENTIDADES HTML ===
Texto procesado: Good & affordable.
¿&amp; fue decodificado?: True

=== CONTROL DE ESPACIOS ===
Texto procesa

In [20]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 4: probar ambas ramas sobre reseñas reales de TRAIN
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Hasta ahora probamos las funciones con frases creadas
# manualmente.
#
# Ahora vamos a comprobar cómo funcionan con reseñas REALES
# pertenecientes exclusivamente a TRAIN.
#
# Haremos lo siguiente:
#
# 1. Seleccionar una pequeña muestra de reseñas de train_df.
#
# 2. Tomar únicamente la columna 'text' como contenido
#    que será transformado.
#
# 3. Aplicar:
#
#       clean_text_sequential()
#       -> rama Word2Vec + BiLSTM
#
#       clean_text_bert()
#       -> rama BERT
#
# 4. Comparar visualmente:
#
#       texto original
#       texto secuencial
#       texto BERT
#
# 5. Comprobar si alguna salida queda vacía.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Queremos detectar problemas de limpieza ANTES de procesar
# miles de reseñas.
#
# También dejamos evidencia de que las dos arquitecturas
# reciben versiones diferentes del mismo campo 'text'.
#
#
# REGLA CRÍTICA DEL CASO:
#
# SOLO 'text' puede alimentar los modelos.
#
# Las siguientes columnas NO pueden ser predictores:
#
# - stars
# - review_id
# - business_id
# - user_id
# - date
# - cool
# - useful
# - funny
#
# sentiment y sentiment_id son únicamente las etiquetas
# objetivo, NO predictores.
#
#
# IMPORTANTE:
#
# - Usamos únicamente TRAIN.
# - NO utilizamos validation en esta prueba.
# - NO utilizamos test.
# - NO modificamos train_df.
# - NO entrenamos Word2Vec.
# - NO entrenamos BiLSTM.
# - NO descargamos ni entrenamos BERT.


# ------------------------------------------------------------
# 1. CREAR UNA PEQUEÑA MUESTRA DE TRAIN
# ------------------------------------------------------------

# sample() selecciona algunas reseñas al azar.
#
# random_state=42 hace que siempre obtengamos
# exactamente la misma muestra si repetimos la celda.

train_sample = (
    train_df
    .sample(
        n=8,
        random_state=42
    )
    .copy()
)


# ------------------------------------------------------------
# 2. CREAR LAS DOS VERSIONES DEL CAMPO text
# ------------------------------------------------------------

# La única columna que transformamos es 'text'.
#
# Creamos dos columnas temporales SOLO para esta muestra.

train_sample["text_sequential"] = (
    train_sample["text"]
    .apply(clean_text_sequential)
)

train_sample["text_bert"] = (
    train_sample["text"]
    .apply(clean_text_bert)
)


# ------------------------------------------------------------
# 3. MOSTRAR LAS RESEÑAS UNA POR UNA
# ------------------------------------------------------------

# Lo mostramos individualmente para que sea más fácil
# entender la diferencia entre las dos ramas.

print("=== MUESTRA REAL DE TRAIN ===")

for _, row in train_sample.iterrows():

    print("\n" + "=" * 70)

    print("REVIEW_ID:")
    print(row["review_id"])

    print("\nSENTIMIENTO OBJETIVO:")
    print(
        row["sentiment"],
        "(" + str(row["sentiment_id"]) + ")"
    )

    print("\nTEXTO ORIGINAL:")
    print(row["text"])

    print("\nRAMA Word2Vec + BiLSTM:")
    print(row["text_sequential"])

    print("\nRAMA BERT:")
    print(row["text_bert"])


# ------------------------------------------------------------
# 4. COMPROBAR POSIBLES SALIDAS VACÍAS
# ------------------------------------------------------------

# La limpieza fuerte de la rama secuencial podría,
# en teoría, dejar algún texto vacío si una reseña
# contuviera solamente stopwords o símbolos.
#
# En esta pequeña muestra comprobamos si ocurre.

empty_sequential_sample = int(
    (
        train_sample["text_sequential"]
        .str.strip()
        .eq("")
    ).sum()
)

empty_bert_sample = int(
    (
        train_sample["text_bert"]
        .str.strip()
        .eq("")
    ).sum()
)


print("\n=== CONTROL DE TEXTOS VACÍOS EN LA MUESTRA ===")

print(
    "Textos secuenciales vacíos:",
    empty_sequential_sample
)

print(
    "Textos BERT vacíos:",
    empty_bert_sample
)


# ------------------------------------------------------------
# 5. COMPROBAR QUE train_df NO FUE MODIFICADO
# ------------------------------------------------------------

# Las nuevas columnas deben existir solamente en
# train_sample, NO en train_df.

print("\n=== CONTROL DE INTEGRIDAD DE TRAIN ===")

print(
    "¿text_sequential existe en train_df?:",
    "text_sequential" in train_df.columns
)

print(
    "¿text_bert existe en train_df?:",
    "text_bert" in train_df.columns
)

print(
    "¿text_sequential existe en train_sample?:",
    "text_sequential" in train_sample.columns
)

print(
    "¿text_bert existe en train_sample?:",
    "text_bert" in train_sample.columns
)


# ------------------------------------------------------------
# 6. DOCUMENTAR QUÉ CAMPOS PUEDEN SER PREDICTORES
# ------------------------------------------------------------

print("\n=== CONTROL DE PREDICTORES ===")

print(
    "Campo permitido como entrada:",
    "text"
)

print(
    "Etiquetas objetivo:",
    "sentiment / sentiment_id"
)

print(
    "stars utilizado como predictor:",
    "NO"
)

print(
    "business_id utilizado como predictor:",
    "NO"
)

print(
    "user_id utilizado como predictor:",
    "NO"
)

print(
    "date utilizado como predictor:",
    "NO"
)

print(
    "cool/useful/funny utilizados como predictores:",
    "NO"
)


# ------------------------------------------------------------
# 7. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST ===")

print(
    "Test utilizado en esta celda:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)


# ------------------------------------------------------------
# 8. ESTADO DE G4
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print(
    "Rama secuencial probada con datos reales de train:",
    "SÍ"
)

print(
    "Rama BERT probada con datos reales de train:",
    "SÍ"
)

print(
    "Train completo procesado:",
    "NO"
)

print(
    "Validation completa procesada:",
    "NO"
)

print(
    "Test procesado:",
    "NO"
)

=== MUESTRA REAL DE TRAIN ===

REVIEW_ID:
fMUn3H7Y-hd14NaamqPByA

SENTIMIENTO OBJETIVO:
NEGATIVO (0)

TEXTO ORIGINAL:
They play two kinds of music here: country and western.  The place is about the size of my one bedroom apartment.  It's a dive for sure.  It's cheap though.  But cash only.  The food is actually not too bad for bar food.  Overall this place doesn't impress much, but considering you don't have a lot of choices for Laveen, you can't really compalin either.

RAMA Word2Vec + BiLSTM:
play two kind music country western place size one bedroom apartment dive sure cheap though cash food actually not bad bar food overall place not impress much considering not lot choice laveen ca not really compalin either

RAMA BERT:
They play two kinds of music here: country and western. The place is about the size of my one bedroom apartment. It's a dive for sure. It's cheap though. But cash only. The food is actually not too bad for bar food. Overall this place doesn't impress much, but cons

In [22]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 5: procesar TRAIN y VALIDATION con las dos ramas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea copias de TRAIN y VALIDATION para trabajar sin
#    modificar los DataFrames originales de G3.
#
# 2. Genera dos versiones del texto:
#
#       text_sequential
#       -> para Word2Vec + BiLSTM
#
#       text_bert
#       -> para BERT
#
# 3. Comprueba que no se pierdan filas.
#
# 4. Comprueba que no aparezcan textos vacíos ni nulos.
#
# 5. Verifica que review_id y etiquetas sigan intactos.
#
# 6. Comprueba que TEST siga sellado y sin procesar.
#
# IMPORTANTE:
#
# - SOLO transformamos la columna text.
# - NO usamos stars, IDs, fecha, cool, useful o funny
#   como predictores.
# - NO procesamos TEST todavía.
# - NO entrenamos Word2Vec.
# - NO entrenamos BiLSTM.
# - NO entrenamos BERT.


# ------------------------------------------------------------
# 1. CREAR COPIAS DE TRAIN Y VALIDATION
# ------------------------------------------------------------

# Trabajamos sobre copias para conservar intactos
# los DataFrames originales de G3.

train_g4 = train_df.copy(deep=True)
val_g4 = val_df.copy(deep=True)


# ------------------------------------------------------------
# 2. PROCESAR TRAIN — RAMA SECUENCIAL
# ------------------------------------------------------------

train_g4["text_sequential"] = (
    train_g4["text"]
    .apply(clean_text_sequential)
)


# ------------------------------------------------------------
# 3. PROCESAR TRAIN — RAMA BERT
# ------------------------------------------------------------

train_g4["text_bert"] = (
    train_g4["text"]
    .apply(clean_text_bert)
)


# ------------------------------------------------------------
# 4. PROCESAR VALIDATION — RAMA SECUENCIAL
# ------------------------------------------------------------

val_g4["text_sequential"] = (
    val_g4["text"]
    .apply(clean_text_sequential)
)


# ------------------------------------------------------------
# 5. PROCESAR VALIDATION — RAMA BERT
# ------------------------------------------------------------

val_g4["text_bert"] = (
    val_g4["text"]
    .apply(clean_text_bert)
)


# ------------------------------------------------------------
# 6. COMPROBAR EL NÚMERO DE FILAS
# ------------------------------------------------------------

print("=== CONTROL DE FILAS ===")

print(
    "TRAIN original:",
    len(train_df)
)

print(
    "TRAIN procesado:",
    len(train_g4)
)

print(
    "¿TRAIN conserva todas las filas?:",
    len(train_df) == len(train_g4)
)

print()

print(
    "VALIDATION original:",
    len(val_df)
)

print(
    "VALIDATION procesado:",
    len(val_g4)
)

print(
    "¿VALIDATION conserva todas las filas?:",
    len(val_df) == len(val_g4)
)


# ------------------------------------------------------------
# 7. BUSCAR TEXTOS VACÍOS EN LA RAMA SECUENCIAL
# ------------------------------------------------------------

train_empty_sequential = int(
    train_g4["text_sequential"]
    .str.strip()
    .eq("")
    .sum()
)

val_empty_sequential = int(
    val_g4["text_sequential"]
    .str.strip()
    .eq("")
    .sum()
)

print("\n=== TEXTOS VACÍOS — RAMA SECUENCIAL ===")

print(
    "TRAIN vacíos:",
    train_empty_sequential
)

print(
    "VALIDATION vacíos:",
    val_empty_sequential
)


# ------------------------------------------------------------
# 8. BUSCAR TEXTOS VACÍOS EN LA RAMA BERT
# ------------------------------------------------------------

train_empty_bert = int(
    train_g4["text_bert"]
    .str.strip()
    .eq("")
    .sum()
)

val_empty_bert = int(
    val_g4["text_bert"]
    .str.strip()
    .eq("")
    .sum()
)

print("\n=== TEXTOS VACÍOS — RAMA BERT ===")

print(
    "TRAIN vacíos:",
    train_empty_bert
)

print(
    "VALIDATION vacíos:",
    val_empty_bert
)


# ------------------------------------------------------------
# 9. COMPROBAR NULOS
# ------------------------------------------------------------

print("\n=== NULOS DESPUÉS DEL PREPROCESAMIENTO ===")

print(
    "TRAIN text_sequential:",
    int(train_g4["text_sequential"].isna().sum())
)

print(
    "TRAIN text_bert:",
    int(train_g4["text_bert"].isna().sum())
)

print(
    "VALIDATION text_sequential:",
    int(val_g4["text_sequential"].isna().sum())
)

print(
    "VALIDATION text_bert:",
    int(val_g4["text_bert"].isna().sum())
)


# ------------------------------------------------------------
# 10. COMPROBAR QUE LOS review_id NO CAMBIARON
# ------------------------------------------------------------

train_ids_unchanged = (
    set(train_g4["review_id"])
    == set(train_df["review_id"])
)

val_ids_unchanged = (
    set(val_g4["review_id"])
    == set(val_df["review_id"])
)

print("\n=== INTEGRIDAD DE LOS SPLITS ===")

print(
    "TRAIN conserva los mismos review_id:",
    train_ids_unchanged
)

print(
    "VALIDATION conserva los mismos review_id:",
    val_ids_unchanged
)


# ------------------------------------------------------------
# 11. COMPROBAR QUE LAS ETIQUETAS NO CAMBIARON
# ------------------------------------------------------------

train_labels_original = (
    train_df
    .sort_values("review_id")["sentiment_id"]
    .astype(int)
    .reset_index(drop=True)
)

train_labels_g4 = (
    train_g4
    .sort_values("review_id")["sentiment_id"]
    .astype(int)
    .reset_index(drop=True)
)

val_labels_original = (
    val_df
    .sort_values("review_id")["sentiment_id"]
    .astype(int)
    .reset_index(drop=True)
)

val_labels_g4 = (
    val_g4
    .sort_values("review_id")["sentiment_id"]
    .astype(int)
    .reset_index(drop=True)
)

print("\n=== INTEGRIDAD DE LAS ETIQUETAS ===")

print(
    "TRAIN conserva las etiquetas:",
    train_labels_original.equals(train_labels_g4)
)

print(
    "VALIDATION conserva las etiquetas:",
    val_labels_original.equals(val_labels_g4)
)


# ------------------------------------------------------------
# 12. COMPARAR LAS DOS RAMAS
# ------------------------------------------------------------

# Algunas reseñas pueden quedar iguales en ambas ramas
# si son muy simples.
#
# Esta cifra es descriptiva, no un error.

same_train = int(
    (
        train_g4["text_sequential"]
        == train_g4["text_bert"]
    ).sum()
)

same_val = int(
    (
        val_g4["text_sequential"]
        == val_g4["text_bert"]
    ).sum()
)

print("\n=== COMPARACIÓN ENTRE RAMAS ===")

print(
    "TRAIN con ambas versiones exactamente iguales:",
    same_train
)

print(
    "VALIDATION con ambas versiones exactamente iguales:",
    same_val
)


# ------------------------------------------------------------
# 13. COMPROBAR QUE TEST SIGUE SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "¿text_sequential existe en test_df?:",
    "text_sequential" in test_df.columns
)

print(
    "¿text_bert existe en test_df?:",
    "text_bert" in test_df.columns
)

print(
    "Test procesado:",
    "NO"
)

print(
    "Test evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 14. CONTROL DE MODELOS
# ------------------------------------------------------------

print("\n=== CONTROL DE MODELOS ===")

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "BERT entrenado:",
    "NO"
)


# ------------------------------------------------------------
# 15. ESTADO DE G4
# ------------------------------------------------------------

print("\n=== ESTADO DE G4 ===")

print(
    "TRAIN procesado en ambas ramas:",
    "SÍ"
)

print(
    "VALIDATION procesada en ambas ramas:",
    "SÍ"
)

print(
    "TEST procesado:",
    "NO"
)

print(
    "G4 finalizado:",
    "TODAVÍA NO"
)

=== CONTROL DE FILAS ===
TRAIN original: 5943
TRAIN procesado: 5943
¿TRAIN conserva todas las filas?: True

VALIDATION original: 1956
VALIDATION procesado: 1956
¿VALIDATION conserva todas las filas?: True

=== TEXTOS VACÍOS — RAMA SECUENCIAL ===
TRAIN vacíos: 0
VALIDATION vacíos: 0

=== TEXTOS VACÍOS — RAMA BERT ===
TRAIN vacíos: 0
VALIDATION vacíos: 0

=== NULOS DESPUÉS DEL PREPROCESAMIENTO ===
TRAIN text_sequential: 0
TRAIN text_bert: 0
VALIDATION text_sequential: 0
VALIDATION text_bert: 0

=== INTEGRIDAD DE LOS SPLITS ===
TRAIN conserva los mismos review_id: True
VALIDATION conserva los mismos review_id: True

=== INTEGRIDAD DE LAS ETIQUETAS ===
TRAIN conserva las etiquetas: True
VALIDATION conserva las etiquetas: True

=== COMPARACIÓN ENTRE RAMAS ===
TRAIN con ambas versiones exactamente iguales: 0
VALIDATION con ambas versiones exactamente iguales: 0

=== CONTROL DEL TEST SELLADO ===
¿text_sequential existe en test_df?: False
¿text_bert existe en test_df?: False
Test procesado: NO

In [23]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 6: guardar los artefactos procesados de TRAIN y VALIDATION
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Guarda TRAIN procesado con las dos ramas:
#       text_sequential
#       text_bert
#
# 2. Guarda VALIDATION procesada con las mismas dos ramas.
#
# 3. Vuelve a leer los archivos para comprobar que:
#       - se pueden recuperar;
#       - conservan las mismas filas;
#       - contienen ambas columnas procesadas.
#
# 4. Comprueba que TEST sigue sellado y sin procesar.
#
# ¿POR QUÉ LO HACEMOS?
#
# Google Colab utiliza memoria temporal.
# Si la sesión se reinicia, podríamos perder train_g4 y val_g4.
#
# Guardarlos nos permite continuar posteriormente desde
# artefactos reproducibles.
#
# IMPORTANTE:
#
# - NO modificamos train_df ni val_df.
# - NO procesamos test_df.
# - NO sobrescribimos el dataset original.
# - NO entrenamos ningún modelo.
# - Solo persistimos resultados ya validados de G4.


# ------------------------------------------------------------
# 1. DEFINIR LAS RUTAS DE SALIDA
# ------------------------------------------------------------

G4_TRAIN_PATH = (
    ARTIFACT_DIR
    / "g4_train_texto_procesado.csv"
)

G4_VALIDATION_PATH = (
    ARTIFACT_DIR
    / "g4_validation_texto_procesado.csv"
)


# ------------------------------------------------------------
# 2. GUARDAR TRAIN PROCESADO
# ------------------------------------------------------------

train_g4.to_csv(
    G4_TRAIN_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. GUARDAR VALIDATION PROCESADA
# ------------------------------------------------------------

val_g4.to_csv(
    G4_VALIDATION_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. COMPROBAR QUE LOS ARCHIVOS EXISTEN
# ------------------------------------------------------------

print("=== ARTEFACTOS DE G4 ===")

print(
    "TRAIN procesado existe:",
    G4_TRAIN_PATH.exists()
)

print(
    "VALIDATION procesada existe:",
    G4_VALIDATION_PATH.exists()
)

print(
    "\nRuta TRAIN:",
    G4_TRAIN_PATH
)

print(
    "Ruta VALIDATION:",
    G4_VALIDATION_PATH
)


# ------------------------------------------------------------
# 5. VOLVER A LEER LOS ARCHIVOS
# ------------------------------------------------------------

train_g4_check = pd.read_csv(
    G4_TRAIN_PATH
)

val_g4_check = pd.read_csv(
    G4_VALIDATION_PATH
)


# ------------------------------------------------------------
# 6. COMPROBAR LAS FILAS RECUPERADAS
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "TRAIN guardado:",
    len(train_g4)
)

print(
    "TRAIN recuperado:",
    len(train_g4_check)
)

print(
    "¿TRAIN coincide?:",
    len(train_g4) == len(train_g4_check)
)

print()

print(
    "VALIDATION guardada:",
    len(val_g4)
)

print(
    "VALIDATION recuperada:",
    len(val_g4_check)
)

print(
    "¿VALIDATION coincide?:",
    len(val_g4) == len(val_g4_check)
)


# ------------------------------------------------------------
# 7. COMPROBAR LAS DOS RAMAS
# ------------------------------------------------------------

print("\n=== CONTROL DE COLUMNAS DE TEXTO ===")

print(
    "TRAIN contiene text_sequential:",
    "text_sequential" in train_g4_check.columns
)

print(
    "TRAIN contiene text_bert:",
    "text_bert" in train_g4_check.columns
)

print(
    "VALIDATION contiene text_sequential:",
    "text_sequential" in val_g4_check.columns
)

print(
    "VALIDATION contiene text_bert:",
    "text_bert" in val_g4_check.columns
)


# ------------------------------------------------------------
# 8. COMPROBAR TEXTOS VACÍOS TRAS RECUPERAR LOS CSV
# ------------------------------------------------------------

train_seq_empty_check = int(
    train_g4_check["text_sequential"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

train_bert_empty_check = int(
    train_g4_check["text_bert"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

val_seq_empty_check = int(
    val_g4_check["text_sequential"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

val_bert_empty_check = int(
    val_g4_check["text_bert"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\n=== TEXTOS VACÍOS TRAS RECUPERACIÓN ===")

print(
    "TRAIN secuencial vacíos:",
    train_seq_empty_check
)

print(
    "TRAIN BERT vacíos:",
    train_bert_empty_check
)

print(
    "VALIDATION secuencial vacíos:",
    val_seq_empty_check
)

print(
    "VALIDATION BERT vacíos:",
    val_bert_empty_check
)


# ------------------------------------------------------------
# 9. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== TEST SELLADO ===")

print(
    "¿text_sequential existe en test_df?:",
    "text_sequential" in test_df.columns
)

print(
    "¿text_bert existe en test_df?:",
    "text_bert" in test_df.columns
)

print(
    "TEST procesado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 10. CONTROL DE MODELOS
# ------------------------------------------------------------

print("\n=== MODELOS ===")

print("Word2Vec entrenado: NO")
print("BiLSTM entrenada: NO")
print("BERT entrenado: NO")


# ------------------------------------------------------------
# 11. ESTADO DE G4
# ------------------------------------------------------------

g4_controls_ok = (
    len(train_g4) == len(train_g4_check)
    and len(val_g4) == len(val_g4_check)
    and "text_sequential" in train_g4_check.columns
    and "text_bert" in train_g4_check.columns
    and "text_sequential" in val_g4_check.columns
    and "text_bert" in val_g4_check.columns
    and train_seq_empty_check == 0
    and train_bert_empty_check == 0
    and val_seq_empty_check == 0
    and val_bert_empty_check == 0
    and "text_sequential" not in test_df.columns
    and "text_bert" not in test_df.columns
)

print("\n=== RESULTADO FINAL DE G4 ===")

print(
    "¿Controles de G4 superados?:",
    g4_controls_ok
)

print(
    "G4 listo para aprobación:",
    "SÍ" if g4_controls_ok else "NO"
)

=== ARTEFACTOS DE G4 ===
TRAIN procesado existe: True
VALIDATION procesada existe: True

Ruta TRAIN: /content/artifacts_yelp/g4_train_texto_procesado.csv
Ruta VALIDATION: /content/artifacts_yelp/g4_validation_texto_procesado.csv

=== CONTROL DE RECUPERACIÓN ===
TRAIN guardado: 5943
TRAIN recuperado: 5943
¿TRAIN coincide?: True

VALIDATION guardada: 1956
VALIDATION recuperada: 1956
¿VALIDATION coincide?: True

=== CONTROL DE COLUMNAS DE TEXTO ===
TRAIN contiene text_sequential: True
TRAIN contiene text_bert: True
VALIDATION contiene text_sequential: True
VALIDATION contiene text_bert: True

=== TEXTOS VACÍOS TRAS RECUPERACIÓN ===
TRAIN secuencial vacíos: 0
TRAIN BERT vacíos: 0
VALIDATION secuencial vacíos: 0
VALIDATION BERT vacíos: 0

=== TEST SELLADO ===
¿text_sequential existe en test_df?: False
¿text_bert existe en test_df?: False
TEST procesado: NO
TEST evaluado: NO

=== MODELOS ===
Word2Vec entrenado: NO
BiLSTM entrenada: NO
BERT entrenado: NO

=== RESULTADO FINAL DE G4 ===
¿Contro

In [24]:
# ============================================================
# G4 — PREPROCESAMIENTO DEL TEXTO
# Paso 7: evidencia formal para auditoría de las dos ramas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Esta celda NO vuelve a procesar TRAIN ni VALIDATION.
#
# Su finalidad es producir evidencia ejecutada para el
# AUDITOR DE DATOS sobre el comportamiento de:
#
#       clean_text_sequential()
#       clean_text_bert()
#
# Comprobaremos:
#
# 1. Cómo está definida cada función.
#
# 2. Que la rama secuencial conserva información de negación.
#
# 3. Que la rama BERT conserva:
#       - negaciones
#       - stopwords
#       - puntuación
#
# 4. Que la rama BERT no recibe text_sequential.
#
# 5. Que TEST continúa sellado.
#
# IMPORTANTE:
#
# - NO modifica train_g4.
# - NO modifica val_g4.
# - NO modifica test_df.
# - NO entrena Word2Vec.
# - NO entrena BiLSTM.
# - NO entrena BERT.


# ------------------------------------------------------------
# 1. MOSTRAR EL CÓDIGO REAL DE LAS DOS FUNCIONES
# ------------------------------------------------------------

# inspect.getsource() intenta recuperar exactamente el código
# de las funciones actualmente definidas en memoria.

import inspect

print("=== CÓDIGO DE clean_text_sequential ===")

try:
    print(inspect.getsource(clean_text_sequential))
except Exception as e:
    print(
        "No fue posible mostrar automáticamente el código:",
        repr(e)
    )


print("\n=== CÓDIGO DE clean_text_bert ===")

try:
    print(inspect.getsource(clean_text_bert))
except Exception as e:
    print(
        "No fue posible mostrar automáticamente el código:",
        repr(e)
    )


# ------------------------------------------------------------
# 2. PRUEBAS CONTROLADAS DE NEGACIÓN — SECUENCIAL
# ------------------------------------------------------------

# Usamos frases artificiales únicamente para comprobar
# el comportamiento de la función.
#
# NO son resultados del modelo.

sequential_negation_tests = {
    "no": "There is no good service here.",
    "not": "This restaurant is not good.",
    "nor": "Neither friendly nor helpful.",
    "never": "I will never return.",
    "contraction_nt": "I don't like this place."
}


print("\n=== NEGACIONES — RAMA SECUENCIAL ===")

sequential_outputs = {}

for name, example in sequential_negation_tests.items():

    processed = clean_text_sequential(example)

    sequential_outputs[name] = processed

    print(f"\nPRUEBA: {name}")
    print("Original:", example)
    print("Procesado:", processed)


# ------------------------------------------------------------
# 3. VALIDACIONES AUTOMÁTICAS DE NEGACIÓN
# ------------------------------------------------------------

# En una contracción como don't, nuestra normalización puede
# convertir la información de n't en el token 'not'.
#
# Por ello comprobamos la PRESERVACIÓN SEMÁNTICA de la
# negación y no exigimos que aparezca literalmente "n't"
# después de eliminar puntuación.

negation_checks = {
    "no_conservado":
        "no" in sequential_outputs["no"].split(),

    "not_conservado":
        "not" in sequential_outputs["not"].split(),

    "nor_conservado":
        "nor" in sequential_outputs["nor"].split(),

    "never_conservado":
        "never" in sequential_outputs["never"].split(),

    "n't_preserva_negacion":
        "not" in sequential_outputs["contraction_nt"].split()
}


print("\n=== VALIDACIÓN AUTOMÁTICA DE NEGACIONES ===")

for control, result in negation_checks.items():
    print(control, ":", result)


# ------------------------------------------------------------
# 4. PRUEBA DE LA RAMA BERT
# ------------------------------------------------------------

# Esta frase contiene deliberadamente:
#
# - una contracción negativa;
# - stopwords;
# - mayúsculas;
# - puntuación;
# - entidad HTML;
# - espacios múltiples.
#
# La limpieza BERT debe ser mínima.

bert_test_text = (
    "I don't REALLY like this restaurant!!! "
    "It is not good &amp; it is never cheap.   "
    "Neither friendly nor helpful?"
)

bert_test_output = clean_text_bert(
    bert_test_text
)


print("\n=== PRUEBA CONTROLADA — RAMA BERT ===")

print("ORIGINAL:")
print(bert_test_text)

print("\nPROCESADO:")
print(bert_test_output)


# ------------------------------------------------------------
# 5. VALIDACIONES AUTOMÁTICAS DE BERT
# ------------------------------------------------------------

bert_checks = {

    # Negaciones
    "conserva_dont":
        "don't" in bert_test_output.lower(),

    "conserva_not":
        "not" in bert_test_output.lower().split(),

    "conserva_never":
        "never" in bert_test_output.lower().split(),

    "conserva_nor":
        "nor" in bert_test_output.lower().split(),

    # Stopwords representativas
    "conserva_stopword_this":
        "this" in bert_test_output.lower().split(),

    "conserva_stopword_it":
        "it" in bert_test_output.lower().split(),

    # Puntuación
    "conserva_exclamacion":
        "!" in bert_test_output,

    "conserva_interrogacion":
        "?" in bert_test_output,

    # HTML
    "decodifica_amp":
        "&amp;" not in bert_test_output
        and "&" in bert_test_output,

    # Espacios
    "normaliza_espacios":
        "   " not in bert_test_output
}


print("\n=== VALIDACIÓN AUTOMÁTICA — RAMA BERT ===")

for control, result in bert_checks.items():
    print(control, ":", result)


# ------------------------------------------------------------
# 6. COMPROBAR QUE BERT PARTE DEL text ORIGINAL
# ------------------------------------------------------------

# En G4 construimos text_bert directamente mediante:
#
# train_g4["text"].apply(clean_text_bert)
#
# Aquí hacemos una comprobación adicional:
# recalculamos text_bert desde text ORIGINAL para una muestra
# y verificamos que coincide con el artefacto ya creado.

bert_recalculated = (
    train_g4["text"]
    .head(20)
    .apply(clean_text_bert)
    .reset_index(drop=True)
)

bert_saved = (
    train_g4["text_bert"]
    .head(20)
    .reset_index(drop=True)
)

bert_from_original_ok = (
    bert_recalculated.equals(bert_saved)
)


print("\n=== ORIGEN DE LA RAMA BERT ===")

print(
    "text_bert coincide al recalcularse directamente desde text:",
    bert_from_original_ok
)

print(
    "BERT utiliza text_sequential como entrada:",
    "NO"
)


# ------------------------------------------------------------
# 7. COMPROBAR QUE TEST SIGUE SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "text_sequential en test_df:",
    "text_sequential" in test_df.columns
)

print(
    "text_bert en test_df:",
    "text_bert" in test_df.columns
)

print(
    "TEST procesado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 8. RESUMEN AUTOMÁTICO DE LOS CONTROLES
# ------------------------------------------------------------

all_sequential_negations_ok = all(
    negation_checks.values()
)

all_bert_checks_ok = all(
    bert_checks.values()
)

g4_audit_evidence_ok = (
    all_sequential_negations_ok
    and all_bert_checks_ok
    and bert_from_original_ok
    and "text_sequential" not in test_df.columns
    and "text_bert" not in test_df.columns
)


print("\n=== RESUMEN PARA AUDITORÍA ===")

print(
    "Negaciones secuenciales superan controles:",
    all_sequential_negations_ok
)

print(
    "Rama BERT supera controles:",
    all_bert_checks_ok
)

print(
    "BERT deriva directamente de text:",
    bert_from_original_ok
)

print(
    "Evidencia adicional G4 superada:",
    g4_audit_evidence_ok
)


# ------------------------------------------------------------
# 9. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO ===")

print(
    "TRAIN/VALIDATION modificados por esta celda:",
    "NO"
)

print(
    "TEST procesado:",
    "NO"
)

print(
    "Modelos entrenados:",
    "NINGUNO"
)

print(
    "G5 iniciado:",
    "NO"
)

=== CÓDIGO DE clean_text_sequential ===
def clean_text_sequential(text):
    """
    Preprocesamiento para la rama Word2Vec + BiLSTM.

    Entrada:
        texto original de una reseña

    Salida:
        texto limpio y normalizado como cadena
    """

    # --------------------------------------------------------
    # 1.1. ASEGURAR QUE TRABAJAMOS CON TEXTO
    # --------------------------------------------------------

    # Convertimos cualquier entrada a str por robustez.

    text = str(text)


    # --------------------------------------------------------
    # 1.2. DECODIFICAR ENTIDADES HTML
    # --------------------------------------------------------

    # Ejemplo:
    #   &amp;  -> &
    #   &quot; -> "

    text = html.unescape(text)


    # --------------------------------------------------------
    # 1.3. CONVERTIR A MINÚSCULAS
    # --------------------------------------------------------

    text = text.lower()


    # -----------------------------------------------

In [25]:
# ============================================================
# ETAPA Y NOMBRE
# Paso X: objetivo
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
# Explicación sencilla...

# IMPORTANTE:
# Qué modifica.
# Qué NO modifica.
# Qué riesgo controla.

# ------------------------------------------------------------
# 1. ...
# ------------------------------------------------------------

In [26]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 1: verificar entradas y preparar el baseline mayoritario
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. ...
# 2. ...
# 3. ...

# ¿POR QUÉ LO HACEMOS?
#
# Explicación sencilla de para qué sirve este paso.

# IMPORTANTE:
#
# - Qué modifica.
# - Qué NO modifica.
# - Qué datos puede utilizar.
# - Qué datos NO puede utilizar.
# - Cómo protegemos TEST.

# ¿QUÉ RIESGO CONTROLAMOS?
#
# Explicación del posible error metodológico que estamos evitando.

# ------------------------------------------------------------
# 1. PRIMER CONTROL
# ------------------------------------------------------------

# Explicación sencilla de lo que hace la siguiente instrucción.

# Código...

# ------------------------------------------------------------
# 2. SEGUNDO CONTROL
# ------------------------------------------------------------

# Explicación...

# Código...

In [27]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 1: verificar entradas y preparar el baseline mayoritario
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba que seguimos utilizando exactamente los splits
#    aprobados en G3.
#
# 2. Verifica que TRAIN y VALIDATION contienen las etiquetas
#    necesarias para evaluar baselines.
#
# 3. Comprueba que TEST continúa sellado.
#
# 4. Calcula, SOLO con TRAIN, cuál es la clase mayoritaria.
#
# 5. Construye el baseline más simple posible:
#
#       "predecir siempre la clase mayoritaria"
#
# 6. Genera predicciones de ese baseline sobre VALIDATION.
#
# IMPORTANTE:
#
# - La clase mayoritaria se determina SOLO con TRAIN.
#
# - NO usamos TEST.
#
# - NO entrenamos todavía Logistic Regression.
#
# - NO usamos stars, IDs, fecha, cool, useful o funny
#   como predictores.
#
# - sentiment_id se usa únicamente como etiqueta objetivo.
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos calcular el baseline utilizando información de
# VALIDATION o TEST.
#
# También dejamos una referencia mínima contra la cual
# interpretar posteriormente la accuracy de modelos más
# complejos.


# ------------------------------------------------------------
# 1. COMPROBAR LOS TAMAÑOS DE TRAIN Y VALIDATION
# ------------------------------------------------------------

print("=== ENTRADAS DE G5 ===")

print(
    "Filas TRAIN:",
    len(train_g4)
)

print(
    "Filas VALIDATION:",
    len(val_g4)
)


# ------------------------------------------------------------
# 2. COMPROBAR QUE LAS ETIQUETAS ESTÁN DISPONIBLES
# ------------------------------------------------------------

print("\n=== CONTROL DE ETIQUETAS ===")

print(
    "sentiment_id existe en TRAIN:",
    "sentiment_id" in train_g4.columns
)

print(
    "sentiment_id existe en VALIDATION:",
    "sentiment_id" in val_g4.columns
)


# ------------------------------------------------------------
# 3. EXTRAER LAS ETIQUETAS
# ------------------------------------------------------------

# y_train:
# etiquetas reales utilizadas para construir el baseline.
#
# y_val:
# etiquetas reales de VALIDATION.
#
# VALIDATION se usa únicamente para medir el baseline,
# no para determinar cuál es la clase mayoritaria.

y_train = (
    train_g4["sentiment_id"]
    .astype(int)
    .to_numpy()
)

y_val = (
    val_g4["sentiment_id"]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 4. CALCULAR LA CLASE MAYORITARIA SOLO CON TRAIN
# ------------------------------------------------------------

# value_counts() cuenta cuántas observaciones hay por clase.
#
# idxmax() devuelve la etiqueta más frecuente.

train_class_counts = (
    train_g4["sentiment_id"]
    .astype(int)
    .value_counts()
    .sort_index()
)

majority_class = int(
    train_class_counts.idxmax()
)


print("\n=== DISTRIBUCIÓN DE CLASES EN TRAIN ===")

print(
    train_class_counts
)


print(
    "\nClase mayoritaria calculada SOLO con TRAIN:",
    majority_class
)


# ------------------------------------------------------------
# 5. CREAR PREDICCIONES DEL BASELINE MAYORITARIO
# ------------------------------------------------------------

# np.full() crea un vector del mismo tamaño que VALIDATION
# en el que todas las predicciones son la clase mayoritaria.
#
# Esto NO es un modelo entrenado.
# Es un control experimental mínimo.

import numpy as np

y_val_pred_majority = np.full(
    shape=len(y_val),
    fill_value=majority_class,
    dtype=int
)


# ------------------------------------------------------------
# 6. COMPROBAR EL TAMAÑO DE LAS PREDICCIONES
# ------------------------------------------------------------

print("\n=== BASELINE MAYORITARIO ===")

print(
    "Predicciones generadas:",
    len(y_val_pred_majority)
)

print(
    "Filas en VALIDATION:",
    len(y_val)
)

print(
    "¿Coinciden los tamaños?:",
    len(y_val_pred_majority) == len(y_val)
)


# ------------------------------------------------------------
# 7. COMPROBAR QUE TODAS LAS PREDICCIONES SON LA MISMA CLASE
# ------------------------------------------------------------

unique_majority_predictions = np.unique(
    y_val_pred_majority
)

print(
    "Clases predichas por el baseline:",
    unique_majority_predictions
)


# ------------------------------------------------------------
# 8. CONTROL DE PREDICTORES
# ------------------------------------------------------------

print("\n=== CONTROL DE PREDICTORES ===")

print(
    "Campo text usado como predictor en este baseline:",
    "NO"
)

print(
    "stars usado como predictor:",
    "NO"
)

print(
    "business_id usado como predictor:",
    "NO"
)

print(
    "user_id usado como predictor:",
    "NO"
)

print(
    "review_id usado como predictor:",
    "NO"
)


# ------------------------------------------------------------
# 9. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado para determinar la clase mayoritaria:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "TEST modificado:",
    "NO"
)


# ------------------------------------------------------------
# 10. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO DE G5 ===")

print(
    "Baseline mayoritario preparado:",
    "SÍ"
)

print(
    "Métricas calculadas:",
    "NO"
)

print(
    "TF-IDF entrenado:",
    "NO"
)

print(
    "Logistic Regression entrenada:",
    "NO"
)

print(
    "G5 finalizado:",
    "NO"
)

=== ENTRADAS DE G5 ===
Filas TRAIN: 5943
Filas VALIDATION: 1956

=== CONTROL DE ETIQUETAS ===
sentiment_id existe en TRAIN: True
sentiment_id existe en VALIDATION: True

=== DISTRIBUCIÓN DE CLASES EN TRAIN ===
sentiment_id
0     993
1     847
2    4103
Name: count, dtype: int64

Clase mayoritaria calculada SOLO con TRAIN: 2

=== BASELINE MAYORITARIO ===
Predicciones generadas: 1956
Filas en VALIDATION: 1956
¿Coinciden los tamaños?: True
Clases predichas por el baseline: [2]

=== CONTROL DE PREDICTORES ===
Campo text usado como predictor en este baseline: NO
stars usado como predictor: NO
business_id usado como predictor: NO
user_id usado como predictor: NO
review_id usado como predictor: NO

=== CONTROL DEL TEST SELLADO ===
TEST utilizado para determinar la clase mayoritaria: NO
TEST evaluado: NO
TEST modificado: NO

=== ESTADO DE G5 ===
Baseline mayoritario preparado: SÍ
Métricas calculadas: NO
TF-IDF entrenado: NO
Logistic Regression entrenada: NO
G5 finalizado: NO


In [28]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 2: crear la función común de métricas y evaluar
#         el baseline de clase mayoritaria en VALIDATION
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Define una función común para evaluar predicciones.
#
# 2. Calcula sobre VALIDATION:
#
#       - accuracy
#       - balanced accuracy
#       - precision macro
#       - recall macro
#       - F1 macro
#       - F1 weighted
#
# 3. Genera el reporte por clase:
#
#       NEGATIVO (0)
#       NEUTRAL  (1)
#       POSITIVO (2)
#
# 4. Calcula:
#
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 5. Evalúa el baseline que siempre predice
#    la clase mayoritaria obtenida SOLO desde TRAIN.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# El dataset está desbalanceado.
#
# Por eso accuracy no debe interpretarse sola.
#
# Una estrategia muy simple puede obtener una accuracy
# aparentemente elevada prediciendo siempre POSITIVO.
#
# Nuestra métrica principal será:
#
#       F1 MACRO
#
# porque calcula el rendimiento de cada clase y después
# les concede el mismo peso.
#
#
# IMPORTANTE:
#
# - Estas son métricas de VALIDATION.
#
# - NO son resultados finales de TEST.
#
# - TEST continúa sellado.
#
# - Esta función podrá reutilizarse posteriormente para
#   evaluar de manera homogénea otros modelos.
#
# - NO se entrena ningún modelo en esta celda.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos comparar modelos utilizando métricas calculadas
# de maneras diferentes.
#
# También evitamos interpretar una accuracy alta como
# evidencia suficiente cuando existen clases desbalanceadas.


# ------------------------------------------------------------
# 1. IMPORTAR LAS MÉTRICAS DE SCIKIT-LEARN
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINIR EL ORDEN OFICIAL DE LAS CLASES
# ------------------------------------------------------------

# Este orden debe mantenerse durante todo el experimento:
#
# 0 = NEGATIVO
# 1 = NEUTRAL
# 2 = POSITIVO

LABEL_IDS = [0, 1, 2]

LABEL_NAMES = [
    "NEGATIVO",
    "NEUTRAL",
    "POSITIVO"
]


# ------------------------------------------------------------
# 3. CREAR LA FUNCIÓN COMÚN DE EVALUACIÓN
# ------------------------------------------------------------

# La función recibe:
#
# y_true -> etiquetas reales
# y_pred -> predicciones
#
# Devuelve:
#
# métricas generales
# reporte por clase
# matriz de confusión absoluta
# matriz de confusión normalizada

def evaluate_predictions(y_true, y_pred):

    # --------------------------------------------------------
    # 3.1. MÉTRICAS GENERALES
    # --------------------------------------------------------

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),

        "precision_macro": precision_score(
            y_true,
            y_pred,
            labels=LABEL_IDS,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            labels=LABEL_IDS,
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            labels=LABEL_IDS,
            average="macro",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            labels=LABEL_IDS,
            average="weighted",
            zero_division=0
        )
    }


    # --------------------------------------------------------
    # 3.2. REPORTE POR CLASE
    # --------------------------------------------------------

    # output_dict=True devuelve el reporte como estructura
    # de datos en lugar de imprimirlo directamente.

    report = classification_report(
        y_true,
        y_pred,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0
    )


    # --------------------------------------------------------
    # 3.3. MATRIZ DE CONFUSIÓN ABSOLUTA
    # --------------------------------------------------------

    cm_absolute = confusion_matrix(
        y_true,
        y_pred,
        labels=LABEL_IDS
    )


    # --------------------------------------------------------
    # 3.4. MATRIZ DE CONFUSIÓN NORMALIZADA
    # --------------------------------------------------------

    # normalize="true":
    # cada fila se divide por el total real de esa clase.
    #
    # Así podremos interpretar qué proporción de cada clase
    # fue asignada a cada categoría.

    cm_normalized = confusion_matrix(
        y_true,
        y_pred,
        labels=LABEL_IDS,
        normalize="true"
    )


    return (
        metrics,
        report,
        cm_absolute,
        cm_normalized
    )


# ------------------------------------------------------------
# 4. EVALUAR EL BASELINE MAYORITARIO
# ------------------------------------------------------------

(
    majority_metrics,
    majority_report,
    majority_cm,
    majority_cm_norm
) = evaluate_predictions(
    y_val,
    y_val_pred_majority
)


# ------------------------------------------------------------
# 5. MOSTRAR LAS MÉTRICAS GENERALES
# ------------------------------------------------------------

print(
    "=== BASELINE MAYORITARIO — "
    "MÉTRICAS EN VALIDATION ==="
)

for metric_name, metric_value in majority_metrics.items():

    print(
        f"{metric_name}: "
        f"{metric_value:.4f}"
    )


# ------------------------------------------------------------
# 6. MOSTRAR EL REPORTE POR CLASE
# ------------------------------------------------------------

# Convertimos el diccionario en DataFrame para que resulte
# más sencillo de leer en el notebook.

majority_report_df = (
    pd.DataFrame(majority_report)
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — VALIDATION ==="
)

print(
    majority_report_df.round(4).to_string()
)


# ------------------------------------------------------------
# 7. MOSTRAR LA MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

majority_cm_df = pd.DataFrame(
    majority_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA ==="
)

print(
    majority_cm_df.to_string()
)


# ------------------------------------------------------------
# 8. MOSTRAR LA MATRIZ DE CONFUSIÓN NORMALIZADA
# ------------------------------------------------------------

majority_cm_norm_df = pd.DataFrame(
    majority_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA ==="
)

print(
    majority_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. DESTACAR LA MÉTRICA PRINCIPAL
# ------------------------------------------------------------

print(
    "\n=== MÉTRICA PRINCIPAL ==="
)

print(
    "F1 macro del baseline mayoritario:",
    round(
        majority_metrics["f1_macro"],
        4
    )
)


# ------------------------------------------------------------
# 10. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "Métricas calculadas sobre:",
    "VALIDATION"
)

print(
    "TEST utilizado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 11. ESTADO DE G5
# ------------------------------------------------------------

print(
    "\n=== ESTADO DE G5 ==="
)

print(
    "Baseline mayoritario evaluado:",
    "SÍ"
)

print(
    "Función común de métricas creada:",
    "SÍ"
)

print(
    "TF-IDF entrenado:",
    "NO"
)

print(
    "Logistic Regression entrenada:",
    "NO"
)

print(
    "G5 finalizado:",
    "NO"
)

=== BASELINE MAYORITARIO — MÉTRICAS EN VALIDATION ===
accuracy: 0.6748
balanced_accuracy: 0.3333
precision_macro: 0.2249
recall_macro: 0.3333
f1_macro: 0.2686
f1_weighted: 0.5438

=== REPORTE POR CLASE — VALIDATION ===
              precision  recall  f1-score    support
NEGATIVO         0.0000  0.0000    0.0000   330.0000
NEUTRAL          0.0000  0.0000    0.0000   306.0000
POSITIVO         0.6748  1.0000    0.8059  1320.0000
accuracy         0.6748  0.6748    0.6748     0.6748
macro avg        0.2249  0.3333    0.2686  1956.0000
weighted avg     0.4554  0.6748    0.5438  1956.0000

=== MATRIZ DE CONFUSIÓN ABSOLUTA ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO              0             0            330
Real_NEUTRAL               0             0            306
Real_POSITIVO              0             0           1320

=== MATRIZ DE CONFUSIÓN NORMALIZADA ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO            0.0           0.

In [29]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 3: ajustar TF-IDF exclusivamente con TRAIN
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Selecciona exclusivamente text_sequential como predictor.
#
# 2. Crea un vectorizador TF-IDF.
#
# 3. AJUSTA el vocabulario de TF-IDF únicamente con TRAIN.
#
# 4. Transforma TRAIN utilizando ese vocabulario.
#
# 5. Transforma VALIDATION utilizando EXACTAMENTE el mismo
#    vocabulario aprendido de TRAIN.
#
# 6. Comprueba las dimensiones resultantes.
#
#
# ¿QUÉ ES TF-IDF?
#
# Los modelos como Logistic Regression no pueden trabajar
# directamente con frases.
#
# TF-IDF convierte cada reseña en un vector numérico.
#
# De forma simplificada:
#
# - una palabra recibe más importancia si aparece en una
#   reseña;
#
# - pero pierde importancia si aparece constantemente
#   en prácticamente todas las reseñas.
#
#
# IMPORTANTE:
#
# - TF-IDF se AJUSTA solamente con TRAIN.
#
# - VALIDATION NO participa en la creación del vocabulario.
#
# - TEST NO se utiliza.
#
# - Solo text_sequential alimenta el modelo.
#
# - sentiment_id es únicamente la etiqueta objetivo.
#
# - Todavía NO entrenamos Logistic Regression.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos FUGA DE INFORMACIÓN.
#
# Si ejecutáramos fit() sobre TRAIN + VALIDATION, el
# vocabulario habría aprendido información de VALIDATION
# antes de evaluar el modelo.
#
# Por eso:
#
# TRAIN:
#     fit_transform()
#
# VALIDATION:
#     transform()
#
# Nunca:
#     fit_transform(TRAIN + VALIDATION)


# ------------------------------------------------------------
# 1. IMPORTAR TF-IDF
# ------------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer


# ------------------------------------------------------------
# 2. DEFINIR LOS PREDICTORES
# ------------------------------------------------------------

# Solo utilizamos la columna de texto procesado.
#
# Ninguna variable estructurada del dataset se incorpora
# como predictor.

X_train_text = (
    train_g4["text_sequential"]
    .astype(str)
)

X_val_text = (
    val_g4["text_sequential"]
    .astype(str)
)


# ------------------------------------------------------------
# 3. COMPROBAR LAS ENTRADAS
# ------------------------------------------------------------

print("=== ENTRADAS DE TF-IDF ===")

print(
    "Textos TRAIN:",
    len(X_train_text)
)

print(
    "Textos VALIDATION:",
    len(X_val_text)
)

print(
    "Columna utilizada como predictor:",
    "text_sequential"
)


# ------------------------------------------------------------
# 4. CREAR EL VECTORIZADOR TF-IDF
# ------------------------------------------------------------

# min_df=2:
# una característica debe aparecer al menos en 2 documentos
# de TRAIN.
#
# ngram_range=(1, 2):
# utilizamos palabras individuales (unigramas) y parejas
# consecutivas de palabras (bigramas).
#
# sublinear_tf=True:
# aplica una escala logarítmica a la frecuencia de términos.
#
# Estas decisiones forman parte de la configuración del
# baseline y deben quedar registradas.

tfidf_vectorizer = TfidfVectorizer(
    min_df=2,
    ngram_range=(1, 2),
    sublinear_tf=True
)


# ------------------------------------------------------------
# 5. AJUSTAR TF-IDF EXCLUSIVAMENTE CON TRAIN
# ------------------------------------------------------------

# fit_transform():
#
# 1. aprende el vocabulario desde TRAIN;
# 2. calcula los pesos IDF desde TRAIN;
# 3. transforma TRAIN a representación numérica.

X_train_tfidf = tfidf_vectorizer.fit_transform(
    X_train_text
)


# ------------------------------------------------------------
# 6. TRANSFORMAR VALIDATION SIN VOLVER A AJUSTAR
# ------------------------------------------------------------

# Aquí utilizamos SOLO transform().
#
# Esto es fundamental:
# VALIDATION no puede modificar el vocabulario ni los IDF.

X_val_tfidf = tfidf_vectorizer.transform(
    X_val_text
)


# ------------------------------------------------------------
# 7. MOSTRAR LAS DIMENSIONES
# ------------------------------------------------------------

print("\n=== DIMENSIONES TF-IDF ===")

print(
    "TRAIN:",
    X_train_tfidf.shape
)

print(
    "VALIDATION:",
    X_val_tfidf.shape
)


# ------------------------------------------------------------
# 8. COMPROBAR EL VOCABULARIO
# ------------------------------------------------------------

vocabulary_size = len(
    tfidf_vectorizer.vocabulary_
)

print(
    "\nTamaño del vocabulario aprendido:",
    vocabulary_size
)


# ------------------------------------------------------------
# 9. VERIFICAR QUE TRAIN Y VALIDATION COMPARTEN
#    EXACTAMENTE EL MISMO ESPACIO DE CARACTERÍSTICAS
# ------------------------------------------------------------

same_feature_space = (
    X_train_tfidf.shape[1]
    == X_val_tfidf.shape[1]
    == vocabulary_size
)

print(
    "¿TRAIN y VALIDATION usan el mismo espacio TF-IDF?:",
    same_feature_space
)


# ------------------------------------------------------------
# 10. CONTROL DE VARIABLES PROHIBIDAS
# ------------------------------------------------------------

forbidden_predictors = [
    "stars",
    "business_id",
    "user_id",
    "review_id",
    "date",
    "cool",
    "useful",
    "funny"
]

used_predictors = [
    "text_sequential"
]

forbidden_used = (
    set(forbidden_predictors)
    & set(used_predictors)
)


print("\n=== CONTROL DE PREDICTORES ===")

print(
    "Predictores utilizados:",
    used_predictors
)

print(
    "Variables prohibidas utilizadas:",
    sorted(forbidden_used)
)

print(
    "¿Solo texto alimenta TF-IDF?:",
    forbidden_used == set()
    and used_predictors == ["text_sequential"]
)


# ------------------------------------------------------------
# 11. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado para ajustar TF-IDF:",
    "NO"
)

print(
    "TEST transformado con TF-IDF:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 12. ESTADO DE G5
# ------------------------------------------------------------

print("\n=== ESTADO DE G5 ===")

print(
    "Baseline mayoritario evaluado:",
    "SÍ"
)

print(
    "TF-IDF ajustado exclusivamente con TRAIN:",
    "SÍ"
)

print(
    "VALIDATION transformada sin fit:",
    "SÍ"
)

print(
    "Logistic Regression entrenada:",
    "NO"
)

print(
    "G5 finalizado:",
    "NO"
)

=== ENTRADAS DE TF-IDF ===
Textos TRAIN: 5943
Textos VALIDATION: 1956
Columna utilizada como predictor: text_sequential

=== DIMENSIONES TF-IDF ===
TRAIN: (5943, 51309)
VALIDATION: (1956, 51309)

Tamaño del vocabulario aprendido: 51309
¿TRAIN y VALIDATION usan el mismo espacio TF-IDF?: True

=== CONTROL DE PREDICTORES ===
Predictores utilizados: ['text_sequential']
Variables prohibidas utilizadas: []
¿Solo texto alimenta TF-IDF?: True

=== CONTROL DEL TEST SELLADO ===
TEST utilizado para ajustar TF-IDF: NO
TEST transformado con TF-IDF: NO
TEST evaluado: NO

=== ESTADO DE G5 ===
Baseline mayoritario evaluado: SÍ
TF-IDF ajustado exclusivamente con TRAIN: SÍ
VALIDATION transformada sin fit: SÍ
Logistic Regression entrenada: NO
G5 finalizado: NO


In [30]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 4: entrenar TF-IDF + Logistic Regression con TRAIN
#         y generar predicciones sobre VALIDATION
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea un modelo Logistic Regression.
#
# 2. Lo entrena utilizando:
#
#       X_train_tfidf -> predictor textual
#       y_train       -> etiqueta objetivo
#
# 3. Después del entrenamiento genera predicciones
#    exclusivamente sobre VALIDATION.
#
# 4. Comprueba que las predicciones:
#       - tienen el tamaño correcto;
#       - pertenecen únicamente a las clases 0, 1 y 2.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# TF-IDF + Logistic Regression es un baseline clásico de
# clasificación de texto.
#
# Nos permitirá comprobar posteriormente si los modelos
# requeridos:
#
#       Word2Vec + BiLSTM
#       BERT
#
# mejoran realmente frente a una solución clásica sencilla.
#
#
# IMPORTANTE:
#
# - SOLO text_sequential alimenta el modelo a través de TF-IDF.
#
# - Logistic Regression se ajusta SOLO con TRAIN.
#
# - VALIDATION se usa para evaluación/desarrollo.
#
# - TEST continúa completamente sellado.
#
# - stars, IDs, date, cool, useful y funny NO son predictores.
#
# - random_state=42 mantiene nuestra semilla oficial.
#
# - Todavía NO calculamos las métricas en esta celda.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos:
#
# - entrenar con VALIDATION;
# - utilizar TEST durante el desarrollo;
# - introducir variables distintas de text;
# - comparar modelos sobre particiones diferentes.


# ------------------------------------------------------------
# 1. IMPORTAR LOGISTIC REGRESSION
# ------------------------------------------------------------

from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 2. DEFINIR LA CONFIGURACIÓN DEL MODELO
# ------------------------------------------------------------

# random_state=42:
# utiliza la semilla oficial del experimento.
#
# max_iter=1000:
# concede suficientes iteraciones al optimizador para
# converger con este espacio TF-IDF.
#
# class_weight="balanced":
# compensa el desbalance de clases utilizando pesos derivados
# automáticamente de las frecuencias de TRAIN.
#
# IMPORTANTE:
# esos pesos se calculan a partir de y_train, nunca de
# VALIDATION ni TEST.

logreg_baseline = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)


# ------------------------------------------------------------
# 3. ENTRENAR EXCLUSIVAMENTE CON TRAIN
# ------------------------------------------------------------

# fit() es el momento en el que el modelo aprende.
#
# X_train_tfidf:
# representación numérica obtenida exclusivamente desde
# text_sequential.
#
# y_train:
# etiqueta proxy de sentimiento.

logreg_baseline.fit(
    X_train_tfidf,
    y_train
)


# ------------------------------------------------------------
# 4. GENERAR PREDICCIONES SOBRE VALIDATION
# ------------------------------------------------------------

# predict() NO reentrena el modelo.
#
# Simplemente utiliza lo aprendido con TRAIN para clasificar
# las reseñas de VALIDATION.

y_val_pred_logreg = logreg_baseline.predict(
    X_val_tfidf
)


# ------------------------------------------------------------
# 5. COMPROBAR EL TAMAÑO DE LAS PREDICCIONES
# ------------------------------------------------------------

print("=== LOGISTIC REGRESSION — ENTRENAMIENTO ===")

print(
    "Filas utilizadas para entrenamiento:",
    X_train_tfidf.shape[0]
)

print(
    "Características TF-IDF:",
    X_train_tfidf.shape[1]
)

print(
    "Modelo entrenado:",
    "SÍ"
)


print("\n=== PREDICCIONES SOBRE VALIDATION ===")

print(
    "Predicciones generadas:",
    len(y_val_pred_logreg)
)

print(
    "Filas reales de VALIDATION:",
    len(y_val)
)

print(
    "¿Coinciden los tamaños?:",
    len(y_val_pred_logreg) == len(y_val)
)


# ------------------------------------------------------------
# 6. COMPROBAR LAS CLASES PREDICHAS
# ------------------------------------------------------------

predicted_classes = np.unique(
    y_val_pred_logreg
)

print(
    "Clases predichas:",
    predicted_classes
)

print(
    "¿Las predicciones pertenecen solo a 0, 1 y 2?:",
    set(predicted_classes).issubset({0, 1, 2})
)


# ------------------------------------------------------------
# 7. MOSTRAR LA DISTRIBUCIÓN DE PREDICCIONES
# ------------------------------------------------------------

# Esto todavía NO mide si son correctas.
#
# Solo nos permite comprobar que el modelo no esté
# prediciendo accidentalmente una única clase.

prediction_distribution = (
    pd.Series(
        y_val_pred_logreg,
        name="sentiment_id"
    )
    .value_counts()
    .sort_index()
)

print(
    "\n=== DISTRIBUCIÓN DE PREDICCIONES EN VALIDATION ==="
)

print(
    prediction_distribution
)


# ------------------------------------------------------------
# 8. CONTROL DEL MODELO
# ------------------------------------------------------------

print("\n=== CONFIGURACIÓN DEL MODELO ===")

print(
    "Modelo:",
    "TF-IDF + Logistic Regression"
)

print(
    "class_weight:",
    logreg_baseline.class_weight
)

print(
    "random_state:",
    logreg_baseline.random_state
)

print(
    "max_iter:",
    logreg_baseline.max_iter
)


# ------------------------------------------------------------
# 9. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado para entrenar Logistic Regression:",
    "NO"
)

print(
    "TEST transformado con TF-IDF:",
    "NO"
)

print(
    "TEST utilizado para seleccionar configuración:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 10. ESTADO DE G5
# ------------------------------------------------------------

print("\n=== ESTADO DE G5 ===")

print(
    "Baseline mayoritario evaluado:",
    "SÍ"
)

print(
    "TF-IDF ajustado con TRAIN:",
    "SÍ"
)

print(
    "Logistic Regression entrenada con TRAIN:",
    "SÍ"
)

print(
    "Predicciones de VALIDATION generadas:",
    "SÍ"
)

print(
    "Métricas Logistic Regression calculadas:",
    "NO"
)

print(
    "G5 finalizado:",
    "NO"
)

=== LOGISTIC REGRESSION — ENTRENAMIENTO ===
Filas utilizadas para entrenamiento: 5943
Características TF-IDF: 51309
Modelo entrenado: SÍ

=== PREDICCIONES SOBRE VALIDATION ===
Predicciones generadas: 1956
Filas reales de VALIDATION: 1956
¿Coinciden los tamaños?: True
Clases predichas: [0 1 2]
¿Las predicciones pertenecen solo a 0, 1 y 2?: True

=== DISTRIBUCIÓN DE PREDICCIONES EN VALIDATION ===
sentiment_id
0     360
1     305
2    1291
Name: count, dtype: int64

=== CONFIGURACIÓN DEL MODELO ===
Modelo: TF-IDF + Logistic Regression
class_weight: balanced
random_state: 42
max_iter: 1000

=== CONTROL DEL TEST SELLADO ===
TEST utilizado para entrenar Logistic Regression: NO
TEST transformado con TF-IDF: NO
TEST utilizado para seleccionar configuración: NO
TEST evaluado: NO

=== ESTADO DE G5 ===
Baseline mayoritario evaluado: SÍ
TF-IDF ajustado con TRAIN: SÍ
Logistic Regression entrenada con TRAIN: SÍ
Predicciones de VALIDATION generadas: SÍ
Métricas Logistic Regression calculadas: NO
G5 f

In [31]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 5: evaluar TF-IDF + Logistic Regression en VALIDATION
#         y compararla con el baseline mayoritario
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Evalúa las predicciones de Logistic Regression sobre
#    VALIDATION.
#
# 2. Utiliza EXACTAMENTE la misma función de métricas que
#    usamos con el baseline mayoritario.
#
# 3. Calcula:
#
#       - accuracy
#       - balanced accuracy
#       - precision macro
#       - recall macro
#       - F1 macro
#       - F1 weighted
#
# 4. Genera:
#
#       - reporte por clase
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 5. Compara el baseline mayoritario contra
#    TF-IDF + Logistic Regression.
#
# ¿POR QUÉ LO HACEMOS?
#
# Queremos comprobar si un modelo clásico de texto mejora
# realmente frente a simplemente predecir siempre POSITIVO.
#
# La métrica PRINCIPAL de comparación es F1 macro.
#
# IMPORTANTE:
#
# - Todas estas métricas pertenecen a VALIDATION.
# - NO son resultados finales de TEST.
# - TEST continúa sellado.
# - NO entrenamos ningún modelo nuevo en esta celda.
# - NO modificamos el modelo ya entrenado.
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos comparar modelos con métricas diferentes.
#
# Ambos baselines se evalúan:
#
#       - sobre la misma VALIDATION;
#       - con las mismas etiquetas;
#       - con la misma función de evaluación.


# ------------------------------------------------------------
# 1. EVALUAR LOGISTIC REGRESSION
# ------------------------------------------------------------

(
    logreg_metrics,
    logreg_report,
    logreg_cm,
    logreg_cm_norm
) = evaluate_predictions(
    y_val,
    y_val_pred_logreg
)


# ------------------------------------------------------------
# 2. MOSTRAR LAS MÉTRICAS GENERALES
# ------------------------------------------------------------

print(
    "=== TF-IDF + LOGISTIC REGRESSION — "
    "MÉTRICAS EN VALIDATION ==="
)

for metric_name, metric_value in logreg_metrics.items():

    print(
        f"{metric_name}: "
        f"{metric_value:.4f}"
    )


# ------------------------------------------------------------
# 3. MOSTRAR EL REPORTE POR CLASE
# ------------------------------------------------------------

logreg_report_df = (
    pd.DataFrame(logreg_report)
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — VALIDATION ==="
)

print(
    logreg_report_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 4. MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

logreg_cm_df = pd.DataFrame(
    logreg_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA ==="
)

print(
    logreg_cm_df.to_string()
)


# ------------------------------------------------------------
# 5. MATRIZ DE CONFUSIÓN NORMALIZADA
# ------------------------------------------------------------

logreg_cm_norm_df = pd.DataFrame(
    logreg_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA ==="
)

print(
    logreg_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 6. CREAR TABLA COMPARATIVA
# ------------------------------------------------------------

# Construimos una tabla utilizando los resultados que YA
# fueron ejecutados para ambos baselines.

baseline_comparison = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "TF-IDF + Logistic Regression"
    ],

    "accuracy": [
        majority_metrics["accuracy"],
        logreg_metrics["accuracy"]
    ],

    "balanced_accuracy": [
        majority_metrics["balanced_accuracy"],
        logreg_metrics["balanced_accuracy"]
    ],

    "precision_macro": [
        majority_metrics["precision_macro"],
        logreg_metrics["precision_macro"]
    ],

    "recall_macro": [
        majority_metrics["recall_macro"],
        logreg_metrics["recall_macro"]
    ],

    "f1_macro": [
        majority_metrics["f1_macro"],
        logreg_metrics["f1_macro"]
    ],

    "f1_weighted": [
        majority_metrics["f1_weighted"],
        logreg_metrics["f1_weighted"]
    ]
})


print(
    "\n=== COMPARACIÓN DE BASELINES — VALIDATION ==="
)

print(
    baseline_comparison
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. CALCULAR LA DIFERENCIA EN F1 MACRO
# ------------------------------------------------------------

# Esta diferencia indica cuánto cambia nuestra métrica
# principal respecto al baseline más simple.
#
# La calculamos directamente desde los resultados ejecutados.

f1_macro_difference = (
    logreg_metrics["f1_macro"]
    - majority_metrics["f1_macro"]
)


print(
    "\n=== COMPARACIÓN EN LA MÉTRICA PRINCIPAL ==="
)

print(
    "F1 macro — clase mayoritaria:",
    round(
        majority_metrics["f1_macro"],
        4
    )
)

print(
    "F1 macro — TF-IDF + Logistic Regression:",
    round(
        logreg_metrics["f1_macro"],
        4
    )
)

print(
    "Diferencia absoluta de F1 macro:",
    round(
        f1_macro_difference,
        4
    )
)


# ------------------------------------------------------------
# 8. IDENTIFICAR EL MEJOR BASELINE EN VALIDATION
# ------------------------------------------------------------

# Esta conclusión se limita expresamente a VALIDATION.
#
# NO significa que sea el mejor modelo final del caso.

if (
    logreg_metrics["f1_macro"]
    > majority_metrics["f1_macro"]
):
    best_baseline_validation = (
        "TF-IDF + Logistic Regression"
    )

elif (
    logreg_metrics["f1_macro"]
    < majority_metrics["f1_macro"]
):
    best_baseline_validation = (
        "Clase mayoritaria"
    )

else:
    best_baseline_validation = (
        "Empate"
    )


print(
    "\nMejor baseline por F1 macro en VALIDATION:",
    best_baseline_validation
)


# ------------------------------------------------------------
# 9. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "Conjunto evaluado:",
    "VALIDATION"
)

print(
    "TEST utilizado en esta comparación:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 10. ESTADO DE G5
# ------------------------------------------------------------

print(
    "\n=== ESTADO DE G5 ==="
)

print(
    "Baseline mayoritario evaluado:",
    "SÍ"
)

print(
    "TF-IDF + Logistic Regression evaluada:",
    "SÍ"
)

print(
    "Comparación por F1 macro realizada:",
    "SÍ"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G5 listo para persistencia y auditoría:",
    "SÍ"
)

print(
    "G5 aprobado:",
    "TODAVÍA NO"
)

=== TF-IDF + LOGISTIC REGRESSION — MÉTRICAS EN VALIDATION ===
accuracy: 0.7740
balanced_accuracy: 0.6773
precision_macro: 0.6642
recall_macro: 0.6773
f1_macro: 0.6703
f1_weighted: 0.7753

=== REPORTE POR CLASE — VALIDATION ===
              precision  recall  f1-score   support
NEGATIVO         0.6611  0.7212    0.6899   330.000
NEUTRAL          0.4492  0.4477    0.4484   306.000
POSITIVO         0.8823  0.8629    0.8725  1320.000
accuracy         0.7740  0.7740    0.7740     0.774
macro avg        0.6642  0.6773    0.6703  1956.000
weighted avg     0.7772  0.7740    0.7753  1956.000

=== MATRIZ DE CONFUSIÓN ABSOLUTA ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO            238            52             40
Real_NEUTRAL              57           137            112
Real_POSITIVO             65           116           1139

=== MATRIZ DE CONFUSIÓN NORMALIZADA ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO         0.7212        0.15

In [32]:
# ============================================================
# G5 — BASELINES REPRODUCIBLES
# Paso 6: persistir resultados, predicciones y configuración
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Guarda las métricas ejecutadas de los dos baselines.
#
# 2. Guarda las predicciones de VALIDATION de:
#       - clase mayoritaria
#       - TF-IDF + Logistic Regression
#
# 3. Guarda los reportes por clase.
#
# 4. Guarda las matrices de confusión absoluta y normalizada.
#
# 5. Guarda la configuración utilizada por TF-IDF y
#    Logistic Regression.
#
# 6. Guarda físicamente el vectorizador TF-IDF y el modelo
#    Logistic Regression ya entrenados.
#
# 7. Comprueba que los archivos realmente existen.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Las variables de Colab viven en memoria temporal.
#
# Estos archivos convierten los resultados ejecutados de G5
# en artefactos reproducibles y auditables.
#
#
# IMPORTANTE:
#
# - NO se vuelve a entrenar ningún modelo.
#
# - NO se modifica TRAIN.
#
# - NO se modifica VALIDATION.
#
# - NO se transforma ni evalúa TEST.
#
# - Las métricas guardadas corresponden a VALIDATION.
#
# - G5 todavía NO queda aprobado automáticamente.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos perder:
#
# - resultados;
# - predicciones;
# - configuración;
# - modelo;
# - vectorizador
#
# si Colab reinicia la sesión.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import json
import joblib
import pandas as pd


# ------------------------------------------------------------
# 2. DEFINIR RUTAS DE LOS ARTEFACTOS
# ------------------------------------------------------------

G5_METRICS_PATH = (
    ARTIFACT_DIR / "g5_metricas_validation.csv"
)

G5_PREDICTIONS_PATH = (
    ARTIFACT_DIR / "g5_predicciones_validation.csv"
)

G5_MAJORITY_REPORT_PATH = (
    ARTIFACT_DIR / "g5_reporte_clase_mayoritaria.csv"
)

G5_LOGREG_REPORT_PATH = (
    ARTIFACT_DIR / "g5_reporte_logistic_regression.csv"
)

G5_MAJORITY_CM_PATH = (
    ARTIFACT_DIR / "g5_matriz_confusion_mayoritaria.csv"
)

G5_MAJORITY_CM_NORM_PATH = (
    ARTIFACT_DIR / "g5_matriz_confusion_mayoritaria_normalizada.csv"
)

G5_LOGREG_CM_PATH = (
    ARTIFACT_DIR / "g5_matriz_confusion_logreg.csv"
)

G5_LOGREG_CM_NORM_PATH = (
    ARTIFACT_DIR / "g5_matriz_confusion_logreg_normalizada.csv"
)

G5_CONFIG_PATH = (
    ARTIFACT_DIR / "g5_configuracion.json"
)

G5_TFIDF_PATH = (
    ARTIFACT_DIR / "g5_tfidf_vectorizer.joblib"
)

G5_LOGREG_MODEL_PATH = (
    ARTIFACT_DIR / "g5_logistic_regression.joblib"
)


# ------------------------------------------------------------
# 3. GUARDAR LA TABLA DE MÉTRICAS
# ------------------------------------------------------------

baseline_comparison.to_csv(
    G5_METRICS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. GUARDAR LAS PREDICCIONES DE VALIDATION
# ------------------------------------------------------------

# Conservamos review_id únicamente como identificador para
# poder auditar cada predicción.
#
# IMPORTANTE:
# review_id NO fue utilizado como predictor.

g5_predictions = pd.DataFrame({
    "review_id": val_g4["review_id"].values,
    "y_true": y_val,
    "pred_majority": y_val_pred_majority,
    "pred_logreg": y_val_pred_logreg
})

g5_predictions.to_csv(
    G5_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. GUARDAR LOS REPORTES POR CLASE
# ------------------------------------------------------------

majority_report_df.to_csv(
    G5_MAJORITY_REPORT_PATH,
    encoding="utf-8"
)

logreg_report_df.to_csv(
    G5_LOGREG_REPORT_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. GUARDAR LAS MATRICES DE CONFUSIÓN
# ------------------------------------------------------------

majority_cm_df.to_csv(
    G5_MAJORITY_CM_PATH,
    encoding="utf-8"
)

majority_cm_norm_df.to_csv(
    G5_MAJORITY_CM_NORM_PATH,
    encoding="utf-8"
)

logreg_cm_df.to_csv(
    G5_LOGREG_CM_PATH,
    encoding="utf-8"
)

logreg_cm_norm_df.to_csv(
    G5_LOGREG_CM_NORM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. GUARDAR LA CONFIGURACIÓN
# ------------------------------------------------------------

g5_config = {
    "stage": "G5",
    "seed": 42,

    "predictor": "text_sequential",

    "labels": {
        "0": "NEGATIVO",
        "1": "NEUTRAL",
        "2": "POSITIVO"
    },

    "primary_metric": "f1_macro",

    "tfidf": {
        "min_df": 2,
        "ngram_range": [1, 2],
        "sublinear_tf": True,
        "vocabulary_size": int(vocabulary_size),
        "fit_split": "train"
    },

    "logistic_regression": {
        "class_weight": "balanced",
        "random_state": 42,
        "max_iter": 1000,
        "fit_split": "train"
    },

    "evaluation_split": "validation",

    "test_used": False
}

with open(
    G5_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        g5_config,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 8. GUARDAR TF-IDF Y LOGISTIC REGRESSION
# ------------------------------------------------------------

joblib.dump(
    tfidf_vectorizer,
    G5_TFIDF_PATH
)

joblib.dump(
    logreg_baseline,
    G5_LOGREG_MODEL_PATH
)


# ------------------------------------------------------------
# 9. COMPROBAR LOS ARTEFACTOS
# ------------------------------------------------------------

g5_artifacts = {
    "Métricas":
        G5_METRICS_PATH,

    "Predicciones VALIDATION":
        G5_PREDICTIONS_PATH,

    "Reporte mayoritario":
        G5_MAJORITY_REPORT_PATH,

    "Reporte Logistic Regression":
        G5_LOGREG_REPORT_PATH,

    "Matriz mayoritaria":
        G5_MAJORITY_CM_PATH,

    "Matriz mayoritaria normalizada":
        G5_MAJORITY_CM_NORM_PATH,

    "Matriz Logistic Regression":
        G5_LOGREG_CM_PATH,

    "Matriz Logistic Regression normalizada":
        G5_LOGREG_CM_NORM_PATH,

    "Configuración":
        G5_CONFIG_PATH,

    "Vectorizador TF-IDF":
        G5_TFIDF_PATH,

    "Modelo Logistic Regression":
        G5_LOGREG_MODEL_PATH
}


print("=== ARTEFACTOS DE G5 ===")

all_g5_artifacts_exist = True

for name, path in g5_artifacts.items():

    exists = path.exists()

    all_g5_artifacts_exist &= exists

    print(
        f"{name}:",
        exists
    )


# ------------------------------------------------------------
# 10. RECUPERAR Y COMPROBAR LAS MÉTRICAS
# ------------------------------------------------------------

metrics_check = pd.read_csv(
    G5_METRICS_PATH
)

predictions_check = pd.read_csv(
    G5_PREDICTIONS_PATH
)


print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Filas tabla de métricas:",
    len(metrics_check)
)

print(
    "Modelos esperados:",
    2
)

print(
    "¿Coinciden?:",
    len(metrics_check) == 2
)

print()

print(
    "Predicciones recuperadas:",
    len(predictions_check)
)

print(
    "Filas VALIDATION:",
    len(y_val)
)

print(
    "¿Coinciden?:",
    len(predictions_check) == len(y_val)
)


# ------------------------------------------------------------
# 11. COMPROBAR IDENTIFICADORES
# ------------------------------------------------------------

print("\n=== CONTROL DE review_id ===")

print(
    "review_id duplicados en predicciones:",
    int(
        predictions_check["review_id"]
        .duplicated()
        .sum()
    )
)


# ------------------------------------------------------------
# 12. MOSTRAR LAS MÉTRICAS GUARDADAS
# ------------------------------------------------------------

print("\n=== MÉTRICAS PERSISTIDAS ===")

print(
    metrics_check
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST incluido en archivo de predicciones:",
    "NO"
)

print(
    "TEST transformado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 14. RESULTADO PARA AUDITORÍA
# ------------------------------------------------------------

g5_persistence_ok = (
    all_g5_artifacts_exist
    and len(metrics_check) == 2
    and len(predictions_check) == len(y_val)
    and predictions_check["review_id"].duplicated().sum() == 0
)


print("\n=== RESULTADO DE PERSISTENCIA G5 ===")

print(
    "¿Todos los controles de persistencia superados?:",
    g5_persistence_ok
)

print(
    "G5 listo para auditoría independiente:",
    "SÍ" if g5_persistence_ok else "NO"
)

print(
    "G5 aprobado:",
    "TODAVÍA NO"
)

print(
    "G6 iniciado:",
    "NO"
)

=== ARTEFACTOS DE G5 ===
Métricas: True
Predicciones VALIDATION: True
Reporte mayoritario: True
Reporte Logistic Regression: True
Matriz mayoritaria: True
Matriz mayoritaria normalizada: True
Matriz Logistic Regression: True
Matriz Logistic Regression normalizada: True
Configuración: True
Vectorizador TF-IDF: True
Modelo Logistic Regression: True

=== CONTROL DE RECUPERACIÓN ===
Filas tabla de métricas: 2
Modelos esperados: 2
¿Coinciden?: True

Predicciones recuperadas: 1956
Filas VALIDATION: 1956
¿Coinciden?: True

=== CONTROL DE review_id ===
review_id duplicados en predicciones: 0

=== MÉTRICAS PERSISTIDAS ===
                      modelo  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
           Clase mayoritaria    0.6748             0.3333           0.2249        0.3333    0.2686       0.5438
TF-IDF + Logistic Regression    0.7740             0.6773           0.6642        0.6773    0.6703       0.7753

=== CONTROL DEL TEST SELLADO ===
TEST incl

In [34]:
# ============================================================
# G6 — PREPARACIÓN DEL ENTORNO
# Corrección: comprobar e instalar Gensim si no está disponible
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba si la librería gensim está instalada
#    en la sesión ACTUAL de Google Colab.
#
# 2. Si gensim ya existe:
#       NO instala nada.
#
# 3. Si gensim no existe:
#       instala únicamente gensim.
#
# 4. Después comprueba la versión disponible.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Word2Vec se implementará mediante:
#
#       gensim.models.Word2Vec
#
# El error:
#
#       ModuleNotFoundError: No module named 'gensim'
#
# significa que la sesión actual de Colab no dispone
# de esa librería.
#
#
# IMPORTANTE:
#
# - NO reinstalamos PyTorch.
# - NO reinstalamos pandas.
# - NO reinstalamos scikit-learn.
# - NO entrenamos Word2Vec.
# - NO entrenamos BiLSTM.
# - NO modificamos el dataset.
# - NO utilizamos VALIDATION.
# - NO utilizamos TEST.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos modificar innecesariamente el entorno de Colab
# y provocar nuevos conflictos entre dependencias.


# ------------------------------------------------------------
# 1. COMPROBAR SI GENSIM ESTÁ DISPONIBLE
# ------------------------------------------------------------

import importlib.util

gensim_available = (
    importlib.util.find_spec("gensim")
    is not None
)

print("=== COMPROBACIÓN DE GENSIM ===")

print(
    "Gensim disponible antes de la comprobación:",
    gensim_available
)


# ------------------------------------------------------------
# 2. INSTALAR GENSIM SOLO SI FALTA
# ------------------------------------------------------------

if not gensim_available:

    print(
        "\nGensim no está instalado."
    )

    print(
        "Se instalará únicamente gensim."
    )

    %pip install -q "gensim==4.4.0"

else:

    print(
        "\nGensim ya está instalado."
    )

    print(
        "No se realizará ninguna instalación."
    )


# ------------------------------------------------------------
# 3. IMPORTAR GENSIM DESPUÉS DE LA COMPROBACIÓN
# ------------------------------------------------------------

import gensim

from gensim.models import Word2Vec


# ------------------------------------------------------------
# 4. MOSTRAR LA VERSIÓN REAL DISPONIBLE
# ------------------------------------------------------------

print("\n=== RESULTADO ===")

print(
    "Gensim importado correctamente:",
    True
)

print(
    "Versión de gensim:",
    gensim.__version__
)

print(
    "Word2Vec disponible:",
    Word2Vec is not None
)


# ------------------------------------------------------------
# 5. RECORDATORIO METODOLÓGICO
# ------------------------------------------------------------

print("\n=== ESTADO DEL EXPERIMENTO ===")

print(
    "Esta celda solo prepara una dependencia del entorno."
)

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "TEST utilizado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

=== COMPROBACIÓN DE GENSIM ===
Gensim disponible antes de la comprobación: False

Gensim no está instalado.
Se instalará únicamente gensim.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 69.3 MB/s eta 0:00:00

=== RESULTADO ===
Gensim importado correctamente: True
Versión de gensim: 4.4.0
Word2Vec disponible: True

=== ESTADO DEL EXPERIMENTO ===
Esta celda solo prepara una dependencia del entorno.
Word2Vec entrenado: NO
BiLSTM entrenada: NO
TEST utilizado: NO
TEST evaluado: NO


In [35]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 1: preparar exclusivamente TRAIN para Word2Vec
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba que estamos utilizando el TRAIN oficial
#    aprobado en G3 y procesado en G4.
#
# 2. Selecciona exclusivamente:
#
#       text_sequential
#
#    como fuente textual para Word2Vec.
#
# 3. Convierte cada reseña procesada en una lista de tokens.
#
# Ejemplo:
#
#       "food not good service"
#
# se convierte en:
#
#       ["food", "not", "good", "service"]
#
# 4. Comprueba que ninguna reseña de TRAIN quede sin tokens.
#
# 5. Calcula estadísticas descriptivas de longitud
#    exclusivamente sobre TRAIN.
#
# 6. Comprueba que VALIDATION y TEST NO forman parte
#    del corpus utilizado para entrenar Word2Vec.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Word2Vec aprende representaciones numéricas de palabras
# observando las palabras que aparecen cerca unas de otras.
#
# Ese aprendizaje debe realizarse exclusivamente con TRAIN.
#
# VALIDATION se reservará posteriormente para seleccionar
# la BiLSTM mediante F1 macro.
#
# TEST seguirá sellado hasta G8.
#
#
# IMPORTANTE:
#
# - NO entrenamos Word2Vec todavía.
#
# - NO construimos todavía la matriz de embeddings.
#
# - NO construimos todavía el vocabulario de la BiLSTM.
#
# - NO calculamos todavía los pesos de clase.
#
# - NO entrenamos la BiLSTM.
#
# - VALIDATION NO entra en el corpus Word2Vec.
#
# - TEST permanece completamente sellado.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Principalmente evitamos FUGA DE INFORMACIÓN.
#
# Si Word2Vec aprendiera palabras o contextos utilizando
# VALIDATION o TEST, habría recibido información de conjuntos
# reservados antes de la evaluación correspondiente.


# ------------------------------------------------------------
# 1. IMPORTAR LIBRERÍAS NECESARIAS
# ------------------------------------------------------------

import random
import numpy as np
import pandas as pd
import gensim

from gensim.models import Word2Vec


# ------------------------------------------------------------
# 2. FIJAR LA SEMILLA OFICIAL
# ------------------------------------------------------------

# Utilizamos la semilla 42 en el experimento siempre que
# la librería correspondiente permita establecerla.
#
# Aquí fijamos las semillas de Python y NumPy.
#
# La semilla específica de Word2Vec también se indicará
# posteriormente en su configuración.

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


print("=== CONFIGURACIÓN INICIAL G6 ===")

print(
    "Semilla:",
    SEED
)

print(
    "gensim:",
    gensim.__version__
)

print(
    "Modelo secuencial previsto:",
    "Word2Vec + BiLSTM"
)


# ------------------------------------------------------------
# 3. COMPROBAR EL TRAIN DE ENTRADA
# ------------------------------------------------------------

# train_g4 debe corresponder al TRAIN que:
#
# - procede del split aprobado en G3;
# - fue procesado en G4;
# - contiene text_sequential.
#
# Esta celda NO elimina ni añade filas.

print("\n=== ENTRADA A WORD2VEC ===")

print(
    "Filas TRAIN:",
    len(train_g4)
)

print(
    "text_sequential disponible:",
    "text_sequential" in train_g4.columns
)

print(
    "sentiment_id disponible:",
    "sentiment_id" in train_g4.columns
)


# ------------------------------------------------------------
# 4. UTILIZAR SOLO text_sequential
# ------------------------------------------------------------

# Word2Vec recibe únicamente el texto de la rama secuencial.
#
# No incorporamos:
#
# stars
# sentiment_id
# review_id
# business_id
# user_id
# date
# cool
# useful
# funny
#
# como información para aprender los embeddings.

train_text_for_w2v = (
    train_g4["text_sequential"]
    .astype(str)
)


# ------------------------------------------------------------
# 5. TOKENIZAR EL CORPUS DE TRAIN
# ------------------------------------------------------------

# El preprocesamiento lingüístico ya se realizó en G4.
#
# Por eso aquí NO:
#
# - eliminamos stopwords otra vez;
# - lematizamos otra vez;
# - eliminamos puntuación otra vez.
#
# Simplemente convertimos cada texto procesado en una
# lista de palabras utilizando espacios como separadores.

train_tokens_w2v = [
    text.split()
    for text in train_text_for_w2v
]


# ------------------------------------------------------------
# 6. COMPROBAR EL NÚMERO DE DOCUMENTOS
# ------------------------------------------------------------

# Debemos obtener exactamente una secuencia de tokens
# por cada reseña de TRAIN.

print("\n=== CORPUS TOKENIZADO ===")

print(
    "Reseñas en TRAIN:",
    len(train_g4)
)

print(
    "Secuencias tokenizadas:",
    len(train_tokens_w2v)
)

print(
    "¿Coinciden?:",
    len(train_g4) == len(train_tokens_w2v)
)


# ------------------------------------------------------------
# 7. COMPROBAR SECUENCIAS VACÍAS
# ------------------------------------------------------------

# Una secuencia vacía tendría esta forma:
#
# []
#
# Esto significaría que una reseña perdió todos sus tokens
# durante el preprocesamiento.
#
# En G4 ya comprobamos que no había textos procesados vacíos,
# pero repetimos aquí el control sobre el objeto que
# realmente recibirá Word2Vec.

empty_token_sequences = sum(
    len(tokens) == 0
    for tokens in train_tokens_w2v
)


print(
    "Secuencias vacías:",
    empty_token_sequences
)


# ------------------------------------------------------------
# 8. CALCULAR LONGITUDES EN TOKENS — SOLO TRAIN
# ------------------------------------------------------------

# Medimos cuántos tokens tiene cada reseña después del
# preprocesamiento secuencial.
#
# IMPORTANTE:
#
# Estas estadísticas NO cambian la longitud de las reseñas.
#
# Tampoco estamos decidiendo aquí el max_length.
#
# La configuración base de la BiLSTM utilizará posteriormente
# una longitud máxima de 300 tokens.

train_token_lengths = np.array([
    len(tokens)
    for tokens in train_tokens_w2v
])


print(
    "\n=== LONGITUD DE TRAIN DESPUÉS DEL PREPROCESAMIENTO ==="
)

print(
    "Promedio:",
    round(
        float(train_token_lengths.mean()),
        2
    )
)

print(
    "Mediana:",
    float(
        np.median(train_token_lengths)
    )
)

print(
    "P90:",
    float(
        np.percentile(
            train_token_lengths,
            90
        )
    )
)

print(
    "P95:",
    float(
        np.percentile(
            train_token_lengths,
            95
        )
    )
)

print(
    "Mínimo:",
    int(
        train_token_lengths.min()
    )
)

print(
    "Máximo:",
    int(
        train_token_lengths.max()
    )
)


# ------------------------------------------------------------
# 9. MOSTRAR UNA PEQUEÑA MUESTRA DE TOKENS
# ------------------------------------------------------------

# Mostramos únicamente las tres primeras reseñas de TRAIN.
#
# Esto nos permite comprobar visualmente que Word2Vec
# recibirá palabras ya procesadas por la rama secuencial.
#
# Limitamos la visualización a los primeros 30 tokens
# para que la salida sea legible.

print("\n=== EJEMPLO DE TOKENS DE TRAIN ===")

for i in range(3):

    print(
        f"\nReseña TRAIN {i + 1}:"
    )

    print(
        train_tokens_w2v[i][:30]
    )


# ------------------------------------------------------------
# 10. CONTROL DE ENTRADAS A WORD2VEC
# ------------------------------------------------------------

# Declaramos explícitamente qué columna se utilizó para
# construir train_tokens_w2v.

word2vec_input_columns = [
    "text_sequential"
]


# Estas columnas NO pueden alimentar Word2Vec.

forbidden_predictors = {
    "stars",
    "sentiment",
    "sentiment_id",
    "review_id",
    "business_id",
    "user_id",
    "date",
    "type",
    "cool",
    "useful",
    "funny"
}


# Calculamos la intersección entre las columnas utilizadas
# y las columnas prohibidas.
#
# El resultado esperado es un conjunto vacío.

forbidden_used = (
    set(word2vec_input_columns)
    & forbidden_predictors
)


print("\n=== CONTROL DE ENTRADAS A WORD2VEC ===")

print(
    "Columnas utilizadas:",
    word2vec_input_columns
)

print(
    "Variables prohibidas utilizadas:",
    sorted(forbidden_used)
)

print(
    "¿Word2Vec recibe únicamente texto?:",
    len(forbidden_used) == 0
)


# ------------------------------------------------------------
# 11. CONTROL DE AISLAMIENTO DE VALIDATION
# ------------------------------------------------------------

# VALIDATION existe y se utilizará posteriormente para
# evaluar y seleccionar la BiLSTM.
#
# Pero NO participa en el corpus utilizado para aprender
# los embeddings Word2Vec.
#
# IMPORTANTE:
# Estas líneas documentan el diseño.
# La evidencia principal está en que train_tokens_w2v
# fue construido exclusivamente desde train_g4.

print("\n=== CONTROL DE VALIDATION ===")

print(
    "Origen de train_tokens_w2v:",
    "train_g4['text_sequential']"
)

print(
    "VALIDATION concatenada con TRAIN:",
    False
)


# ------------------------------------------------------------
# 12. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

# TEST continúa reservado para G8.
#
# No debemos:
#
# - incorporarlo al vocabulario;
# - entrenar Word2Vec con él;
# - utilizarlo para hiperparámetros;
# - evaluarlo.
#
# En esta celda no se referencia ningún objeto de TEST.

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "Objeto de entrada a Word2Vec:",
    "train_tokens_w2v"
)

print(
    "TEST incorporado al corpus en esta celda:",
    False
)

print(
    "TEST evaluado en esta celda:",
    False
)


# ------------------------------------------------------------
# 13. REGISTRAR LA CONFIGURACIÓN DE WORD2VEC
# ------------------------------------------------------------

# Esta es la configuración base establecida para el caso.
#
# sg=1:
# utiliza Skip-gram.
#
# vector_size=100:
# cada palabra tendrá un vector de 100 dimensiones.
#
# window=5:
# Word2Vec observará un contexto de hasta 5 palabras.
#
# min_count=2:
# una palabra debe aparecer al menos 2 veces en TRAIN
# para entrar en el vocabulario Word2Vec.
#
# negative=10:
# utiliza 10 muestras negativas.
#
# epochs=10:
# Word2Vec recorrerá el corpus durante 10 épocas.
#
# seed=42:
# semilla oficial del experimento.
#
# IMPORTANTE:
# Aquí únicamente REGISTRAMOS la configuración.
# Todavía NO entrenamos Word2Vec.

W2V_CONFIG = {
    "sg": 1,
    "vector_size": 100,
    "window": 5,
    "min_count": 2,
    "negative": 10,
    "epochs": 10,
    "seed": 42
}


print("\n=== CONFIGURACIÓN PREVISTA DE WORD2VEC ===")

print(
    W2V_CONFIG
)


# ------------------------------------------------------------
# 14. VALIDACIÓN AUTOMÁTICA DEL PASO
# ------------------------------------------------------------

# No nos limitamos a imprimir mensajes.
# Construimos controles derivados de los objetos reales
# creados por esta celda.

g6_step1_ok = (
    len(train_g4) == len(train_tokens_w2v)
    and empty_token_sequences == 0
    and len(forbidden_used) == 0
    and word2vec_input_columns == ["text_sequential"]
    and W2V_CONFIG["sg"] == 1
    and W2V_CONFIG["vector_size"] == 100
    and W2V_CONFIG["window"] == 5
    and W2V_CONFIG["min_count"] == 2
    and W2V_CONFIG["negative"] == 10
    and W2V_CONFIG["epochs"] == 10
    and W2V_CONFIG["seed"] == 42
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 1 ===")

print(
    "¿Todos los controles verificables superados?:",
    g6_step1_ok
)


# ------------------------------------------------------------
# 15. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Corpus TRAIN preparado:",
    "SÍ" if g6_step1_ok else "REVISAR"
)

print(
    "Word2Vec entrenado:",
    "NO"
)

print(
    "Vocabulario BiLSTM construido:",
    "NO"
)

print(
    "Matriz de embeddings construida:",
    "NO"
)

print(
    "Pesos de clase calculados:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "NO"
)

=== CONFIGURACIÓN INICIAL G6 ===
Semilla: 42
gensim: 4.4.0
Modelo secuencial previsto: Word2Vec + BiLSTM

=== ENTRADA A WORD2VEC ===
Filas TRAIN: 5943
text_sequential disponible: True
sentiment_id disponible: True

=== CORPUS TOKENIZADO ===
Reseñas en TRAIN: 5943
Secuencias tokenizadas: 5943
¿Coinciden?: True
Secuencias vacías: 0

=== LONGITUD DE TRAIN DESPUÉS DEL PREPROCESAMIENTO ===
Promedio: 67.73
Mediana: 53.0
P90: 139.0
P95: 176.89999999999964
Mínimo: 1
Máximo: 530

=== EJEMPLO DE TOKENS DE TRAIN ===

Reseña TRAIN 1:
['wife', 'took', 'birthday', 'breakfast', 'excellent', 'weather', 'perfect', 'made', 'sitting', 'outside', 'overlooking', 'ground', 'absolute', 'pleasure', 'waitress', 'excellent', 'food', 'arrived', 'quickly', 'semi', 'busy', 'saturday', 'morning', 'looked', 'like', 'place', 'fill', 'pretty', 'quickly', 'earlier']

Reseña TRAIN 2:
['love', 'gyro', 'plate', 'rice', 'good', 'also', 'dig', 'candy', 'selection']

Reseña TRAIN 3:
['rosie', 'dakota', 'love', 'chaparral', '

In [36]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 2: entrenar Word2Vec exclusivamente con TRAIN
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea un modelo Word2Vec con la configuración oficial.
#
# 2. Construye su vocabulario utilizando ÚNICAMENTE:
#
#       train_tokens_w2v
#
# 3. Entrena Word2Vec durante 10 épocas.
#
# 4. Comprueba:
#
#       - tamaño del vocabulario;
#       - dimensión de los embeddings;
#       - configuración efectiva;
#       - número de documentos usados;
#
# 5. Verifica que Word2Vec pueda recuperar vectores
#    numéricos reales para palabras del vocabulario.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Word2Vec aprenderá un vector de 100 dimensiones para cada
# palabra suficientemente frecuente de TRAIN.
#
# Posteriormente esos vectores se integrarán REALMENTE como
# matriz de embeddings de la BiLSTM.
#
#
# IMPORTANTE:
#
# - Word2Vec se entrena SOLO con TRAIN.
#
# - VALIDATION NO entra en build_vocab().
#
# - VALIDATION NO entra en train().
#
# - TEST NO entra en build_vocab().
#
# - TEST NO entra en train().
#
# - sentiment_id NO se utiliza para entrenar Word2Vec.
#
# - Todavía NO construimos la matriz de embeddings BiLSTM.
#
# - Todavía NO entrenamos la BiLSTM.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos que Word2Vec aprenda vocabulario o contextos
# procedentes de VALIDATION o TEST.
#
# También usamos workers=1 para mejorar la reproducibilidad:
# Word2Vec puede presentar variaciones cuando entrena en
# múltiples hilos.


# ------------------------------------------------------------
# 1. FIJAR NUEVAMENTE LAS SEMILLAS
# ------------------------------------------------------------

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# 2. CREAR EL MODELO WORD2VEC
# ------------------------------------------------------------

# Utilizamos exactamente la configuración aprobada:
#
# sg=1              -> Skip-gram
# vector_size=100   -> vectores de 100 dimensiones
# window=5          -> ventana contextual
# min_count=2       -> palabra presente al menos 2 veces
# negative=10       -> negative sampling
# seed=42           -> reproducibilidad
#
# workers=1:
# limita el entrenamiento a un hilo para reducir
# variaciones no deterministas entre ejecuciones.

w2v_model = Word2Vec(
    vector_size=W2V_CONFIG["vector_size"],
    window=W2V_CONFIG["window"],
    min_count=W2V_CONFIG["min_count"],
    sg=W2V_CONFIG["sg"],
    negative=W2V_CONFIG["negative"],
    seed=W2V_CONFIG["seed"],
    workers=1
)


# ------------------------------------------------------------
# 3. CONSTRUIR EL VOCABULARIO SOLO CON TRAIN
# ------------------------------------------------------------

# build_vocab() aprende QUÉ palabras forman parte
# del vocabulario Word2Vec.
#
# La única fuente es train_tokens_w2v.

w2v_model.build_vocab(
    train_tokens_w2v
)


# ------------------------------------------------------------
# 4. REGISTRAR INFORMACIÓN ANTES DEL ENTRENAMIENTO
# ------------------------------------------------------------

print("=== WORD2VEC — VOCABULARIO CONSTRUIDO ===")

print(
    "Documentos utilizados:",
    w2v_model.corpus_count
)

print(
    "Documentos esperados de TRAIN:",
    len(train_tokens_w2v)
)

print(
    "¿Coinciden?:",
    w2v_model.corpus_count == len(train_tokens_w2v)
)

print(
    "Tamaño del vocabulario Word2Vec:",
    len(w2v_model.wv)
)


# ------------------------------------------------------------
# 5. ENTRENAR WORD2VEC
# ------------------------------------------------------------

# train() realiza el aprendizaje de los embeddings.
#
# total_examples:
# utiliza el número de documentos que Word2Vec registró
# durante build_vocab().
#
# epochs:
# exactamente 10, según la configuración del caso.

w2v_model.train(
    train_tokens_w2v,
    total_examples=w2v_model.corpus_count,
    epochs=W2V_CONFIG["epochs"]
)


# ------------------------------------------------------------
# 6. COMPROBAR LA CONFIGURACIÓN EFECTIVA
# ------------------------------------------------------------

print("\n=== CONFIGURACIÓN EFECTIVA DE WORD2VEC ===")

print(
    "Arquitectura:",
    "skip-gram" if w2v_model.sg == 1 else "CBOW"
)

print(
    "sg:",
    w2v_model.sg
)

print(
    "vector_size:",
    w2v_model.vector_size
)

print(
    "window:",
    w2v_model.window
)

print(
    "min_count:",
    w2v_model.min_count
)

print(
    "negative:",
    w2v_model.negative
)

print(
    "epochs ejecutadas:",
    W2V_CONFIG["epochs"]
)

print(
    "seed:",
    w2v_model.seed
)

print(
    "workers:",
    w2v_model.workers
)


# ------------------------------------------------------------
# 7. COMPROBAR QUE EXISTEN EMBEDDINGS REALES
# ------------------------------------------------------------

# Elegimos la primera palabra del vocabulario únicamente
# como prueba estructural.
#
# NO estamos evaluando calidad semántica todavía.

first_word = (
    w2v_model.wv.index_to_key[0]
)

first_vector = (
    w2v_model.wv[first_word]
)


print("\n=== CONTROL DE EMBEDDINGS ===")

print(
    "Palabra de prueba:",
    first_word
)

print(
    "Dimensión de su vector:",
    first_vector.shape
)

print(
    "Primeros 5 valores del vector:"
)

print(
    first_vector[:5]
)


# ------------------------------------------------------------
# 8. COMPROBAR LA MATRIZ COMPLETA DE WORD2VEC
# ------------------------------------------------------------

print("\n=== MATRIZ DE VECTORES WORD2VEC ===")

print(
    "Forma:",
    w2v_model.wv.vectors.shape
)

print(
    "Número de palabras:",
    w2v_model.wv.vectors.shape[0]
)

print(
    "Dimensiones por palabra:",
    w2v_model.wv.vectors.shape[1]
)


# ------------------------------------------------------------
# 9. CONTROL AUTOMÁTICO DE CONFIGURACIÓN
# ------------------------------------------------------------

w2v_config_ok = (
    w2v_model.sg == 1
    and w2v_model.vector_size == 100
    and w2v_model.window == 5
    and w2v_model.min_count == 2
    and w2v_model.negative == 10
    and W2V_CONFIG["epochs"] == 10
    and w2v_model.seed == 42
    and w2v_model.workers == 1
    and w2v_model.corpus_count == len(train_tokens_w2v)
    and w2v_model.wv.vectors.shape[1] == 100
)


print("\n=== VALIDACIÓN AUTOMÁTICA DE WORD2VEC ===")

print(
    "¿Configuración y corpus correctos?:",
    w2v_config_ok
)


# ------------------------------------------------------------
# 10. CONTROL DE VALIDATION
# ------------------------------------------------------------

# Estos mensajes documentan el flujo.
#
# La evidencia principal es que tanto build_vocab()
# como train() recibieron train_tokens_w2v.

print("\n=== CONTROL DE VALIDATION ===")

print(
    "Objeto usado en build_vocab():",
    "train_tokens_w2v"
)

print(
    "Objeto usado en train():",
    "train_tokens_w2v"
)

print(
    "VALIDATION utilizada para Word2Vec:",
    False
)


# ------------------------------------------------------------
# 11. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado para build_vocab():",
    False
)

print(
    "TEST utilizado para entrenar Word2Vec:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 12. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Corpus TRAIN preparado:",
    "SÍ"
)

print(
    "Word2Vec entrenado:",
    "SÍ" if w2v_config_ok else "REVISAR"
)

print(
    "Vocabulario Word2Vec aprendido solo de TRAIN:",
    "SÍ" if w2v_config_ok else "REVISAR"
)

print(
    "Matriz de embeddings BiLSTM construida:",
    "NO"
)

print(
    "Pesos de clase calculados:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "NO"
)

=== WORD2VEC — VOCABULARIO CONSTRUIDO ===
Documentos utilizados: 5943
Documentos esperados de TRAIN: 5943
¿Coinciden?: True
Tamaño del vocabulario Word2Vec: 11548

=== CONFIGURACIÓN EFECTIVA DE WORD2VEC ===
Arquitectura: skip-gram
sg: 1
vector_size: 100
window: 5
min_count: 2
negative: 10
epochs ejecutadas: 10
seed: 42
workers: 1

=== CONTROL DE EMBEDDINGS ===
Palabra de prueba: not
Dimensión de su vector: (100,)
Primeros 5 valores del vector:
[-0.04591847  0.05893326 -0.1672869   0.19271728 -0.23463692]

=== MATRIZ DE VECTORES WORD2VEC ===
Forma: (11548, 100)
Número de palabras: 11548
Dimensiones por palabra: 100

=== VALIDACIÓN AUTOMÁTICA DE WORD2VEC ===
¿Configuración y corpus correctos?: True

=== CONTROL DE VALIDATION ===
Objeto usado en build_vocab(): train_tokens_w2v
Objeto usado en train(): train_tokens_w2v
VALIDATION utilizada para Word2Vec: False

=== CONTROL DEL TEST SELLADO ===
TEST utilizado para build_vocab(): False
TEST utilizado para entrenar Word2Vec: False
TEST evalua

In [37]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 3: construir vocabulario secuencial y matriz de embeddings
#         exclusivamente a partir de TRAIN
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Construye un vocabulario para la BiLSTM usando SOLO TRAIN.
#
# 2. Reserva índices especiales:
#
#       PAD = 0
#       UNK = 1
#
# 3. Añade al vocabulario todas las palabras de TRAIN
#    que aparecen en Word2Vec.
#
# 4. Construye una matriz de embeddings de 100 dimensiones.
#
# 5. Copia en esa matriz los vectores aprendidos por Word2Vec.
#
# 6. Comprueba cuántas palabras tienen cobertura Word2Vec.
#
# 7. Verifica que VALIDATION y TEST no participan
#    en la construcción del vocabulario ni de los embeddings.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# La BiLSTM no puede trabajar directamente con palabras.
#
# Necesita que cada token se convierta en un índice entero.
#
# Ejemplo:
#
#       "food" -> 27
#       "not"  -> 58
#
# Posteriormente, esos índices consultarán una matriz
# de embeddings.
#
# Esa matriz debe contener los vectores aprendidos
# realmente por Word2Vec.
#
#
# IMPORTANTE:
#
# - El vocabulario se construye SOLO con TRAIN.
#
# - VALIDATION no añade palabras nuevas al vocabulario.
#
# - TEST no añade palabras nuevas al vocabulario.
#
# - Word2Vec ya fue entrenado SOLO con TRAIN.
#
# - Todavía NO entrenamos la BiLSTM.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos fuga de vocabulario.
#
# Si construyéramos el vocabulario usando VALIDATION o TEST,
# estaríamos incorporando información de conjuntos reservados.


# ------------------------------------------------------------
# 1. DEFINIR TOKENS ESPECIALES
# ------------------------------------------------------------

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1


# ------------------------------------------------------------
# 2. CONSTRUIR EL VOCABULARIO DESDE TRAIN
# ------------------------------------------------------------

# Empezamos con los tokens especiales.

word_to_idx = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}


# Añadimos únicamente palabras presentes en Word2Vec.
#
# Como Word2Vec fue construido con min_count=2,
# las palabras demasiado raras no aparecerán en w2v_model.wv.

for word in w2v_model.wv.index_to_key:

    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)


# ------------------------------------------------------------
# 3. CREAR EL VOCABULARIO INVERSO
# ------------------------------------------------------------

idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}


# ------------------------------------------------------------
# 4. MOSTRAR TAMAÑO DEL VOCABULARIO
# ------------------------------------------------------------

vocab_size = len(word_to_idx)

print("=== VOCABULARIO SECUENCIAL ===")

print(
    "Tamaño total del vocabulario:",
    vocab_size
)

print(
    "Tokens especiales incluidos:",
    [PAD_TOKEN, UNK_TOKEN]
)

print(
    "Índice PAD:",
    PAD_IDX
)

print(
    "Índice UNK:",
    UNK_IDX
)


# ------------------------------------------------------------
# 5. CREAR LA MATRIZ DE EMBEDDINGS
# ------------------------------------------------------------

EMBEDDING_DIM = 100

embedding_matrix = np.zeros(
    (vocab_size, EMBEDDING_DIM),
    dtype=np.float32
)


# ------------------------------------------------------------
# 6. INICIALIZAR EL VECTOR UNK
# ------------------------------------------------------------

# PAD queda como vector de ceros.
#
# UNK se inicializa con valores pequeños aleatorios.
#
# Usamos la semilla oficial.

rng = np.random.default_rng(SEED)

embedding_matrix[UNK_IDX] = rng.normal(
    loc=0.0,
    scale=0.05,
    size=EMBEDDING_DIM
).astype(np.float32)


# ------------------------------------------------------------
# 7. COPIAR LOS VECTORES WORD2VEC REALES
# ------------------------------------------------------------

covered_words = 0

for word, idx in word_to_idx.items():

    if word in {PAD_TOKEN, UNK_TOKEN}:
        continue

    if word in w2v_model.wv:

        embedding_matrix[idx] = (
            w2v_model.wv[word]
        )

        covered_words += 1


# ------------------------------------------------------------
# 8. CALCULAR COBERTURA DEL VOCABULARIO
# ------------------------------------------------------------

# Excluimos PAD y UNK del denominador.

real_vocab_words = (
    vocab_size - 2
)

coverage_ratio = (
    covered_words / real_vocab_words
    if real_vocab_words > 0
    else 0.0
)


print("\n=== COBERTURA WORD2VEC ===")

print(
    "Palabras cubiertas:",
    covered_words
)

print(
    "Palabras reales del vocabulario:",
    real_vocab_words
)

print(
    "Cobertura:",
    round(
        coverage_ratio * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 9. COMPROBAR LA MATRIZ
# ------------------------------------------------------------

print("\n=== MATRIZ DE EMBEDDINGS BiLSTM ===")

print(
    "Forma:",
    embedding_matrix.shape
)

print(
    "Dimensión esperada:",
    EMBEDDING_DIM
)

print(
    "Vector PAD es todo ceros:",
    bool(
        np.allclose(
            embedding_matrix[PAD_IDX],
            0.0
        )
    )
)

print(
    "Vector UNK tiene dimensión correcta:",
    embedding_matrix[UNK_IDX].shape
)


# ------------------------------------------------------------
# 10. VERIFICAR QUE UN VECTOR COINCIDE CON WORD2VEC
# ------------------------------------------------------------

# Elegimos una palabra real del vocabulario
# para demostrar que la matriz de embeddings
# contiene el vector aprendido por Word2Vec.

sample_word = w2v_model.wv.index_to_key[0]

sample_idx = word_to_idx[sample_word]

vector_matches = np.allclose(
    embedding_matrix[sample_idx],
    w2v_model.wv[sample_word]
)


print("\n=== INTEGRACIÓN REAL DE WORD2VEC ===")

print(
    "Palabra de prueba:",
    sample_word
)

print(
    "Índice en vocabulario BiLSTM:",
    sample_idx
)

print(
    "¿Vector de la matriz coincide con Word2Vec?:",
    vector_matches
)


# ------------------------------------------------------------
# 11. CONTROL DE VALIDATION Y TEST
# ------------------------------------------------------------

print("\n=== CONTROL DE AISLAMIENTO ===")

print(
    "Fuente del vocabulario:",
    "w2v_model.wv.index_to_key"
)

print(
    "Word2Vec fue entrenado solo con TRAIN:",
    True
)

print(
    "VALIDATION usada para ampliar vocabulario:",
    False
)

print(
    "TEST usado para ampliar vocabulario:",
    False
)


# ------------------------------------------------------------
# 12. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g6_step3_ok = (
    PAD_IDX == 0
    and UNK_IDX == 1
    and embedding_matrix.shape[0] == vocab_size
    and embedding_matrix.shape[1] == 100
    and np.allclose(
        embedding_matrix[PAD_IDX],
        0.0
    )
    and vector_matches
    and coverage_ratio == 1.0
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 3 ===")

print(
    "¿Controles superados?:",
    g6_step3_ok
)


# ------------------------------------------------------------
# 13. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Word2Vec entrenado:",
    "SÍ"
)

print(
    "Vocabulario BiLSTM construido:",
    "SÍ"
)

print(
    "Matriz de embeddings construida:",
    "SÍ" if g6_step3_ok else "REVISAR"
)

print(
    "Cobertura Word2Vec registrada:",
    "SÍ"
)

print(
    "Pesos de clase calculados:",
    "NO"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "NO"
)

=== VOCABULARIO SECUENCIAL ===
Tamaño total del vocabulario: 11550
Tokens especiales incluidos: ['<PAD>', '<UNK>']
Índice PAD: 0
Índice UNK: 1

=== COBERTURA WORD2VEC ===
Palabras cubiertas: 11548
Palabras reales del vocabulario: 11548
Cobertura: 100.0 %

=== MATRIZ DE EMBEDDINGS BiLSTM ===
Forma: (11550, 100)
Dimensión esperada: 100
Vector PAD es todo ceros: True
Vector UNK tiene dimensión correcta: (100,)

=== INTEGRACIÓN REAL DE WORD2VEC ===
Palabra de prueba: not
Índice en vocabulario BiLSTM: 2
¿Vector de la matriz coincide con Word2Vec?: True

=== CONTROL DE AISLAMIENTO ===
Fuente del vocabulario: w2v_model.wv.index_to_key
Word2Vec fue entrenado solo con TRAIN: True
VALIDATION usada para ampliar vocabulario: False
TEST usado para ampliar vocabulario: False

=== VALIDACIÓN AUTOMÁTICA DEL PASO 3 ===
¿Controles superados?: True

=== ESTADO DE G6 ===
Word2Vec entrenado: SÍ
Vocabulario BiLSTM construido: SÍ
Matriz de embeddings construida: SÍ
Cobertura Word2Vec registrada: SÍ
Pesos de 

In [38]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 4: crear secuencias numéricas y calcular pesos de clase
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Convierte los tokens de TRAIN y VALIDATION a índices.
#
# 2. Usa el vocabulario construido SOLO con TRAIN.
#
# 3. Toda palabra desconocida pasa a:
#
#       <UNK> -> índice 1
#
# 4. Aplica longitud máxima:
#
#       MAX_LEN = 300
#
# 5. Hace padding con:
#
#       <PAD> -> índice 0
#
# 6. Calcula los pesos de clase EXCLUSIVAMENTE con TRAIN.
#
# 7. Comprueba tamaños, clases y aislamiento de TEST.
#
#
# IMPORTANTE:
#
# - El vocabulario NO se amplía con VALIDATION.
#
# - TEST no se procesa.
#
# - Los pesos de clase se calculan SOLO con y_train.
#
# - Todavía NO se entrena la BiLSTM.


# ------------------------------------------------------------
# 1. DEFINIR LONGITUD MÁXIMA
# ------------------------------------------------------------

MAX_LEN = 300

print("=== CONFIGURACIÓN DE SECUENCIAS ===")
print("MAX_LEN:", MAX_LEN)
print("PAD_IDX:", PAD_IDX)
print("UNK_IDX:", UNK_IDX)


# ------------------------------------------------------------
# 2. FUNCIÓN PARA CONVERTIR TEXTO A ÍNDICES
# ------------------------------------------------------------

def text_to_indices(text, word_to_idx, max_len):

    tokens = str(text).split()

    indices = [
        word_to_idx.get(token, UNK_IDX)
        for token in tokens
    ]

    # truncamiento
    indices = indices[:max_len]

    # padding
    if len(indices) < max_len:
        indices = indices + [
            PAD_IDX
        ] * (max_len - len(indices))

    return indices


# ------------------------------------------------------------
# 3. CONVERTIR TRAIN
# ------------------------------------------------------------

X_train_seq = np.array([
    text_to_indices(
        text,
        word_to_idx,
        MAX_LEN
    )
    for text in train_g4["text_sequential"]
], dtype=np.int64)


# ------------------------------------------------------------
# 4. CONVERTIR VALIDATION
# ------------------------------------------------------------

X_val_seq = np.array([
    text_to_indices(
        text,
        word_to_idx,
        MAX_LEN
    )
    for text in val_g4["text_sequential"]
], dtype=np.int64)


# ------------------------------------------------------------
# 5. COMPROBAR FORMAS
# ------------------------------------------------------------

print("\n=== FORMAS DE LAS SECUENCIAS ===")

print(
    "TRAIN:",
    X_train_seq.shape
)

print(
    "VALIDATION:",
    X_val_seq.shape
)

print(
    "¿TRAIN tiene MAX_LEN columnas?:",
    X_train_seq.shape[1] == MAX_LEN
)

print(
    "¿VALIDATION tiene MAX_LEN columnas?:",
    X_val_seq.shape[1] == MAX_LEN
)


# ------------------------------------------------------------
# 6. COMPROBAR ÍNDICES VÁLIDOS
# ------------------------------------------------------------

max_index_train = int(
    X_train_seq.max()
)

max_index_val = int(
    X_val_seq.max()
)

print("\n=== CONTROL DE ÍNDICES ===")

print(
    "Máximo índice TRAIN:",
    max_index_train
)

print(
    "Máximo índice VALIDATION:",
    max_index_val
)

print(
    "Tamaño vocabulario:",
    vocab_size
)

print(
    "¿Índices TRAIN válidos?:",
    max_index_train < vocab_size
)

print(
    "¿Índices VALIDATION válidos?:",
    max_index_val < vocab_size
)


# ------------------------------------------------------------
# 7. CALCULAR TRUNCAMIENTO EN TRAIN
# ------------------------------------------------------------

train_truncated = int(
    sum(
        len(tokens) > MAX_LEN
        for tokens in train_tokens_w2v
    )
)

train_truncated_pct = (
    train_truncated
    / len(train_tokens_w2v)
    * 100
)


# ------------------------------------------------------------
# 8. TOKENIZAR VALIDATION SOLO PARA MEDIR LONGITUD
# ------------------------------------------------------------

# Esto NO construye vocabulario.
# Solo mide la longitud de text_sequential.

val_tokens_seq = [
    str(text).split()
    for text in val_g4["text_sequential"]
]

val_truncated = int(
    sum(
        len(tokens) > MAX_LEN
        for tokens in val_tokens_seq
    )
)

val_truncated_pct = (
    val_truncated
    / len(val_tokens_seq)
    * 100
)


print("\n=== TRUNCAMIENTO A 300 TOKENS ===")

print(
    "TRAIN truncadas:",
    train_truncated
)

print(
    "TRAIN porcentaje:",
    round(train_truncated_pct, 2),
    "%"
)

print(
    "VALIDATION truncadas:",
    val_truncated
)

print(
    "VALIDATION porcentaje:",
    round(val_truncated_pct, 2),
    "%"
)


# ------------------------------------------------------------
# 9. MEDIR USO DE UNk EN VALIDATION
# ------------------------------------------------------------

# Como el vocabulario viene SOLO de TRAIN,
# palabras nuevas en VALIDATION pasan a <UNK>.

val_nonpad_tokens = int(
    np.sum(X_val_seq != PAD_IDX)
)

val_unk_tokens = int(
    np.sum(X_val_seq == UNK_IDX)
)

val_unk_pct = (
    val_unk_tokens
    / val_nonpad_tokens
    * 100
    if val_nonpad_tokens > 0
    else 0.0
)


print("\n=== COBERTURA DEL VOCABULARIO EN VALIDATION ===")

print(
    "Tokens no PAD:",
    val_nonpad_tokens
)

print(
    "Tokens UNK:",
    val_unk_tokens
)

print(
    "Porcentaje UNK:",
    round(val_unk_pct, 2),
    "%"
)


# ------------------------------------------------------------
# 10. CALCULAR PESOS DE CLASE SOLO CON TRAIN
# ------------------------------------------------------------

from sklearn.utils.class_weight import compute_class_weight

classes = np.array(
    [0, 1, 2]
)

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)


print("\n=== PESOS DE CLASE — SOLO TRAIN ===")

for class_id, weight in zip(
    classes,
    class_weights_np
):
    print(
        f"Clase {class_id}:",
        round(float(weight), 6)
    )


# ------------------------------------------------------------
# 11. COMPROBAR DISTRIBUCIÓN QUE GENERÓ LOS PESOS
# ------------------------------------------------------------

train_class_counts_check = (
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)

print("\n=== DISTRIBUCIÓN USADA PARA LOS PESOS ===")

print(
    train_class_counts_check
)

print(
    "Origen de los pesos:",
    "y_train"
)


# ------------------------------------------------------------
# 12. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g6_step4_ok = (
    X_train_seq.shape
    == (len(train_g4), MAX_LEN)

    and X_val_seq.shape
    == (len(val_g4), MAX_LEN)

    and max_index_train < vocab_size

    and max_index_val < vocab_size

    and len(class_weights_np) == 3

    and np.all(
        class_weights_np > 0
    )
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 4 ===")

print(
    "¿Controles superados?:",
    g6_step4_ok
)


# ------------------------------------------------------------
# 13. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST convertido a secuencias:",
    False
)

print(
    "TEST usado para pesos de clase:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 14. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Secuencias TRAIN creadas:",
    "SÍ"
)

print(
    "Secuencias VALIDATION creadas:",
    "SÍ"
)

print(
    "Vocabulario construido solo con TRAIN:",
    "SÍ"
)

print(
    "Pesos de clase calculados solo con TRAIN:",
    "SÍ" if g6_step4_ok else "REVISAR"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "NO"
)


=== CONFIGURACIÓN DE SECUENCIAS ===
MAX_LEN: 300
PAD_IDX: 0
UNK_IDX: 1

=== FORMAS DE LAS SECUENCIAS ===
TRAIN: (5943, 300)
VALIDATION: (1956, 300)
¿TRAIN tiene MAX_LEN columnas?: True
¿VALIDATION tiene MAX_LEN columnas?: True

=== CONTROL DE ÍNDICES ===
Máximo índice TRAIN: 11549
Máximo índice VALIDATION: 11549
Tamaño vocabulario: 11550
¿Índices TRAIN válidos?: True
¿Índices VALIDATION válidos?: True

=== TRUNCAMIENTO A 300 TOKENS ===
TRAIN truncadas: 39
TRAIN porcentaje: 0.66 %
VALIDATION truncadas: 23
VALIDATION porcentaje: 1.18 %

=== COBERTURA DEL VOCABULARIO EN VALIDATION ===
Tokens no PAD: 138483
Tokens UNK: 5628
Porcentaje UNK: 4.06 %

=== PESOS DE CLASE — SOLO TRAIN ===
Clase 0: 1.994965
Clase 1: 2.338843
Clase 2: 0.482817

=== DISTRIBUCIÓN USADA PARA LOS PESOS ===
0     993
1     847
2    4103
Name: count, dtype: int64
Origen de los pesos: y_train

=== VALIDACIÓN AUTOMÁTICA DEL PASO 4 ===
¿Controles superados?: True

=== CONTROL DEL TEST SELLADO ===
TEST convertido a secuenci

In [39]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 5: preparar DataLoaders y construir la BiLSTM
#         utilizando realmente la matriz Word2Vec
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba si CUDA está disponible.
#
# 2. Convierte las secuencias y etiquetas de TRAIN y
#    VALIDATION a tensores de PyTorch.
#
# 3. Crea DataLoaders:
#
#       TRAIN      -> batch 64, shuffle=True
#       VALIDATION -> batch 64, shuffle=False
#
# 4. Define una BiLSTM REAL.
#
# 5. Inicializa su capa Embedding utilizando directamente
#    embedding_matrix, que construimos con Word2Vec.
#
# 6. Configura:
#
#       embedding = 100 dimensiones
#       BiLSTM bidireccional
#       hidden total = 128
#       dropout = 0.30
#       3 clases
#
# 7. Crea:
#
#       CrossEntropyLoss ponderada
#       Adam optimizer
#
# 8. Comprueba mediante código que la matriz de embeddings
#    del modelo coincide inicialmente con Word2Vec.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# El requisito no se cumple simplemente entrenando Word2Vec.
#
# Word2Vec debe integrarse REALMENTE dentro de la BiLSTM.
#
# Por eso la capa:
#
#       nn.Embedding
#
# se inicializa utilizando:
#
#       embedding_matrix
#
#
# IMPORTANTE:
#
# - Todavía NO entrenamos la BiLSTM.
#
# - Word2Vec ya fue entrenado SOLO con TRAIN.
#
# - El vocabulario fue construido SOLO con TRAIN.
#
# - Los pesos de clase fueron calculados SOLO con TRAIN.
#
# - VALIDATION se utilizará únicamente para evaluación
#   y early stopping.
#
# - TEST NO se convierte ni utiliza.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Comprobamos ANTES del entrenamiento que:
#
# - la arquitectura sea realmente bidireccional;
# - Word2Vec esté realmente conectado a la BiLSTM;
# - las dimensiones sean correctas;
# - la pérdida ponderada use los pesos de TRAIN;
# - TEST no participe.


# ------------------------------------------------------------
# 1. IMPORTAR PYTORCH
# ------------------------------------------------------------

import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


# ------------------------------------------------------------
# 2. FIJAR SEMILLAS
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 3. CONFIGURAR DISPOSITIVO
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=== DISPOSITIVO ===")

print(
    "Dispositivo:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "GPU:",
        "NO DISPONIBLE"
    )


# ------------------------------------------------------------
# 4. CONVERTIR TRAIN A TENSORES
# ------------------------------------------------------------

# X_train_seq contiene índices enteros.
#
# Para nn.Embedding necesitamos dtype=torch.long.
#
# Las etiquetas también deben ser long para
# CrossEntropyLoss.

X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)


# ------------------------------------------------------------
# 5. CONVERTIR VALIDATION A TENSORES
# ------------------------------------------------------------

X_val_tensor = torch.tensor(
    X_val_seq,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)


# ------------------------------------------------------------
# 6. COMPROBAR FORMAS DE LOS TENSORES
# ------------------------------------------------------------

print("\n=== TENSORES ===")

print(
    "X_train:",
    X_train_tensor.shape
)

print(
    "y_train:",
    y_train_tensor.shape
)

print(
    "X_validation:",
    X_val_tensor.shape
)

print(
    "y_validation:",
    y_val_tensor.shape
)


# ------------------------------------------------------------
# 7. CREAR DATASETS
# ------------------------------------------------------------

train_dataset_seq = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset_seq = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


# ------------------------------------------------------------
# 8. CREAR DATALOADERS
# ------------------------------------------------------------

BATCH_SIZE = 64


# Creamos un generador con semilla para que el shuffle
# de TRAIN sea reproducible.

train_generator = torch.Generator()

train_generator.manual_seed(
    SEED
)


train_loader_seq = DataLoader(
    train_dataset_seq,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator
)


val_loader_seq = DataLoader(
    val_dataset_seq,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("\n=== DATALOADERS ===")

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Batches TRAIN:",
    len(train_loader_seq)
)

print(
    "Batches VALIDATION:",
    len(val_loader_seq)
)


# ------------------------------------------------------------
# 9. DEFINIR LA ARQUITECTURA BiLSTM
# ------------------------------------------------------------

class Word2VecBiLSTM(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_size_per_direction=64,
        num_classes=3,
        dropout=0.30
    ):

        super().__init__()


        # ----------------------------------------------------
        # EMBEDDING
        # ----------------------------------------------------
        #
        # from_pretrained() inicializa la capa directamente
        # con nuestra matriz Word2Vec.
        #
        # freeze=False:
        # permite que estos embeddings puedan ajustarse
        # durante el entrenamiento supervisado.
        #
        # padding_idx=0:
        # corresponde a <PAD>.

        embedding_tensor = torch.tensor(
            embedding_matrix,
            dtype=torch.float32
        )

        self.embedding = nn.Embedding.from_pretrained(
            embeddings=embedding_tensor,
            freeze=False,
            padding_idx=PAD_IDX
        )


        # ----------------------------------------------------
        # BiLSTM
        # ----------------------------------------------------
        #
        # hidden_size=64 por dirección.
        #
        # Como bidirectional=True:
        #
        #       64 hacia adelante
        #     + 64 hacia atrás
        #     -------------------
        #       128 total
        #
        # Esto satisface hidden total 128.

        self.lstm = nn.LSTM(
            input_size=EMBEDDING_DIM,
            hidden_size=hidden_size_per_direction,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )


        # ----------------------------------------------------
        # DROPOUT
        # ----------------------------------------------------

        self.dropout = nn.Dropout(
            dropout
        )


        # ----------------------------------------------------
        # CAPA FINAL DE CLASIFICACIÓN
        # ----------------------------------------------------
        #
        # Entrada:
        #
        # 64 * 2 = 128 características
        #
        # Salida:
        #
        # 3 logits:
        #
        # 0 = NEGATIVO
        # 1 = NEUTRAL
        # 2 = POSITIVO

        self.classifier = nn.Linear(
            hidden_size_per_direction * 2,
            num_classes
        )


    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    def forward(self, input_ids):

        # ----------------------------------------------------
        # 1. OBTENER EMBEDDINGS
        # ----------------------------------------------------

        embedded = self.embedding(
            input_ids
        )


        # ----------------------------------------------------
        # 2. PASAR POR LA BiLSTM
        # ----------------------------------------------------
        #
        # h_n tiene forma:
        #
        # [num_directions, batch, hidden_size]
        #
        # Como tenemos una sola capa bidireccional:
        #
        # h_n[-2] = estado final forward
        # h_n[-1] = estado final backward

        _, (h_n, _) = self.lstm(
            embedded
        )


        # ----------------------------------------------------
        # 3. CONCATENAR AMBAS DIRECCIONES
        # ----------------------------------------------------

        forward_hidden = h_n[-2]

        backward_hidden = h_n[-1]

        combined_hidden = torch.cat(
            (
                forward_hidden,
                backward_hidden
            ),
            dim=1
        )


        # ----------------------------------------------------
        # 4. DROPOUT
        # ----------------------------------------------------

        combined_hidden = self.dropout(
            combined_hidden
        )


        # ----------------------------------------------------
        # 5. CLASIFICACIÓN
        # ----------------------------------------------------

        logits = self.classifier(
            combined_hidden
        )

        return logits


# ------------------------------------------------------------
# 10. CREAR EL MODELO
# ------------------------------------------------------------

HIDDEN_PER_DIRECTION = 64

DROPOUT = 0.30

NUM_CLASSES = 3


bilstm_model = Word2VecBiLSTM(
    embedding_matrix=embedding_matrix,
    hidden_size_per_direction=HIDDEN_PER_DIRECTION,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)


bilstm_model = bilstm_model.to(
    device
)


# ------------------------------------------------------------
# 11. CREAR LOS PESOS DE CLASE PARA PYTORCH
# ------------------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=device
)


# ------------------------------------------------------------
# 12. CREAR CrossEntropyLoss PONDERADA
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# 13. CREAR OPTIMIZADOR
# ------------------------------------------------------------

LEARNING_RATE = 1e-3


optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=LEARNING_RATE
)


# ------------------------------------------------------------
# 14. MOSTRAR CONFIGURACIÓN DE LA BiLSTM
# ------------------------------------------------------------

print("\n=== CONFIGURACIÓN BiLSTM ===")

print(
    "Embedding dimension:",
    EMBEDDING_DIM
)

print(
    "Hidden por dirección:",
    HIDDEN_PER_DIRECTION
)

print(
    "Hidden total:",
    HIDDEN_PER_DIRECTION * 2
)

print(
    "Bidireccional:",
    bilstm_model.lstm.bidirectional
)

print(
    "Dropout:",
    bilstm_model.dropout.p
)

print(
    "Número de clases:",
    NUM_CLASSES
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Loss:",
    type(criterion).__name__
)

print(
    "Pesos de clase en loss:",
    class_weights_tensor.detach().cpu().numpy()
)


# ------------------------------------------------------------
# 15. COMPROBAR LA INTEGRACIÓN REAL DE WORD2VEC
# ------------------------------------------------------------

# Antes del entrenamiento, elegimos otra vez una palabra
# conocida y verificamos que el vector dentro de nn.Embedding
# coincide con la matriz Word2Vec construida.
#
# IMPORTANTE:
# hacemos esta comprobación ANTES de optimizer.step(),
# porque después los embeddings podrían empezar a cambiar.

sample_word_bilstm = (
    w2v_model.wv.index_to_key[0]
)

sample_idx_bilstm = (
    word_to_idx[sample_word_bilstm]
)


embedding_from_model = (
    bilstm_model
    .embedding
    .weight[
        sample_idx_bilstm
    ]
    .detach()
    .cpu()
    .numpy()
)


embedding_from_matrix = (
    embedding_matrix[
        sample_idx_bilstm
    ]
)


embedding_integration_ok = np.allclose(
    embedding_from_model,
    embedding_from_matrix
)


print("\n=== INTEGRACIÓN Word2Vec -> BiLSTM ===")

print(
    "Palabra de prueba:",
    sample_word_bilstm
)

print(
    "Índice:",
    sample_idx_bilstm
)

print(
    "¿Embedding del modelo coincide con Word2Vec?:",
    embedding_integration_ok
)


# ------------------------------------------------------------
# 16. HACER UN FORWARD DE PRUEBA SIN ENTRENAR
# ------------------------------------------------------------

# Tomamos un único batch de TRAIN.
#
# model.eval():
# desactiva temporalmente el dropout.
#
# torch.no_grad():
# evita calcular gradientes.
#
# Esto NO entrena el modelo.
#
# Solo comprobamos que produce 3 logits por reseña.

bilstm_model.eval()

sample_batch_X, sample_batch_y = next(
    iter(train_loader_seq)
)

sample_batch_X = sample_batch_X.to(
    device
)


with torch.no_grad():

    sample_logits = bilstm_model(
        sample_batch_X
    )


print("\n=== FORWARD DE PRUEBA ===")

print(
    "Forma del batch de entrada:",
    sample_batch_X.shape
)

print(
    "Forma de logits:",
    sample_logits.shape
)

print(
    "¿Hay 3 logits por reseña?:",
    sample_logits.shape[1] == 3
)


# ------------------------------------------------------------
# 17. CONTAR PARÁMETROS
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in bilstm_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in bilstm_model.parameters()
    if p.requires_grad
)


print("\n=== PARÁMETROS DEL MODELO ===")

print(
    "Parámetros totales:",
    total_parameters
)

print(
    "Parámetros entrenables:",
    trainable_parameters
)


# ------------------------------------------------------------
# 18. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g6_step5_ok = (
    X_train_tensor.shape
    == (len(train_g4), MAX_LEN)

    and X_val_tensor.shape
    == (len(val_g4), MAX_LEN)

    and BATCH_SIZE == 64

    and bilstm_model.embedding.embedding_dim == 100

    and bilstm_model.lstm.bidirectional is True

    and HIDDEN_PER_DIRECTION * 2 == 128

    and abs(
        bilstm_model.dropout.p - 0.30
    ) < 1e-9

    and LEARNING_RATE == 1e-3

    and isinstance(
        criterion,
        nn.CrossEntropyLoss
    )

    and embedding_integration_ok

    and sample_logits.shape[1] == 3
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 5 ===")

print(
    "¿Controles superados?:",
    g6_step5_ok
)


# ------------------------------------------------------------
# 19. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST convertido a TensorDataset:",
    False
)

print(
    "TEST convertido a DataLoader:",
    False
)

print(
    "TEST usado en forward:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 20. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Word2Vec entrenado:",
    "SÍ"
)

print(
    "Matriz Word2Vec integrada en BiLSTM:",
    "SÍ" if embedding_integration_ok else "REVISAR"
)

print(
    "DataLoaders TRAIN/VALIDATION creados:",
    "SÍ"
)

print(
    "CrossEntropyLoss ponderada creada:",
    "SÍ"
)

print(
    "BiLSTM construida:",
    "SÍ"
)

print(
    "BiLSTM entrenada:",
    "NO"
)

print(
    "Early stopping ejecutado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "NO"
)

=== DISPOSITIVO ===
Dispositivo: cuda
GPU: Tesla T4

=== TENSORES ===
X_train: torch.Size([5943, 300])
y_train: torch.Size([5943])
X_validation: torch.Size([1956, 300])
y_validation: torch.Size([1956])

=== DATALOADERS ===
Batch size: 64
Batches TRAIN: 93
Batches VALIDATION: 31

=== CONFIGURACIÓN BiLSTM ===
Embedding dimension: 100
Hidden por dirección: 64
Hidden total: 128
Bidireccional: True
Dropout: 0.3
Número de clases: 3
Learning rate: 0.001
Loss: CrossEntropyLoss
Pesos de clase en loss: [1.9949647  2.3388429  0.48281744]

=== INTEGRACIÓN Word2Vec -> BiLSTM ===
Palabra de prueba: not
Índice: 2
¿Embedding del modelo coincide con Word2Vec?: True

=== FORWARD DE PRUEBA ===
Forma del batch de entrada: torch.Size([64, 300])
Forma de logits: torch.Size([64, 3])
¿Hay 3 logits por reseña?: True

=== PARÁMETROS DEL MODELO ===
Parámetros totales: 1240379
Parámetros entrenables: 1240379

=== VALIDACIÓN AUTOMÁTICA DEL PASO 5 ===
¿Controles superados?: True

=== CONTROL DEL TEST SELLADO ===
TE

In [40]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 6: entrenar la BiLSTM con early stopping y F1 macro
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Entrena la BiLSTM utilizando SOLO TRAIN.
#
# 2. Después de cada época evalúa sobre VALIDATION.
#
# 3. Calcula F1 macro de VALIDATION.
#
# 4. Aplica gradient clipping para evitar gradientes
#    excesivamente grandes.
#
# 5. Guarda en memoria el mejor estado del modelo según:
#
#       F1 MACRO DE VALIDATION
#
# 6. Aplica early stopping con:
#
#       máximo 10 épocas
#       paciencia = 2
#
# 7. Al terminar, restaura automáticamente el mejor modelo.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# TRAIN es el único conjunto que modifica los pesos.
#
# VALIDATION sirve únicamente para:
#
# - medir rendimiento durante desarrollo;
# - seleccionar la mejor época;
# - activar early stopping.
#
# TEST sigue completamente sellado.
#
#
# IMPORTANTE:
#
# - La métrica de selección es F1 macro.
#
# - Accuracy NO selecciona el checkpoint.
#
# - TEST NO participa.
#
# - Los pesos de clase proceden únicamente de TRAIN.
#
# - Word2Vec ya fue entrenado únicamente con TRAIN.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# - Sobreentrenamiento.
# - Selección mediante una métrica inadecuada.
# - Gradientes demasiado grandes.
# - Contaminación del TEST.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import copy
import time
import numpy as np
import torch

from sklearn.metrics import f1_score


# ------------------------------------------------------------
# 2. CONFIGURACIÓN DE ENTRENAMIENTO
# ------------------------------------------------------------

MAX_EPOCHS = 10

PATIENCE = 2

GRAD_CLIP = 1.0


print("=== CONFIGURACIÓN DE ENTRENAMIENTO ===")

print(
    "Máximo de épocas:",
    MAX_EPOCHS
)

print(
    "Paciencia early stopping:",
    PATIENCE
)

print(
    "Gradient clipping:",
    GRAD_CLIP
)

print(
    "Métrica de selección:",
    "F1 macro de VALIDATION"
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Batch size:",
    BATCH_SIZE
)


# ------------------------------------------------------------
# 3. FUNCIÓN DE ENTRENAMIENTO DE UNA ÉPOCA
# ------------------------------------------------------------

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    grad_clip
):

    # Activamos modo entrenamiento.
    #
    # Esto activa dropout y cálculo de gradientes.

    model.train()

    total_loss = 0.0

    total_examples = 0


    for batch_X, batch_y in loader:

        # Movemos el batch al dispositivo.
        batch_X = batch_X.to(
            device
        )

        batch_y = batch_y.to(
            device
        )


        # Eliminamos gradientes del paso anterior.
        optimizer.zero_grad()


        # Forward.
        logits = model(
            batch_X
        )


        # Calcular pérdida.
        loss = criterion(
            logits,
            batch_y
        )


        # Backpropagation.
        loss.backward()


        # ----------------------------------------------------
        # GRADIENT CLIPPING
        # ----------------------------------------------------
        #
        # Limita la norma de los gradientes.
        #
        # Esto resulta especialmente útil en redes recurrentes.

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=grad_clip
        )


        # Actualizar parámetros.
        optimizer.step()


        # Acumulamos la pérdida ponderada por batch.
        batch_size_current = (
            batch_y.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size_current
        )

        total_examples += (
            batch_size_current
        )


    epoch_loss = (
        total_loss
        / total_examples
    )

    return epoch_loss


# ------------------------------------------------------------
# 4. FUNCIÓN DE EVALUACIÓN EN VALIDATION
# ------------------------------------------------------------

def evaluate_bilstm(
    model,
    loader,
    criterion,
    device
):

    # Modo evaluación:
    # desactiva dropout.

    model.eval()


    total_loss = 0.0

    total_examples = 0

    all_true = []

    all_pred = []


    # No necesitamos gradientes durante validation.
    with torch.no_grad():

        for batch_X, batch_y in loader:

            batch_X = batch_X.to(
                device
            )

            batch_y = batch_y.to(
                device
            )


            logits = model(
                batch_X
            )


            loss = criterion(
                logits,
                batch_y
            )


            predictions = torch.argmax(
                logits,
                dim=1
            )


            batch_size_current = (
                batch_y.size(0)
            )


            total_loss += (
                loss.item()
                * batch_size_current
            )

            total_examples += (
                batch_size_current
            )


            all_true.extend(
                batch_y
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            all_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


    validation_loss = (
        total_loss
        / total_examples
    )


    validation_f1_macro = f1_score(
        all_true,
        all_pred,
        average="macro",
        labels=[0, 1, 2],
        zero_division=0
    )


    return (
        validation_loss,
        validation_f1_macro,
        np.array(all_true),
        np.array(all_pred)
    )


# ------------------------------------------------------------
# 5. PREPARAR EARLY STOPPING
# ------------------------------------------------------------

best_val_f1 = -np.inf

best_epoch = None

best_model_state = None

epochs_without_improvement = 0


training_history = []


# ------------------------------------------------------------
# 6. REGISTRAR TIEMPO
# ------------------------------------------------------------

training_start_time = time.time()


print("\n=== ENTRENAMIENTO Word2Vec + BiLSTM ===")


# ------------------------------------------------------------
# 7. BUCLE DE ENTRENAMIENTO
# ------------------------------------------------------------

for epoch in range(
    1,
    MAX_EPOCHS + 1
):


    # --------------------------------------------------------
    # 7.1. ENTRENAR CON TRAIN
    # --------------------------------------------------------

    train_loss = train_one_epoch(
        model=bilstm_model,
        loader=train_loader_seq,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        grad_clip=GRAD_CLIP
    )


    # --------------------------------------------------------
    # 7.2. EVALUAR CON VALIDATION
    # --------------------------------------------------------

    (
        val_loss,
        val_f1_macro,
        _,
        _
    ) = evaluate_bilstm(
        model=bilstm_model,
        loader=val_loader_seq,
        criterion=criterion,
        device=device
    )


    # --------------------------------------------------------
    # 7.3. REGISTRAR LA ÉPOCA
    # --------------------------------------------------------

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro
    }


    training_history.append(
        epoch_record
    )


    print(
        f"Época {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f}"
    )


    # --------------------------------------------------------
    # 7.4. COMPROBAR SI ES EL MEJOR MODELO
    # --------------------------------------------------------

    if val_f1_macro > best_val_f1:

        best_val_f1 = val_f1_macro

        best_epoch = epoch


        # Guardamos una COPIA del estado actual.
        best_model_state = copy.deepcopy(
            bilstm_model.state_dict()
        )


        epochs_without_improvement = 0


        print(
            "  -> Nuevo mejor checkpoint "
            "por F1 macro de VALIDATION."
        )


    else:

        epochs_without_improvement += 1


        print(
            "  -> Sin mejora. "
            f"Paciencia: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # --------------------------------------------------------
    # 7.5. EARLY STOPPING
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            "\nEarly stopping activado."
        )

        break


# ------------------------------------------------------------
# 8. CALCULAR TIEMPO TOTAL
# ------------------------------------------------------------

training_end_time = time.time()

bilstm_training_seconds = (
    training_end_time
    - training_start_time
)


# ------------------------------------------------------------
# 9. RESTAURAR EL MEJOR CHECKPOINT
# ------------------------------------------------------------

# El modelo que conservamos no es necesariamente
# el de la última época.
#
# Restauramos el que obtuvo mayor F1 macro en VALIDATION.

assert best_model_state is not None, (
    "No se guardó ningún checkpoint."
)


bilstm_model.load_state_dict(
    best_model_state
)


# ------------------------------------------------------------
# 10. CONVERTIR HISTORIAL A DATAFRAME
# ------------------------------------------------------------

training_history_df = pd.DataFrame(
    training_history
)


# ------------------------------------------------------------
# 11. MOSTRAR HISTORIAL COMPLETO
# ------------------------------------------------------------

print("\n=== HISTORIAL DE ENTRENAMIENTO ===")

print(
    training_history_df
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. MOSTRAR EL MEJOR CHECKPOINT
# ------------------------------------------------------------

print("\n=== MEJOR CHECKPOINT ===")

print(
    "Mejor época:",
    best_epoch
)

print(
    "Mejor F1 macro de VALIDATION:",
    round(
        float(best_val_f1),
        4
    )
)

print(
    "Épocas ejecutadas:",
    len(training_history_df)
)

print(
    "Máximo permitido:",
    MAX_EPOCHS
)

print(
    "Paciencia:",
    PATIENCE
)


# ------------------------------------------------------------
# 13. MOSTRAR TIEMPO DE ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== TIEMPO ===")

print(
    "Tiempo total de entrenamiento (segundos):",
    round(
        bilstm_training_seconds,
        2
    )
)


# ------------------------------------------------------------
# 14. COMPROBAR QUE EL MEJOR MODELO QUEDÓ RESTAURADO
# ------------------------------------------------------------

(
    best_val_loss_check,
    best_val_f1_check,
    y_val_bilstm_true,
    y_val_bilstm_pred
) = evaluate_bilstm(
    model=bilstm_model,
    loader=val_loader_seq,
    criterion=criterion,
    device=device
)


print("\n=== COMPROBACIÓN DEL MODELO RESTAURADO ===")

print(
    "F1 macro restaurado:",
    round(
        float(best_val_f1_check),
        4
    )
)

print(
    "F1 macro registrado como mejor:",
    round(
        float(best_val_f1),
        4
    )
)

print(
    "¿Coinciden?:",
    np.isclose(
        best_val_f1_check,
        best_val_f1
    )
)


# ------------------------------------------------------------
# 15. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g6_step6_ok = (
    best_epoch is not None

    and best_model_state is not None

    and len(training_history_df)
        <= MAX_EPOCHS

    and np.isclose(
        best_val_f1_check,
        best_val_f1
    )

    and best_epoch
        <= len(training_history_df)

    and PATIENCE == 2

    and GRAD_CLIP == 1.0
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 6 ===")

print(
    "¿Controles superados?:",
    g6_step6_ok
)


# ------------------------------------------------------------
# 16. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado durante entrenamiento:",
    False
)

print(
    "TEST utilizado para early stopping:",
    False
)

print(
    "TEST utilizado para seleccionar checkpoint:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 17. ESTADO DE G6
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Word2Vec entrenado:",
    "SÍ"
)

print(
    "BiLSTM entrenada:",
    "SÍ" if g6_step6_ok else "REVISAR"
)

print(
    "Gradient clipping aplicado:",
    "SÍ"
)

print(
    "Early stopping aplicado:",
    "SÍ"
)

print(
    "Selección por F1 macro de VALIDATION:",
    "SÍ"
)

print(
    "Mejor checkpoint restaurado:",
    "SÍ" if g6_step6_ok else "REVISAR"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 finalizado:",
    "TODAVÍA NO"
)

=== CONFIGURACIÓN DE ENTRENAMIENTO ===
Máximo de épocas: 10
Paciencia early stopping: 2
Gradient clipping: 1.0
Métrica de selección: F1 macro de VALIDATION
Learning rate: 0.001
Batch size: 64

=== ENTRENAMIENTO Word2Vec + BiLSTM ===
Época 01 | train_loss=1.0529 | val_loss=0.9609 | val_f1_macro=0.5201
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 02 | train_loss=0.8716 | val_loss=0.8423 | val_f1_macro=0.5639
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 03 | train_loss=0.7493 | val_loss=0.8589 | val_f1_macro=0.6291
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 04 | train_loss=0.6736 | val_loss=0.8298 | val_f1_macro=0.5845
  -> Sin mejora. Paciencia: 1/2
Época 05 | train_loss=0.5511 | val_loss=0.8418 | val_f1_macro=0.5824
  -> Sin mejora. Paciencia: 2/2

Early stopping activado.

=== HISTORIAL DE ENTRENAMIENTO ===
 epoch  train_loss  val_loss  val_f1_macro
     1      1.0529    0.9609        0.5201
     2      0.8716    0.8423        0.5639

In [41]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 7: evaluar el mejor checkpoint sobre VALIDATION
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Utiliza el mejor checkpoint restaurado de la BiLSTM.
#
# 2. Genera predicciones sobre VALIDATION.
#
# 3. Evalúa esas predicciones con EXACTAMENTE la misma
#    función común utilizada en G5:
#
#       evaluate_predictions()
#
# 4. Calcula:
#
#       - accuracy
#       - balanced accuracy
#       - precision macro
#       - recall macro
#       - F1 macro
#       - F1 weighted
#
# 5. Genera:
#
#       - reporte por clase
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 6. Compara Word2Vec + BiLSTM contra los baselines
#    únicamente en VALIDATION.
#
#
# IMPORTANTE:
#
# - NO entrenamos de nuevo la BiLSTM.
# - NO tocamos TEST.
# - NO seleccionamos modelo usando TEST.
# - La comparación sigue gobernada por F1 macro.


# ------------------------------------------------------------
# 1. GENERAR PREDICCIONES DEL CHECKPOINT RESTAURADO
# ------------------------------------------------------------

(
    bilstm_val_loss,
    bilstm_val_f1_macro_check,
    y_val_bilstm_true,
    y_val_bilstm_pred
) = evaluate_bilstm(
    model=bilstm_model,
    loader=val_loader_seq,
    criterion=criterion,
    device=device
)


# ------------------------------------------------------------
# 2. COMPROBAR TAMAÑOS
# ------------------------------------------------------------

print("=== PREDICCIONES BiLSTM — VALIDATION ===")

print(
    "Etiquetas reales:",
    len(y_val_bilstm_true)
)

print(
    "Predicciones:",
    len(y_val_bilstm_pred)
)

print(
    "¿Coinciden?:",
    len(y_val_bilstm_true)
    == len(y_val_bilstm_pred)
    == len(y_val)
)


# ------------------------------------------------------------
# 3. EVALUAR CON LA FUNCIÓN COMÚN
# ------------------------------------------------------------

(
    bilstm_metrics,
    bilstm_report,
    bilstm_cm,
    bilstm_cm_norm
) = evaluate_predictions(
    y_val_bilstm_true,
    y_val_bilstm_pred
)


# ------------------------------------------------------------
# 4. MOSTRAR MÉTRICAS GENERALES
# ------------------------------------------------------------

print(
    "\n=== Word2Vec + BiLSTM — MÉTRICAS EN VALIDATION ==="
)

for metric_name, metric_value in bilstm_metrics.items():

    print(
        f"{metric_name}: "
        f"{metric_value:.4f}"
    )


# ------------------------------------------------------------
# 5. COMPROBAR CONSISTENCIA DE F1 MACRO
# ------------------------------------------------------------

print(
    "\n=== CONSISTENCIA DEL CHECKPOINT ==="
)

print(
    "F1 macro obtenido durante early stopping:",
    round(
        float(best_val_f1),
        4
    )
)

print(
    "F1 macro recalculado con función común:",
    round(
        float(bilstm_metrics["f1_macro"]),
        4
    )
)

print(
    "¿Coinciden?:",
    np.isclose(
        best_val_f1,
        bilstm_metrics["f1_macro"]
    )
)


# ------------------------------------------------------------
# 6. REPORTE POR CLASE
# ------------------------------------------------------------

bilstm_report_df = (
    pd.DataFrame(bilstm_report)
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — VALIDATION ==="
)

print(
    bilstm_report_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 7. MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

bilstm_cm_df = pd.DataFrame(
    bilstm_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA ==="
)

print(
    bilstm_cm_df.to_string()
)


# ------------------------------------------------------------
# 8. MATRIZ DE CONFUSIÓN NORMALIZADA
# ------------------------------------------------------------

bilstm_cm_norm_df = pd.DataFrame(
    bilstm_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA ==="
)

print(
    bilstm_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. COMPARACIÓN CON LOS BASELINES
# ------------------------------------------------------------

g6_validation_comparison = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "TF-IDF + Logistic Regression",
        "Word2Vec + BiLSTM"
    ],

    "accuracy": [
        majority_metrics["accuracy"],
        logreg_metrics["accuracy"],
        bilstm_metrics["accuracy"]
    ],

    "balanced_accuracy": [
        majority_metrics["balanced_accuracy"],
        logreg_metrics["balanced_accuracy"],
        bilstm_metrics["balanced_accuracy"]
    ],

    "precision_macro": [
        majority_metrics["precision_macro"],
        logreg_metrics["precision_macro"],
        bilstm_metrics["precision_macro"]
    ],

    "recall_macro": [
        majority_metrics["recall_macro"],
        logreg_metrics["recall_macro"],
        bilstm_metrics["recall_macro"]
    ],

    "f1_macro": [
        majority_metrics["f1_macro"],
        logreg_metrics["f1_macro"],
        bilstm_metrics["f1_macro"]
    ],

    "f1_weighted": [
        majority_metrics["f1_weighted"],
        logreg_metrics["f1_weighted"],
        bilstm_metrics["f1_weighted"]
    ]
})


print(
    "\n=== COMPARACIÓN EN VALIDATION ==="
)

print(
    g6_validation_comparison
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. IDENTIFICAR EL MEJOR POR F1 MACRO
# ------------------------------------------------------------

best_row = (
    g6_validation_comparison
    .sort_values(
        "f1_macro",
        ascending=False
    )
    .iloc[0]
)

print(
    "\n=== MEJOR EN VALIDATION POR F1 MACRO ==="
)

print(
    "Modelo:",
    best_row["modelo"]
)

print(
    "F1 macro:",
    round(
        float(best_row["f1_macro"]),
        4
    )
)

print(
    "Interpretación:",
    "mejor en este split y configuración"
)


# ------------------------------------------------------------
# 11. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "Conjunto evaluado:",
    "VALIDATION"
)

print(
    "TEST utilizado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 12. ESTADO DE G6
# ------------------------------------------------------------

print(
    "\n=== ESTADO DE G6 ==="
)

print(
    "Word2Vec entrenado:",
    "SÍ"
)

print(
    "BiLSTM entrenada:",
    "SÍ"
)

print(
    "Mejor checkpoint evaluado:",
    "SÍ"
)

print(
    "Métricas homogéneas calculadas:",
    "SÍ"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G6 listo para persistencia:",
    "SÍ"
)

print(
    "G6 aprobado:",
    "TODAVÍA NO"
)

=== PREDICCIONES BiLSTM — VALIDATION ===
Etiquetas reales: 1956
Predicciones: 1956
¿Coinciden?: True

=== Word2Vec + BiLSTM — MÉTRICAS EN VALIDATION ===
accuracy: 0.7434
balanced_accuracy: 0.6220
precision_macro: 0.6391
recall_macro: 0.6220
f1_macro: 0.6291
f1_weighted: 0.7454

=== CONSISTENCIA DEL CHECKPOINT ===
F1 macro obtenido durante early stopping: 0.6291
F1 macro recalculado con función común: 0.6291
¿Coinciden?: True

=== REPORTE POR CLASE — VALIDATION ===
              precision  recall  f1-score    support
NEGATIVO         0.7203  0.6242    0.6688   330.0000
NEUTRAL          0.3481  0.3856    0.3659   306.0000
POSITIVO         0.8490  0.8561    0.8525  1320.0000
accuracy         0.7434  0.7434    0.7434     0.7434
macro avg        0.6391  0.6220    0.6291  1956.0000
weighted avg     0.7489  0.7434    0.7454  1956.0000

=== MATRIZ DE CONFUSIÓN ABSOLUTA ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO            206            69             55
Real_N

In [42]:
# ============================================================
# G6 — WORD2VEC + BiLSTM
# Paso 8: persistir artefactos del modelo secuencial
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Guarda el modelo Word2Vec ya entrenado.
#
# 2. Guarda el mejor checkpoint de la BiLSTM.
#
# 3. Guarda el historial de entrenamiento.
#
# 4. Guarda las métricas de VALIDATION.
#
# 5. Guarda las predicciones de VALIDATION.
#
# 6. Guarda el reporte por clase.
#
# 7. Guarda:
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 8. Guarda la configuración completa de G6.
#
# 9. Vuelve a leer varios artefactos para comprobar
#    que realmente quedaron persistidos.
#
#
# ¿POR QUÉ LO HACEMOS?
#
# Google Colab trabaja con memoria temporal.
#
# Si la sesión se reinicia, podríamos perder:
#
# - Word2Vec;
# - BiLSTM;
# - historial;
# - predicciones;
# - métricas.
#
# Persistirlos nos permite demostrar que G6 es reproducible
# y entregar evidencia al auditor.
#
#
# IMPORTANTE:
#
# - NO volvemos a entrenar Word2Vec.
#
# - NO volvemos a entrenar la BiLSTM.
#
# - NO modificamos TRAIN.
#
# - NO modificamos VALIDATION.
#
# - NO procesamos TEST.
#
# - NO evaluamos TEST.
#
# - Las métricas guardadas corresponden exclusivamente
#   a VALIDATION.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos perder el mejor checkpoint y garantizamos
# trazabilidad entre:
#
#       configuración
#       entrenamiento
#       predicciones
#       métricas
#       modelo persistido


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import json
import torch
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINIR RUTAS DE LOS ARTEFACTOS
# ------------------------------------------------------------

G6_W2V_PATH = (
    ARTIFACT_DIR
    / "g6_word2vec.model"
)

G6_BILSTM_PATH = (
    ARTIFACT_DIR
    / "g6_bilstm_best.pt"
)

G6_HISTORY_PATH = (
    ARTIFACT_DIR
    / "g6_training_history.csv"
)

G6_METRICS_PATH = (
    ARTIFACT_DIR
    / "g6_metricas_validation.csv"
)

G6_PREDICTIONS_PATH = (
    ARTIFACT_DIR
    / "g6_predicciones_validation.csv"
)

G6_REPORT_PATH = (
    ARTIFACT_DIR
    / "g6_reporte_clases_validation.csv"
)

G6_CM_PATH = (
    ARTIFACT_DIR
    / "g6_matriz_confusion_validation.csv"
)

G6_CM_NORM_PATH = (
    ARTIFACT_DIR
    / "g6_matriz_confusion_validation_normalizada.csv"
)

G6_CONFIG_PATH = (
    ARTIFACT_DIR
    / "g6_configuracion.json"
)


# ------------------------------------------------------------
# 3. GUARDAR WORD2VEC
# ------------------------------------------------------------

# save() conserva el modelo completo de Gensim,
# incluyendo vocabulario y vectores.

w2v_model.save(
    str(G6_W2V_PATH)
)


# ------------------------------------------------------------
# 4. GUARDAR EL MEJOR CHECKPOINT DE LA BiLSTM
# ------------------------------------------------------------

# Guardamos:
#
# - pesos del modelo;
# - mejor época;
# - mejor F1 macro;
# - dimensiones necesarias para reconstruir la arquitectura.

torch.save(
    {
        "model_state_dict":
            bilstm_model.state_dict(),

        "best_epoch":
            int(best_epoch),

        "best_val_f1_macro":
            float(best_val_f1),

        "vocab_size":
            int(vocab_size),

        "embedding_dim":
            int(EMBEDDING_DIM),

        "hidden_per_direction":
            int(HIDDEN_PER_DIRECTION),

        "hidden_total":
            int(HIDDEN_PER_DIRECTION * 2),

        "dropout":
            float(DROPOUT),

        "num_classes":
            int(NUM_CLASSES),

        "max_len":
            int(MAX_LEN),

        "pad_idx":
            int(PAD_IDX),

        "unk_idx":
            int(UNK_IDX)
    },
    G6_BILSTM_PATH
)


# ------------------------------------------------------------
# 5. GUARDAR EL HISTORIAL DE ENTRENAMIENTO
# ------------------------------------------------------------

training_history_df.to_csv(
    G6_HISTORY_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. GUARDAR LAS MÉTRICAS DE VALIDATION
# ------------------------------------------------------------

# Estas métricas son las calculadas con la función común
# evaluate_predictions().
#
# NO son métricas de TEST.

g6_metrics_df = pd.DataFrame([
    {
        "modelo":
            "Word2Vec + BiLSTM",

        "accuracy":
            bilstm_metrics["accuracy"],

        "balanced_accuracy":
            bilstm_metrics["balanced_accuracy"],

        "precision_macro":
            bilstm_metrics["precision_macro"],

        "recall_macro":
            bilstm_metrics["recall_macro"],

        "f1_macro":
            bilstm_metrics["f1_macro"],

        "f1_weighted":
            bilstm_metrics["f1_weighted"],

        "best_epoch":
            int(best_epoch),

        "training_seconds":
            float(bilstm_training_seconds)
    }
])


g6_metrics_df.to_csv(
    G6_METRICS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. GUARDAR LAS PREDICCIONES DE VALIDATION
# ------------------------------------------------------------

# review_id se utiliza únicamente para trazabilidad.
#
# NO fue predictor de la BiLSTM.

g6_predictions_df = pd.DataFrame({
    "review_id":
        val_g4["review_id"].values,

    "y_true":
        y_val_bilstm_true,

    "y_pred_bilstm":
        y_val_bilstm_pred
})


g6_predictions_df.to_csv(
    G6_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. GUARDAR EL REPORTE POR CLASE
# ------------------------------------------------------------

bilstm_report_df.to_csv(
    G6_REPORT_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. GUARDAR MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

bilstm_cm_df.to_csv(
    G6_CM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 10. GUARDAR MATRIZ DE CONFUSIÓN NORMALIZADA
# ------------------------------------------------------------

bilstm_cm_norm_df.to_csv(
    G6_CM_NORM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 11. GUARDAR LA CONFIGURACIÓN COMPLETA DE G6
# ------------------------------------------------------------

g6_config = {

    "stage":
        "G6",

    "model_name":
        "Word2Vec + BiLSTM",

    "seed":
        42,

    "primary_metric":
        "f1_macro",

    "input_predictor":
        "text_sequential",

    "word2vec": {

        "sg":
            1,

        "architecture":
            "skip-gram",

        "vector_size":
            100,

        "window":
            5,

        "min_count":
            2,

        "negative":
            10,

        "epochs":
            10,

        "workers":
            1,

        "fit_split":
            "train",

        "vocabulary_size":
            int(len(w2v_model.wv)),

        "coverage_pct":
            float(
                coverage_ratio * 100
            )
    },

    "bilstm": {

        "embedding_dim":
            100,

        "hidden_per_direction":
            64,

        "hidden_total":
            128,

        "bidirectional":
            True,

        "dropout":
            0.30,

        "batch_size":
            64,

        "learning_rate":
            1e-3,

        "max_epochs":
            10,

        "patience":
            2,

        "max_len":
            300,

        "gradient_clip":
            1.0,

        "num_classes":
            3
    },

    "class_weights": [
        float(x)
        for x in class_weights_np
    ],

    "class_weights_source":
        "train",

    "loss":
        "CrossEntropyLoss weighted",

    "validation_selection":
        "f1_macro",

    "best_epoch":
        int(best_epoch),

    "best_val_f1_macro":
        float(best_val_f1),

    "train_truncation_pct":
        float(train_truncated_pct),

    "validation_truncation_pct":
        float(val_truncated_pct),

    "validation_unk_pct":
        float(val_unk_pct),

    "evaluation_split":
        "validation",

    "test_used":
        False
}


with open(
    G6_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        g6_config,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 12. COMPROBAR QUE TODOS LOS ARTEFACTOS EXISTEN
# ------------------------------------------------------------

g6_artifacts = {

    "Word2Vec":
        G6_W2V_PATH,

    "Checkpoint BiLSTM":
        G6_BILSTM_PATH,

    "Historial entrenamiento":
        G6_HISTORY_PATH,

    "Métricas VALIDATION":
        G6_METRICS_PATH,

    "Predicciones VALIDATION":
        G6_PREDICTIONS_PATH,

    "Reporte por clase":
        G6_REPORT_PATH,

    "Matriz confusión":
        G6_CM_PATH,

    "Matriz confusión normalizada":
        G6_CM_NORM_PATH,

    "Configuración":
        G6_CONFIG_PATH
}


print("=== ARTEFACTOS DE G6 ===")

all_g6_artifacts_exist = True


for name, path in g6_artifacts.items():

    exists = path.exists()

    all_g6_artifacts_exist = (
        all_g6_artifacts_exist
        and exists
    )

    print(
        f"{name}:",
        exists
    )


# ------------------------------------------------------------
# 13. VOLVER A LEER LOS ARTEFACTOS PRINCIPALES
# ------------------------------------------------------------

g6_metrics_check = pd.read_csv(
    G6_METRICS_PATH
)

g6_predictions_check = pd.read_csv(
    G6_PREDICTIONS_PATH
)

g6_history_check = pd.read_csv(
    G6_HISTORY_PATH
)

checkpoint_check = torch.load(
    G6_BILSTM_PATH,
    map_location="cpu"
)


# ------------------------------------------------------------
# 14. COMPROBAR RECUPERACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Filas tabla de métricas:",
    len(g6_metrics_check)
)

print(
    "Predicciones recuperadas:",
    len(g6_predictions_check)
)

print(
    "Filas VALIDATION:",
    len(y_val)
)

print(
    "¿Predicciones coinciden con VALIDATION?:",
    len(g6_predictions_check)
    == len(y_val)
)

print(
    "review_id duplicados:",
    int(
        g6_predictions_check[
            "review_id"
        ]
        .duplicated()
        .sum()
    )
)

print(
    "Épocas recuperadas:",
    len(g6_history_check)
)

print(
    "Mejor época recuperada:",
    checkpoint_check[
        "best_epoch"
    ]
)

print(
    "Mejor F1 macro recuperado:",
    round(
        checkpoint_check[
            "best_val_f1_macro"
        ],
        4
    )
)


# ------------------------------------------------------------
# 15. COMPROBAR CONFIGURACIÓN DEL CHECKPOINT
# ------------------------------------------------------------

print("\n=== CONTROL DEL CHECKPOINT ===")

print(
    "Embedding dim:",
    checkpoint_check[
        "embedding_dim"
    ]
)

print(
    "Hidden total:",
    checkpoint_check[
        "hidden_total"
    ]
)

print(
    "Dropout:",
    checkpoint_check[
        "dropout"
    ]
)

print(
    "MAX_LEN:",
    checkpoint_check[
        "max_len"
    ]
)

print(
    "Número de clases:",
    checkpoint_check[
        "num_classes"
    ]
)


# ------------------------------------------------------------
# 16. MOSTRAR MÉTRICAS PERSISTIDAS
# ------------------------------------------------------------

print("\n=== MÉTRICAS G6 PERSISTIDAS ===")

print(
    g6_metrics_check
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 17. VALIDACIÓN AUTOMÁTICA DE PERSISTENCIA
# ------------------------------------------------------------

g6_persistence_ok = (

    all_g6_artifacts_exist

    and len(g6_metrics_check) == 1

    and len(g6_predictions_check)
        == len(y_val)

    and g6_predictions_check[
        "review_id"
    ].duplicated().sum() == 0

    and checkpoint_check[
        "best_epoch"
    ] == best_epoch

    and np.isclose(
        checkpoint_check[
            "best_val_f1_macro"
        ],
        best_val_f1
    )

    and checkpoint_check[
        "embedding_dim"
    ] == 100

    and checkpoint_check[
        "hidden_total"
    ] == 128

    and np.isclose(
        checkpoint_check[
            "dropout"
        ],
        0.30
    )

    and checkpoint_check[
        "max_len"
    ] == 300

    and checkpoint_check[
        "num_classes"
    ] == 3
)


print("\n=== RESULTADO DE PERSISTENCIA G6 ===")

print(
    "¿Controles superados?:",
    g6_persistence_ok
)

print(
    "G6 listo para auditoría independiente:",
    "SÍ"
    if g6_persistence_ok
    else "NO"
)


# ------------------------------------------------------------
# 18. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST incluido en predicciones:",
    "NO"
)

print(
    "TEST usado para selección:",
    "NO"
)

print(
    "TEST usado para early stopping:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 19. ESTADO FINAL DE ESTA CELDA
# ------------------------------------------------------------

print("\n=== ESTADO DE G6 ===")

print(
    "Word2Vec persistido:",
    "SÍ"
)

print(
    "BiLSTM persistida:",
    "SÍ"
)

print(
    "Historial persistido:",
    "SÍ"
)

print(
    "Predicciones VALIDATION persistidas:",
    "SÍ"
)

print(
    "Métricas VALIDATION persistidas:",
    "SÍ"
)

print(
    "G6 listo para auditoría:",
    "SÍ"
    if g6_persistence_ok
    else "NO"
)

print(
    "G6 aprobado:",
    "TODAVÍA NO"
)

print(
    "G7 iniciado:",
    "NO"
)

=== ARTEFACTOS DE G6 ===
Word2Vec: True
Checkpoint BiLSTM: True
Historial entrenamiento: True
Métricas VALIDATION: True
Predicciones VALIDATION: True
Reporte por clase: True
Matriz confusión: True
Matriz confusión normalizada: True
Configuración: True

=== CONTROL DE RECUPERACIÓN ===
Filas tabla de métricas: 1
Predicciones recuperadas: 1956
Filas VALIDATION: 1956
¿Predicciones coinciden con VALIDATION?: True
review_id duplicados: 0
Épocas recuperadas: 5
Mejor época recuperada: 3
Mejor F1 macro recuperado: 0.6291

=== CONTROL DEL CHECKPOINT ===
Embedding dim: 100
Hidden total: 128
Dropout: 0.3
MAX_LEN: 300
Número de clases: 3

=== MÉTRICAS G6 PERSISTIDAS ===
           modelo  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  f1_weighted  best_epoch  training_seconds
Word2Vec + BiLSTM    0.7434              0.622           0.6391         0.622    0.6291       0.7454           3            3.6521

=== RESULTADO DE PERSISTENCIA G6 ===
¿Controles superados?: True
G6 lis

In [43]:
# ============================================================
# CONTROL DE COMPUERTA
# Evidencia verificable de que G7 todavía NO se ha iniciado
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Esta celda NO inicia G7.
#
# Su única finalidad es comprobar si existen señales
# verificables de que G7 ya haya comenzado.
#
# Revisaremos:
#
# 1. Si existen artefactos de G7 en ARTIFACT_DIR.
#
# 2. Si existen variables típicas de un pipeline BERT:
#
#       bert_model
#       bert_tokenizer
#       train_dataset_bert
#       val_dataset_bert
#       trainer
#
# 3. Si existe algún archivo cuyo nombre empiece por:
#
#       g7_
#
#
# IMPORTANTE:
#
# - NO descargamos BERT.
# - NO creamos tokenizer.
# - NO creamos datasets BERT.
# - NO entrenamos nada.
# - NO utilizamos TEST.
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Producimos evidencia ejecutada y verificable del estado
# previo a G7, sin iniciar G7 para demostrarlo.


# ------------------------------------------------------------
# 1. BUSCAR ARTEFACTOS DE G7
# ------------------------------------------------------------

g7_artifacts = list(
    ARTIFACT_DIR.glob("g7_*")
)

print("=== ARTEFACTOS G7 ===")

print(
    "Cantidad de artefactos g7_*:",
    len(g7_artifacts)
)

print(
    "Artefactos encontrados:",
    [p.name for p in g7_artifacts]
)


# ------------------------------------------------------------
# 2. COMPROBAR VARIABLES TÍPICAS DE G7
# ------------------------------------------------------------

g7_variable_names = [
    "bert_model",
    "bert_tokenizer",
    "train_dataset_bert",
    "val_dataset_bert",
    "trainer",
    "bert_training_args",
    "y_val_pred_bert"
]

g7_variables_present = {
    name: name in globals()
    for name in g7_variable_names
}

print("\n=== VARIABLES G7 ===")

for name, present in g7_variables_present.items():
    print(
        f"{name}:",
        present
    )


# ------------------------------------------------------------
# 3. DETERMINAR SI EXISTE EVIDENCIA DE G7 INICIADO
# ------------------------------------------------------------

g7_started_evidence = (
    len(g7_artifacts) > 0
    or any(g7_variables_present.values())
)


print("\n=== RESULTADO DE CONTROL ===")

print(
    "¿Existe evidencia verificable de que G7 haya iniciado?:",
    g7_started_evidence
)

print(
    "¿G7 permanece no iniciado según los objetos revisados?:",
    not g7_started_evidence
)


# ------------------------------------------------------------
# 4. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST utilizado en esta celda:",
    False
)

print(
    "TEST evaluado:",
    False
)

=== ARTEFACTOS G7 ===
Cantidad de artefactos g7_*: 0
Artefactos encontrados: []

=== VARIABLES G7 ===
bert_model: False
bert_tokenizer: False
train_dataset_bert: False
val_dataset_bert: False
trainer: False
bert_training_args: False
y_val_pred_bert: False

=== RESULTADO DE CONTROL ===
¿Existe evidencia verificable de que G7 haya iniciado?: False
¿G7 permanece no iniciado según los objetos revisados?: True

=== CONTROL DEL TEST SELLADO ===
TEST utilizado en esta celda: False
TEST evaluado: False


In [44]:
# ============================================================
# G7 — BERT
# Paso 1: verificar entorno y preparar tokenizer oficial
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Comprueba CUDA y versiones relevantes.
#
# 2. Importa las herramientas oficiales de Hugging Face.
#
# 3. Carga el tokenizer oficial de:
#
#       google-bert/bert-base-uncased
#
# 4. Selecciona exclusivamente:
#
#       text_bert
#
#    como entrada textual.
#
# 5. Comprueba tamaños de TRAIN y VALIDATION.
#
# 6. Verifica que TEST siga sellado.
#
#
# IMPORTANTE:
#
# - NO entrenamos BERT todavía.
#
# - NO descargamos ni usamos TEST.
#
# - NO usamos text_sequential.
#
# - NO eliminamos stopwords, puntuación ni negaciones.
#
# - NO aplicamos stemming ni lematización.
#
# - sentiment_id es únicamente la etiqueta objetivo.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos:
#
# - mezclar la rama secuencial con BERT;
# - utilizar un tokenizer distinto del oficial;
# - tocar TEST antes de G8.


# ------------------------------------------------------------
# 1. IMPORTAR LIBRERÍAS
# ------------------------------------------------------------

import torch
import transformers

from transformers import AutoTokenizer


# ------------------------------------------------------------
# 2. COMPROBAR DISPOSITIVO
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=== ENTORNO G7 ===")

print(
    "Dispositivo:",
    device
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print(
        "GPU:",
        "NO DISPONIBLE"
    )

print(
    "torch:",
    torch.__version__
)

print(
    "transformers:",
    transformers.__version__
)


# ------------------------------------------------------------
# 3. DEFINIR EL MODELO OFICIAL
# ------------------------------------------------------------

BERT_MODEL_NAME = (
    "google-bert/bert-base-uncased"
)

print("\n=== MODELO BERT ===")

print(
    "Modelo:",
    BERT_MODEL_NAME
)


# ------------------------------------------------------------
# 4. CARGAR TOKENIZER OFICIAL
# ------------------------------------------------------------

bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_NAME,
    use_fast=True
)

print(
    "Tokenizer cargado:",
    type(bert_tokenizer).__name__
)

print(
    "Tokenizer fast:",
    bert_tokenizer.is_fast
)


# ------------------------------------------------------------
# 5. DEFINIR ENTRADAS DE TRAIN Y VALIDATION
# ------------------------------------------------------------

# BERT recibe directamente la rama text_bert.
#
# No recibe text_sequential.

X_train_bert_text = (
    train_g4["text_bert"]
    .astype(str)
)

X_val_bert_text = (
    val_g4["text_bert"]
    .astype(str)
)

y_train_bert = (
    train_g4["sentiment_id"]
    .astype(int)
    .to_numpy()
)

y_val_bert = (
    val_g4["sentiment_id"]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 6. COMPROBAR TAMAÑOS
# ------------------------------------------------------------

print("\n=== ENTRADAS BERT ===")

print(
    "TRAIN textos:",
    len(X_train_bert_text)
)

print(
    "VALIDATION textos:",
    len(X_val_bert_text)
)

print(
    "TRAIN etiquetas:",
    len(y_train_bert)
)

print(
    "VALIDATION etiquetas:",
    len(y_val_bert)
)


# ------------------------------------------------------------
# 7. CONTROL DE PREDICTORES
# ------------------------------------------------------------

bert_input_columns = [
    "text_bert"
]

forbidden_predictors = {
    "stars",
    "sentiment",
    "sentiment_id",
    "review_id",
    "business_id",
    "user_id",
    "date",
    "type",
    "cool",
    "useful",
    "funny",
    "text_sequential"
}

forbidden_used = (
    set(bert_input_columns)
    & forbidden_predictors
)

print("\n=== CONTROL DE ENTRADAS BERT ===")

print(
    "Columnas utilizadas:",
    bert_input_columns
)

print(
    "Variables prohibidas utilizadas:",
    sorted(forbidden_used)
)

print(
    "¿BERT recibe únicamente text_bert?:",
    len(forbidden_used) == 0
    and bert_input_columns == ["text_bert"]
)


# ------------------------------------------------------------
# 8. MOSTRAR EJEMPLO DE TOKENIZACIÓN
# ------------------------------------------------------------

sample_text = X_train_bert_text.iloc[0]

sample_tokens = bert_tokenizer.tokenize(
    sample_text
)

print("\n=== EJEMPLO DE TOKENIZACIÓN BERT ===")

print(
    "Texto original:"
)

print(
    sample_text[:300]
)

print(
    "\nPrimeros tokens:"
)

print(
    sample_tokens[:30]
)


# ------------------------------------------------------------
# 9. CONFIGURACIÓN PREVISTA
# ------------------------------------------------------------

BERT_CONFIG = {
    "model_name":
        "google-bert/bert-base-uncased",

    "max_length":
        256,

    "batch_size":
        8,

    "gradient_accumulation_steps":
        2,

    "learning_rate":
        2e-5,

    "weight_decay":
        0.01,

    "warmup_ratio":
        0.1,

    "epochs":
        3,

    "seed":
        42,

    "primary_metric":
        "f1_macro"
}

print("\n=== CONFIGURACIÓN PREVISTA BERT ===")

print(
    BERT_CONFIG
)


# ------------------------------------------------------------
# 10. CONTROL DE TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST preparado para BERT:",
    False
)

print(
    "TEST tokenizado:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 11. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_step1_ok = (
    len(X_train_bert_text) == len(train_g4)
    and len(X_val_bert_text) == len(val_g4)
    and len(y_train_bert) == len(train_g4)
    and len(y_val_bert) == len(val_g4)
    and bert_tokenizer.is_fast
    and BERT_MODEL_NAME
        == "google-bert/bert-base-uncased"
    and len(forbidden_used) == 0
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 1 ===")

print(
    "¿Controles superados?:",
    g7_step1_ok
)


# ------------------------------------------------------------
# 12. ESTADO DE G7
# ------------------------------------------------------------

print("\n=== ESTADO DE G7 ===")

print(
    "Tokenizer oficial cargado:",
    "SÍ"
)

print(
    "TRAIN preparado:",
    "SÍ"
)

print(
    "VALIDATION preparada:",
    "SÍ"
)

print(
    "BERT entrenado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 finalizado:",
    "NO"
)

=== ENTORNO G7 ===
Dispositivo: cuda
GPU: Tesla T4
torch: 2.11.0+cu128
transformers: 5.15.0

=== MODELO BERT ===
Modelo: google-bert/bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer cargado: BertTokenizer
Tokenizer fast: True

=== ENTRADAS BERT ===
TRAIN textos: 5943
VALIDATION textos: 1956
TRAIN etiquetas: 5943
VALIDATION etiquetas: 1956

=== CONTROL DE ENTRADAS BERT ===
Columnas utilizadas: ['text_bert']
Variables prohibidas utilizadas: []
¿BERT recibe únicamente text_bert?: True

=== EJEMPLO DE TOKENIZACIÓN BERT ===
Texto original:
My wife took me here on my birthday for breakfast and it was excellent. The weather was perfect which made sitting outside overlooking their grounds an absolute pleasure. Our waitress was excellent and our food arrived quickly on the semi-busy Saturday morning. It looked like the place fills up pret

Primeros tokens:
['my', 'wife', 'took', 'me', 'here', 'on', 'my', 'birthday', 'for', 'breakfast', 'and', 'it', 'was', 'excellent', '.', 'the', 'weather', 'was', 'perfect', 'which', 'made', 'sitting', 'outside', 'overlooking', 'their', 'grounds', 'an', 'absolute', 'pleasure', '.']

=== CONFIGURACIÓN PREVISTA BERT ===
{'model_nam

In [45]:
# ============================================================
# G7 — BERT
# Paso 2: medir truncamiento y preparar datasets
#         con tokenización oficial
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Tokeniza TRAIN y VALIDATION con el tokenizer oficial
#    de bert-base-uncased para MEDIR sus longitudes reales.
#
# 2. Calcula:
#
#       - longitud promedio
#       - mediana
#       - P90
#       - P95
#       - máximo
#
#    en TOKENS BERT.
#
# 3. Calcula cuántas reseñas superan:
#
#       MAX_LENGTH = 256
#
# 4. Crea datasets de PyTorch para TRAIN y VALIDATION.
#
# 5. Cada elemento del dataset se tokeniza con:
#
#       truncation=True
#       max_length=256
#       padding=False
#
# 6. Prepara DataCollatorWithPadding para que el padding
#    se realice DINÁMICAMENTE por batch.
#
#
# ¿QUÉ SIGNIFICA PADDING DINÁMICO?
#
# Supongamos que en un batch tenemos reseñas de:
#
#       40 tokens
#       70 tokens
#       90 tokens
#
# En vez de rellenarlas todas hasta 256,
# el batch se rellena solamente hasta 90.
#
# Esto reduce memoria y cálculo innecesario.
#
#
# IMPORTANTE:
#
# - TRAIN y VALIDATION usan text_bert.
#
# - NO usamos text_sequential.
#
# - NO eliminamos stopwords.
#
# - NO eliminamos puntuación.
#
# - NO eliminamos negaciones.
#
# - NO aplicamos stemming.
#
# - NO aplicamos lematización.
#
# - TEST NO se tokeniza.
#
# - BERT todavía NO se entrena.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Documentamos cuánto texto será truncado antes de entrenar
# y comprobamos que el pipeline utiliza el tokenizer oficial
# de forma consistente en TRAIN y VALIDATION.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding


# ------------------------------------------------------------
# 2. DEFINIR LONGITUD MÁXIMA
# ------------------------------------------------------------

MAX_LENGTH_BERT = 256


print("=== CONFIGURACIÓN DE TOKENIZACIÓN ===")

print(
    "Modelo:",
    BERT_MODEL_NAME
)

print(
    "MAX_LENGTH:",
    MAX_LENGTH_BERT
)

print(
    "Padding previsto:",
    "dinámico por batch"
)

print(
    "Truncamiento previsto:",
    True
)


# ------------------------------------------------------------
# 3. FUNCIÓN PARA MEDIR LONGITUDES BERT
# ------------------------------------------------------------

# Para medir el porcentaje que NECESITARÍA truncamiento,
# primero tokenizamos SIN truncar.
#
# return_length=True devuelve la cantidad de tokens
# generados por el tokenizer oficial.
#
# IMPORTANTE:
# Esto solo mide longitudes.
# No entrena el modelo.

def get_bert_lengths(texts, tokenizer):

    encoding = tokenizer(
        list(texts),
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True
    )

    return np.array(
        encoding["length"],
        dtype=np.int32
    )


# ------------------------------------------------------------
# 4. MEDIR LONGITUDES DE TRAIN
# ------------------------------------------------------------

train_bert_lengths = get_bert_lengths(
    X_train_bert_text,
    bert_tokenizer
)


# ------------------------------------------------------------
# 5. MEDIR LONGITUDES DE VALIDATION
# ------------------------------------------------------------

val_bert_lengths = get_bert_lengths(
    X_val_bert_text,
    bert_tokenizer
)


# ------------------------------------------------------------
# 6. MOSTRAR ESTADÍSTICAS DE TRAIN
# ------------------------------------------------------------

print(
    "\n=== LONGITUD EN TOKENS BERT — TRAIN ==="
)

print(
    "Reseñas:",
    len(train_bert_lengths)
)

print(
    "Promedio:",
    round(
        float(train_bert_lengths.mean()),
        2
    )
)

print(
    "Mediana:",
    float(
        np.median(train_bert_lengths)
    )
)

print(
    "P90:",
    float(
        np.percentile(
            train_bert_lengths,
            90
        )
    )
)

print(
    "P95:",
    float(
        np.percentile(
            train_bert_lengths,
            95
        )
    )
)

print(
    "Máximo:",
    int(
        train_bert_lengths.max()
    )
)


# ------------------------------------------------------------
# 7. MOSTRAR ESTADÍSTICAS DE VALIDATION
# ------------------------------------------------------------

print(
    "\n=== LONGITUD EN TOKENS BERT — VALIDATION ==="
)

print(
    "Reseñas:",
    len(val_bert_lengths)
)

print(
    "Promedio:",
    round(
        float(val_bert_lengths.mean()),
        2
    )
)

print(
    "Mediana:",
    float(
        np.median(val_bert_lengths)
    )
)

print(
    "P90:",
    float(
        np.percentile(
            val_bert_lengths,
            90
        )
    )
)

print(
    "P95:",
    float(
        np.percentile(
            val_bert_lengths,
            95
        )
    )
)

print(
    "Máximo:",
    int(
        val_bert_lengths.max()
    )
)


# ------------------------------------------------------------
# 8. CALCULAR TRUNCAMIENTO EN TRAIN
# ------------------------------------------------------------

# Una reseña necesitará truncamiento si tiene
# más de 256 tokens BERT.

train_bert_truncated = int(
    np.sum(
        train_bert_lengths
        > MAX_LENGTH_BERT
    )
)

train_bert_truncated_pct = (
    train_bert_truncated
    / len(train_bert_lengths)
    * 100
)


# ------------------------------------------------------------
# 9. CALCULAR TRUNCAMIENTO EN VALIDATION
# ------------------------------------------------------------

val_bert_truncated = int(
    np.sum(
        val_bert_lengths
        > MAX_LENGTH_BERT
    )
)

val_bert_truncated_pct = (
    val_bert_truncated
    / len(val_bert_lengths)
    * 100
)


print(
    "\n=== TRUNCAMIENTO PREVISTO A 256 TOKENS ==="
)

print(
    "TRAIN reseñas truncadas:",
    train_bert_truncated
)

print(
    "TRAIN porcentaje:",
    round(
        train_bert_truncated_pct,
        2
    ),
    "%"
)

print(
    "VALIDATION reseñas truncadas:",
    val_bert_truncated
)

print(
    "VALIDATION porcentaje:",
    round(
        val_bert_truncated_pct,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 10. DEFINIR DATASET PARA BERT
# ------------------------------------------------------------

# Este Dataset almacena:
#
#       texto
#       etiqueta
#
# Cada vez que PyTorch solicita una reseña:
#
# 1. toma text_bert;
# 2. utiliza bert_tokenizer;
# 3. trunca a 256;
# 4. NO hace padding todavía.
#
# El padding se hará posteriormente por batch mediante
# DataCollatorWithPadding.

class YelpBertDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):

        self.texts = list(texts)

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(
        self,
        idx
    ):

        text = self.texts[idx]

        label = int(
            self.labels[idx]
        )


        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False
        )


        encoding["labels"] = label

        return encoding


# ------------------------------------------------------------
# 11. CREAR DATASET DE TRAIN
# ------------------------------------------------------------

train_dataset_bert = YelpBertDataset(
    texts=X_train_bert_text,
    labels=y_train_bert,
    tokenizer=bert_tokenizer,
    max_length=MAX_LENGTH_BERT
)


# ------------------------------------------------------------
# 12. CREAR DATASET DE VALIDATION
# ------------------------------------------------------------

val_dataset_bert = YelpBertDataset(
    texts=X_val_bert_text,
    labels=y_val_bert,
    tokenizer=bert_tokenizer,
    max_length=MAX_LENGTH_BERT
)


# ------------------------------------------------------------
# 13. CREAR DATA COLLATOR PARA PADDING DINÁMICO
# ------------------------------------------------------------

bert_data_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer,
    padding=True,
    return_tensors="pt"
)


# ------------------------------------------------------------
# 14. COMPROBAR TAMAÑOS DE LOS DATASETS
# ------------------------------------------------------------

print(
    "\n=== DATASETS BERT ==="
)

print(
    "TRAIN:",
    len(train_dataset_bert)
)

print(
    "VALIDATION:",
    len(val_dataset_bert)
)

print(
    "¿TRAIN coincide?:",
    len(train_dataset_bert)
    == len(train_g4)
)

print(
    "¿VALIDATION coincide?:",
    len(val_dataset_bert)
    == len(val_g4)
)


# ------------------------------------------------------------
# 15. INSPECCIONAR UN ELEMENTO DE TRAIN
# ------------------------------------------------------------

sample_bert_item = (
    train_dataset_bert[0]
)


print(
    "\n=== EJEMPLO DE ELEMENTO BERT ==="
)

print(
    "Claves:",
    sample_bert_item.keys()
)

print(
    "Longitud input_ids:",
    len(
        sample_bert_item[
            "input_ids"
        ]
    )
)

print(
    "Etiqueta:",
    sample_bert_item[
        "labels"
    ]
)

print(
    "¿Longitud <= 256?:",
    len(
        sample_bert_item[
            "input_ids"
        ]
    )
    <= MAX_LENGTH_BERT
)


# ------------------------------------------------------------
# 16. PROBAR PADDING DINÁMICO CON UN MINI-BATCH
# ------------------------------------------------------------

# Tomamos 4 reseñas distintas.
#
# El collator debe generar tensores con la misma longitud,
# pero esa longitud debe ser como máximo 256.

mini_batch_examples = [
    train_dataset_bert[i]
    for i in range(4)
]


mini_batch_bert = bert_data_collator(
    mini_batch_examples
)


print(
    "\n=== PRUEBA DE PADDING DINÁMICO ==="
)

print(
    "Forma input_ids:",
    mini_batch_bert[
        "input_ids"
    ].shape
)

print(
    "Forma attention_mask:",
    mini_batch_bert[
        "attention_mask"
    ].shape
)

print(
    "Forma labels:",
    mini_batch_bert[
        "labels"
    ].shape
)

print(
    "Longitud del mini-batch:",
    mini_batch_bert[
        "input_ids"
    ].shape[1]
)

print(
    "¿Longitud <= 256?:",
    mini_batch_bert[
        "input_ids"
    ].shape[1]
    <= MAX_LENGTH_BERT
)


# ------------------------------------------------------------
# 17. COMPROBAR LAS CLASES
# ------------------------------------------------------------

bert_train_classes = np.unique(
    y_train_bert
)

bert_val_classes = np.unique(
    y_val_bert
)


print(
    "\n=== CONTROL DE CLASES ==="
)

print(
    "TRAIN:",
    bert_train_classes
)

print(
    "VALIDATION:",
    bert_val_classes
)

print(
    "¿TRAIN contiene 0, 1 y 2?:",
    set(
        bert_train_classes.tolist()
    ) == {0, 1, 2}
)

print(
    "¿VALIDATION contiene 0, 1 y 2?:",
    set(
        bert_val_classes.tolist()
    ) == {0, 1, 2}
)


# ------------------------------------------------------------
# 18. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_step2_ok = (

    len(train_bert_lengths)
        == len(train_g4)

    and len(val_bert_lengths)
        == len(val_g4)

    and len(train_dataset_bert)
        == len(train_g4)

    and len(val_dataset_bert)
        == len(val_g4)

    and MAX_LENGTH_BERT == 256

    and len(
        sample_bert_item[
            "input_ids"
        ]
    ) <= MAX_LENGTH_BERT

    and mini_batch_bert[
        "input_ids"
    ].shape[1] <= MAX_LENGTH_BERT

    and set(
        bert_train_classes.tolist()
    ) == {0, 1, 2}

    and set(
        bert_val_classes.tolist()
    ) == {0, 1, 2}
)


print(
    "\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 2 ==="
)

print(
    "¿Controles superados?:",
    g7_step2_ok
)


# ------------------------------------------------------------
# 19. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "TEST incluido en medición de longitudes:",
    False
)

print(
    "Dataset BERT de TEST creado:",
    False
)

print(
    "TEST tokenizado:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 20. ESTADO DE G7
# ------------------------------------------------------------

print(
    "\n=== ESTADO DE G7 ==="
)

print(
    "Tokenizer oficial:",
    "SÍ"
)

print(
    "Truncamiento documentado:",
    "SÍ"
)

print(
    "Dataset TRAIN BERT:",
    "SÍ"
)

print(
    "Dataset VALIDATION BERT:",
    "SÍ"
)

print(
    "Padding dinámico preparado:",
    "SÍ"
)

print(
    "BERT entrenado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 finalizado:",
    "NO"
)

=== CONFIGURACIÓN DE TOKENIZACIÓN ===
Modelo: google-bert/bert-base-uncased
MAX_LENGTH: 256
Padding previsto: dinámico por batch
Truncamiento previsto: True


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (916 > 512). Running this sequence through the model will result in indexing errors



=== LONGITUD EN TOKENS BERT — TRAIN ===
Reseñas: 5943
Promedio: 167.81
Mediana: 130.0
P90: 342.0
P95: 441.0
Máximo: 1260

=== LONGITUD EN TOKENS BERT — VALIDATION ===
Reseñas: 1956
Promedio: 177.03
Mediana: 134.0
P90: 371.0
P95: 474.25
Máximo: 1189

=== TRUNCAMIENTO PREVISTO A 256 TOKENS ===
TRAIN reseñas truncadas: 1139
TRAIN porcentaje: 19.17 %
VALIDATION reseñas truncadas: 418
VALIDATION porcentaje: 21.37 %

=== DATASETS BERT ===
TRAIN: 5943
VALIDATION: 1956
¿TRAIN coincide?: True
¿VALIDATION coincide?: True

=== EJEMPLO DE ELEMENTO BERT ===
Claves: KeysView({'input_ids': [101, 2026, 2564, 2165, 2033, 2182, 2006, 2026, 5798, 2005, 6350, 1998, 2009, 2001, 6581, 1012, 1996, 4633, 2001, 3819, 2029, 2081, 3564, 2648, 12549, 2037, 5286, 2019, 7619, 5165, 1012, 2256, 13877, 2001, 6581, 1998, 2256, 2833, 3369, 2855, 2006, 1996, 4100, 1011, 5697, 5095, 2851, 1012, 2009, 2246, 2066, 1996, 2173, 17469, 2039, 3492, 2855, 2061, 1996, 3041, 2017, 2131, 2182, 1996, 2488, 1012, 2079, 4426, 1037, 

In [46]:
# ============================================================
# G7 — BERT
# Paso 3: construir BERT para 3 clases y preparar
#         CrossEntropyLoss ponderada SOLO con TRAIN
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Calcula pesos de clase exclusivamente desde y_train_bert.
#
# 2. Carga:
#
#       google-bert/bert-base-uncased
#
#    para clasificación de 3 clases.
#
# 3. Comprueba el orden oficial:
#
#       0 = NEGATIVO
#       1 = NEUTRAL
#       2 = POSITIVO
#
# 4. Prepara una pérdida CrossEntropyLoss ponderada.
#
# 5. Comprueba el número de parámetros.
#
# 6. Hace un forward de prueba sin entrenar.
#
#
# IMPORTANTE:
#
# - NO usamos VALIDATION para calcular pesos.
# - NO usamos TEST.
# - NO entrenamos BERT todavía.
# - BERT recibe text_bert.
# - La salida debe tener 3 logits por reseña.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import numpy as np
import torch
import torch.nn as nn

from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification
)


# ------------------------------------------------------------
# 2. CALCULAR PESOS DE CLASE SOLO CON TRAIN
# ------------------------------------------------------------

bert_classes = np.array(
    [0, 1, 2]
)

bert_class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=bert_classes,
    y=y_train_bert
)


print("=== PESOS DE CLASE BERT — SOLO TRAIN ===")

for class_id, weight in zip(
    bert_classes,
    bert_class_weights_np
):
    print(
        f"Clase {class_id}:",
        round(float(weight), 6)
    )


# ------------------------------------------------------------
# 3. MOSTRAR DISTRIBUCIÓN USADA
# ------------------------------------------------------------

bert_train_class_counts = (
    pd.Series(y_train_bert)
    .value_counts()
    .sort_index()
)

print(
    "\n=== DISTRIBUCIÓN DE TRAIN USADA PARA LOS PESOS ==="
)

print(
    bert_train_class_counts
)

print(
    "Origen:",
    "y_train_bert"
)


# ------------------------------------------------------------
# 4. DEFINIR MAPEO DE ETIQUETAS
# ------------------------------------------------------------

id2label = {
    0: "NEGATIVO",
    1: "NEUTRAL",
    2: "POSITIVO"
}

label2id = {
    "NEGATIVO": 0,
    "NEUTRAL": 1,
    "POSITIVO": 2
}


# ------------------------------------------------------------
# 5. CREAR CONFIGURACIÓN BERT
# ------------------------------------------------------------

bert_config = AutoConfig.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


# ------------------------------------------------------------
# 6. CARGAR MODELO PREENTRENADO
# ------------------------------------------------------------

bert_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        BERT_MODEL_NAME,
        config=bert_config
    )
)


bert_model = bert_model.to(
    device
)


# ------------------------------------------------------------
# 7. CREAR PESOS PARA PYTORCH
# ------------------------------------------------------------

bert_class_weights_tensor = torch.tensor(
    bert_class_weights_np,
    dtype=torch.float32,
    device=device
)


# ------------------------------------------------------------
# 8. CREAR CrossEntropyLoss PONDERADA
# ------------------------------------------------------------

bert_criterion = nn.CrossEntropyLoss(
    weight=bert_class_weights_tensor
)


# ------------------------------------------------------------
# 9. MOSTRAR CONFIGURACIÓN DEL MODELO
# ------------------------------------------------------------

print("\n=== CONFIGURACIÓN DEL MODELO BERT ===")

print(
    "Modelo:",
    BERT_MODEL_NAME
)

print(
    "Número de clases:",
    bert_model.config.num_labels
)

print(
    "id2label:",
    bert_model.config.id2label
)

print(
    "label2id:",
    bert_model.config.label2id
)

print(
    "Loss:",
    type(bert_criterion).__name__
)

print(
    "Pesos de clase:",
    bert_class_weights_tensor
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# 10. CONTAR PARÁMETROS
# ------------------------------------------------------------

bert_total_parameters = sum(
    p.numel()
    for p in bert_model.parameters()
)

bert_trainable_parameters = sum(
    p.numel()
    for p in bert_model.parameters()
    if p.requires_grad
)


print("\n=== PARÁMETROS BERT ===")

print(
    "Parámetros totales:",
    bert_total_parameters
)

print(
    "Parámetros entrenables:",
    bert_trainable_parameters
)


# ------------------------------------------------------------
# 11. PREPARAR MINI-BATCH DE PRUEBA
# ------------------------------------------------------------

mini_batch_bert_device = {
    key: value.to(device)
    for key, value
    in mini_batch_bert.items()
    if key != "labels"
}


# ------------------------------------------------------------
# 12. FORWARD DE PRUEBA SIN ENTRENAR
# ------------------------------------------------------------

bert_model.eval()

with torch.no_grad():

    bert_test_outputs = bert_model(
        **mini_batch_bert_device
    )


bert_test_logits = (
    bert_test_outputs.logits
)


print("\n=== FORWARD DE PRUEBA ===")

print(
    "Forma logits:",
    bert_test_logits.shape
)

print(
    "¿Hay 3 logits por reseña?:",
    bert_test_logits.shape[1] == 3
)


# ------------------------------------------------------------
# 13. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_step3_ok = (
    len(bert_class_weights_np) == 3

    and np.all(
        bert_class_weights_np > 0
    )

    and bert_model.config.num_labels == 3

    and bert_model.config.id2label[0]
        == "NEGATIVO"

    and bert_model.config.id2label[1]
        == "NEUTRAL"

    and bert_model.config.id2label[2]
        == "POSITIVO"

    and isinstance(
        bert_criterion,
        nn.CrossEntropyLoss
    )

    and bert_test_logits.shape[1] == 3
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 3 ===")

print(
    "¿Controles superados?:",
    g7_step3_ok
)


# ------------------------------------------------------------
# 14. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST usado para pesos de clase:",
    False
)

print(
    "TEST usado para cargar/configurar BERT:",
    False
)

print(
    "TEST usado en forward:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 15. ESTADO DE G7
# ------------------------------------------------------------

print("\n=== ESTADO DE G7 ===")

print(
    "BERT cargado:",
    "SÍ"
)

print(
    "Clasificación de 3 clases configurada:",
    "SÍ"
)

print(
    "CrossEntropyLoss ponderada preparada:",
    "SÍ"
)

print(
    "Pesos calculados solo con TRAIN:",
    "SÍ"
)

print(
    "BERT entrenado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 finalizado:",
    "NO"
)

=== PESOS DE CLASE BERT — SOLO TRAIN ===
Clase 0: 1.994965
Clase 1: 2.338843
Clase 2: 0.482817

=== DISTRIBUCIÓN DE TRAIN USADA PARA LOS PESOS ===
0     993
1     847
2    4103
Name: count, dtype: int64
Origen: y_train_bert


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== CONFIGURACIÓN DEL MODELO BERT ===
Modelo: google-bert/bert-base-uncased
Número de clases: 3
id2label: {0: 'NEGATIVO', 1: 'NEUTRAL', 2: 'POSITIVO'}
label2id: {'NEGATIVO': 0, 'NEUTRAL': 1, 'POSITIVO': 2}
Loss: CrossEntropyLoss
Pesos de clase: [1.9949647  2.3388429  0.48281744]

=== PARÁMETROS BERT ===
Parámetros totales: 109484547
Parámetros entrenables: 109484547

=== FORWARD DE PRUEBA ===
Forma logits: torch.Size([4, 3])
¿Hay 3 logits por reseña?: True

=== VALIDACIÓN AUTOMÁTICA DEL PASO 3 ===
¿Controles superados?: True

=== CONTROL DEL TEST SELLADO ===
TEST usado para pesos de clase: False
TEST usado para cargar/configurar BERT: False
TEST usado en forward: False
TEST evaluado: False

=== ESTADO DE G7 ===
BERT cargado: SÍ
Clasificación de 3 clases configurada: SÍ
CrossEntropyLoss ponderada preparada: SÍ
Pesos calculados solo con TRAIN: SÍ
BERT entrenado: NO
TEST evaluado: NO
G7 finalizado: NO


In [47]:
# ============================================================
# G7 — BERT
# Paso 4: fine-tuning real de BERT con TRAIN
#         y selección por F1 macro de VALIDATION
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Crea DataLoaders para TRAIN y VALIDATION con:
#
#       batch_size = 8
#       padding dinámico
#
# 2. Configura:
#
#       learning_rate = 2e-5
#       weight_decay = 0.01
#       gradient accumulation = 2
#       warmup ratio = 0.1
#       epochs = 3
#
# 3. Entrena BERT SOLO con TRAIN.
#
# 4. Utiliza la CrossEntropyLoss ponderada que ya
#    construimos exclusivamente con pesos de TRAIN.
#
# 5. Evalúa después de cada época en VALIDATION.
#
# 6. Selecciona el mejor checkpoint exclusivamente por:
#
#       F1 MACRO DE VALIDATION
#
# 7. Usa AMP únicamente porque CUDA está disponible.
#
# 8. Restaura al final el mejor checkpoint.
#
#
# IMPORTANTE:
#
# - TEST NO participa.
#
# - VALIDATION NO actualiza pesos.
#
# - TEST NO se tokeniza.
#
# - TEST NO se utiliza para seleccionar épocas.
#
# - La métrica principal continúa siendo F1 macro.
#
# - Esto SÍ es fine-tuning real:
#   los parámetros de BERT son entrenables.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos:
#
# - entrenar con VALIDATION;
# - seleccionar por accuracy;
# - utilizar TEST durante desarrollo;
# - olvidar los pesos de clase;
# - exceder memoria de GPU mediante acumulación de gradientes.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import copy
import math
import time
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from sklearn.metrics import f1_score

from transformers import (
    get_linear_schedule_with_warmup
)


# ------------------------------------------------------------
# 2. FIJAR SEMILLAS
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 3. CONFIGURACIÓN DE ENTRENAMIENTO
# ------------------------------------------------------------

BERT_BATCH_SIZE = 8

BERT_GRAD_ACCUM_STEPS = 2

BERT_LEARNING_RATE = 2e-5

BERT_WEIGHT_DECAY = 0.01

BERT_WARMUP_RATIO = 0.1

BERT_EPOCHS = 3

BERT_GRAD_CLIP = 1.0


print("=== CONFIGURACIÓN DE ENTRENAMIENTO BERT ===")

print(
    "Batch size:",
    BERT_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    BERT_GRAD_ACCUM_STEPS
)

print(
    "Batch efectivo:",
    BERT_BATCH_SIZE
    * BERT_GRAD_ACCUM_STEPS
)

print(
    "Learning rate:",
    BERT_LEARNING_RATE
)

print(
    "Weight decay:",
    BERT_WEIGHT_DECAY
)

print(
    "Warmup ratio:",
    BERT_WARMUP_RATIO
)

print(
    "Épocas:",
    BERT_EPOCHS
)

print(
    "Gradient clipping:",
    BERT_GRAD_CLIP
)

print(
    "Métrica de selección:",
    "F1 macro de VALIDATION"
)

print(
    "AMP:",
    torch.cuda.is_available()
)


# ------------------------------------------------------------
# 4. CREAR DATALOADER DE TRAIN
# ------------------------------------------------------------

# shuffle=True:
# TRAIN se mezcla en cada época.
#
# El padding dinámico se realiza mediante
# bert_data_collator.

bert_train_generator = torch.Generator()

bert_train_generator.manual_seed(
    SEED
)


train_loader_bert = DataLoader(
    train_dataset_bert,
    batch_size=BERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=bert_data_collator,
    generator=bert_train_generator
)


# ------------------------------------------------------------
# 5. CREAR DATALOADER DE VALIDATION
# ------------------------------------------------------------

# VALIDATION nunca se mezcla ni entrena.

val_loader_bert = DataLoader(
    val_dataset_bert,
    batch_size=BERT_BATCH_SIZE,
    shuffle=False,
    collate_fn=bert_data_collator
)


print("\n=== DATALOADERS BERT ===")

print(
    "Batches TRAIN:",
    len(train_loader_bert)
)

print(
    "Batches VALIDATION:",
    len(val_loader_bert)
)


# ------------------------------------------------------------
# 6. CREAR OPTIMIZADOR
# ------------------------------------------------------------

# AdamW aplica weight decay de forma adecuada.
#
# Solo recibe los parámetros del modelo BERT.

bert_optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=BERT_LEARNING_RATE,
    weight_decay=BERT_WEIGHT_DECAY
)


# ------------------------------------------------------------
# 7. CALCULAR PASOS TOTALES DEL OPTIMIZADOR
# ------------------------------------------------------------

# Como acumulamos gradientes durante 2 batches,
# optimizer.step() no ocurre en cada batch.
#
# Calculamos cuántas actualizaciones reales habrá.

updates_per_epoch = math.ceil(
    len(train_loader_bert)
    / BERT_GRAD_ACCUM_STEPS
)

total_training_steps = (
    updates_per_epoch
    * BERT_EPOCHS
)

warmup_steps = int(
    total_training_steps
    * BERT_WARMUP_RATIO
)


print("\n=== PASOS DE OPTIMIZACIÓN ===")

print(
    "Actualizaciones por época:",
    updates_per_epoch
)

print(
    "Total de actualizaciones:",
    total_training_steps
)

print(
    "Warmup steps:",
    warmup_steps
)


# ------------------------------------------------------------
# 8. CREAR SCHEDULER
# ------------------------------------------------------------

bert_scheduler = get_linear_schedule_with_warmup(
    optimizer=bert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)


# ------------------------------------------------------------
# 9. CONFIGURAR AMP
# ------------------------------------------------------------

# AMP se activa solo cuando CUDA está disponible.
#
# En Tesla T4 nos permite ahorrar memoria y acelerar
# el fine-tuning.

use_amp = torch.cuda.is_available()

if use_amp:

    bert_scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    bert_scaler = None


# ------------------------------------------------------------
# 10. FUNCIÓN DE ENTRENAMIENTO DE UNA ÉPOCA
# ------------------------------------------------------------

def train_bert_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scheduler,
    device,
    grad_accum_steps,
    grad_clip,
    use_amp,
    scaler
):

    model.train()

    optimizer.zero_grad()

    total_loss = 0.0

    total_examples = 0


    for step, batch in enumerate(
        loader,
        start=1
    ):

        # ----------------------------------------------------
        # MOVER EL BATCH A GPU/CPU
        # ----------------------------------------------------

        labels = batch[
            "labels"
        ].to(device)


        model_inputs = {
            key: value.to(device)
            for key, value
            in batch.items()
            if key != "labels"
        }


        # ----------------------------------------------------
        # FORWARD CON AMP
        # ----------------------------------------------------

        if use_amp:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(
                    **model_inputs
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels
                )

                loss_for_backward = (
                    loss
                    / grad_accum_steps
                )


            scaler.scale(
                loss_for_backward
            ).backward()


        else:

            outputs = model(
                **model_inputs
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

            loss_for_backward = (
                loss
                / grad_accum_steps
            )

            loss_for_backward.backward()


        # ----------------------------------------------------
        # ACUMULAR PÉRDIDA REAL
        # ----------------------------------------------------

        batch_size_current = (
            labels.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size_current
        )

        total_examples += (
            batch_size_current
        )


        # ----------------------------------------------------
        # ACTUALIZAR PESOS SOLO CUANDO TOCA
        # ----------------------------------------------------

        should_update = (
            step % grad_accum_steps == 0
            or step == len(loader)
        )


        if should_update:

            if use_amp:

                # Desescala antes del gradient clipping.

                scaler.unscale_(
                    optimizer
                )


            # -----------------------------------------------
            # GRADIENT CLIPPING
            # -----------------------------------------------

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=grad_clip
            )


            if use_amp:

                scaler.step(
                    optimizer
                )

                scaler.update()

            else:

                optimizer.step()


            scheduler.step()

            optimizer.zero_grad()


    epoch_loss = (
        total_loss
        / total_examples
    )


    return epoch_loss


# ------------------------------------------------------------
# 11. FUNCIÓN DE EVALUACIÓN EN VALIDATION
# ------------------------------------------------------------

def evaluate_bert(
    model,
    loader,
    criterion,
    device,
    use_amp
):

    model.eval()

    total_loss = 0.0

    total_examples = 0

    all_true = []

    all_pred = []


    with torch.no_grad():

        for batch in loader:

            labels = batch[
                "labels"
            ].to(device)


            model_inputs = {
                key: value.to(device)
                for key, value
                in batch.items()
                if key != "labels"
            }


            if use_amp:

                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    outputs = model(
                        **model_inputs
                    )

                    logits = outputs.logits

                    loss = criterion(
                        logits,
                        labels
                    )


            else:

                outputs = model(
                    **model_inputs
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels
                )


            predictions = torch.argmax(
                logits,
                dim=1
            )


            batch_size_current = (
                labels.size(0)
            )


            total_loss += (
                loss.item()
                * batch_size_current
            )

            total_examples += (
                batch_size_current
            )


            all_true.extend(
                labels
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            all_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


    val_loss = (
        total_loss
        / total_examples
    )


    val_f1_macro = f1_score(
        all_true,
        all_pred,
        average="macro",
        labels=[0, 1, 2],
        zero_division=0
    )


    return (
        val_loss,
        val_f1_macro,
        np.array(all_true),
        np.array(all_pred)
    )


# ------------------------------------------------------------
# 12. PREPARAR SELECCIÓN DEL MEJOR CHECKPOINT
# ------------------------------------------------------------

best_bert_val_f1 = -np.inf

best_bert_epoch = None

best_bert_state = None

bert_training_history = []


# ------------------------------------------------------------
# 13. REGISTRAR TIEMPO
# ------------------------------------------------------------

bert_training_start = time.time()


print("\n=== FINE-TUNING BERT ===")


# ------------------------------------------------------------
# 14. ENTRENAR DURANTE 3 ÉPOCAS
# ------------------------------------------------------------

for epoch in range(
    1,
    BERT_EPOCHS + 1
):


    # --------------------------------------------------------
    # 14.1 TRAIN
    # --------------------------------------------------------

    train_loss = train_bert_one_epoch(
        model=bert_model,
        loader=train_loader_bert,
        criterion=bert_criterion,
        optimizer=bert_optimizer,
        scheduler=bert_scheduler,
        device=device,
        grad_accum_steps=BERT_GRAD_ACCUM_STEPS,
        grad_clip=BERT_GRAD_CLIP,
        use_amp=use_amp,
        scaler=bert_scaler
    )


    # --------------------------------------------------------
    # 14.2 VALIDATION
    # --------------------------------------------------------

    (
        val_loss,
        val_f1_macro,
        _,
        _
    ) = evaluate_bert(
        model=bert_model,
        loader=val_loader_bert,
        criterion=bert_criterion,
        device=device,
        use_amp=use_amp
    )


    # --------------------------------------------------------
    # 14.3 GUARDAR HISTORIAL
    # --------------------------------------------------------

    bert_epoch_record = {
        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "val_f1_macro":
            val_f1_macro
    }


    bert_training_history.append(
        bert_epoch_record
    )


    print(
        f"Época {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f}"
    )


    # --------------------------------------------------------
    # 14.4 SELECCIONAR POR F1 MACRO
    # --------------------------------------------------------

    if val_f1_macro > best_bert_val_f1:

        best_bert_val_f1 = (
            val_f1_macro
        )

        best_bert_epoch = (
            epoch
        )

        best_bert_state = (
            copy.deepcopy(
                bert_model.state_dict()
            )
        )


        print(
            "  -> Nuevo mejor checkpoint "
            "por F1 macro de VALIDATION."
        )


# ------------------------------------------------------------
# 15. CALCULAR TIEMPO TOTAL
# ------------------------------------------------------------

bert_training_end = time.time()

bert_training_seconds = (
    bert_training_end
    - bert_training_start
)


# ------------------------------------------------------------
# 16. RESTAURAR EL MEJOR CHECKPOINT
# ------------------------------------------------------------

assert best_bert_state is not None, (
    "No se guardó ningún checkpoint BERT."
)


bert_model.load_state_dict(
    best_bert_state
)


# ------------------------------------------------------------
# 17. CONVERTIR HISTORIAL A DATAFRAME
# ------------------------------------------------------------

bert_training_history_df = (
    pd.DataFrame(
        bert_training_history
    )
)


# ------------------------------------------------------------
# 18. MOSTRAR HISTORIAL
# ------------------------------------------------------------

print("\n=== HISTORIAL BERT ===")

print(
    bert_training_history_df
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 19. MOSTRAR MEJOR CHECKPOINT
# ------------------------------------------------------------

print("\n=== MEJOR CHECKPOINT BERT ===")

print(
    "Mejor época:",
    best_bert_epoch
)

print(
    "Mejor F1 macro VALIDATION:",
    round(
        float(
            best_bert_val_f1
        ),
        4
    )
)

print(
    "Épocas ejecutadas:",
    len(
        bert_training_history_df
    )
)


# ------------------------------------------------------------
# 20. MOSTRAR TIEMPO
# ------------------------------------------------------------

print("\n=== TIEMPO BERT ===")

print(
    "Tiempo total de entrenamiento (segundos):",
    round(
        bert_training_seconds,
        2
    )
)


# ------------------------------------------------------------
# 21. COMPROBAR EL MODELO RESTAURADO
# ------------------------------------------------------------

(
    bert_best_val_loss_check,
    bert_best_val_f1_check,
    y_val_bert_true,
    y_val_bert_pred
) = evaluate_bert(
    model=bert_model,
    loader=val_loader_bert,
    criterion=bert_criterion,
    device=device,
    use_amp=use_amp
)


print("\n=== COMPROBACIÓN DEL CHECKPOINT BERT ===")

print(
    "F1 macro restaurado:",
    round(
        float(
            bert_best_val_f1_check
        ),
        4
    )
)

print(
    "F1 macro registrado como mejor:",
    round(
        float(
            best_bert_val_f1
        ),
        4
    )
)

print(
    "¿Coinciden?:",
    np.isclose(
        bert_best_val_f1_check,
        best_bert_val_f1
    )
)


# ------------------------------------------------------------
# 22. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_step4_ok = (

    len(
        bert_training_history_df
    ) == BERT_EPOCHS

    and BERT_BATCH_SIZE == 8

    and BERT_GRAD_ACCUM_STEPS == 2

    and BERT_LEARNING_RATE == 2e-5

    and BERT_WEIGHT_DECAY == 0.01

    and BERT_WARMUP_RATIO == 0.1

    and BERT_EPOCHS == 3

    and best_bert_epoch is not None

    and best_bert_state is not None

    and np.isclose(
        bert_best_val_f1_check,
        best_bert_val_f1
    )

    and use_amp
        == torch.cuda.is_available()
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 4 ===")

print(
    "¿Controles superados?:",
    g7_step4_ok
)


# ------------------------------------------------------------
# 23. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST incluido en DataLoader:",
    False
)

print(
    "TEST utilizado durante fine-tuning:",
    False
)

print(
    "TEST utilizado para seleccionar checkpoint:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 24. ESTADO DE G7
# ------------------------------------------------------------

print("\n=== ESTADO DE G7 ===")

print(
    "BERT fine-tuned:",
    "SÍ"
    if g7_step4_ok
    else "REVISAR"
)

print(
    "CrossEntropyLoss ponderada utilizada:",
    "SÍ"
)

print(
    "Gradient accumulation:",
    "SÍ"
)

print(
    "Warmup:",
    "SÍ"
)

print(
    "AMP con CUDA:",
    "SÍ"
    if use_amp
    else "NO APLICA"
)

print(
    "Selección por F1 macro de VALIDATION:",
    "SÍ"
)

print(
    "Mejor checkpoint restaurado:",
    "SÍ"
    if g7_step4_ok
    else "REVISAR"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 finalizado:",
    "TODAVÍA NO"
)

=== CONFIGURACIÓN DE ENTRENAMIENTO BERT ===
Batch size: 8
Gradient accumulation: 2
Batch efectivo: 16
Learning rate: 2e-05
Weight decay: 0.01
Warmup ratio: 0.1
Épocas: 3
Gradient clipping: 1.0
Métrica de selección: F1 macro de VALIDATION
AMP: True

=== DATALOADERS BERT ===
Batches TRAIN: 743
Batches VALIDATION: 245

=== PASOS DE OPTIMIZACIÓN ===
Actualizaciones por época: 372
Total de actualizaciones: 1116
Warmup steps: 111

=== FINE-TUNING BERT ===
Época 01 | train_loss=0.8028 | val_loss=0.6583 | val_f1_macro=0.6953
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 02 | train_loss=0.5051 | val_loss=0.6441 | val_f1_macro=0.7109
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 03 | train_loss=0.3358 | val_loss=0.7497 | val_f1_macro=0.7210
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.

=== HISTORIAL BERT ===
 epoch  train_loss  val_loss  val_f1_macro
     1      0.8028    0.6583        0.6953
     2      0.5051    0.6441        0.7109
     3      0.3358

In [48]:
# ============================================================
# G7 — BERT
# Paso 4: fine-tuning real de BERT
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Entrena google-bert/bert-base-uncased con TRAIN.
# VALIDATION se usa únicamente para evaluar cada época
# y seleccionar el mejor checkpoint por F1 macro.
#
# Configuración:
# - batch = 8
# - acumulación de gradientes = 2
# - lr = 2e-5
# - weight decay = 0.01
# - warmup = 0.1
# - épocas = 3
# - gradient clipping = 1.0
# - AMP solo con CUDA
#
# IMPORTANTE:
# - utiliza la CrossEntropyLoss ponderada ya creada
# - NO utiliza TEST
# - NO tokeniza TEST
# - NO evalúa TEST

import copy
import math
import time
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup


# ------------------------------------------------------------
# 1. SEMILLA
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 2. CONFIGURACIÓN
# ------------------------------------------------------------

BERT_BATCH_SIZE = 8
BERT_GRAD_ACCUM_STEPS = 2
BERT_LEARNING_RATE = 2e-5
BERT_WEIGHT_DECAY = 0.01
BERT_WARMUP_RATIO = 0.1
BERT_EPOCHS = 3
BERT_GRAD_CLIP = 1.0

print("=== CONFIGURACIÓN DE ENTRENAMIENTO BERT ===")
print("Batch size:", BERT_BATCH_SIZE)
print("Gradient accumulation:", BERT_GRAD_ACCUM_STEPS)
print(
    "Batch efectivo:",
    BERT_BATCH_SIZE * BERT_GRAD_ACCUM_STEPS
)
print("Learning rate:", BERT_LEARNING_RATE)
print("Weight decay:", BERT_WEIGHT_DECAY)
print("Warmup ratio:", BERT_WARMUP_RATIO)
print("Épocas:", BERT_EPOCHS)
print("Gradient clipping:", BERT_GRAD_CLIP)
print(
    "Métrica de selección:",
    "F1 macro de VALIDATION"
)
print(
    "AMP:",
    torch.cuda.is_available()
)


# ------------------------------------------------------------
# 3. DATALOADER TRAIN
# ------------------------------------------------------------

bert_train_generator = torch.Generator()
bert_train_generator.manual_seed(SEED)

train_loader_bert = DataLoader(
    train_dataset_bert,
    batch_size=BERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=bert_data_collator,
    generator=bert_train_generator
)


# ------------------------------------------------------------
# 4. DATALOADER VALIDATION
# ------------------------------------------------------------

val_loader_bert = DataLoader(
    val_dataset_bert,
    batch_size=BERT_BATCH_SIZE,
    shuffle=False,
    collate_fn=bert_data_collator
)

print("\n=== DATALOADERS BERT ===")
print(
    "Batches TRAIN:",
    len(train_loader_bert)
)
print(
    "Batches VALIDATION:",
    len(val_loader_bert)
)


# ------------------------------------------------------------
# 5. OPTIMIZADOR
# ------------------------------------------------------------

bert_optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=BERT_LEARNING_RATE,
    weight_decay=BERT_WEIGHT_DECAY
)


# ------------------------------------------------------------
# 6. PASOS DE OPTIMIZACIÓN
# ------------------------------------------------------------

updates_per_epoch = math.ceil(
    len(train_loader_bert)
    / BERT_GRAD_ACCUM_STEPS
)

total_training_steps = (
    updates_per_epoch
    * BERT_EPOCHS
)

warmup_steps = int(
    total_training_steps
    * BERT_WARMUP_RATIO
)

print("\n=== PASOS DE OPTIMIZACIÓN ===")
print(
    "Actualizaciones por época:",
    updates_per_epoch
)
print(
    "Total de actualizaciones:",
    total_training_steps
)
print(
    "Warmup steps:",
    warmup_steps
)


# ------------------------------------------------------------
# 7. SCHEDULER
# ------------------------------------------------------------

bert_scheduler = get_linear_schedule_with_warmup(
    optimizer=bert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)


# ------------------------------------------------------------
# 8. AMP
# ------------------------------------------------------------

use_amp = torch.cuda.is_available()

if use_amp:
    bert_scaler = torch.amp.GradScaler("cuda")
else:
    bert_scaler = None


# ------------------------------------------------------------
# 9. FUNCIÓN DE ENTRENAMIENTO
# ------------------------------------------------------------

def train_bert_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scheduler,
    device,
    grad_accum_steps,
    grad_clip,
    use_amp,
    scaler
):

    model.train()
    optimizer.zero_grad()

    total_loss = 0.0
    total_examples = 0

    for step, batch in enumerate(
        loader,
        start=1
    ):

        labels = batch["labels"].to(device)

        model_inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        if use_amp:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(
                    **model_inputs
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels
                )

                loss_for_backward = (
                    loss
                    / grad_accum_steps
                )

            scaler.scale(
                loss_for_backward
            ).backward()

        else:

            outputs = model(
                **model_inputs
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

            loss_for_backward = (
                loss
                / grad_accum_steps
            )

            loss_for_backward.backward()

        batch_size_current = labels.size(0)

        total_loss += (
            loss.item()
            * batch_size_current
        )

        total_examples += batch_size_current

        should_update = (
            step % grad_accum_steps == 0
            or step == len(loader)
        )

        if should_update:

            if use_amp:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=grad_clip
            )

            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad()

    return total_loss / total_examples


# ------------------------------------------------------------
# 10. FUNCIÓN DE VALIDACIÓN
# ------------------------------------------------------------

def evaluate_bert(
    model,
    loader,
    criterion,
    device,
    use_amp
):

    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_true = []
    all_pred = []

    with torch.no_grad():

        for batch in loader:

            labels = batch["labels"].to(device)

            model_inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            if use_amp:

                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    outputs = model(
                        **model_inputs
                    )

                    logits = outputs.logits

                    loss = criterion(
                        logits,
                        labels
                    )

            else:

                outputs = model(
                    **model_inputs
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels
                )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size_current = labels.size(0)

            total_loss += (
                loss.item()
                * batch_size_current
            )

            total_examples += batch_size_current

            all_true.extend(
                labels
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            all_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

    val_loss = (
        total_loss
        / total_examples
    )

    val_f1_macro = f1_score(
        all_true,
        all_pred,
        average="macro",
        labels=[0, 1, 2],
        zero_division=0
    )

    return (
        val_loss,
        val_f1_macro,
        np.array(all_true),
        np.array(all_pred)
    )


# ------------------------------------------------------------
# 11. VARIABLES PARA EL MEJOR CHECKPOINT
# ------------------------------------------------------------

best_bert_val_f1 = -np.inf
best_bert_epoch = None
best_bert_state = None

bert_training_history = []


# ------------------------------------------------------------
# 12. INICIAR CRONÓMETRO
# ------------------------------------------------------------

bert_training_start = time.time()

print("\n=== FINE-TUNING BERT ===")


# ------------------------------------------------------------
# 13. ENTRENAMIENTO — 3 ÉPOCAS
# ------------------------------------------------------------

for epoch in range(
    1,
    BERT_EPOCHS + 1
):

    train_loss = train_bert_one_epoch(
        model=bert_model,
        loader=train_loader_bert,
        criterion=bert_criterion,
        optimizer=bert_optimizer,
        scheduler=bert_scheduler,
        device=device,
        grad_accum_steps=BERT_GRAD_ACCUM_STEPS,
        grad_clip=BERT_GRAD_CLIP,
        use_amp=use_amp,
        scaler=bert_scaler
    )

    (
        val_loss,
        val_f1_macro,
        _,
        _
    ) = evaluate_bert(
        model=bert_model,
        loader=val_loader_bert,
        criterion=bert_criterion,
        device=device,
        use_amp=use_amp
    )

    bert_training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro
    })

    print(
        f"Época {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f}"
    )

    if val_f1_macro > best_bert_val_f1:

        best_bert_val_f1 = val_f1_macro
        best_bert_epoch = epoch

        best_bert_state = copy.deepcopy(
            bert_model.state_dict()
        )

        print(
            "  -> Nuevo mejor checkpoint "
            "por F1 macro de VALIDATION."
        )


# ------------------------------------------------------------
# 14. TIEMPO TOTAL
# ------------------------------------------------------------

bert_training_seconds = (
    time.time()
    - bert_training_start
)


# ------------------------------------------------------------
# 15. RESTAURAR MEJOR CHECKPOINT
# ------------------------------------------------------------

assert best_bert_state is not None, (
    "No se guardó ningún checkpoint BERT."
)

bert_model.load_state_dict(
    best_bert_state
)


# ------------------------------------------------------------
# 16. HISTORIAL
# ------------------------------------------------------------

bert_training_history_df = pd.DataFrame(
    bert_training_history
)

print("\n=== HISTORIAL BERT ===")

print(
    bert_training_history_df
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 17. MEJOR CHECKPOINT
# ------------------------------------------------------------

print("\n=== MEJOR CHECKPOINT BERT ===")

print(
    "Mejor época:",
    best_bert_epoch
)

print(
    "Mejor F1 macro VALIDATION:",
    round(
        float(best_bert_val_f1),
        4
    )
)

print(
    "Épocas ejecutadas:",
    len(bert_training_history_df)
)


# ------------------------------------------------------------
# 18. TIEMPO
# ------------------------------------------------------------

print("\n=== TIEMPO BERT ===")

print(
    "Tiempo total de entrenamiento (segundos):",
    round(
        bert_training_seconds,
        2
    )
)


# ------------------------------------------------------------
# 19. VERIFICAR CHECKPOINT RESTAURADO
# ------------------------------------------------------------

(
    bert_best_val_loss_check,
    bert_best_val_f1_check,
    y_val_bert_true,
    y_val_bert_pred
) = evaluate_bert(
    model=bert_model,
    loader=val_loader_bert,
    criterion=bert_criterion,
    device=device,
    use_amp=use_amp
)

print(
    "\n=== COMPROBACIÓN DEL CHECKPOINT BERT ==="
)

print(
    "F1 macro restaurado:",
    round(
        float(bert_best_val_f1_check),
        4
    )
)

print(
    "F1 macro registrado como mejor:",
    round(
        float(best_bert_val_f1),
        4
    )
)

print(
    "¿Coinciden?:",
    np.isclose(
        bert_best_val_f1_check,
        best_bert_val_f1
    )
)


# ------------------------------------------------------------
# 20. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_step4_ok = (
    len(bert_training_history_df)
        == BERT_EPOCHS

    and BERT_BATCH_SIZE == 8

    and BERT_GRAD_ACCUM_STEPS == 2

    and BERT_LEARNING_RATE == 2e-5

    and BERT_WEIGHT_DECAY == 0.01

    and BERT_WARMUP_RATIO == 0.1

    and BERT_EPOCHS == 3

    and best_bert_epoch is not None

    and best_bert_state is not None

    and np.isclose(
        bert_best_val_f1_check,
        best_bert_val_f1
    )

    and use_amp
        == torch.cuda.is_available()
)

print(
    "\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 4 ==="
)

print(
    "¿Controles superados?:",
    g7_step4_ok
)


# ------------------------------------------------------------
# 21. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "TEST incluido en DataLoader:",
    False
)

print(
    "TEST utilizado durante fine-tuning:",
    False
)

print(
    "TEST utilizado para seleccionar checkpoint:",
    False
)

print(
    "TEST evaluado:",
    False
)


# ------------------------------------------------------------
# 22. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO DE G7 ===")

print(
    "BERT fine-tuned:",
    "SÍ" if g7_step4_ok else "REVISAR"
)

print(
    "CrossEntropyLoss ponderada utilizada:",
    "SÍ"
)

print(
    "Gradient accumulation:",
    "SÍ"
)

print(
    "Warmup:",
    "SÍ"
)

print(
    "AMP con CUDA:",
    "SÍ" if use_amp else "NO APLICA"
)

print(
    "Selección por F1 macro de VALIDATION:",
    "SÍ"
)

print(
    "Mejor checkpoint restaurado:",
    "SÍ" if g7_step4_ok else "REVISAR"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 finalizado:",
    "TODAVÍA NO"
)

=== CONFIGURACIÓN DE ENTRENAMIENTO BERT ===
Batch size: 8
Gradient accumulation: 2
Batch efectivo: 16
Learning rate: 2e-05
Weight decay: 0.01
Warmup ratio: 0.1
Épocas: 3
Gradient clipping: 1.0
Métrica de selección: F1 macro de VALIDATION
AMP: True

=== DATALOADERS BERT ===
Batches TRAIN: 743
Batches VALIDATION: 245

=== PASOS DE OPTIMIZACIÓN ===
Actualizaciones por época: 372
Total de actualizaciones: 1116
Warmup steps: 111

=== FINE-TUNING BERT ===
Época 01 | train_loss=0.2921 | val_loss=1.0259 | val_f1_macro=0.7212
  -> Nuevo mejor checkpoint por F1 macro de VALIDATION.
Época 02 | train_loss=0.2297 | val_loss=1.1887 | val_f1_macro=0.7085
Época 03 | train_loss=0.1412 | val_loss=1.4585 | val_f1_macro=0.7120

=== HISTORIAL BERT ===
 epoch  train_loss  val_loss  val_f1_macro
     1      0.2921    1.0259        0.7212
     2      0.2297    1.1887        0.7085
     3      0.1412    1.4585        0.7120

=== MEJOR CHECKPOINT BERT ===
Mejor época: 1
Mejor F1 macro VALIDATION: 0.7212
Épocas 

In [49]:
# ============================================================
# G7 — BERT
# Paso 5: evaluar el mejor checkpoint BERT en VALIDATION
#         con la misma función común de métricas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Utiliza el mejor checkpoint BERT ya restaurado.
#
# 2. Genera predicciones sobre VALIDATION.
#
# 3. Evalúa esas predicciones con EXACTAMENTE la misma
#    función común utilizada para:
#
#       - clase mayoritaria
#       - TF-IDF + Logistic Regression
#       - Word2Vec + BiLSTM
#
# 4. Calcula:
#
#       - accuracy
#       - balanced accuracy
#       - precision macro
#       - recall macro
#       - F1 macro
#       - F1 weighted
#
# 5. Genera:
#
#       - reporte por clase
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 6. Compara todos los modelos únicamente en VALIDATION.
#
#
# IMPORTANTE:
#
# - NO se vuelve a entrenar BERT.
#
# - NO se modifica el checkpoint.
#
# - TEST NO se utiliza.
#
# - F1 macro sigue siendo la métrica principal.
#
# - La conclusión debe limitarse a:
#
#       "mejor en este split y configuración"


# ------------------------------------------------------------
# 1. GENERAR PREDICCIONES DEL CHECKPOINT RESTAURADO
# ------------------------------------------------------------

(
    bert_val_loss_final,
    bert_val_f1_macro_final,
    y_val_bert_true,
    y_val_bert_pred
) = evaluate_bert(
    model=bert_model,
    loader=val_loader_bert,
    criterion=bert_criterion,
    device=device,
    use_amp=use_amp
)


# ------------------------------------------------------------
# 2. COMPROBAR TAMAÑOS
# ------------------------------------------------------------

print("=== PREDICCIONES BERT — VALIDATION ===")

print(
    "Etiquetas reales:",
    len(y_val_bert_true)
)

print(
    "Predicciones:",
    len(y_val_bert_pred)
)

print(
    "Filas VALIDATION:",
    len(y_val)
)

print(
    "¿Coinciden?:",
    len(y_val_bert_true)
    == len(y_val_bert_pred)
    == len(y_val)
)


# ------------------------------------------------------------
# 3. EVALUAR CON LA FUNCIÓN COMÚN
# ------------------------------------------------------------

(
    bert_metrics,
    bert_report,
    bert_cm,
    bert_cm_norm
) = evaluate_predictions(
    y_val_bert_true,
    y_val_bert_pred
)


# ------------------------------------------------------------
# 4. MOSTRAR MÉTRICAS GENERALES
# ------------------------------------------------------------

print(
    "\n=== BERT — MÉTRICAS EN VALIDATION ==="
)

for metric_name, metric_value in bert_metrics.items():

    print(
        f"{metric_name}: "
        f"{metric_value:.4f}"
    )


# ------------------------------------------------------------
# 5. COMPROBAR CONSISTENCIA DEL F1 MACRO
# ------------------------------------------------------------

print(
    "\n=== CONSISTENCIA DEL CHECKPOINT BERT ==="
)

print(
    "F1 macro usado para seleccionar checkpoint:",
    round(
        float(best_bert_val_f1),
        4
    )
)

print(
    "F1 macro recalculado con función común:",
    round(
        float(bert_metrics["f1_macro"]),
        4
    )
)

print(
    "¿Coinciden?:",
    np.isclose(
        best_bert_val_f1,
        bert_metrics["f1_macro"]
    )
)


# ------------------------------------------------------------
# 6. REPORTE POR CLASE
# ------------------------------------------------------------

bert_report_df = (
    pd.DataFrame(bert_report)
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — BERT VALIDATION ==="
)

print(
    bert_report_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 7. MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

bert_cm_df = pd.DataFrame(
    bert_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA — BERT ==="
)

print(
    bert_cm_df.to_string()
)


# ------------------------------------------------------------
# 8. MATRIZ DE CONFUSIÓN NORMALIZADA
# ------------------------------------------------------------

bert_cm_norm_df = pd.DataFrame(
    bert_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA — BERT ==="
)

print(
    bert_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. COMPARACIÓN DE TODOS LOS MODELOS EN VALIDATION
# ------------------------------------------------------------

g7_validation_comparison = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "TF-IDF + Logistic Regression",
        "Word2Vec + BiLSTM",
        "BERT"
    ],

    "accuracy": [
        majority_metrics["accuracy"],
        logreg_metrics["accuracy"],
        bilstm_metrics["accuracy"],
        bert_metrics["accuracy"]
    ],

    "balanced_accuracy": [
        majority_metrics["balanced_accuracy"],
        logreg_metrics["balanced_accuracy"],
        bilstm_metrics["balanced_accuracy"],
        bert_metrics["balanced_accuracy"]
    ],

    "precision_macro": [
        majority_metrics["precision_macro"],
        logreg_metrics["precision_macro"],
        bilstm_metrics["precision_macro"],
        bert_metrics["precision_macro"]
    ],

    "recall_macro": [
        majority_metrics["recall_macro"],
        logreg_metrics["recall_macro"],
        bilstm_metrics["recall_macro"],
        bert_metrics["recall_macro"]
    ],

    "f1_macro": [
        majority_metrics["f1_macro"],
        logreg_metrics["f1_macro"],
        bilstm_metrics["f1_macro"],
        bert_metrics["f1_macro"]
    ],

    "f1_weighted": [
        majority_metrics["f1_weighted"],
        logreg_metrics["f1_weighted"],
        bilstm_metrics["f1_weighted"],
        bert_metrics["f1_weighted"]
    ]
})


print(
    "\n=== COMPARACIÓN GLOBAL EN VALIDATION ==="
)

print(
    g7_validation_comparison
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. IDENTIFICAR EL MEJOR POR F1 MACRO
# ------------------------------------------------------------

best_validation_row = (
    g7_validation_comparison
    .sort_values(
        "f1_macro",
        ascending=False
    )
    .iloc[0]
)

print(
    "\n=== MEJOR MODELO EN VALIDATION POR F1 MACRO ==="
)

print(
    "Modelo:",
    best_validation_row["modelo"]
)

print(
    "F1 macro:",
    round(
        float(
            best_validation_row["f1_macro"]
        ),
        4
    )
)

print(
    "Interpretación:",
    "mejor en este split y configuración"
)


# ------------------------------------------------------------
# 11. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print(
    "\n=== CONTROL DEL TEST SELLADO ==="
)

print(
    "Conjunto evaluado:",
    "VALIDATION"
)

print(
    "TEST utilizado:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 12. ESTADO DE G7
# ------------------------------------------------------------

print(
    "\n=== ESTADO DE G7 ==="
)

print(
    "BERT fine-tuned:",
    "SÍ"
)

print(
    "Mejor checkpoint evaluado:",
    "SÍ"
)

print(
    "Métricas homogéneas calculadas:",
    "SÍ"
)

print(
    "Comparación por F1 macro realizada:",
    "SÍ"
)

print(
    "TEST evaluado:",
    "NO"
)

print(
    "G7 listo para persistencia:",
    "SÍ"
)

print(
    "G7 aprobado:",
    "TODAVÍA NO"
)

=== PREDICCIONES BERT — VALIDATION ===
Etiquetas reales: 1956
Predicciones: 1956
Filas VALIDATION: 1956
¿Coinciden?: True

=== BERT — MÉTRICAS EN VALIDATION ===
accuracy: 0.8108
balanced_accuracy: 0.7289
precision_macro: 0.7149
recall_macro: 0.7289
f1_macro: 0.7212
f1_weighted: 0.8151

=== CONSISTENCIA DEL CHECKPOINT BERT ===
F1 macro usado para seleccionar checkpoint: 0.7212
F1 macro recalculado con función común: 0.7212
¿Coinciden?: True

=== REPORTE POR CLASE — BERT VALIDATION ===
              precision  recall  f1-score    support
NEGATIVO         0.7715  0.7879    0.7796   330.0000
NEUTRAL          0.4564  0.5131    0.4831   306.0000
POSITIVO         0.9169  0.8856    0.9010  1320.0000
accuracy         0.8108  0.8108    0.8108     0.8108
macro avg        0.7149  0.7289    0.7212  1956.0000
weighted avg     0.8203  0.8108    0.8151  1956.0000

=== MATRIZ DE CONFUSIÓN ABSOLUTA — BERT ===
               Pred_NEGATIVO  Pred_NEUTRAL  Pred_POSITIVO
Real_NEGATIVO            260         

In [50]:
# ============================================================
# G7 — BERT
# Paso 6: persistir modelo, tokenizer, métricas,
#         predicciones, historial y configuración
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Guarda el mejor modelo BERT ya restaurado.
#
# 2. Guarda el tokenizer oficial.
#
# 3. Guarda el historial de entrenamiento.
#
# 4. Guarda las métricas de VALIDATION.
#
# 5. Guarda las predicciones de VALIDATION.
#
# 6. Guarda:
#       - reporte por clase
#       - matriz de confusión absoluta
#       - matriz de confusión normalizada
#
# 7. Guarda la configuración completa de G7.
#
# 8. Comprueba que los artefactos realmente existen.
#
#
# IMPORTANTE:
#
# - NO vuelve a entrenar BERT.
#
# - NO modifica TRAIN.
#
# - NO modifica VALIDATION.
#
# - NO tokeniza TEST.
#
# - NO evalúa TEST.
#
# - review_id se usa únicamente para trazabilidad.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos perder el mejor checkpoint de BERT y dejamos
# evidencia reproducible para la auditoría independiente.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import json
import pandas as pd
import numpy as np
import torch


# ------------------------------------------------------------
# 2. DEFINIR DIRECTORIOS Y RUTAS
# ------------------------------------------------------------

G7_MODEL_DIR = (
    ARTIFACT_DIR
    / "g7_bert_best_model"
)

G7_TOKENIZER_DIR = (
    ARTIFACT_DIR
    / "g7_bert_tokenizer"
)

G7_HISTORY_PATH = (
    ARTIFACT_DIR
    / "g7_training_history.csv"
)

G7_METRICS_PATH = (
    ARTIFACT_DIR
    / "g7_metricas_validation.csv"
)

G7_PREDICTIONS_PATH = (
    ARTIFACT_DIR
    / "g7_predicciones_validation.csv"
)

G7_REPORT_PATH = (
    ARTIFACT_DIR
    / "g7_reporte_clases_validation.csv"
)

G7_CM_PATH = (
    ARTIFACT_DIR
    / "g7_matriz_confusion_validation.csv"
)

G7_CM_NORM_PATH = (
    ARTIFACT_DIR
    / "g7_matriz_confusion_validation_normalizada.csv"
)

G7_CONFIG_PATH = (
    ARTIFACT_DIR
    / "g7_configuracion.json"
)


# ------------------------------------------------------------
# 3. GUARDAR EL MEJOR MODELO BERT
# ------------------------------------------------------------

# save_pretrained() guarda:
#
# - pesos
# - configuración
#
# del checkpoint que ya fue restaurado como mejor.

bert_model.save_pretrained(
    G7_MODEL_DIR
)


# ------------------------------------------------------------
# 4. GUARDAR TOKENIZER
# ------------------------------------------------------------

bert_tokenizer.save_pretrained(
    G7_TOKENIZER_DIR
)


# ------------------------------------------------------------
# 5. GUARDAR HISTORIAL DE ENTRENAMIENTO
# ------------------------------------------------------------

bert_training_history_df.to_csv(
    G7_HISTORY_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. GUARDAR MÉTRICAS DE VALIDATION
# ------------------------------------------------------------

g7_metrics_df = pd.DataFrame([
    {
        "modelo":
            "BERT",

        "model_name":
            BERT_MODEL_NAME,

        "accuracy":
            bert_metrics["accuracy"],

        "balanced_accuracy":
            bert_metrics["balanced_accuracy"],

        "precision_macro":
            bert_metrics["precision_macro"],

        "recall_macro":
            bert_metrics["recall_macro"],

        "f1_macro":
            bert_metrics["f1_macro"],

        "f1_weighted":
            bert_metrics["f1_weighted"],

        "best_epoch":
            int(best_bert_epoch),

        "training_seconds":
            float(bert_training_seconds)
    }
])


g7_metrics_df.to_csv(
    G7_METRICS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. GUARDAR PREDICCIONES DE VALIDATION
# ------------------------------------------------------------

# review_id aparece únicamente como identificador.
#
# No fue predictor de BERT.

g7_predictions_df = pd.DataFrame({
    "review_id":
        val_g4["review_id"].values,

    "y_true":
        y_val_bert_true,

    "y_pred_bert":
        y_val_bert_pred
})


g7_predictions_df.to_csv(
    G7_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. GUARDAR REPORTE POR CLASE
# ------------------------------------------------------------

bert_report_df.to_csv(
    G7_REPORT_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. GUARDAR MATRIZ DE CONFUSIÓN ABSOLUTA
# ------------------------------------------------------------

bert_cm_df.to_csv(
    G7_CM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 10. GUARDAR MATRIZ NORMALIZADA
# ------------------------------------------------------------

bert_cm_norm_df.to_csv(
    G7_CM_NORM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 11. GUARDAR CONFIGURACIÓN COMPLETA DE G7
# ------------------------------------------------------------

g7_config = {

    "stage":
        "G7",

    "model_name":
        BERT_MODEL_NAME,

    "seed":
        42,

    "primary_metric":
        "f1_macro",

    "input_predictor":
        "text_bert",

    "labels": {
        "0": "NEGATIVO",
        "1": "NEUTRAL",
        "2": "POSITIVO"
    },

    "tokenization": {

        "tokenizer":
            BERT_MODEL_NAME,

        "max_length":
            256,

        "padding":
            "dynamic",

        "truncation":
            True,

        "train_truncation_pct":
            float(
                train_bert_truncated_pct
            ),

        "validation_truncation_pct":
            float(
                val_bert_truncated_pct
            )
    },

    "training": {

        "batch_size":
            8,

        "gradient_accumulation_steps":
            2,

        "effective_batch_size":
            16,

        "learning_rate":
            2e-5,

        "weight_decay":
            0.01,

        "warmup_ratio":
            0.1,

        "epochs":
            3,

        "gradient_clip":
            1.0,

        "amp":
            bool(use_amp)
    },

    "loss":
        "CrossEntropyLoss weighted",

    "class_weights": [
        float(x)
        for x in bert_class_weights_np
    ],

    "class_weights_source":
        "train",

    "selection_metric":
        "f1_macro_validation",

    "best_epoch":
        int(best_bert_epoch),

    "best_val_f1_macro":
        float(best_bert_val_f1),

    "evaluation_split":
        "validation",

    "test_used":
        False
}


with open(
    G7_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        g7_config,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 12. COMPROBAR ARTEFACTOS
# ------------------------------------------------------------

g7_artifacts = {

    "Modelo BERT":
        G7_MODEL_DIR,

    "Tokenizer BERT":
        G7_TOKENIZER_DIR,

    "Historial":
        G7_HISTORY_PATH,

    "Métricas VALIDATION":
        G7_METRICS_PATH,

    "Predicciones VALIDATION":
        G7_PREDICTIONS_PATH,

    "Reporte por clase":
        G7_REPORT_PATH,

    "Matriz confusión":
        G7_CM_PATH,

    "Matriz confusión normalizada":
        G7_CM_NORM_PATH,

    "Configuración":
        G7_CONFIG_PATH
}


print("=== ARTEFACTOS DE G7 ===")

all_g7_artifacts_exist = True


for name, path in g7_artifacts.items():

    exists = path.exists()

    all_g7_artifacts_exist = (
        all_g7_artifacts_exist
        and exists
    )

    print(
        f"{name}:",
        exists
    )


# ------------------------------------------------------------
# 13. RECUPERAR MÉTRICAS Y PREDICCIONES
# ------------------------------------------------------------

g7_metrics_check = pd.read_csv(
    G7_METRICS_PATH
)

g7_predictions_check = pd.read_csv(
    G7_PREDICTIONS_PATH
)

g7_history_check = pd.read_csv(
    G7_HISTORY_PATH
)


# ------------------------------------------------------------
# 14. COMPROBAR RECUPERACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Filas métricas:",
    len(g7_metrics_check)
)

print(
    "Predicciones recuperadas:",
    len(g7_predictions_check)
)

print(
    "Filas VALIDATION:",
    len(y_val)
)

print(
    "¿Predicciones coinciden?:",
    len(g7_predictions_check)
    == len(y_val)
)

print(
    "review_id duplicados:",
    int(
        g7_predictions_check[
            "review_id"
        ]
        .duplicated()
        .sum()
    )
)

print(
    "Épocas recuperadas:",
    len(g7_history_check)
)


# ------------------------------------------------------------
# 15. MOSTRAR MÉTRICAS PERSISTIDAS
# ------------------------------------------------------------

print("\n=== MÉTRICAS G7 PERSISTIDAS ===")

print(
    g7_metrics_check
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 16. COMPROBAR CONFIGURACIÓN GUARDADA
# ------------------------------------------------------------

with open(
    G7_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    g7_config_check = json.load(f)


print("\n=== CONFIGURACIÓN RECUPERADA ===")

print(
    "Modelo:",
    g7_config_check[
        "model_name"
    ]
)

print(
    "Métrica principal:",
    g7_config_check[
        "primary_metric"
    ]
)

print(
    "MAX_LENGTH:",
    g7_config_check[
        "tokenization"
    ][
        "max_length"
    ]
)

print(
    "Batch:",
    g7_config_check[
        "training"
    ][
        "batch_size"
    ]
)

print(
    "Acumulación:",
    g7_config_check[
        "training"
    ][
        "gradient_accumulation_steps"
    ]
)

print(
    "Learning rate:",
    g7_config_check[
        "training"
    ][
        "learning_rate"
    ]
)

print(
    "Weight decay:",
    g7_config_check[
        "training"
    ][
        "weight_decay"
    ]
)

print(
    "Warmup:",
    g7_config_check[
        "training"
    ][
        "warmup_ratio"
    ]
)

print(
    "Mejor época:",
    g7_config_check[
        "best_epoch"
    ]
)

print(
    "Mejor F1 macro:",
    round(
        g7_config_check[
            "best_val_f1_macro"
        ],
        4
    )
)


# ------------------------------------------------------------
# 17. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g7_persistence_ok = (

    all_g7_artifacts_exist

    and len(g7_metrics_check) == 1

    and len(g7_predictions_check)
        == len(y_val)

    and g7_predictions_check[
        "review_id"
    ].duplicated().sum() == 0

    and len(g7_history_check)
        == BERT_EPOCHS

    and g7_config_check[
        "model_name"
    ] == BERT_MODEL_NAME

    and g7_config_check[
        "primary_metric"
    ] == "f1_macro"

    and g7_config_check[
        "best_epoch"
    ] == best_bert_epoch

    and np.isclose(
        g7_config_check[
            "best_val_f1_macro"
        ],
        best_bert_val_f1
    )
)


print("\n=== RESULTADO DE PERSISTENCIA G7 ===")

print(
    "¿Controles superados?:",
    g7_persistence_ok
)

print(
    "G7 listo para auditoría independiente:",
    "SÍ"
    if g7_persistence_ok
    else "NO"
)


# ------------------------------------------------------------
# 18. CONTROL DEL TEST SELLADO
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "TEST incluido en predicciones:",
    "NO"
)

print(
    "TEST usado para seleccionar checkpoint:",
    "NO"
)

print(
    "TEST evaluado:",
    "NO"
)


# ------------------------------------------------------------
# 19. ESTADO DE G7
# ------------------------------------------------------------

print("\n=== ESTADO DE G7 ===")

print(
    "Modelo BERT persistido:",
    "SÍ"
)

print(
    "Tokenizer persistido:",
    "SÍ"
)

print(
    "Historial persistido:",
    "SÍ"
)

print(
    "Predicciones VALIDATION persistidas:",
    "SÍ"
)

print(
    "Métricas VALIDATION persistidas:",
    "SÍ"
)

print(
    "G7 listo para auditoría:",
    "SÍ"
    if g7_persistence_ok
    else "NO"
)

print(
    "G7 aprobado:",
    "TODAVÍA NO"
)

print(
    "G8 iniciado:",
    "NO"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

=== ARTEFACTOS DE G7 ===
Modelo BERT: True
Tokenizer BERT: True
Historial: True
Métricas VALIDATION: True
Predicciones VALIDATION: True
Reporte por clase: True
Matriz confusión: True
Matriz confusión normalizada: True
Configuración: True

=== CONTROL DE RECUPERACIÓN ===
Filas métricas: 1
Predicciones recuperadas: 1956
Filas VALIDATION: 1956
¿Predicciones coinciden?: True
review_id duplicados: 0
Épocas recuperadas: 3

=== MÉTRICAS G7 PERSISTIDAS ===
modelo                    model_name  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  f1_weighted  best_epoch  training_seconds
  BERT google-bert/bert-base-uncased    0.8108             0.7289           0.7149        0.7289    0.7212       0.8151           1          274.2617

=== CONFIGURACIÓN RECUPERADA ===
Modelo: google-bert/bert-base-uncased
Métrica principal: f1_macro
MAX_LENGTH: 256
Batch: 8
Acumulación: 2
Learning rate: 2e-05
Weight decay: 0.01
Warmup: 0.1
Mejor época: 1
Mejor F1 macro: 0.7212

=== RESULTADO DE

In [51]:
# ============================================================
# G7 — CONTROL FINAL DE COMPUERTA
# Evidencia verificable de que G8 todavía NO se ha iniciado
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Busca archivos/artefactos cuyo nombre empiece por g8_.
#
# 2. Comprueba si existen variables típicas que indicarían
#    que TEST ya fue preparado o evaluado en G8.
#
# 3. NO crea ningún Dataset de TEST.
#
# 4. NO tokeniza TEST.
#
# 5. NO genera predicciones de TEST.
#
# 6. NO evalúa TEST.
#
#
# IMPORTANTE:
#
# Esta celda NO inicia G8.
#
# Solo inspecciona el estado actual de la sesión y de la
# carpeta de artefactos.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Creamos evidencia objetiva de que G8 todavía no comenzó,
# sin abrir ni utilizar TEST.


# ------------------------------------------------------------
# 1. BUSCAR ARTEFACTOS DE G8
# ------------------------------------------------------------

g8_artifacts_found = list(
    ARTIFACT_DIR.glob("g8_*")
)


print("=== ARTEFACTOS G8 ===")

print(
    "Cantidad de artefactos g8_*:",
    len(g8_artifacts_found)
)

print(
    "Artefactos encontrados:",
    [
        path.name
        for path in g8_artifacts_found
    ]
)


# ------------------------------------------------------------
# 2. VARIABLES QUE INDICARÍAN QUE G8 YA COMENZÓ
# ------------------------------------------------------------

g8_variable_names = [

    # posibles objetos de TEST BERT
    "test_dataset_bert",
    "test_loader_bert",
    "X_test_bert_text",
    "y_test_bert",

    # posibles secuencias del modelo secuencial
    "X_test_seq",
    "X_test_tensor",
    "test_dataset_seq",
    "test_loader_seq",

    # predicciones finales
    "y_test_pred_majority",
    "y_test_pred_logreg",
    "y_test_pred_bilstm",
    "y_test_pred_bert",

    # métricas finales
    "test_metrics",
    "bert_test_metrics",
    "bilstm_test_metrics",

    # comparación final
    "g8_test_comparison"
]


g8_variables_present = {
    name: name in globals()
    for name in g8_variable_names
}


print("\n=== VARIABLES G8 ===")

for name, present in g8_variables_present.items():

    print(
        f"{name}:",
        present
    )


# ------------------------------------------------------------
# 3. DETERMINAR SI EXISTE EVIDENCIA DE G8 INICIADO
# ------------------------------------------------------------

g8_started_evidence = (

    len(g8_artifacts_found) > 0

    or any(
        g8_variables_present.values()
    )
)


print("\n=== RESULTADO DEL CONTROL ===")

print(
    "¿Existe evidencia verificable de que G8 haya iniciado?:",
    g8_started_evidence
)

print(
    "¿G8 permanece no iniciado según los objetos revisados?:",
    not g8_started_evidence
)


# ------------------------------------------------------------
# 4. CONFIRMAR QUE ESTA CELDA NO UTILIZA TEST
# ------------------------------------------------------------

print("\n=== CONTROL DEL TEST SELLADO ===")

print(
    "Dataset TEST creado por esta celda:",
    False
)

print(
    "TEST tokenizado por esta celda:",
    False
)

print(
    "Predicciones TEST generadas por esta celda:",
    False
)

print(
    "TEST evaluado por esta celda:",
    False
)


# ------------------------------------------------------------
# 5. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO DE COMPUERTA ===")

print(
    "G7 permanece pendiente únicamente de auditoría final."
)

print(
    "G8 iniciado por esta celda:",
    False
)

print(
    "TEST evaluado:",
    False
)

=== ARTEFACTOS G8 ===
Cantidad de artefactos g8_*: 0
Artefactos encontrados: []

=== VARIABLES G8 ===
test_dataset_bert: False
test_loader_bert: False
X_test_bert_text: False
y_test_bert: False
X_test_seq: False
X_test_tensor: False
test_dataset_seq: False
test_loader_seq: False
y_test_pred_majority: False
y_test_pred_logreg: False
y_test_pred_bilstm: False
y_test_pred_bert: False
test_metrics: False
bert_test_metrics: False
bilstm_test_metrics: False
g8_test_comparison: False

=== RESULTADO DEL CONTROL ===
¿Existe evidencia verificable de que G8 haya iniciado?: False
¿G8 permanece no iniciado según los objetos revisados?: True

=== CONTROL DEL TEST SELLADO ===
Dataset TEST creado por esta celda: False
TEST tokenizado por esta celda: False
Predicciones TEST generadas por esta celda: False
TEST evaluado por esta celda: False

=== ESTADO DE COMPUERTA ===
G7 permanece pendiente únicamente de auditoría final.
G8 iniciado por esta celda: False
TEST evaluado: False


In [52]:
# ============================================================
# G8 — EVALUACIÓN FINAL EN TEST
# Paso 1: abrir y preparar TEST con los pipelines ya aprobados
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Recupera el TEST sellado creado y aprobado en G3.
#
# 2. Comprueba que contiene exactamente las 2.099 filas
#    esperadas.
#
# 3. Aplica al TEST las DOS funciones de texto ya aprobadas:
#
#       clean_text_sequential()
#       clean_text_bert()
#
# 4. Genera:
#
#       text_sequential
#       text_bert
#
# 5. Comprueba que no existan textos vacíos.
#
# 6. Comprueba que las etiquetas 0, 1 y 2 estén presentes.
#
# 7. Verifica que TEST no modifica:
#
#       - Word2Vec
#       - vocabulario BiLSTM
#       - TF-IDF
#       - Logistic Regression
#       - BERT
#       - pesos de clase
#
#
# MUY IMPORTANTE:
#
# Esta celda ABRE TEST únicamente para preparar sus entradas.
#
# TODAVÍA:
#
# - NO genera predicciones.
# - NO calcula accuracy.
# - NO calcula F1.
# - NO calcula matrices de confusión.
# - NO selecciona ningún modelo.
#
#
# ¿POR QUÉ LO HACEMOS ASÍ?
#
# Todos los modelos y decisiones ya quedaron cerrados
# utilizando TRAIN y VALIDATION.
#
# TEST puede ahora transformarse usando exclusivamente
# objetos ya aprendidos desde TRAIN.
#
#
# ¿QUÉ RIESGO CONTROLAMOS?
#
# Evitamos que TEST se convierta accidentalmente en un
# nuevo conjunto de desarrollo.
#
# Desde G8:
#
# TEST solo sirve para la evaluación FINAL.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINIR LA RUTA DEL TEST SELLADO
# ------------------------------------------------------------

G3_TEST_PATH = (
    ARTIFACT_DIR
    / "g3_test_SELLADO.csv"
)


print("=== APERTURA CONTROLADA DE TEST ===")

print(
    "Archivo TEST existe:",
    G3_TEST_PATH.exists()
)

print(
    "Ruta:",
    G3_TEST_PATH
)


# ------------------------------------------------------------
# 3. COMPROBAR QUE EL ARCHIVO EXISTE
# ------------------------------------------------------------

assert G3_TEST_PATH.exists(), (
    "No se encuentra g3_test_SELLADO.csv. "
    "No continúes hasta recuperar el artefacto aprobado de G3."
)


# ------------------------------------------------------------
# 4. RECUPERAR TEST
# ------------------------------------------------------------

test_g8 = pd.read_csv(
    G3_TEST_PATH
)


print("\n=== TEST RECUPERADO ===")

print(
    "Filas:",
    len(test_g8)
)

print(
    "Columnas:",
    len(test_g8.columns)
)


# ------------------------------------------------------------
# 5. COMPROBAR EL TAMAÑO ESPERADO
# ------------------------------------------------------------

EXPECTED_TEST_ROWS = 2099


print(
    "Filas esperadas:",
    EXPECTED_TEST_ROWS
)

print(
    "¿Coincide el tamaño?:",
    len(test_g8)
    == EXPECTED_TEST_ROWS
)


assert len(test_g8) == EXPECTED_TEST_ROWS, (
    "El TEST recuperado no contiene las 2.099 filas "
    "aprobadas en G3."
)


# ------------------------------------------------------------
# 6. COMPROBAR COLUMNAS NECESARIAS
# ------------------------------------------------------------

required_test_columns = [
    "review_id",
    "business_id",
    "text",
    "sentiment",
    "sentiment_id"
]


missing_test_columns = [
    column
    for column in required_test_columns
    if column not in test_g8.columns
]


print("\n=== CONTROL DE COLUMNAS ===")

print(
    "Columnas requeridas ausentes:",
    missing_test_columns
)


assert len(missing_test_columns) == 0, (
    "Faltan columnas necesarias en TEST."
)


# ------------------------------------------------------------
# 7. COMPROBAR QUE TEST TODAVÍA NO TIENE LAS RAMAS G4
# ------------------------------------------------------------

print("\n=== ESTADO DE TEST ANTES DEL PROCESAMIENTO ===")

print(
    "¿text_sequential ya existe?:",
    "text_sequential"
    in test_g8.columns
)

print(
    "¿text_bert ya existe?:",
    "text_bert"
    in test_g8.columns
)


# ------------------------------------------------------------
# 8. CREAR UNA COPIA DE TRABAJO DE TEST
# ------------------------------------------------------------

# No alteramos el archivo g3_test_SELLADO.csv.
#
# Todo el procesamiento ocurre sobre una copia en memoria.

test_g8_work = test_g8.copy(
    deep=True
)


# ------------------------------------------------------------
# 9. APLICAR LA RAMA SECUENCIAL APROBADA EN G4
# ------------------------------------------------------------

# Utilizamos EXACTAMENTE la función ya definida y auditada:
#
# clean_text_sequential()
#
# NO se vuelve a ajustar ningún objeto.

test_g8_work[
    "text_sequential"
] = (
    test_g8_work[
        "text"
    ]
    .astype(str)
    .apply(
        clean_text_sequential
    )
)


# ------------------------------------------------------------
# 10. APLICAR LA RAMA BERT APROBADA EN G4
# ------------------------------------------------------------

# Utilizamos EXACTAMENTE:
#
# clean_text_bert()
#
# BERT conserva texto natural con limpieza mínima.

test_g8_work[
    "text_bert"
] = (
    test_g8_work[
        "text"
    ]
    .astype(str)
    .apply(
        clean_text_bert
    )
)


# ------------------------------------------------------------
# 11. COMPROBAR QUE TEST CONSERVA TODAS SUS FILAS
# ------------------------------------------------------------

print("\n=== CONTROL DE FILAS ===")

print(
    "TEST antes:",
    len(test_g8)
)

print(
    "TEST procesado:",
    len(test_g8_work)
)

print(
    "¿Conserva todas las filas?:",
    len(test_g8)
    == len(test_g8_work)
)


# ------------------------------------------------------------
# 12. COMPROBAR TEXTOS VACÍOS
# ------------------------------------------------------------

test_seq_empty = int(
    test_g8_work[
        "text_sequential"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)


test_bert_empty = int(
    test_g8_work[
        "text_bert"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)


print("\n=== TEXTOS VACÍOS ===")

print(
    "Rama secuencial:",
    test_seq_empty
)

print(
    "Rama BERT:",
    test_bert_empty
)


# ------------------------------------------------------------
# 13. COMPROBAR NULOS
# ------------------------------------------------------------

print("\n=== NULOS ===")

print(
    "text_sequential:",
    int(
        test_g8_work[
            "text_sequential"
        ]
        .isna()
        .sum()
    )
)

print(
    "text_bert:",
    int(
        test_g8_work[
            "text_bert"
        ]
        .isna()
        .sum()
    )
)


# ------------------------------------------------------------
# 14. COMPROBAR LAS CLASES
# ------------------------------------------------------------

test_classes = np.unique(
    test_g8_work[
        "sentiment_id"
    ]
    .astype(int)
    .to_numpy()
)


print("\n=== CLASES DE TEST ===")

print(
    "Clases presentes:",
    test_classes
)

print(
    "¿Contiene 0, 1 y 2?:",
    set(
        test_classes.tolist()
    ) == {0, 1, 2}
)


# ------------------------------------------------------------
# 15. COMPROBAR DISTRIBUCIÓN
# ------------------------------------------------------------

# Esta distribución ya existía desde G3.
#
# Mostrarla ahora NO implica seleccionar modelos.
#
# Solo verifica que recuperamos el TEST correcto.

test_class_distribution = (
    test_g8_work[
        "sentiment"
    ]
    .value_counts()
)


print("\n=== DISTRIBUCIÓN DE TEST ===")

print(
    test_class_distribution
)


# ------------------------------------------------------------
# 16. PREPARAR LAS ETIQUETAS FINALES
# ------------------------------------------------------------

# Estas etiquetas se utilizarán una única vez
# para calcular las métricas finales.

y_test_final = (
    test_g8_work[
        "sentiment_id"
    ]
    .astype(int)
    .to_numpy()
)


print("\n=== ETIQUETAS TEST ===")

print(
    "Cantidad:",
    len(y_test_final)
)

print(
    "Clases:",
    np.unique(
        y_test_final
    )
)


# ------------------------------------------------------------
# 17. CONTROL DE OBJETOS YA ENTRENADOS
# ------------------------------------------------------------

# Comprobamos que siguen existiendo.
#
# Esta celda NO llama a fit(), train() ni optimizer.step().

trained_objects_status = {

    "TF-IDF":
        "tfidf_vectorizer"
        in globals(),

    "Logistic Regression":
        "logreg_baseline"
        in globals(),

    "Word2Vec":
        "w2v_model"
        in globals(),

    "BiLSTM":
        "bilstm_model"
        in globals(),

    "BERT":
        "bert_model"
        in globals(),

    "Tokenizer BERT":
        "bert_tokenizer"
        in globals()
}


print("\n=== MODELOS/OBJETOS YA ENTRENADOS ===")

for name, present in trained_objects_status.items():

    print(
        f"{name}:",
        present
    )


# ------------------------------------------------------------
# 18. CONTROL EXPLÍCITO DE NO REENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== CONTROL DE DESARROLLO CERRADO ===")

print(
    "TF-IDF reajustado con TEST:",
    False
)

print(
    "Logistic Regression reentrenada con TEST:",
    False
)

print(
    "Word2Vec reentrenado con TEST:",
    False
)

print(
    "Vocabulario BiLSTM ampliado con TEST:",
    False
)

print(
    "Pesos de clase recalculados con TEST:",
    False
)

print(
    "BERT reentrenado con TEST:",
    False
)


# ------------------------------------------------------------
# 19. VALIDACIÓN AUTOMÁTICA DEL PASO
# ------------------------------------------------------------

g8_step1_ok = (

    len(test_g8_work)
        == EXPECTED_TEST_ROWS

    and len(
        y_test_final
    ) == EXPECTED_TEST_ROWS

    and test_seq_empty == 0

    and test_bert_empty == 0

    and set(
        test_classes.tolist()
    ) == {0, 1, 2}

    and all(
        trained_objects_status.values()
    )
)


print("\n=== VALIDACIÓN AUTOMÁTICA DEL PASO 1 ===")

print(
    "¿Controles superados?:",
    g8_step1_ok
)


# ------------------------------------------------------------
# 20. ESTADO DE G8
# ------------------------------------------------------------

print("\n=== ESTADO DE G8 ===")

print(
    "TEST abierto de forma controlada:",
    "SÍ"
)

print(
    "Rama secuencial preparada:",
    "SÍ"
)

print(
    "Rama BERT preparada:",
    "SÍ"
)

print(
    "Modelos modificados:",
    "NO"
)

print(
    "Predicciones finales generadas:",
    "NO"
)

print(
    "Métricas finales calculadas:",
    "NO"
)

print(
    "TEST evaluado:",
    "TODAVÍA NO"
)

print(
    "G8 finalizado:",
    "NO"
)

=== APERTURA CONTROLADA DE TEST ===
Archivo TEST existe: True
Ruta: /content/artifacts_yelp/g3_test_SELLADO.csv

=== TEST RECUPERADO ===
Filas: 2099
Columnas: 13
Filas esperadas: 2099
¿Coincide el tamaño?: True

=== CONTROL DE COLUMNAS ===
Columnas requeridas ausentes: []

=== ESTADO DE TEST ANTES DEL PROCESAMIENTO ===
¿text_sequential ya existe?: False
¿text_bert ya existe?: False

=== CONTROL DE FILAS ===
TEST antes: 2099
TEST procesado: 2099
¿Conserva todas las filas?: True

=== TEXTOS VACÍOS ===
Rama secuencial: 0
Rama BERT: 0

=== NULOS ===
text_sequential: 0
text_bert: 0

=== CLASES DE TEST ===
Clases presentes: [0 1 2]
¿Contiene 0, 1 y 2?: True

=== DISTRIBUCIÓN DE TEST ===
sentiment
POSITIVO    1439
NEGATIVO     352
NEUTRAL      308
Name: count, dtype: int64

=== ETIQUETAS TEST ===
Cantidad: 2099
Clases: [0 1 2]

=== MODELOS/OBJETOS YA ENTRENADOS ===
TF-IDF: True
Logistic Regression: True
Word2Vec: True
BiLSTM: True
BERT: True
Tokenizer BERT: True

=== CONTROL DE DESARROLLO CER

In [53]:
# ============================================================
# G8 — EVALUACIÓN FINAL EN TEST
# Paso 2: generar predicciones finales y métricas homogéneas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Evalúa una única vez en TEST:
#
# 1. Baseline de clase mayoritaria
# 2. TF-IDF + Logistic Regression
# 3. Word2Vec + BiLSTM
# 4. BERT
#
# Todos usan:
#
# - los objetos ya entrenados;
# - los splits ya aprobados;
# - la misma codificación de clases;
# - la misma función evaluate_predictions().
#
#
# IMPORTANTE:
#
# - NO se ejecuta fit().
# - NO se ejecuta build_vocab().
# - NO se reentrena Word2Vec.
# - NO se reentrena Logistic Regression.
# - NO se reentrena BiLSTM.
# - NO se reentrena BERT.
# - NO se recalculan pesos de clase.
# - NO se modifica ningún checkpoint.
#
# Esta es la evaluación FINAL de TEST.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import time
import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


# ------------------------------------------------------------
# 2. PREPARAR ETIQUETAS DE TEST
# ------------------------------------------------------------

y_test = (
    test_g8_work["sentiment_id"]
    .astype(int)
    .to_numpy()
)

print("=== G8 — TEST FINAL ===")

print(
    "Filas TEST:",
    len(y_test)
)

print(
    "Clases:",
    np.unique(y_test)
)


# ============================================================
# MODELO 1 — CLASE MAYORITARIA
# ============================================================

# ------------------------------------------------------------
# 3. GENERAR PREDICCIONES DEL BASELINE MAYORITARIO
# ------------------------------------------------------------

# majority_class fue calculada previamente SOLO con TRAIN.

test_majority_start = time.time()

y_test_pred_majority = np.full(
    shape=len(y_test),
    fill_value=majority_class,
    dtype=int
)

test_majority_seconds = (
    time.time()
    - test_majority_start
)


# ------------------------------------------------------------
# 4. EVALUAR BASELINE MAYORITARIO
# ------------------------------------------------------------

(
    majority_test_metrics,
    majority_test_report,
    majority_test_cm,
    majority_test_cm_norm
) = evaluate_predictions(
    y_test,
    y_test_pred_majority
)


# ============================================================
# MODELO 2 — TF-IDF + LOGISTIC REGRESSION
# ============================================================

# ------------------------------------------------------------
# 5. TRANSFORMAR TEST CON TF-IDF YA AJUSTADO
# ------------------------------------------------------------

# MUY IMPORTANTE:
# aquí usamos transform(), NO fit_transform().

X_test_text_logreg = (
    test_g8_work["text_sequential"]
    .astype(str)
)

test_logreg_start = time.time()

X_test_tfidf = (
    tfidf_vectorizer.transform(
        X_test_text_logreg
    )
)


# ------------------------------------------------------------
# 6. PREDICCIONES LOGISTIC REGRESSION
# ------------------------------------------------------------

y_test_pred_logreg = (
    logreg_baseline.predict(
        X_test_tfidf
    )
)

test_logreg_seconds = (
    time.time()
    - test_logreg_start
)


# ------------------------------------------------------------
# 7. EVALUAR LOGISTIC REGRESSION
# ------------------------------------------------------------

(
    logreg_test_metrics,
    logreg_test_report,
    logreg_test_cm,
    logreg_test_cm_norm
) = evaluate_predictions(
    y_test,
    y_test_pred_logreg
)


# ============================================================
# MODELO 3 — WORD2VEC + BiLSTM
# ============================================================

# ------------------------------------------------------------
# 8. CONVERTIR TEST A SECUENCIAS
# ------------------------------------------------------------

# Utilizamos word_to_idx YA construido desde TRAIN.
#
# Las palabras nuevas de TEST pasan a UNK.
#
# NO se amplía el vocabulario.

X_test_seq = np.array([
    text_to_indices(
        text,
        word_to_idx,
        MAX_LEN
    )
    for text in test_g8_work["text_sequential"]
], dtype=np.int64)


# ------------------------------------------------------------
# 9. MEDIR TRUNCAMIENTO Y UNK EN TEST
# ------------------------------------------------------------

test_tokens_seq = [
    str(text).split()
    for text in test_g8_work["text_sequential"]
]

test_seq_truncated = int(
    sum(
        len(tokens) > MAX_LEN
        for tokens in test_tokens_seq
    )
)

test_seq_truncated_pct = (
    test_seq_truncated
    / len(test_tokens_seq)
    * 100
)

test_nonpad_tokens = int(
    np.sum(
        X_test_seq != PAD_IDX
    )
)

test_unk_tokens = int(
    np.sum(
        X_test_seq == UNK_IDX
    )
)

test_unk_pct = (
    test_unk_tokens
    / test_nonpad_tokens
    * 100
    if test_nonpad_tokens > 0
    else 0.0
)


# ------------------------------------------------------------
# 10. CREAR TENSOR/DATALOADER DE TEST
# ------------------------------------------------------------

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

test_dataset_seq = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader_seq = DataLoader(
    test_dataset_seq,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 11. PREDICCIONES BiLSTM
# ------------------------------------------------------------

bilstm_model.eval()

y_test_pred_bilstm = []

test_bilstm_start = time.time()

with torch.no_grad():

    for batch_X, batch_y in test_loader_seq:

        batch_X = batch_X.to(
            device
        )

        logits = bilstm_model(
            batch_X
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        y_test_pred_bilstm.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

test_bilstm_seconds = (
    time.time()
    - test_bilstm_start
)

y_test_pred_bilstm = np.array(
    y_test_pred_bilstm
)


# ------------------------------------------------------------
# 12. EVALUAR BiLSTM
# ------------------------------------------------------------

(
    bilstm_test_metrics,
    bilstm_test_report,
    bilstm_test_cm,
    bilstm_test_cm_norm
) = evaluate_predictions(
    y_test,
    y_test_pred_bilstm
)


# ============================================================
# MODELO 4 — BERT
# ============================================================

# ------------------------------------------------------------
# 13. CREAR DATASET DE TEST BERT
# ------------------------------------------------------------

# Se usa el tokenizer YA fijado.
#
# max_length=256
# truncation=True
# padding dinámico
#
# NO hay fit.

X_test_bert_text = (
    test_g8_work["text_bert"]
    .astype(str)
)

test_dataset_bert = YelpBertDataset(
    texts=X_test_bert_text,
    labels=y_test,
    tokenizer=bert_tokenizer,
    max_length=MAX_LENGTH_BERT
)


# ------------------------------------------------------------
# 14. CREAR DATALOADER BERT TEST
# ------------------------------------------------------------

test_loader_bert = DataLoader(
    test_dataset_bert,
    batch_size=BERT_BATCH_SIZE,
    shuffle=False,
    collate_fn=bert_data_collator
)


# ------------------------------------------------------------
# 15. MEDIR TRUNCAMIENTO BERT EN TEST
# ------------------------------------------------------------

test_bert_lengths = get_bert_lengths(
    X_test_bert_text,
    bert_tokenizer
)

test_bert_truncated = int(
    np.sum(
        test_bert_lengths
        > MAX_LENGTH_BERT
    )
)

test_bert_truncated_pct = (
    test_bert_truncated
    / len(test_bert_lengths)
    * 100
)


# ------------------------------------------------------------
# 16. GENERAR PREDICCIONES BERT
# ------------------------------------------------------------

bert_model.eval()

y_test_pred_bert = []

test_bert_start = time.time()

with torch.no_grad():

    for batch in test_loader_bert:

        labels = batch[
            "labels"
        ].to(device)

        model_inputs = {
            key: value.to(device)
            for key, value
            in batch.items()
            if key != "labels"
        }

        if use_amp:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = bert_model(
                    **model_inputs
                )

        else:

            outputs = bert_model(
                **model_inputs
            )

        logits = outputs.logits

        predictions = torch.argmax(
            logits,
            dim=1
        )

        y_test_pred_bert.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

test_bert_seconds = (
    time.time()
    - test_bert_start
)

y_test_pred_bert = np.array(
    y_test_pred_bert
)


# ------------------------------------------------------------
# 17. EVALUAR BERT
# ------------------------------------------------------------

(
    bert_test_metrics,
    bert_test_report,
    bert_test_cm,
    bert_test_cm_norm
) = evaluate_predictions(
    y_test,
    y_test_pred_bert
)


# ============================================================
# COMPARACIÓN FINAL
# ============================================================

# ------------------------------------------------------------
# 18. CREAR TABLA DE MÉTRICAS TEST
# ------------------------------------------------------------

g8_test_comparison = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "TF-IDF + Logistic Regression",
        "Word2Vec + BiLSTM",
        "BERT"
    ],

    "accuracy": [
        majority_test_metrics[
            "accuracy"
        ],
        logreg_test_metrics[
            "accuracy"
        ],
        bilstm_test_metrics[
            "accuracy"
        ],
        bert_test_metrics[
            "accuracy"
        ]
    ],

    "balanced_accuracy": [
        majority_test_metrics[
            "balanced_accuracy"
        ],
        logreg_test_metrics[
            "balanced_accuracy"
        ],
        bilstm_test_metrics[
            "balanced_accuracy"
        ],
        bert_test_metrics[
            "balanced_accuracy"
        ]
    ],

    "precision_macro": [
        majority_test_metrics[
            "precision_macro"
        ],
        logreg_test_metrics[
            "precision_macro"
        ],
        bilstm_test_metrics[
            "precision_macro"
        ],
        bert_test_metrics[
            "precision_macro"
        ]
    ],

    "recall_macro": [
        majority_test_metrics[
            "recall_macro"
        ],
        logreg_test_metrics[
            "recall_macro"
        ],
        bilstm_test_metrics[
            "recall_macro"
        ],
        bert_test_metrics[
            "recall_macro"
        ]
    ],

    "f1_macro": [
        majority_test_metrics[
            "f1_macro"
        ],
        logreg_test_metrics[
            "f1_macro"
        ],
        bilstm_test_metrics[
            "f1_macro"
        ],
        bert_test_metrics[
            "f1_macro"
        ]
    ],

    "f1_weighted": [
        majority_test_metrics[
            "f1_weighted"
        ],
        logreg_test_metrics[
            "f1_weighted"
        ],
        bilstm_test_metrics[
            "f1_weighted"
        ],
        bert_test_metrics[
            "f1_weighted"
        ]
    ]
})


# ------------------------------------------------------------
# 19. MOSTRAR RESULTADOS FINALES
# ------------------------------------------------------------

print("\n=== MÉTRICAS FINALES — TEST ===")

print(
    g8_test_comparison
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 20. IDENTIFICAR EL MEJOR EN TEST POR F1 MACRO
# ------------------------------------------------------------

best_test_row = (
    g8_test_comparison
    .sort_values(
        "f1_macro",
        ascending=False
    )
    .iloc[0]
)


print("\n=== MEJOR MODELO EN TEST POR F1 MACRO ===")

print(
    "Modelo:",
    best_test_row[
        "modelo"
    ]
)

print(
    "F1 macro:",
    round(
        float(
            best_test_row[
                "f1_macro"
            ]
        ),
        4
    )
)

print(
    "Interpretación:",
    "mejor en este split y configuración"
)


# ------------------------------------------------------------
# 21. MOSTRAR REPORTE BERT TEST
# ------------------------------------------------------------

bert_test_report_df = (
    pd.DataFrame(
        bert_test_report
    )
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — BERT TEST ==="
)

print(
    bert_test_report_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 22. MOSTRAR REPORTE BiLSTM TEST
# ------------------------------------------------------------

bilstm_test_report_df = (
    pd.DataFrame(
        bilstm_test_report
    )
    .transpose()
)

print(
    "\n=== REPORTE POR CLASE — BiLSTM TEST ==="
)

print(
    bilstm_test_report_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 23. MATRIZ BERT TEST
# ------------------------------------------------------------

bert_test_cm_df = pd.DataFrame(
    bert_test_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

bert_test_cm_norm_df = pd.DataFrame(
    bert_test_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA — BERT TEST ==="
)

print(
    bert_test_cm_df.to_string()
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA — BERT TEST ==="
)

print(
    bert_test_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 24. MATRIZ BiLSTM TEST
# ------------------------------------------------------------

bilstm_test_cm_df = pd.DataFrame(
    bilstm_test_cm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

bilstm_test_cm_norm_df = pd.DataFrame(
    bilstm_test_cm_norm,
    index=[
        "Real_NEGATIVO",
        "Real_NEUTRAL",
        "Real_POSITIVO"
    ],
    columns=[
        "Pred_NEGATIVO",
        "Pred_NEUTRAL",
        "Pred_POSITIVO"
    ]
)

print(
    "\n=== MATRIZ DE CONFUSIÓN ABSOLUTA — BiLSTM TEST ==="
)

print(
    bilstm_test_cm_df.to_string()
)

print(
    "\n=== MATRIZ DE CONFUSIÓN NORMALIZADA — BiLSTM TEST ==="
)

print(
    bilstm_test_cm_norm_df
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 25. TIEMPOS
# ------------------------------------------------------------

print("\n=== TIEMPOS DE INFERENCIA TEST ===")

print(
    "Clase mayoritaria:",
    round(
        test_majority_seconds,
        4
    ),
    "s"
)

print(
    "TF-IDF + Logistic Regression:",
    round(
        test_logreg_seconds,
        4
    ),
    "s"
)

print(
    "Word2Vec + BiLSTM:",
    round(
        test_bilstm_seconds,
        4
    ),
    "s"
)

print(
    "BERT:",
    round(
        test_bert_seconds,
        4
    ),
    "s"
)


# ------------------------------------------------------------
# 26. TRUNCAMIENTO Y COBERTURA
# ------------------------------------------------------------

print("\n=== COBERTURA / TRUNCAMIENTO TEST ===")

print(
    "BiLSTM reseñas truncadas:",
    test_seq_truncated
)

print(
    "BiLSTM truncamiento:",
    round(
        test_seq_truncated_pct,
        2
    ),
    "%"
)

print(
    "BiLSTM tokens UNK:",
    test_unk_tokens
)

print(
    "BiLSTM porcentaje UNK:",
    round(
        test_unk_pct,
        2
    ),
    "%"
)

print(
    "BERT reseñas truncadas:",
    test_bert_truncated
)

print(
    "BERT truncamiento:",
    round(
        test_bert_truncated_pct,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 27. COMPROBAR TAMAÑOS DE PREDICCIÓN
# ------------------------------------------------------------

g8_prediction_sizes_ok = (
    len(
        y_test_pred_majority
    ) == len(y_test)

    and len(
        y_test_pred_logreg
    ) == len(y_test)

    and len(
        y_test_pred_bilstm
    ) == len(y_test)

    and len(
        y_test_pred_bert
    ) == len(y_test)
)


print(
    "\n=== CONTROL DE PREDICCIONES ==="
)

print(
    "Predicciones clase mayoritaria:",
    len(
        y_test_pred_majority
    )
)

print(
    "Predicciones Logistic Regression:",
    len(
        y_test_pred_logreg
    )
)

print(
    "Predicciones BiLSTM:",
    len(
        y_test_pred_bilstm
    )
)

print(
    "Predicciones BERT:",
    len(
        y_test_pred_bert
    )
)

print(
    "Filas TEST:",
    len(y_test)
)

print(
    "¿Todos los tamaños coinciden?:",
    g8_prediction_sizes_ok
)


# ------------------------------------------------------------
# 28. ESTADO FINAL DE LA EVALUACIÓN
# ------------------------------------------------------------

print("\n=== ESTADO DE G8 — EVALUACIÓN ===")

print(
    "TEST evaluado:",
    "SÍ"
)

print(
    "Número de evaluaciones finales de TEST previstas:",
    1
)

print(
    "Modelos reentrenados después de ver TEST:",
    "NO"
)

print(
    "Hiperparámetros modificados después de ver TEST:",
    "NO"
)

print(
    "F1 macro utilizada para comparación:",
    "SÍ"
)

print(
    "G8 listo para persistencia y auditoría:",
    "SÍ"
)

=== G8 — TEST FINAL ===
Filas TEST: 2099
Clases: [0 1 2]

=== MÉTRICAS FINALES — TEST ===
                      modelo  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
           Clase mayoritaria    0.6856             0.3333           0.2285        0.3333    0.2712       0.5577
TF-IDF + Logistic Regression    0.7894             0.6943           0.6806        0.6943    0.6871       0.7921
           Word2Vec + BiLSTM    0.7542             0.6269           0.6423        0.6269    0.6326       0.7572
                        BERT    0.8113             0.7273           0.7088        0.7273    0.7166       0.8181

=== MEJOR MODELO EN TEST POR F1 MACRO ===
Modelo: BERT
F1 macro: 0.7166
Interpretación: mejor en este split y configuración

=== REPORTE POR CLASE — BERT TEST ===
              precision  recall  f1-score    support
NEGATIVO         0.7827  0.7983    0.7904   352.0000
NEUTRAL          0.4189  0.5032    0.4572   308.0000
POSITIVO         0.9248  0.

In [54]:
# ============================================================
# G8 — EVALUACIÓN FINAL EN TEST
# Paso 3: persistir predicciones, métricas, reportes y matrices
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# 1. Guarda las métricas finales de TEST de los cuatro modelos.
#
# 2. Guarda todas las predicciones finales de TEST.
#
# 3. Guarda los reportes por clase.
#
# 4. Guarda las matrices de confusión absolutas y normalizadas.
#
# 5. Guarda información de:
#
#       - tiempos de inferencia
#       - truncamiento
#       - porcentaje UNK
#
# 6. Guarda una configuración final de G8.
#
# 7. Comprueba que todos los artefactos existen
#    y que las 2.099 predicciones están completas.
#
#
# IMPORTANTE:
#
# - NO se vuelve a generar ninguna predicción.
#
# - NO se vuelve a evaluar ningún modelo.
#
# - NO se reentrena nada.
#
# - NO se modifican hiperparámetros.
#
# - Esta celda solo persiste los resultados ya obtenidos.


# ------------------------------------------------------------
# 1. IMPORTAR UTILIDADES
# ------------------------------------------------------------

import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINIR RUTAS
# ------------------------------------------------------------

G8_METRICS_PATH = (
    ARTIFACT_DIR
    / "g8_metricas_test.csv"
)

G8_PREDICTIONS_PATH = (
    ARTIFACT_DIR
    / "g8_predicciones_test.csv"
)

G8_BERT_REPORT_PATH = (
    ARTIFACT_DIR
    / "g8_reporte_bert_test.csv"
)

G8_BILSTM_REPORT_PATH = (
    ARTIFACT_DIR
    / "g8_reporte_bilstm_test.csv"
)

G8_BERT_CM_PATH = (
    ARTIFACT_DIR
    / "g8_matriz_confusion_bert_test.csv"
)

G8_BERT_CM_NORM_PATH = (
    ARTIFACT_DIR
    / "g8_matriz_confusion_bert_test_normalizada.csv"
)

G8_BILSTM_CM_PATH = (
    ARTIFACT_DIR
    / "g8_matriz_confusion_bilstm_test.csv"
)

G8_BILSTM_CM_NORM_PATH = (
    ARTIFACT_DIR
    / "g8_matriz_confusion_bilstm_test_normalizada.csv"
)

G8_CONFIG_PATH = (
    ARTIFACT_DIR
    / "g8_configuracion_final.json"
)


# ------------------------------------------------------------
# 3. GUARDAR MÉTRICAS FINALES
# ------------------------------------------------------------

g8_test_comparison.to_csv(
    G8_METRICS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. GUARDAR PREDICCIONES DE TEST
# ------------------------------------------------------------

# review_id se usa únicamente como identificador.
#
# No fue predictor de ningún modelo.

g8_predictions_df = pd.DataFrame({
    "review_id":
        test_g8_work["review_id"].values,

    "y_true":
        y_test,

    "pred_majority":
        y_test_pred_majority,

    "pred_logreg":
        y_test_pred_logreg,

    "pred_bilstm":
        y_test_pred_bilstm,

    "pred_bert":
        y_test_pred_bert
})


g8_predictions_df.to_csv(
    G8_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. GUARDAR REPORTES POR CLASE
# ------------------------------------------------------------

bert_test_report_df.to_csv(
    G8_BERT_REPORT_PATH,
    encoding="utf-8"
)

bilstm_test_report_df.to_csv(
    G8_BILSTM_REPORT_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. GUARDAR MATRICES BERT
# ------------------------------------------------------------

bert_test_cm_df.to_csv(
    G8_BERT_CM_PATH,
    encoding="utf-8"
)

bert_test_cm_norm_df.to_csv(
    G8_BERT_CM_NORM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. GUARDAR MATRICES BiLSTM
# ------------------------------------------------------------

bilstm_test_cm_df.to_csv(
    G8_BILSTM_CM_PATH,
    encoding="utf-8"
)

bilstm_test_cm_norm_df.to_csv(
    G8_BILSTM_CM_NORM_PATH,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. GUARDAR CONFIGURACIÓN FINAL DE G8
# ------------------------------------------------------------

g8_config = {

    "stage":
        "G8",

    "evaluation_split":
        "test",

    "test_rows":
        int(len(y_test)),

    "primary_metric":
        "f1_macro",

    "models": [
        "Clase mayoritaria",
        "TF-IDF + Logistic Regression",
        "Word2Vec + BiLSTM",
        "BERT"
    ],

    "best_model_test":
        str(
            best_test_row[
                "modelo"
            ]
        ),

    "best_f1_macro_test":
        float(
            best_test_row[
                "f1_macro"
            ]
        ),

    "inference_seconds": {

        "majority":
            float(
                test_majority_seconds
            ),

        "logreg":
            float(
                test_logreg_seconds
            ),

        "bilstm":
            float(
                test_bilstm_seconds
            ),

        "bert":
            float(
                test_bert_seconds
            )
    },

    "bilstm_test": {

        "max_len":
            int(MAX_LEN),

        "truncated_reviews":
            int(
                test_seq_truncated
            ),

        "truncation_pct":
            float(
                test_seq_truncated_pct
            ),

        "unk_tokens":
            int(
                test_unk_tokens
            ),

        "unk_pct":
            float(
                test_unk_pct
            )
    },

    "bert_test": {

        "max_length":
            int(
                MAX_LENGTH_BERT
            ),

        "truncated_reviews":
            int(
                test_bert_truncated
            ),

        "truncation_pct":
            float(
                test_bert_truncated_pct
            )
    },

    "models_retrained_after_test":
        False,

    "hyperparameters_changed_after_test":
        False,

    "test_evaluations_planned":
        1
}


with open(
    G8_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        g8_config,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 9. COMPROBAR ARTEFACTOS
# ------------------------------------------------------------

g8_artifacts = {

    "Métricas TEST":
        G8_METRICS_PATH,

    "Predicciones TEST":
        G8_PREDICTIONS_PATH,

    "Reporte BERT":
        G8_BERT_REPORT_PATH,

    "Reporte BiLSTM":
        G8_BILSTM_REPORT_PATH,

    "Matriz BERT":
        G8_BERT_CM_PATH,

    "Matriz BERT normalizada":
        G8_BERT_CM_NORM_PATH,

    "Matriz BiLSTM":
        G8_BILSTM_CM_PATH,

    "Matriz BiLSTM normalizada":
        G8_BILSTM_CM_NORM_PATH,

    "Configuración final":
        G8_CONFIG_PATH
}


print("=== ARTEFACTOS DE G8 ===")

all_g8_artifacts_exist = True


for name, path in g8_artifacts.items():

    exists = path.exists()

    all_g8_artifacts_exist = (
        all_g8_artifacts_exist
        and exists
    )

    print(
        f"{name}:",
        exists
    )


# ------------------------------------------------------------
# 10. RECUPERAR MÉTRICAS Y PREDICCIONES
# ------------------------------------------------------------

g8_metrics_check = pd.read_csv(
    G8_METRICS_PATH
)

g8_predictions_check = pd.read_csv(
    G8_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# 11. CONTROL DE RECUPERACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE RECUPERACIÓN ===")

print(
    "Modelos en tabla de métricas:",
    len(g8_metrics_check)
)

print(
    "Modelos esperados:",
    4
)

print(
    "¿Coinciden?:",
    len(g8_metrics_check) == 4
)

print(
    "Predicciones recuperadas:",
    len(g8_predictions_check)
)

print(
    "Filas TEST:",
    len(y_test)
)

print(
    "¿Coinciden?:",
    len(g8_predictions_check)
    == len(y_test)
)

print(
    "review_id duplicados:",
    int(
        g8_predictions_check[
            "review_id"
        ]
        .duplicated()
        .sum()
    )
)


# ------------------------------------------------------------
# 12. COMPROBAR TODAS LAS COLUMNAS DE PREDICCIÓN
# ------------------------------------------------------------

prediction_columns = [
    "pred_majority",
    "pred_logreg",
    "pred_bilstm",
    "pred_bert"
]

prediction_nulls = {
    column:
        int(
            g8_predictions_check[
                column
            ]
            .isna()
            .sum()
        )
    for column
    in prediction_columns
}


print("\n=== CONTROL DE PREDICCIONES PERSISTIDAS ===")

for column, n_nulls in prediction_nulls.items():

    print(
        f"{column} nulos:",
        n_nulls
    )


# ------------------------------------------------------------
# 13. MOSTRAR MÉTRICAS RECUPERADAS
# ------------------------------------------------------------

print("\n=== MÉTRICAS TEST RECUPERADAS ===")

print(
    g8_metrics_check
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 14. COMPROBAR CONFIGURACIÓN
# ------------------------------------------------------------

with open(
    G8_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    g8_config_check = json.load(
        f
    )


print("\n=== CONFIGURACIÓN FINAL RECUPERADA ===")

print(
    "Conjunto de evaluación:",
    g8_config_check[
        "evaluation_split"
    ]
)

print(
    "Filas TEST:",
    g8_config_check[
        "test_rows"
    ]
)

print(
    "Métrica principal:",
    g8_config_check[
        "primary_metric"
    ]
)

print(
    "Mejor modelo TEST:",
    g8_config_check[
        "best_model_test"
    ]
)

print(
    "Mejor F1 macro TEST:",
    round(
        g8_config_check[
            "best_f1_macro_test"
        ],
        4
    )
)

print(
    "Modelos reentrenados después de TEST:",
    g8_config_check[
        "models_retrained_after_test"
    ]
)

print(
    "Hiperparámetros cambiados después de TEST:",
    g8_config_check[
        "hyperparameters_changed_after_test"
    ]
)


# ------------------------------------------------------------
# 15. VALIDACIÓN AUTOMÁTICA
# ------------------------------------------------------------

g8_persistence_ok = (

    all_g8_artifacts_exist

    and len(
        g8_metrics_check
    ) == 4

    and len(
        g8_predictions_check
    ) == len(y_test)

    and g8_predictions_check[
        "review_id"
    ].duplicated().sum() == 0

    and all(
        value == 0
        for value
        in prediction_nulls.values()
    )

    and g8_config_check[
        "evaluation_split"
    ] == "test"

    and g8_config_check[
        "primary_metric"
    ] == "f1_macro"

    and g8_config_check[
        "models_retrained_after_test"
    ] is False

    and g8_config_check[
        "hyperparameters_changed_after_test"
    ] is False
)


print("\n=== RESULTADO DE PERSISTENCIA G8 ===")

print(
    "¿Controles superados?:",
    g8_persistence_ok
)

print(
    "G8 listo para auditoría independiente:",
    "SÍ"
    if g8_persistence_ok
    else "NO"
)


# ------------------------------------------------------------
# 16. ESTADO FINAL DE G8
# ------------------------------------------------------------

print("\n=== ESTADO DE G8 ===")

print(
    "TEST evaluado:",
    "SÍ"
)

print(
    "Predicciones finales persistidas:",
    "SÍ"
)

print(
    "Métricas finales persistidas:",
    "SÍ"
)

print(
    "Modelos reentrenados tras TEST:",
    "NO"
)

print(
    "Hiperparámetros cambiados tras TEST:",
    "NO"
)

print(
    "G8 listo para auditoría:",
    "SÍ"
    if g8_persistence_ok
    else "NO"
)

print(
    "G8 aprobado:",
    "TODAVÍA NO"
)

print(
    "G9 iniciado:",
    "NO"
)

=== ARTEFACTOS DE G8 ===
Métricas TEST: True
Predicciones TEST: True
Reporte BERT: True
Reporte BiLSTM: True
Matriz BERT: True
Matriz BERT normalizada: True
Matriz BiLSTM: True
Matriz BiLSTM normalizada: True
Configuración final: True

=== CONTROL DE RECUPERACIÓN ===
Modelos en tabla de métricas: 4
Modelos esperados: 4
¿Coinciden?: True
Predicciones recuperadas: 2099
Filas TEST: 2099
¿Coinciden?: True
review_id duplicados: 0

=== CONTROL DE PREDICCIONES PERSISTIDAS ===
pred_majority nulos: 0
pred_logreg nulos: 0
pred_bilstm nulos: 0
pred_bert nulos: 0

=== MÉTRICAS TEST RECUPERADAS ===
                      modelo  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
           Clase mayoritaria    0.6856             0.3333           0.2285        0.3333    0.2712       0.5577
TF-IDF + Logistic Regression    0.7894             0.6943           0.6806        0.6943    0.6871       0.7921
           Word2Vec + BiLSTM    0.7542             0.6269           0.6

In [59]:
# ============================================================
# G9 — ANÁLISIS DE ERRORES
# Paso 1: recuperar las predicciones finales ya persistidas
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Recupera las predicciones de TEST que YA fueron calculadas
# y aprobadas en G8.
#
# IMPORTANTE:
# - NO ejecuta BERT.
# - NO ejecuta BiLSTM.
# - NO genera nuevas predicciones.
# - NO calcula nuevamente las métricas de TEST.
# - NO entrena ningún modelo.
# - NO modifica hiperparámetros.
#
# G9 comienza exclusivamente como análisis de resultados
# ya persistidos.


import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. LOCALIZAR PREDICCIONES PERSISTIDAS DE G8
# ------------------------------------------------------------

G8_PREDICTIONS_PATH = (
    ARTIFACT_DIR
    / "g8_predicciones_test.csv"
)


print("=== G9 — ENTRADA ===")

print(
    "Archivo de predicciones G8 existe:",
    G8_PREDICTIONS_PATH.exists()
)

print(
    "Ruta:",
    G8_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# 2. RECUPERAR PREDICCIONES
# ------------------------------------------------------------

g9_predictions = pd.read_csv(
    G8_PREDICTIONS_PATH
)


print("\n=== PREDICCIONES RECUPERADAS ===")

print(
    "Filas:",
    len(g9_predictions)
)

print(
    "Columnas:",
    g9_predictions.columns.tolist()
)


# ------------------------------------------------------------
# 3. CONTROLES BÁSICOS
# ------------------------------------------------------------

print("\n=== CONTROL DE TRAZABILIDAD ===")

print(
    "review_id disponible:",
    "review_id" in g9_predictions.columns
)

if "review_id" in g9_predictions.columns:

    print(
        "review_id duplicados:",
        g9_predictions["review_id"]
        .duplicated()
        .sum()
    )


prediction_columns = [
    "pred_majority",
    "pred_logreg",
    "pred_bilstm",
    "pred_bert"
]


print("\n=== COLUMNAS DE PREDICCIÓN ===")

for column in prediction_columns:

    print(
        f"{column}:",
        column in g9_predictions.columns
    )


# ------------------------------------------------------------
# 4. COMPROBAR ETIQUETA REAL
# ------------------------------------------------------------

possible_target_columns = [
    "sentiment_id",
    "y_true",
    "true_label",
    "label"
]


target_columns_found = [
    column
    for column in possible_target_columns
    if column in g9_predictions.columns
]


print("\n=== ETIQUETA REAL ===")

print(
    "Columnas candidatas encontradas:",
    target_columns_found
)


# ------------------------------------------------------------
# 5. COMPROBAR SI EL TEXTO FUE PERSISTIDO
# ------------------------------------------------------------

possible_text_columns = [
    "text",
    "text_bert",
    "text_sequential"
]


text_columns_found = [
    column
    for column in possible_text_columns
    if column in g9_predictions.columns
]


print("\n=== TEXTO DISPONIBLE ===")

print(
    "Columnas de texto encontradas:",
    text_columns_found
)


# ------------------------------------------------------------
# 6. DISTRIBUCIÓN DE PREDICCIONES BERT
# ------------------------------------------------------------

if "pred_bert" in g9_predictions.columns:

    print(
        "\n=== DISTRIBUCIÓN DE PREDICCIONES BERT ==="
    )

    print(
        g9_predictions[
            "pred_bert"
        ]
        .value_counts()
        .sort_index()
    )


# ------------------------------------------------------------
# 7. IDENTIFICAR ERRORES BERT
# ------------------------------------------------------------

if len(target_columns_found) == 1:

    TARGET_COLUMN = target_columns_found[0]

    g9_predictions[
        "bert_error"
    ] = (
        g9_predictions[
            TARGET_COLUMN
        ]
        !=
        g9_predictions[
            "pred_bert"
        ]
    )


    print("\n=== ERRORES BERT ===")

    print(
        "Predicciones correctas:",
        (~g9_predictions["bert_error"]).sum()
    )

    print(
        "Errores:",
        g9_predictions["bert_error"].sum()
    )

    print(
        "Total:",
        len(g9_predictions)
    )

else:

    TARGET_COLUMN = None

    print(
        "\nNo se crea todavía bert_error porque "
        "la columna objetivo debe identificarse "
        "de forma inequívoca."
    )


# ------------------------------------------------------------
# 8. CONTROL DE NO REEVALUACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE G9 ===")

print(
    "BERT ejecutado en esta celda:",
    False
)

print(
    "BiLSTM ejecutada en esta celda:",
    False
)

print(
    "Predicciones TEST recalculadas:",
    False
)

print(
    "Métricas TEST recalculadas:",
    False
)

print(
    "Modelos entrenados:",
    False
)

print(
    "Hiperparámetros modificados:",
    False
)


# ------------------------------------------------------------
# 9. ESTADO
# ------------------------------------------------------------

print("\n=== ESTADO DE G9 ===")

print(
    "Predicciones G8 recuperadas:",
    "SÍ"
)

print(
    "Análisis de errores iniciado:",
    "SÍ"
)

print(
    "TEST reinferido:",
    "NO"
)

print(
    "G9 finalizado:",
    "NO"
)

=== G9 — ENTRADA ===
Archivo de predicciones G8 existe: True
Ruta: /content/artifacts_yelp/g8_predicciones_test.csv

=== PREDICCIONES RECUPERADAS ===
Filas: 2099
Columnas: ['review_id', 'y_true', 'pred_majority', 'pred_logreg', 'pred_bilstm', 'pred_bert']

=== CONTROL DE TRAZABILIDAD ===
review_id disponible: True
review_id duplicados: 0

=== COLUMNAS DE PREDICCIÓN ===
pred_majority: True
pred_logreg: True
pred_bilstm: True
pred_bert: True

=== ETIQUETA REAL ===
Columnas candidatas encontradas: ['y_true']

=== TEXTO DISPONIBLE ===
Columnas de texto encontradas: []

=== DISTRIBUCIÓN DE PREDICCIONES BERT ===
pred_bert
0     359
1     370
2    1370
Name: count, dtype: int64

=== ERRORES BERT ===
Predicciones correctas: 1703
Errores: 396
Total: 2099

=== CONTROL DE G9 ===
BERT ejecutado en esta celda: False
BiLSTM ejecutada en esta celda: False
Predicciones TEST recalculadas: False
Métricas TEST recalculadas: False
Modelos entrenados: False
Hiperparámetros modificados: False

=== ESTADO DE

In [60]:
# ============================================================
# G9 — ANÁLISIS DE ERRORES
# Paso 2: recuperar texto de TEST y unirlo a predicciones G8
# ============================================================

# ¿QUÉ HACE ESTA CELDA?
#
# Recupera el texto ORIGINAL de las reseñas de TEST y lo une
# mediante review_id a las predicciones YA calculadas en G8.
#
# Después caracteriza los errores ya existentes de BERT:
#
# - errores por clase;
# - especial atención a NEUTRAL;
# - negación;
# - posibles reseñas mixtas;
# - textos largos;
# - candidatos para revisión cualitativa.
#
# IMPORTANTE:
#
# - NO ejecuta BERT.
# - NO ejecuta BiLSTM.
# - NO genera nuevas predicciones.
# - NO recalcula métricas de TEST.
# - NO entrena.
# - NO modifica modelos.
# - NO modifica el archivo TEST original.
#
# Las detecciones de negación y reseña mixta son indicadores
# heurísticos para análisis, NO nuevas etiquetas.


import pandas as pd
import numpy as np
import re


# ------------------------------------------------------------
# 1. RECUPERAR TEST SELLADO EN UNA COPIA DE ANÁLISIS
# ------------------------------------------------------------

G3_TEST_PATH = (
    ARTIFACT_DIR
    / "g3_test_SELLADO.csv"
)


print("=== FUENTE DE TEXTO PARA G9 ===")

print(
    "Archivo TEST existe:",
    G3_TEST_PATH.exists()
)


test_g9_source = pd.read_csv(
    G3_TEST_PATH
)


print(
    "Filas TEST:",
    len(test_g9_source)
)

print(
    "Columnas TEST:",
    test_g9_source.columns.tolist()
)


# ------------------------------------------------------------
# 2. SELECCIONAR SOLO LO NECESARIO
# ------------------------------------------------------------

required_columns = [
    "review_id",
    "text"
]


missing_columns = [
    col
    for col in required_columns
    if col not in test_g9_source.columns
]


print("\n=== CONTROL DE COLUMNAS ===")

print(
    "Columnas requeridas ausentes:",
    missing_columns
)


assert len(missing_columns) == 0, (
    "Faltan columnas necesarias para G9."
)


g9_text_source = (
    test_g9_source[
        required_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. CONTROLES ANTES DEL MERGE
# ------------------------------------------------------------

print("\n=== CONTROL PRE-MERGE ===")

print(
    "review_id duplicados en texto TEST:",
    g9_text_source["review_id"]
    .duplicated()
    .sum()
)

print(
    "review_id duplicados en predicciones:",
    g9_predictions["review_id"]
    .duplicated()
    .sum()
)


assert (
    g9_text_source["review_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    g9_predictions["review_id"]
    .duplicated()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 4. UNIR TEXTO Y PREDICCIONES
# ------------------------------------------------------------

g9_error_analysis = (
    g9_predictions
    .merge(
        g9_text_source,
        on="review_id",
        how="left",
        validate="one_to_one"
    )
)


print("\n=== MERGE G9 ===")

print(
    "Filas después del merge:",
    len(g9_error_analysis)
)

print(
    "Filas esperadas:",
    len(g9_predictions)
)

print(
    "¿Conserva todas las predicciones?:",
    len(g9_error_analysis)
    ==
    len(g9_predictions)
)

print(
    "Textos nulos:",
    g9_error_analysis["text"]
    .isna()
    .sum()
)


assert (
    len(g9_error_analysis)
    ==
    len(g9_predictions)
)

assert (
    g9_error_analysis["text"]
    .isna()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 5. CREAR VARIABLES DE ERROR
# ------------------------------------------------------------

g9_error_analysis[
    "bert_error"
] = (
    g9_error_analysis["y_true"]
    !=
    g9_error_analysis["pred_bert"]
)


g9_error_analysis[
    "bilstm_error"
] = (
    g9_error_analysis["y_true"]
    !=
    g9_error_analysis["pred_bilstm"]
)


g9_error_analysis[
    "logreg_error"
] = (
    g9_error_analysis["y_true"]
    !=
    g9_error_analysis["pred_logreg"]
)


# ------------------------------------------------------------
# 6. LONGITUD DEL TEXTO
# ------------------------------------------------------------

g9_error_analysis[
    "word_count"
] = (
    g9_error_analysis["text"]
    .astype(str)
    .str.split()
    .str.len()
)


LONG_REVIEW_THRESHOLD = (
    g9_error_analysis["word_count"]
    .quantile(0.90)
)


g9_error_analysis[
    "long_review"
] = (
    g9_error_analysis["word_count"]
    >=
    LONG_REVIEW_THRESHOLD
)


# ------------------------------------------------------------
# 7. INDICADOR HEURÍSTICO DE NEGACIÓN
# ------------------------------------------------------------

negation_pattern = (
    r"\b("
    r"no|not|never|nor|neither|"
    r"don't|doesn't|didn't|"
    r"isn't|wasn't|weren't|"
    r"can't|couldn't|"
    r"won't|wouldn't|"
    r"shouldn't|"
    r"haven't|hasn't|hadn't"
    r")\b"
)


g9_error_analysis[
    "has_negation"
] = (
    g9_error_analysis["text"]
    .astype(str)
    .str.lower()
    .str.contains(
        negation_pattern,
        regex=True,
        na=False
    )
)


# ------------------------------------------------------------
# 8. INDICADOR HEURÍSTICO DE RESEÑA MIXTA
# ------------------------------------------------------------

contrast_pattern = (
    r"\b("
    r"but|however|although|though|"
    r"yet|except|while|overall"
    r")\b"
)


g9_error_analysis[
    "mixed_signal"
] = (
    g9_error_analysis["text"]
    .astype(str)
    .str.lower()
    .str.contains(
        contrast_pattern,
        regex=True,
        na=False
    )
)


# ------------------------------------------------------------
# 9. EXTRAER ERRORES BERT
# ------------------------------------------------------------

bert_errors = (
    g9_error_analysis[
        g9_error_analysis[
            "bert_error"
        ]
    ]
    .copy()
)


bilstm_errors = (
    g9_error_analysis[
        g9_error_analysis[
            "bilstm_error"
        ]
    ]
    .copy()
)


neutral_errors = (
    bert_errors[
        bert_errors[
            "y_true"
        ] == 1
    ]
    .copy()
)


negation_errors = (
    bert_errors[
        bert_errors[
            "has_negation"
        ]
    ]
    .copy()
)


mixed_review_errors = (
    bert_errors[
        bert_errors[
            "mixed_signal"
        ]
    ]
    .copy()
)


long_review_errors = (
    bert_errors[
        bert_errors[
            "long_review"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. RESUMEN GENERAL
# ------------------------------------------------------------

print("\n=== RESUMEN DE ERRORES ===")

print(
    "Total TEST:",
    len(g9_error_analysis)
)

print(
    "Errores BERT:",
    len(bert_errors)
)

print(
    "Errores BiLSTM:",
    len(bilstm_errors)
)

print(
    "Errores Logistic Regression:",
    int(
        g9_error_analysis[
            "logreg_error"
        ].sum()
    )
)


# ------------------------------------------------------------
# 11. ERRORES BERT SEGÚN CLASE REAL
# ------------------------------------------------------------

print(
    "\n=== ERRORES BERT POR CLASE REAL ==="
)


bert_errors_by_class = (
    bert_errors[
        "y_true"
    ]
    .value_counts()
    .sort_index()
)


print(
    bert_errors_by_class
)


# ------------------------------------------------------------
# 12. MATRIZ DE TIPOS DE ERROR BERT
# ------------------------------------------------------------

print(
    "\n=== CONFUSIONES BERT ENTRE CLASES ==="
)


bert_confusion_types = (
    bert_errors
    .groupby(
        [
            "y_true",
            "pred_bert"
        ]
    )
    .size()
    .reset_index(
        name="cantidad"
    )
    .sort_values(
        "cantidad",
        ascending=False
    )
)


print(
    bert_confusion_types
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 13. NEUTRALIDAD
# ------------------------------------------------------------

total_neutral = int(
    (
        g9_error_analysis[
            "y_true"
        ] == 1
    )
    .sum()
)


correct_neutral = int(
    (
        (
            g9_error_analysis[
                "y_true"
            ] == 1
        )
        &
        (
            g9_error_analysis[
                "pred_bert"
            ] == 1
        )
    )
    .sum()
)


print(
    "\n=== ANÁLISIS DE NEUTRALIDAD — BERT ==="
)

print(
    "Reseñas NEUTRAL reales:",
    total_neutral
)

print(
    "NEUTRAL correctamente clasificadas:",
    correct_neutral
)

print(
    "NEUTRAL mal clasificadas:",
    len(neutral_errors)
)


if total_neutral > 0:

    print(
        "Porcentaje de error en NEUTRAL:",
        round(
            len(neutral_errors)
            /
            total_neutral
            *
            100,
            2
        ),
        "%"
    )


# ------------------------------------------------------------
# 14. NEGACIÓN
# ------------------------------------------------------------

total_with_negation = int(
    g9_error_analysis[
        "has_negation"
    ].sum()
)


errors_with_negation = len(
    negation_errors
)


print(
    "\n=== NEGACIÓN — BERT ==="
)

print(
    "Reseñas con indicador de negación:",
    total_with_negation
)

print(
    "Errores BERT entre ellas:",
    errors_with_negation
)


if total_with_negation > 0:

    print(
        "Tasa de error con negación:",
        round(
            errors_with_negation
            /
            total_with_negation
            *
            100,
            2
        ),
        "%"
    )


# ------------------------------------------------------------
# 15. RESEÑAS MIXTAS
# ------------------------------------------------------------

total_mixed = int(
    g9_error_analysis[
        "mixed_signal"
    ].sum()
)


errors_mixed = len(
    mixed_review_errors
)


print(
    "\n=== RESEÑAS CON SEÑAL MIXTA — BERT ==="
)

print(
    "Reseñas detectadas:",
    total_mixed
)

print(
    "Errores BERT entre ellas:",
    errors_mixed
)


if total_mixed > 0:

    print(
        "Tasa de error:",
        round(
            errors_mixed
            /
            total_mixed
            *
            100,
            2
        ),
        "%"
    )


# ------------------------------------------------------------
# 16. TEXTOS LARGOS
# ------------------------------------------------------------

total_long = int(
    g9_error_analysis[
        "long_review"
    ].sum()
)


errors_long = len(
    long_review_errors
)


print(
    "\n=== TEXTOS LARGOS — BERT ==="
)

print(
    "Umbral P90 de palabras:",
    LONG_REVIEW_THRESHOLD
)

print(
    "Reseñas consideradas largas:",
    total_long
)

print(
    "Errores BERT entre textos largos:",
    errors_long
)


if total_long > 0:

    print(
        "Tasa de error en textos largos:",
        round(
            errors_long
            /
            total_long
            *
            100,
            2
        ),
        "%"
    )


# ------------------------------------------------------------
# 17. COMPARACIÓN DE TASAS DE ERROR
# ------------------------------------------------------------

overall_bert_error_rate = (
    len(bert_errors)
    /
    len(g9_error_analysis)
    *
    100
)


print(
    "\n=== REFERENCIA GENERAL BERT ==="
)

print(
    "Tasa global de error:",
    round(
        overall_bert_error_rate,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 18. MUESTRA PARA REVISIÓN CUALITATIVA
# ------------------------------------------------------------

LABEL_NAMES = {
    0: "NEGATIVO",
    1: "NEUTRAL",
    2: "POSITIVO"
}


qualitative_sample = (
    bert_errors[
        [
            "review_id",
            "y_true",
            "pred_bert",
            "text",
            "word_count",
            "has_negation",
            "mixed_signal",
            "long_review"
        ]
    ]
    .copy()
)


qualitative_sample[
    "real_label"
] = (
    qualitative_sample[
        "y_true"
    ]
    .map(
        LABEL_NAMES
    )
)


qualitative_sample[
    "predicted_label"
] = (
    qualitative_sample[
        "pred_bert"
    ]
    .map(
        LABEL_NAMES
    )
)


print(
    "\n=== EJEMPLOS DE ERRORES BERT ==="
)


sample_to_show = (
    qualitative_sample
    .sort_values(
        [
            "has_negation",
            "mixed_signal",
            "long_review"
        ],
        ascending=False
    )
    .head(10)
)


for i, row in sample_to_show.iterrows():

    print(
        "\n----------------------------------------"
    )

    print(
        "REAL:",
        row["real_label"]
    )

    print(
        "PREDICCIÓN:",
        row["predicted_label"]
    )

    print(
        "Negación:",
        row["has_negation"]
    )

    print(
        "Señal mixta:",
        row["mixed_signal"]
    )

    print(
        "Texto largo:",
        row["long_review"]
    )

    print(
        "Palabras:",
        row["word_count"]
    )

    print(
        "Texto:"
    )

    print(
        str(row["text"])[:1000]
    )


# ------------------------------------------------------------
# 19. PERSISTIR ARTEFACTOS G9
# ------------------------------------------------------------

G9_ANALYSIS_PATH = (
    ARTIFACT_DIR
    / "g9_error_analysis.csv"
)


G9_BERT_ERRORS_PATH = (
    ARTIFACT_DIR
    / "g9_bert_errors.csv"
)


G9_QUALITATIVE_PATH = (
    ARTIFACT_DIR
    / "g9_qualitative_sample.csv"
)


g9_error_analysis.to_csv(
    G9_ANALYSIS_PATH,
    index=False
)


bert_errors.to_csv(
    G9_BERT_ERRORS_PATH,
    index=False
)


qualitative_sample.to_csv(
    G9_QUALITATIVE_PATH,
    index=False
)


print(
    "\n=== ARTEFACTOS G9 ==="
)

print(
    "Análisis completo:",
    G9_ANALYSIS_PATH.exists()
)

print(
    "Errores BERT:",
    G9_BERT_ERRORS_PATH.exists()
)

print(
    "Muestra cualitativa:",
    G9_QUALITATIVE_PATH.exists()
)


# ------------------------------------------------------------
# 20. CONTROL FINAL
# ------------------------------------------------------------

print(
    "\n=== CONTROL DE NO REEVALUACIÓN ==="
)

print(
    "Predicciones TEST nuevas:",
    False
)

print(
    "Métricas TEST recalculadas:",
    False
)

print(
    "BERT ejecutado:",
    False
)

print(
    "BiLSTM ejecutada:",
    False
)

print(
    "Modelos modificados:",
    False
)


print(
    "\n=== ESTADO DE G9 ==="
)

print(
    "Errores identificados:",
    "SÍ"
)

print(
    "Neutralidad analizada:",
    "SÍ"
)

print(
    "Negación analizada:",
    "SÍ"
)

print(
    "Reseñas mixtas analizadas:",
    "SÍ"
)

print(
    "Textos largos analizados:",
    "SÍ"
)

print(
    "Revisión cualitativa preparada:",
    "SÍ"
)

print(
    "Sarcasmo:",
    "PENDIENTE DE REVISIÓN CUALITATIVA"
)

print(
    "Ruido de etiqueta proxy:",
    "PENDIENTE DE REVISIÓN CUALITATIVA"
)

print(
    "G9 finalizado:",
    "TODAVÍA NO"
)

=== FUENTE DE TEXTO PARA G9 ===
Archivo TEST existe: True
Filas TEST: 2099
Columnas TEST: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment', 'sentiment_id', 'split']

=== CONTROL DE COLUMNAS ===
Columnas requeridas ausentes: []

=== CONTROL PRE-MERGE ===
review_id duplicados en texto TEST: 0
review_id duplicados en predicciones: 0

=== MERGE G9 ===
Filas después del merge: 2099
Filas esperadas: 2099
¿Conserva todas las predicciones?: True
Textos nulos: 0

=== RESUMEN DE ERRORES ===
Total TEST: 2099
Errores BERT: 396
Errores BiLSTM: 516
Errores Logistic Regression: 442

=== ERRORES BERT POR CLASE REAL ===
y_true
0     71
1    153
2    172
Name: count, dtype: int64

=== CONFUSIONES BERT ENTRE CLASES ===
 y_true  pred_bert  cantidad
      2          1       158
      1          2        89
      1          0        64
      0          1        57
      0          2        14
      2          0        14

=== ANÁLISIS DE NEUTRAL


----------------------------------------
REAL: POSITIVO
PREDICCIÓN: NEUTRAL
Negación: True
Señal mixta: True
Texto largo: True
Palabras: 310
Texto:
Cliff notes
- Cheap: $1-2/plate
- Fast: It's ready-to-go and on a conveyor
- Acceptable quality: I'm here weekly and haven't gotten sick lol

This place is definitely good for what it is, "it" being a budget-friendly sushi place that's great for a quick bite.  In a response to the lower-rating reviews stating how the food is just okay at best or authentic or blah blah blah, well think of it like wal-mart vs. *inserthighendretailerhere*.  This place has a business model based on selling in large volumes (hence the low prices like wal-mart) so they're gonna have to "reduce costs" somehow (so-so quality food, not-so-attentive staff, and slightly smaller than average portions), but it's still a solid bang-for-your-buck kinda place.

On top of this, I can be in and out quickly since everything is already prepared, perfect for those quick 30 min

In [61]:
# ============================================================
# G9 — Paso 3
# Revisión cualitativa: sarcasmo y ruido de etiqueta proxy
# + persistencia del resumen final de G9
# ============================================================

import pandas as pd
import numpy as np
import json
import re


# ------------------------------------------------------------
# 1. CONTROLES DE ENTRADA
# ------------------------------------------------------------

print("=== G9 — REVISIÓN CUALITATIVA FINAL ===")

assert "g9_error_analysis" in globals()
assert "bert_errors" in globals()

required_cols = [
    "review_id",
    "y_true",
    "pred_bert",
    "text",
    "word_count",
    "has_negation",
    "mixed_signal",
    "long_review"
]

missing = [
    c for c in required_cols
    if c not in bert_errors.columns
]

print("Columnas requeridas ausentes:", missing)

assert len(missing) == 0


LABEL_NAMES = {
    0: "NEGATIVO",
    1: "NEUTRAL",
    2: "POSITIVO"
}


# ------------------------------------------------------------
# 2. CANDIDATOS A SARCASMO
# ------------------------------------------------------------
#
# IMPORTANTE:
# Esto NO afirma que una reseña sea sarcástica.
# Solo recupera candidatos para inspección cualitativa.
# ------------------------------------------------------------

sarcasm_terms = [
    r"\bsarcasm\b",
    r"\bsarcastic\b",
    r"\byeah right\b",
    r"\bwhat a joke\b",
    r"\bjust great\b",
    r"\bthanks a lot\b"
]

sarcasm_pattern = "(?:" + "|".join(sarcasm_terms) + ")"


sarcasm_candidates = (
    bert_errors[
        bert_errors["text"]
        .astype(str)
        .str.lower()
        .str.contains(
            sarcasm_pattern,
            regex=True,
            na=False
        )
    ]
    .copy()
)


print("\n=== SARCASMO — CANDIDATOS PARA REVISIÓN ===")

print(
    "Candidatos encontrados entre errores BERT:",
    len(sarcasm_candidates)
)

print(
    "Interpretación:",
    "indicador heurístico; NO detección automática de sarcasmo"
)


# ------------------------------------------------------------
# 3. CANDIDATOS A RUIDO DE ETIQUETA PROXY
# ------------------------------------------------------------
#
# Las clases proceden de estrellas:
#
# 1–2 -> NEGATIVO (0)
# 3   -> NEUTRAL  (1)
# 4–5 -> POSITIVO (2)
#
# Una discrepancia modelo/proxy NO demuestra que la etiqueta
# sea incorrecta. Se seleccionan casos ambiguos para revisión.
# ------------------------------------------------------------

proxy_label_candidates = (
    bert_errors[
        (
            (bert_errors["y_true"] == 1)
            |
            (bert_errors["mixed_signal"])
        )
    ]
    .copy()
)


print("\n=== RUIDO DE ETIQUETA PROXY — CANDIDATOS ===")

print(
    "Candidatos encontrados:",
    len(proxy_label_candidates)
)

print(
    "Interpretación:",
    "casos potencialmente ambiguos; "
    "NO prueba automática de etiqueta incorrecta"
)


# ------------------------------------------------------------
# 4. TABLA CUALITATIVA DE SARCASMO
# ------------------------------------------------------------

sarcasm_review = sarcasm_candidates[
    [
        "review_id",
        "y_true",
        "pred_bert",
        "text",
        "word_count",
        "has_negation",
        "mixed_signal",
        "long_review"
    ]
].copy()


sarcasm_review["real_label"] = (
    sarcasm_review["y_true"].map(LABEL_NAMES)
)

sarcasm_review["predicted_label"] = (
    sarcasm_review["pred_bert"].map(LABEL_NAMES)
)


# ------------------------------------------------------------
# 5. TABLA CUALITATIVA DE ETIQUETA PROXY
# ------------------------------------------------------------

proxy_review = proxy_label_candidates[
    [
        "review_id",
        "y_true",
        "pred_bert",
        "text",
        "word_count",
        "has_negation",
        "mixed_signal",
        "long_review"
    ]
].copy()


proxy_review["real_label"] = (
    proxy_review["y_true"].map(LABEL_NAMES)
)

proxy_review["predicted_label"] = (
    proxy_review["pred_bert"].map(LABEL_NAMES)
)


# ------------------------------------------------------------
# 6. MOSTRAR CANDIDATOS A SARCASMO
# ------------------------------------------------------------

print("\n=== EJEMPLOS CANDIDATOS A SARCASMO ===")

if len(sarcasm_review) == 0:

    print(
        "No se encontraron candidatos mediante "
        "los indicadores utilizados."
    )

else:

    for _, row in sarcasm_review.head(10).iterrows():

        print("\n----------------------------------------")
        print("REAL:", row["real_label"])
        print("PREDICCIÓN BERT:", row["predicted_label"])
        print("Palabras:", row["word_count"])
        print("Texto:")
        print(str(row["text"])[:1200])


# ------------------------------------------------------------
# 7. MUESTRA DE POSIBLE AMBIGÜEDAD DE LA ETIQUETA PROXY
# ------------------------------------------------------------

print(
    "\n=== EJEMPLOS PARA REVISIÓN DE ETIQUETA PROXY ==="
)


proxy_sample = (
    proxy_review
    .sort_values(
        [
            "mixed_signal",
            "has_negation",
            "long_review"
        ],
        ascending=False
    )
    .head(15)
)


for _, row in proxy_sample.iterrows():

    print("\n----------------------------------------")
    print("ETIQUETA PROXY:", row["real_label"])
    print("PREDICCIÓN BERT:", row["predicted_label"])
    print("Negación:", row["has_negation"])
    print("Señal mixta:", row["mixed_signal"])
    print("Texto largo:", row["long_review"])
    print("Texto:")
    print(str(row["text"])[:1200])


# ------------------------------------------------------------
# 8. RESUMEN CUANTITATIVO FINAL DE G9
# ------------------------------------------------------------

n_total = len(g9_error_analysis)
n_bert_errors = len(bert_errors)

n_neutral = int(
    (g9_error_analysis["y_true"] == 1).sum()
)

n_neutral_errors = int(
    (
        (g9_error_analysis["y_true"] == 1)
        &
        (g9_error_analysis["pred_bert"] != 1)
    ).sum()
)

n_negation = int(
    g9_error_analysis["has_negation"].sum()
)

n_negation_errors = int(
    (
        g9_error_analysis["has_negation"]
        &
        g9_error_analysis["bert_error"]
    ).sum()
)

n_mixed = int(
    g9_error_analysis["mixed_signal"].sum()
)

n_mixed_errors = int(
    (
        g9_error_analysis["mixed_signal"]
        &
        g9_error_analysis["bert_error"]
    ).sum()
)

n_long = int(
    g9_error_analysis["long_review"].sum()
)

n_long_errors = int(
    (
        g9_error_analysis["long_review"]
        &
        g9_error_analysis["bert_error"]
    ).sum()
)


def pct(num, den):
    if den == 0:
        return None
    return round(100 * num / den, 2)


g9_summary = {
    "total_test": int(n_total),

    "bert_errors": int(n_bert_errors),

    "bert_global_error_rate_pct":
        pct(n_bert_errors, n_total),

    "neutral_total": int(n_neutral),

    "neutral_errors": int(n_neutral_errors),

    "neutral_error_rate_pct":
        pct(n_neutral_errors, n_neutral),

    "negation_total": int(n_negation),

    "negation_errors": int(n_negation_errors),

    "negation_error_rate_pct":
        pct(n_negation_errors, n_negation),

    "mixed_signal_total": int(n_mixed),

    "mixed_signal_errors": int(n_mixed_errors),

    "mixed_signal_error_rate_pct":
        pct(n_mixed_errors, n_mixed),

    "long_review_total": int(n_long),

    "long_review_errors": int(n_long_errors),

    "long_review_error_rate_pct":
        pct(n_long_errors, n_long),

    "sarcasm_candidates":
        int(len(sarcasm_candidates)),

    "proxy_label_candidates":
        int(len(proxy_label_candidates)),

    "sarcasm_interpretation":
        "qualitative_candidates_only",

    "proxy_label_interpretation":
        "qualitative_candidates_only",

    "test_reinference":
        False,

    "metrics_recalculated":
        False,

    "models_modified":
        False
}


print("\n=== RESUMEN FINAL G9 ===")

for key, value in g9_summary.items():
    print(f"{key}: {value}")


# ------------------------------------------------------------
# 9. PERSISTENCIA
# ------------------------------------------------------------

G9_SARCASM_PATH = (
    ARTIFACT_DIR
    / "g9_sarcasm_candidates.csv"
)

G9_PROXY_PATH = (
    ARTIFACT_DIR
    / "g9_proxy_label_candidates.csv"
)

G9_SUMMARY_PATH = (
    ARTIFACT_DIR
    / "g9_summary.json"
)


sarcasm_review.to_csv(
    G9_SARCASM_PATH,
    index=False
)

proxy_review.to_csv(
    G9_PROXY_PATH,
    index=False
)

with open(
    G9_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        g9_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 10. CONTROL DE ARTEFACTOS
# ------------------------------------------------------------

print("\n=== ARTEFACTOS FINALES G9 ===")

print(
    "Análisis completo:",
    G9_ANALYSIS_PATH.exists()
)

print(
    "Errores BERT:",
    G9_BERT_ERRORS_PATH.exists()
)

print(
    "Muestra cualitativa:",
    G9_QUALITATIVE_PATH.exists()
)

print(
    "Candidatos sarcasmo:",
    G9_SARCASM_PATH.exists()
)

print(
    "Candidatos etiqueta proxy:",
    G9_PROXY_PATH.exists()
)

print(
    "Resumen G9:",
    G9_SUMMARY_PATH.exists()
)


# ------------------------------------------------------------
# 11. CONTROL DE NO REEVALUACIÓN
# ------------------------------------------------------------

print("\n=== CONTROL DE NO REEVALUACIÓN ===")

print("BERT ejecutado:", False)
print("BiLSTM ejecutada:", False)
print("Predicciones TEST generadas:", False)
print("Métricas TEST recalculadas:", False)
print("Modelos reentrenados:", False)
print("Hiperparámetros modificados:", False)


# ------------------------------------------------------------
# 12. ESTADO
# ------------------------------------------------------------

g9_controls_ok = all([
    G9_ANALYSIS_PATH.exists(),
    G9_BERT_ERRORS_PATH.exists(),
    G9_QUALITATIVE_PATH.exists(),
    G9_SARCASM_PATH.exists(),
    G9_PROXY_PATH.exists(),
    G9_SUMMARY_PATH.exists(),
    len(g9_error_analysis) == 2099,
    len(bert_errors) == 396
])


print("\n=== ESTADO DE G9 ===")

print(
    "Neutralidad analizada:",
    "SÍ"
)

print(
    "Negación analizada:",
    "SÍ"
)

print(
    "Reseñas mixtas analizadas:",
    "SÍ"
)

print(
    "Textos largos analizados:",
    "SÍ"
)

print(
    "Sarcasmo revisado mediante candidatos:",
    "SÍ"
)

print(
    "Ruido de etiqueta proxy revisado mediante candidatos:",
    "SÍ"
)

print(
    "¿Controles de G9 superados?:",
    g9_controls_ok
)

print(
    "G9 listo para auditoría:",
    "SÍ" if g9_controls_ok else "NO"
)

print(
    "G9 aprobado:",
    "TODAVÍA NO"
)

print(
    "G10 iniciado:",
    "NO"
)

=== G9 — REVISIÓN CUALITATIVA FINAL ===
Columnas requeridas ausentes: []

=== SARCASMO — CANDIDATOS PARA REVISIÓN ===
Candidatos encontrados entre errores BERT: 1
Interpretación: indicador heurístico; NO detección automática de sarcasmo

=== RUIDO DE ETIQUETA PROXY — CANDIDATOS ===
Candidatos encontrados: 338
Interpretación: casos potencialmente ambiguos; NO prueba automática de etiqueta incorrecta

=== EJEMPLOS CANDIDATOS A SARCASMO ===

----------------------------------------
REAL: NEGATIVO
PREDICCIÓN BERT: NEUTRAL
Palabras: 272
Texto:
I'd love to give these guys a better review, because they were very friendly and professional.  However, I was very dissapointed with the service I received.  I'll type the one sarcastic comment that I can't get out of my head and then I'll get to the more substantial feedback.  I thought a detail was supposed to be a little more DETAILED.

Here are the positives, because I want to give credit where credit is due.  The guys come to you and are very po

---

# Resumen final de resultados

## Evaluación en TEST

| Modelo | Accuracy | Balanced accuracy | Precision macro | Recall macro | F1 macro | F1 weighted |
|---|---:|---:|---:|---:|---:|---:|
| Clase mayoritaria | 0.6856 | 0.3333 | 0.2285 | 0.3333 | 0.2712 | 0.5577 |
| TF-IDF + Logistic Regression | 0.7894 | 0.6943 | 0.6806 | 0.6943 | 0.6871 | 0.7921 |
| Word2Vec + BiLSTM | 0.7542 | 0.6269 | 0.6423 | 0.6269 | 0.6326 | 0.7572 |
| **BERT** | **0.8113** | **0.7273** | **0.7088** | **0.7273** | **0.7166** | **0.8181** |

### Resultado principal

**BERT fue el modelo con mayor F1 macro en este split y configuración.**

Esta conclusión no implica superioridad universal.

## Rendimiento BERT por clase en TEST

| Clase | Precision | Recall | F1 | Soporte |
|---|---:|---:|---:|---:|
| NEGATIVO | 0.7827 | 0.7983 | 0.7904 | 352 |
| NEUTRAL | 0.4189 | 0.5032 | 0.4572 | 308 |
| POSITIVO | 0.9248 | 0.8805 | 0.9021 | 1439 |

## Análisis de errores BERT

- Total TEST: **2099**
- Errores BERT: **396**
- Tasa global de error: **18.87 %**
- Error en clase NEUTRAL: **49.68 %**
- Error en textos con indicador de negación: **21.87 %**
- Error en reseñas con señal mixta: **22.88 %**
- Error en textos largos: **32.38 %**
- Candidatos cualitativos a sarcasmo: **1**
- Candidatos a posible ambigüedad de la etiqueta proxy: **338**

La neutralidad fue la principal dificultad observada. Las detecciones de negación, señal mixta, sarcasmo y posible ruido de etiqueta son análisis descriptivos/heurísticos y no prueban causalidad.

## Limitaciones

- Las estrellas funcionan como etiqueta proxy de sentimiento.
- Existe desbalance de clases.
- La clase NEUTRAL presenta mayor dificultad.
- BERT utiliza `max_length=256`, por lo que parte de las reseñas largas se trunca.
- Los resultados corresponden exclusivamente a este split y configuración.

---

## Conclusión

El baseline TF-IDF + Logistic Regression resultó competitivo y superó a Word2Vec + BiLSTM. Sin embargo, BERT obtuvo el mayor F1 macro final (**0.7166**) y la mayor accuracy (**0.8113**) en TEST.

**Conclusión final: BERT fue el mejor modelo en este split y configuración según F1 macro.**
